# MultiModalSpectralTransformer
- Sort IBM Benchmarking into its own category
- 

### Config

In [ ]:
# Core libraries
import copy
import glob
import json
import os
import pickle
import random
import statistics
from argparse import Namespace
from collections import defaultdict
from datetime import datetime
import tempfile
import math
import ast
import time
import shutil
from math import ceil
import re
from typing import List, Dict, Any, Union, Tuple


# Data processing and scientific computing
import numpy as np
import pandas as pd
from tqdm import tqdm
from tqdm.autonotebook import tqdm

# Machine learning and data visualization
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from sklearn.model_selection import train_test_split
import umap
from sklearn.neighbors import RadiusNeighborsRegressor
from scipy.spatial.distance import cosine
from sklearn.neighbors import NearestNeighbors

# PyTorch and related libraries
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.utils.data.distributed import DistributedSampler
from torch.optim.lr_scheduler import ReduceLROnPlateau
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from pytorch_lightning.profiler import SimpleProfiler, AdvancedProfiler
from pytorch_lightning.loggers import WandbLogger

# RDKit for cheminformatics
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem, Draw, Descriptors, MolFromSmiles, MolToSmiles
from rdkit.Chem.Descriptors import MolWt
from rdkit.Chem.rdMolDescriptors import CalcMolFormula
from rdkit.DataStructs import FingerprintSimilarity, TanimotoSimilarity

# Weights & Biases for experiment tracking
import wandb

# Miscellaneous
from IPython.display import HTML, SVG, display

# Setting up environment
%matplotlib inline
torch.cuda.device_count()
#wandb.login()

# Custom utility modules
import utils_MMT.clip_functions_v15_4 as cf
import utils_MMT.MT_functions_v15_4 as mtf
import utils_MMT.validate_generate_MMT_v15_4 as vgmmt
import utils_MMT.run_batch_gen_val_MMT_v15_4 as rbgvm
import utils_MMT.clustering_visualization_v15_4 as cv
import utils_MMT.plotting_v15_4 as pt
import utils_MMT.execution_function_v15_4 as ex
import utils_MMT.train_test_functions_pl_v15_4 as ttf
import utils_MMT.ir_simulation_v15_4 as irs
import utils_MMT.helper_functions_pl_v15_4 as hf
import utils_MMT.mmt_result_test_functions_15_4 as mrtf
import utils_MMT.experiment_function_v15_4 as exp_func
from utils_MMT.mmt_result_test_functions_15_4 import *
import utils_MMT.improvement_cycle_neg_examples_v15_4 as icne
import shutil

In [ ]:

def load_json_dics():
    with open('./itos.json', 'r') as f:
        itos = json.load(f)
    with open('./stoi.json', 'r') as f:
        stoi = json.load(f)
    with open('./stoi_MF.json', 'r') as f:
        stoi_MF = json.load(f)
    with open('./itos_MF.json', 'r') as f:
        itos_MF = json.load(f)    
    return itos, stoi, stoi_MF, itos_MF
    
itos, stoi, stoi_MF, itos_MF = load_json_dics()
rand_num = str(random.randint(1, 10000000))

In [ ]:
IR_config_dict = {
    "gpu": list(range(torch.cuda.device_count())),
    "test_path": [os.path.abspath("./models/chemprop-ir/ir_models_data/solvation_example/solvation_spectra.csv")],
    "use_compound_names": [False],
    "preds_path": [os.path.abspath("./models/chemprop-ir/ir_models_data/ir_preds_test_2.csv")],
    "checkpoint_dir": [os.path.abspath("./models/chemprop-ir/ir_models_data/experiment_model/model_files")],
    "spectra_type": ["experimental"],
    "spectra_type_nr": [0],
    "checkpoint_path": [None],
    "batch_size": [50],
    "no_cuda": [[False]],
    "features_generator": [None],
    "features_path": [None],
    "max_data_size": [100],
    "ensemble_variance": [False],
    "ensemble_variance_conv": [0.0],
}

import os

# Determine the project root directory (where the notebook is located)
project_root = os.path.dirname(os.path.abspath('__file__'))

hyperparameters = {
    # General project information
    "project": ["MMST_V1"],  # Name of the project for wandb monitoring
    "ran_num":[rand_num],
    "device": ["cuda"], # device on which training takes place
    "gpu_num":[1], # number of GPUs for training with pytorch lightning
    "num_workers":[4], # Needs to stay 1 otherwise code crashes - ToDO
    "data_type":["sgnn"], #["sgnn", "exp", "acd", "real", "inference"], Different data types to select
    "execution_type":["validate_MMT"], #[ "plot_similarities", "simulate_real", "test_performance", "SMI_generation_MMT", "SMI_generation_MF", "data_generation", "transformer_training","transformer_improvement", "clip_training", "clip_improvement", "validate_MMT"] # different networks to select for training
    "syn_data_simulated": [False],  # For the improvment cycle a ticker that shows whether data has been simulated or not.
    "training_type":["clip"], #["clip","transformer"] # different networks to select for training

    # Encoding dicts
    "itos_path": [os.path.join(project_root, "itos.json")],
    "stoi_path": [os.path.join(project_root, "stoi.json")],
    "itos_MF_path": [os.path.join(project_root, "itos_MF.json")],
    "stoi_MF_path": [os.path.join(project_root, "stoi_MF.json")],
    
    ### Data settings
    "input_dim_1H":[2], # Imput dimensions of the 1H data
    "input_dim_13C": [1], # Imput dimensions of the 13C data
    "input_dim_HSQC": [2], # Imput dimensions of the HSQC data
    "input_dim_COSY": [2],  # Imput dimensions of the COSY data
    "input_dim_IR": [1000],  # Imput dimensions of the IR data
    "MF_vocab_size": [len(stoi_MF)],  # New, size of the vocabulary for molecular formulas
    "MS_vocab_size": [len(stoi)],  # New, size of the vocabulary for molecular formulas
    "tr_te_split":[0.9], # Train-Test split
    "padding_points_number":[64], # Padding number for the embedding layer into the network
    "data_size": [1000], # number of datapoints for the training ZINC:3975764/ PubChem1797828
    "test_size": [10], # number of datapoints for the training ZINC: 3975764
    "model_save_dir": [os.path.join(project_root, "experiments/exp_1/model_save_dir")],
    "ML_dump_folder": [os.path.join(project_root, "experiments/exp_1/dump")],
    "model_save_interval": [10000],
    
    # Option 1 SGNN
    "use_real_data": [False],
    "ref_data_type": ["1H"],
    "csv_train_path": [os.path.join(project_root, "data/ZINK_dataset/ML_NMR_5M_XL_1H_comb_train_V8.csv")],
    "csv_1H_path_SGNN": [os.path.join(project_root, "data/ZINK_dataset/ML_NMR_5M_XL_1H_comb_train_V8.csv")],
    "csv_13C_path_SGNN": [os.path.join(project_root, "data/ZINK_dataset/ML_NMR_5M_XL_13C_train_V8.csv")],    
    "csv_HSQC_path_SGNN": [os.path.join(project_root, "data/ZINK_dataset/ML_NMR_5M_XL_HSQC_train_V8.csv")],    
    "csv_COSY_path_SGNN": [os.path.join(project_root, "data/ZINK_dataset/ML_NMR_5M_XL_COSY_train_V8.csv")],      
    "csv_IR_MF_path": [''],
    "csv_path_val": [os.path.join(project_root, "data/ZINK_dataset/ML_NMR_5M_XL_1H_comb_test_V8.csv")], 
    "IR_data_folder": [os.path.join(project_root, "data/ZINK_dataset/IR_spectra_NN")],
    "pickle_file_path": [""],
    "dl_mode": ['val'],
    "isomericSmiles": [False],
    
    #### Transformer Settings ####
    # Training and model settings
    "training_mode":["1H_13C_HSQC_COSY_IR_MF_MW"], #["edding_src_1H = torch.zeros((feature_dim, current_ba"], Modalities selected for training
    "blank_percentage":[0.0], # percentage of spectra that are blanked out during training for better generalizability of the network to various datatypes
    "batch_size":[64], # number needs to be the same as number of GPUs 
    "num_epochs": [10], # number of epochs for training
    "lr_pretraining": [1e-4], # Pretraining learning rate
    "lr_finetuning": [5e-5], # Finetuning learning rate
    "load_model": [True], # if model should be loaded from path
    "checkpoint_path":[os.path.join(project_root,"models/mmst/base_models/1_0_V8i_MMTi_RAW_MW_DROP_Loss_0.112.ckpt")], #V8
    "save_model": [True], # if model should be saved
    
    # Model architecture
    "in_size": [len(stoi)],
    "hidden_size": [128],
    "out_size": [len(stoi)],
    "num_encoder_layers": [6], #8
    "num_decoder_layers": [6], #8
    "num_heads": [16], #8  ### number of attention heads
    "forward_expansion": [4], #4
    "max_len": [128], # maximum length of the generated sequence
    "drop_out": [0.1],
    "fingerprint_size": [512], # Dimensions of encoder output for CLIP contrastive training    
    "gen_SMI_sequence":[True], # If the model generates a sequence with the SMILES current model for evaluation
    "sampling_method":["mix"], # weight_mol_weight ["multinomial", "greedy". "mix"]  
    "training_setup":["pretraining"], # ["pretraining","finetuning"]
    "smi_randomizer":[False], # if smiles are randomized or canonical during training
    
    ### SGNN Feedback
    "sgnn_feedback":[False], # if SGNN generates 1H and 13C spectrum on the fly on the generated smiles -> "gen_SMI_sequence":[True]
    "matching":["HungDist"], #["MinSum","EucDist","HungDist"], # HSQC point matching technique used
    "padding":["NN"], # ["Zero","Trunc","NN"], # HSQC padding technique used -> see publication: XXX
    # Weight feedback
    "train_weight_min":[None], # Calculate on the fly - Used for the weight loss calculation for scaling
    "train_weight_max":[None], # Calculate on the fly - Used for the weight loss calculation for scaling
    # Training Loss Weighting options
    "weight_validity": [0.0], # up to 1
    "weight_SMI": [1.0], # up to 1
    "weight_FP": [0.0], # up to 1
    "weight_MW": [0], # up to 100
    "weight_sgnn": [0.0], # up to 10
    "weight_tanimoto": [0.0], # up to 1
    "change_loss_weights":[False], # if selected the weights get ajusted along the training
    "increment":[0.01], # increment on how much it gets ajusted during training -> TODO
    "batch_frequency":[10000], # Frequency how often it gets ajusted -> TODO
    
    ### For Validation
    "beam_size": [1],  
    "multinom_runs": [1], 
    "temperature":[1],
    "gen_len":[64],
    "pkl_save_folder":[os.path.join(project_root, "exp_1/pkl_save_folder")],
    
    ### Molformer options 
    "MF_max_trails":[500],
    "MF_tanimoto_filter":[0.1],
    "MF_filter_higher":[1], # False = 0 True = 1
    "MF_delta_weight":[5],
    "MF_generations":[30],
    "MF_model_path":[os.path.join(project_root,"models/mol2mol/Alessandro_big/weights_pubchem_with_counts_and_rank_sanitized.ckpt")],
    "MF_vocab":[os.path.join(project_root,"models/mol2mol/Alessandro_big/vocab_new.pkl")],
    "MF_csv_source_folder_location":[os.path.join(project_root,"deep-molecular-optimization/data/MMP")],
    "MF_csv_source_file_name":["test_selection_2"],
    "MF_methods":["MMP"], #["MMP", "scaffold", "MMP_scaffold"],      
    "max_scaffold_generations":[10], #
    
    ### MMT batch generation
    "MMT_batch":[32], # how big is the batch of copies of the same inputs that is processed by MMT 
    "MMT_generations":[4], # need to be multiple of MMT_batch -> number of valid generated molecules
    "n_samples":[10], # number of molecules that should be processed for data generation - needs to be smaller than dataloader size
    "gen_mol_csv_folder_path": [os.path.join(project_root,"data/SGNN_gen_folder")],  # Updated to use local path in the repository
    
    ### Fine-tuning improvement options
    "train_data_blend":[0], # how many additional molecules should be added to the new dataset from the original training dataset
    "train_data_blend_CLIP":[1000], # how many additional molecules should be added to the new dataset from the original training dataset
    

    ### Data generation SGNN -> 1H, 13C, HSQC, COSY
    "SGNN_gen_folder_path":[os.path.join(project_root, "experiments/exp_1/SGNN_gen_folder")],
    "SGNN_csv_gen_smi":[os.path.join(project_root,"data/test_data/IBM_SMI_data_top20.csv")],
    "SGNN_size_filter":[550],
    "SGNN_csv_save_folder": [os.path.join(project_root,"experiments/exp_1/SGNN_gen_folder")],
    "IR_save_folder":[os.path.join(project_root, "experiments/exp_1/IR_data")],

    ##################################################
    #### LEGACY parameters for Future experiments ####
    ##################################################
    ### CLIP Model for contrastive learning ChemBerta VS MultiModalSpectralTransformer
    #### CLIP Settings ####
    ### ChemBerta
    "model_version":[os.path.join(project_root,"models/mmst/OLD/Chemberta_source")],   # Source of pretrained chemberta from paper
    "CB_model_path":[os.path.join(project_root,"models/mmst/OLD/Large_300_15.pth")], # path to pretrained Chemberta model
    "num_class":[1024], #
    "num_linear_layers":[0], # number of linear layers in architecture before num_class output
    "use_dropout":[True],
    "use_relu":[False],
    "loss_fn":["BCEWithLogitsLoss"], #"MSELoss", "BCELoss", 
    "CB_embedding": [1024], #1024
    # PCA
    "fp_dim_reduction":[False], #True
    "pca_components":[300],  
    #"CB_model_name": ["Large_300_15"],

    ### Multimodal Transformer
    "MT_model_path":[os.path.join(project_root,"models/mmst/OLD/MultimodalTransformer_time_1706856620.3718672_Loss_0.202.pth")],  # path to pretrained Multimodal Transformer model  
    #"MT_model_name": ["SpectrumBERT_PCA_large_3.6"],
    "MT_embedding": [512], #512
    ### Projection Head
    "projection_dim": [512],
    "dropout": [0.1],
    
    ### Train parameters
    # Dataloader settings
    "similarity_threshold":[0.6], # Filtere that selects just molecules with a tanimotosimilarity higher than that number
    "max_search_size":[10000], # Size of the data that will be searched to find the similar molecules  # 100000
    "weight_delta":[50], # Filter to molecules with a +/- delta weight of that numbeTraceback (most recent call last):
    "CLIP_batch_size":[128],  #,64,128,256 ### batch size for the CLIP training
    "CLIP_NUM_EPOCHS": [10],    # Number of training epochs
    "CLIP_temperature": [1],
    #"CB_projection_lr": [1e-3], # projection head learning rate for Chemberta
    "MT_projection_lr": [1e-3], # projection head learning rate for Multimodal Transfomer
    "CB_lr": [1e-4], # Chemberta Learning Rate
    "MT_lr": [1e-5], # Multimodal Transfomer Learning Rate
    "weight_decay": [1e-3], # Weight decay for projection heads -> TODO why just on those
    "patience": [1],   # not integrated yet
    "factor": [0.8],   # not integrated yet
    "CLIP_continue_training":[True],
    "CLIP_model_path":[os.path.join(project_root,"models/mmst/OLD/MultimodalCLIP_Epoch_9_Loss0.096.ckpt")],   
    "CLIP_model_save_dir":[os.path.join(project_root,"models/mmst/OLD/test_CLIP")],
    }


def save_config(config, path):
    with open(path, 'w') as f:
        json.dump(config, f)

def load_config(path):
    try:
        with open(path, 'r') as f:
            return json.load(f)
    except FileNotFoundError:
        return None    

def parse_arguments(hyperparameters):
    # Using dictionary comprehension to simplify your code
    parsed_args = {key: val[0] for key, val in hyperparameters.items()}
    return Namespace(**parsed_args)

config = parse_arguments(hyperparameters)
ir_config_path = os.path.join(project_root,'utils_MMT/ir_config_V8.json')
save_config(IR_config_dict, ir_config_path)
IR_config_dict = load_config(ir_config_path)
IR_config = parse_arguments(IR_config_dict)
irs.modify_predict_args(IR_config)

config_path = os.path.join(project_root,'utils_MMT/config_V8.json')
save_config(hyperparameters, config_path)
config_dict = load_config(config_path)
config = parse_arguments(config_dict)

In [ ]:
# Import necessary libraries (assuming these are already imported in your script)
import matplotlib.pyplot as plt
import numpy as np
import os
import pickle
import statistics

# Correct path to the IBM comparison data
ibm_data_path = '/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/___FIGURES_PAPERS/Figures_Paper_2/precomputed_raw_data/20250612_IBM_comparison'

# Load the IBM comparison data
file_path = os.path.join(ibm_data_path, '1.0_prob_dict_results_IBM.pkl')
with open(file_path, 'rb') as file:
    prob_dict_results_ibm = pickle.load(file)

file_path = os.path.join(ibm_data_path, '1.1_results_dict_IBM.pkl')
with open(file_path, 'rb') as file:
    results_dict_ibm = pickle.load(file)

# Plot 1: Violin plot for correct SMILES sample probability
fig, ax = plt.subplots(figsize=(8, 10))
data_for_violin = [prob_dict_results_ibm["aggregated_corr_prob_multi"]]
parts = ax.violinplot(data_for_violin, showmeans=True, showmedians=False, showextrema=False)

# Customizing colors
color = "#8CB0FE"
for pc in parts['bodies']:
    pc.set_facecolor(color)
    pc.set_edgecolor('black')
    pc.set_alpha(0.7)

# Customizing the axes and labels
ax.set_title('IBM: Correct SMILES Sample Probability', fontsize=22)
ax.set_ylabel('Probability of Correct SMILES', fontsize=22)
ax.set_xticks([1])
ax.set_xticklabels(['IBM Model'], fontsize=22)
ax.tick_params(axis='both', which='major', labelsize=22)

mean_value = np.mean(data_for_violin[0])
ax.text(1, mean_value, f'{mean_value:.2f}', ha='center', va='bottom', fontsize=22)

# Add grid and set the limits
ax.grid(axis='y', linestyle='--', alpha=0.7)
ax.set_ylim(0, 1)

plt.tight_layout()
# Commented out save code
# save_path = os.path.abspath('./_FIGURES/IBM_Violin_Prob.png')
# plt.savefig(save_path, format='png')
plt.show()

# Plot 2: Violin plot for Tanimoto similarity
fig, ax = plt.subplots(figsize=(8, 10))
data_for_violin = [results_dict_ibm["tanimoto_scores_"]]
parts = ax.violinplot(data_for_violin, showmeans=True, showmedians=False, showextrema=False)

# Customizing colors
color = "#8CB0FE"
for pc in parts['bodies']:
    pc.set_facecolor(color)
    pc.set_edgecolor('black')
    pc.set_alpha(0.7)

# Customizing the axes and labels
ax.set_title('IBM: Greedy Sampled Average Tanimoto Similarity', fontsize=22)
ax.set_ylabel('Average Tanimoto Similarity', fontsize=22)
ax.set_xticks([1])
ax.set_xticklabels(['IBM Model'], fontsize=22)
ax.tick_params(axis='both', which='major', labelsize=22)

mean_value = np.mean(data_for_violin[0])
ax.text(1, mean_value, f'{mean_value:.2f}', ha='center', va='bottom', fontsize=22)

# Add grid and set the limits
ax.grid(axis='y', linestyle='--', alpha=0.7)
ax.set_ylim(0, 1)

plt.tight_layout()
# Commented out save code
# save_path = os.path.abspath('./_FIGURES/IBM_Tanimoto_Violin.png')
# plt.savefig(save_path, format='png')
plt.show()

# Plot 3: Bar chart for invalid SMILES count
fig, ax = plt.subplots(figsize=(8, 10))
failed_count = len(results_dict_ibm["failed"])
total_entries = len(results_dict_ibm["gen_conv_SMI_list"])
percentage = (failed_count / total_entries) * 100 if total_entries > 0 else 0

# Define colors for the bars
color = "#8CB0FE"

# Plotting the bar
bar = ax.bar([0], [failed_count], width=0.35, color=color, edgecolor='black')

# Adding value labels inside and percentage on top of each bar
ax.text(0, failed_count / 2, f'{percentage:.1f}%', 
        rotation=90, ha='center', va='center', fontsize=22)

# Set the title and labels
ax.set_title('IBM: Greedy Sampled Number of Invalid SMILES', fontsize=22)
ax.set_ylabel('Invalid Molecules', fontsize=22)
ax.set_xticks([0])
ax.set_xticklabels(['IBM Model'], fontsize=22)
ax.tick_params(axis='both', which='major', labelsize=22)

# Adding grid lines for better readability
ax.grid(axis='y', linestyle='--', alpha=0.7)

# Set y-limit slightly higher than max for label visibility
ax.set_ylim(0, failed_count * 1.25)

plt.tight_layout()
# Commented out save code
# save_path = os.path.abspath('./_FIGURES/IBM_Invalid_molecules.png')
# plt.savefig(save_path, format='png')
plt.show()

In [ ]:
results_dict_ibm.keys()

In [ ]:
len(results_dict_ibm["tanimoto_scores_all"])

### 0.0 Test Sized Models and Different Training Data


##### Data size effect - Training times effect


In [ ]:


"""
# V8i Raw 1M
#config.checkpoint_path = os.path.abspath("./models/mmst/experiment_models/different_trainings/0.0_V8i_MMTi_1Mio-epoch=07-loss=0.10.ckpt")
#same batches as 20M mol seen
config.checkpoint_path = os.path.abspath("./models/mmst/experiment_models/different_trainings/0.0_V8i_MMTi_1Mio-epoch=19-loss=0.04.ckpt")


model_MMT = mrtf.load_MMT_model(config)
prob_dict_results_0a, results_dict_0a = mrtf.run_model_analysis(config, model_MMT, val_dataloader_multi, stoi, itos)
print(np.mean(results_dict_0a["tanimoto_sim"]))

import pickle
# Save the data to a file
#file_prob_dict_path = os.path.abspath('./past_experiments/ChemXriv/0.0_Experiment_Training_Strategy/20240430_Epoch_7/0.0_prob_dict_results_1M_L_epoch_7.pkl')
file_prob_dict_path = os.path.abspath('./past_experiments/ChemXriv/0.0_Experiment_Training_Strategy/20240430_Epoch_20/0.0_prob_dict_results_1M_L_20M_mol.pkl') 
with open(file_prob_dict_path, 'wb') as file:
    pickle.dump(prob_dict_results_0a, file)
    

# Save the data to a file
#file_results_dict_path = os.path.abspath('./past_experiments/ChemXriv/0.0_Experiment_Training_Strategy/20240430_Epoch_7/0.1_results_dict_1M_L_epoch_7.pkl')
file_results_dict_path = os.path.abspath('./past_experiments/ChemXriv/0.0_Experiment_Training_Strategy/20240430_Epoch_20/0.1_results_dict_1M_L_20M_mol.pkl') 
with open(file_results_dict_path, 'wb') as file:
    pickle.dump(results_dict_0a, file)"""

In [ ]:
"""
# V8i Raw 2M
#config.checkpoint_path = os.path.abspath("./models/mmst/experiment_models/different_trainings/0.0_V8i_MMTi_2Mio-epoch=07-loss=0.06.ckpt")
#same batches as 20M mol seen
config.checkpoint_path = os.path.abspath("./models/mmst/experiment_models/different_trainings/0.0_V8i_MMTi_2Mio-epoch=09-loss=0.04.ckpt")

model_MMT = mrtf.load_MMT_model(config)
prob_dict_results_0b, results_dict_0b = mrtf.run_model_analysis(config, model_MMT, val_dataloader_multi, stoi, itos)
print(np.mean(results_dict_0b["tanimoto_sim"]))


import pickle
# Save the data to a file
#file_prob_dict_path = os.path.abspath('./past_experiments/ChemXriv/0.0_Experiment_Training_Strategy/20240430_Epoch_7/0.0_prob_dict_results_2M_L_epoch_7.pkl')  
file_prob_dict_path = os.path.abspath('./past_experiments/ChemXriv/0.0_Experiment_Training_Strategy/20240430_Epoch_20/0.0_prob_dict_results_2M_L_20M_mol.pkl') 
with open(file_prob_dict_path, 'wb') as file:
    pickle.dump(prob_dict_results_0b, file)
    

# Save the data to a file
#file_results_dict_path = os.path.abspath('./past_experiments/ChemXriv/0.0_Experiment_Training_Strategy/20240430_Epoch_7/0.1_results_dict_2M_L_epoch_7.pkl')  
file_results_dict_path = os.path.abspath('./past_experiments/ChemXriv/0.0_Experiment_Training_Strategy/20240430_Epoch_20/0.1_results_dict_2M_L_20M_mol.pkl') 
with open(file_results_dict_path, 'wb') as file:
    pickle.dump(results_dict_0b, file)"""

In [ ]:
"""
# V8i Raw 4M
#config.checkpoint_path = os.path.abspath("./models/mmst/experiment_models/different_trainings/0.0_V8i_MMTi_4Mio-epoch=07-loss=0.02.ckpt")
#same batches as 20M mol seen
config.checkpoint_path = os.path.abspath("./models/mmst/experiment_models/different_trainings/0.0_V8i_MMTi_4Mio-epoch=04-loss=0.04.ckpt")

model_MMT = mrtf.load_MMT_model(config)
prob_dict_results_0b, results_dict_0b = mrtf.run_model_analysis(config, model_MMT, val_dataloader_multi, stoi, itos)

import pickle
# Save the data to a file
#file_prob_dict_path = os.path.abspath('./past_experiments/ChemXriv/0.0_Experiment_Training_Strategy/20240430_Epoch_7/0.0_prob_dict_results_4M_L_epoch_7.pkl') 
file_prob_dict_path = os.path.abspath('./past_experiments/ChemXriv/0.0_Experiment_Training_Strategy/20240430_Epoch_20/0.0_prob_dict_results_4M_L_20M_mol.pkl') 
with open(file_prob_dict_path, 'wb') as file:
    pickle.dump(prob_dict_results_0b, file)
    

# Save the data to a file
#file_results_dict_path = os.path.abspath('./past_experiments/ChemXriv/0.0_Experiment_Training_Strategy/20240430_Epoch_7/0.1_results_dict_4M_L_epoch_7.pkl') 
file_results_dict_path = os.path.abspath('./past_experiments/ChemXriv/0.0_Experiment_Training_Strategy/20240430_Epoch_20/0.1_results_dict_4M_L_20M_mol.pkl') 
with open(file_results_dict_path, 'wb') as file:
    pickle.dump(results_dict_0b, file)"""

#### Load saved data - EPOCH 7
- Different data sizes



In [ ]:
import pickle
import os
import gc
import numpy as np
import statistics
import matplotlib.pyplot as plt

### Epoch 7 for all molecules seen - File-based loading
results_dict_results = []
prob_dict_results = []
# Define file paths for prob_dict files
prob_dict_file_paths = [
    './past_experiments/ChemXriv/0.0_Experiment_Training_Strategy/20240430_Epoch_7/0.0_prob_dict_results_0.1M_L_epoch_7.pkl',
    './past_experiments/ChemXriv/0.0_Experiment_Training_Strategy/20240430_Epoch_7/0.0_prob_dict_results_1M_L_epoch_7.pkl',
    './past_experiments/ChemXriv/0.0_Experiment_Training_Strategy/20240430_Epoch_7/0.0_prob_dict_results_2M_L_epoch_7.pkl',
    './past_experiments/ChemXriv/0.0_Experiment_Training_Strategy/20240430_Epoch_7/0.0_prob_dict_results_4M_L_epoch_7.pkl'
]

# Load prob_dict files
for i, file_path in enumerate(prob_dict_file_paths):
    print(f"Loading prob_dict file {i+1}/{len(prob_dict_file_paths)}: {os.path.basename(file_path)}")
    
    try:
        full_path = os.path.abspath(file_path)
        with open(full_path, 'rb') as file:
            prob_dict = pickle.load(file)
            prob_dict_results.append(prob_dict)
        print(f"  - Successfully loaded {labels[i]} prob_dict")
    except FileNotFoundError:
        print(f"  - File not found: {file_path}")
    except Exception as e:
        print(f"  - Error loading {file_path}: {e}")

# Define file paths for results_dict files
results_dict_file_paths = [
    './past_experiments/ChemXriv/0.0_Experiment_Training_Strategy/20240430_Epoch_7/0.1_results_dict_0.1M_L_epoch_7.pkl',
    './past_experiments/ChemXriv/0.0_Experiment_Training_Strategy/20240430_Epoch_7/0.1_results_dict_1M_L_epoch_7.pkl',
    './past_experiments/ChemXriv/0.0_Experiment_Training_Strategy/20240430_Epoch_7/0.1_results_dict_2M_L_epoch_7.pkl',
    './past_experiments/ChemXriv/0.0_Experiment_Training_Strategy/20240430_Epoch_7/0.1_results_dict_4M_L_epoch_7.pkl'
]

# Load results_dict files
for i, file_path in enumerate(results_dict_file_paths):
    print(f"Loading results_dict file {i+1}/{len(results_dict_file_paths)}: {os.path.basename(file_path)}")
    
    try:
        full_path = os.path.abspath(file_path)
        with open(full_path, 'rb') as file:
            results_dict = pickle.load(file)
            results_dict_results.append(results_dict)
        print(f"  - Successfully loaded {labels[i]} results_dict")
    except FileNotFoundError:
        print(f"  - File not found: {file_path}")
    except Exception as e:
        print(f"  - Error loading {file_path}: {e}")

print(f"\nLoaded {len(prob_dict_results)} prob_dict files and {len(results_dict_results)} results_dict files")


##### Probability of Correct SMILES -  Violin Chart

In [ ]:

# Assign to original variable names for compatibility
prob_dict_results_2 = prob_dict_results

# Calculate mean and standard deviation for each dictionary
mean_results = []
std_results = []
data_for_violin = []

for prob_dict in prob_dict_results_2:
    mean_prob = np.mean(prob_dict["aggregated_corr_prob_multi"])
    mean_results.append(mean_prob)
    std_value = statistics.stdev(prob_dict["aggregated_corr_prob_multi"])
    std_results.append(std_value)
    data_for_violin.append(prob_dict["aggregated_corr_prob_multi"])

# Create the violin plot
fig, ax = plt.subplots(figsize=(6, 10))
parts = ax.violinplot(data_for_violin, showmeans=True, showmedians=False, showextrema=False)

# Customizing colors
color = "#8CB0FE"  # Use a single color for all the violin plots
for pc in parts['bodies']:
    pc.set_facecolor(color)
    pc.set_edgecolor('black')
    pc.set_alpha(0.7)

# Customizing the axes and labels
#ax.set_title('Correct SMILES Sample Probability', fontsize=22)
ax.set_ylabel('Probability of Correct SMILES', fontsize=22)
ax.set_xticks(np.arange(1, len(labels) + 1))
ax.set_xticklabels(labels, rotation=45, ha='center', fontsize=22)
ax.tick_params(axis='both', which='major', labelsize=22)

mean_values = [np.mean(data) if data is not None else 0 for data in data_for_violin]

# Add mean values as text
for pos, mean_value in zip(np.arange(1, len(mean_values) + 1), mean_values):
    ax.text(pos, mean_value, f'{mean_value:.2f}', ha='center', va='bottom', fontsize=22)

# Add grid and set the limits
ax.grid(axis='y', linestyle='--', alpha=0.7)
ax.set_ylim(0, 1)  # Adjust based on your data's range

plt.tight_layout()
save_path = os.path.abspath('./_FIGURES/0.0_Violin_Epoch_7_v8.png')
plt.savefig(save_path, format='png')
plt.show()

print(f"\nPlot saved to: {save_path}")
print("\nEpoch 7 probability analysis completed!")

##### Tanimoto Chart - Violine Chart

In [ ]:
# Tanimoto Similarity Violin Plot
labels = ['0.1M Epoch 7', '1M Epoch 7', '2M Epoch 7', '4M Epoch 7']
results_dict_2 = results_dict_results  # Use your loaded data

# Prepare data - check for empty datasets
data_for_violin_raw = [d["tanimoto_sim"] for d in results_dict_2]

# Filter out empty datasets
data_for_violin = []
filtered_labels = []
for i, data in enumerate(data_for_violin_raw):
    if len(data) > 0:
        data_for_violin.append(data)
        filtered_labels.append(labels[i])
    else:
        print(f"Warning: {labels[i]} has no Tanimoto data, skipping...")

# Use filtered labels
labels = filtered_labels

# Calculate means for each dataset
mean_values = [np.mean(data) for data in data_for_violin]

# Create the violin plot
fig, ax = plt.subplots(figsize=(6, 10))
parts = ax.violinplot(data_for_violin, showmeans=True, showmedians=False, showextrema=False)

# Customizing colors to all lightseagreen
color = "#8CB0FE"
for pc in parts['bodies']:
    pc.set_facecolor(color)
    pc.set_edgecolor('black')
    pc.set_alpha(0.7)

# Customizing the axes and labels
#ax.set_title('Greedy Sampled Average Tanimoto Similarity', fontsize=22)
ax.set_ylabel('Average Tanimoto Similarity', fontsize=22)
ax.set_xticks(np.arange(1, len(labels) + 1))
ax.set_xticklabels(labels, rotation=45, ha='center', fontsize=22)

# Add grid and set the limits
ax.grid(axis='y', linestyle='--', alpha=0.7)
ax.set_ylim(0, 1)
ax.tick_params(axis='both', which='major', labelsize=22)

# Add mean values as text
for pos, mean_value in zip(np.arange(1, len(mean_values) + 1), mean_values):
    ax.text(pos, mean_value, f'{mean_value:.2f}', ha='center', va='bottom', fontsize=22)

plt.tight_layout()
save_path = os.path.abspath('./_FIGURES/0.0_Tanimoto_Violin_plot_Epoch_7_v8.png')
plt.savefig(save_path, format='png')
plt.show()

##### Correct Molecules - Bar Chart

In [ ]:
# Initialize lists to store the extracted data
correct_counts = []
total_counts = []
filtered_labels = []

print("Processing loaded data to count correct molecules (Tanimoto = 1.0)...")

# Process each loaded results_dict
for i, (results_dict, label) in enumerate(zip(results_dict_results, labels)):
    print(f"Processing dataset {i+1}/{len(results_dict_results)}: {label}")
    
    # Extract Tanimoto similarity data - using the same key as your violin plot
    tanimoto_sim = results_dict.get("tanimoto_sim", [])
    
    # Only process if data exists
    if len(tanimoto_sim) > 0:
        # Count molecules with Tanimoto similarity = 1.0 (perfect matches)
        correct_count = sum(1 for sim in tanimoto_sim if sim == 1.0)
        total_count = len(tanimoto_sim)
        
        correct_counts.append(correct_count)
        total_counts.append(total_count)
        filtered_labels.append(label)
        
        accuracy_percentage = (correct_count / total_count) * 100 if total_count > 0 else 0
        
        print(f"  - Total molecules: {total_count}")
        print(f"  - Correct molecules: {correct_count}")
        print(f"  - Accuracy: {accuracy_percentage:.1f}%")
    else:
        print(f"  - No Tanimoto data found for {label}")

print(f"\nProcessed {len(correct_counts)} datasets with Tanimoto data.")

# Check if we have any data
if not correct_counts:
    print("Error: All datasets are empty. Unable to create bar plot.")
else:
    # Calculate percentages
    accuracy_percentages = [(correct / total) * 100 if total > 0 else 0 
                           for correct, total in zip(correct_counts, total_counts)]
    
    # Define colors and styling to match invalid molecules plot
    color2 = "#8CB0FE"
    bar_width = 0.35
    positions = np.arange(len(filtered_labels))
    
    # Create the plot with same dimensions as invalid molecules plot
    fig, ax = plt.subplots(figsize=(6, 10))
    
    # Create bars with consistent styling
    bars = ax.bar(positions, accuracy_percentages, align='center', alpha=0.7, 
                  ecolor='black', capsize=10, edgecolor='black', color=color2)
    
    # Add absolute count labels inside each bar (vertical, black, non-bold)
    for i, bar in enumerate(bars):
        yval = bar.get_height()
        absolute_count = correct_counts[i]
        ax.text(bar.get_x() + bar.get_width() / 2, yval / 2, f'{absolute_count}', 
                rotation=90, ha='center', va='center', fontsize=22, color='black')

    # Add percentage labels on top of each bar
    for i, bar in enumerate(bars):
        yval = bar.get_height()
        percentage = accuracy_percentages[i]
        ax.text(bar.get_x() + bar.get_width() / 2, yval + max(accuracy_percentages) * 0.02, f'{percentage:.1f}%', 
                ha='center', va='bottom', fontsize=22, color='black')
    
    # Set the title and labels to match style
    #plt.title(f'Greedy Sampled Accuracy (Tanimoto = 1.0)           ', fontsize=22)
    plt.ylabel('Accuracy (%) (Tanimoto = 1.0) ', fontsize=22)
    plt.xticks(positions, filtered_labels, ha='center', fontsize=22)
    ax.set_xticklabels(filtered_labels, rotation=45, ha='center', fontsize=22)
    
    # Adding grid lines for better readability
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    
    # Set y-limit slightly higher than max for percentage labels on top
    ax.set_ylim(0, max(accuracy_percentages) * 1.15)
    ax.tick_params(axis='both', which='major', labelsize=22)
    
    # Style the spines to match your other plots
    for spine in ax.spines.values():
        spine.set_edgecolor('gray')
        spine.set_alpha(0.8)
        
    # Set aspect ratio
    ax.set_aspect(aspect='auto')
    
    # Adjust layout
    plt.tight_layout()
    
    # Save the figure
    save_path = os.path.abspath('./_FIGURES/0.0_Training_Strategy_Correct_Molecules_Epoch_7_v8.png')
    plt.savefig(save_path, format='png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"\nPlot saved to: {save_path}")

# Print summary statistics
print("\nSummary Statistics:")
print("="*60)
for label, correct, total, percentage in zip(filtered_labels, correct_counts, total_counts, accuracy_percentages):
    print(f"{label:15s}: {correct:4d}/{total:4d} ({percentage:5.1f}%)")

print("\nTraining strategy correct molecules analysis completed!")

##### Invalid molecules - Bar Chart

In [ ]:
# Invalid Molecules Bar Plot
labels = ['0.1M Epoch 7', '1M Epoch 7', '2M Epoch 7', '4M Epoch 7']
results_dict_2 = results_dict_results  # Use your loaded data
# Prepare data
mean_results_2 = [len(d["failed"]) for d in results_dict_2]  # Assuming "failed" is a key in each dictionary
total_entries = len(results_dict_2[0]["gen_conv_SMI_list"])  # Total for percentage calculation

# Calculate percentages for each dataset
percentages = [(num_failed / total_entries) * 100 for num_failed in mean_results_2]

# Define colors for the bars
color2 = "#8CB0FE"
# Set bar width
bar_width = 0.35
positions = np.arange(len(labels))

# Create the plot
fig, ax = plt.subplots(figsize=(6, 10))

# Plotting the bars
bars = ax.bar(positions, percentages, align='center', alpha=0.7, 
              ecolor='black', capsize=10, edgecolor='black', color=color2)

# Add absolute count labels inside each bar (vertical, black, non-bold)
for i, bar in enumerate(bars):
    yval = bar.get_height()
    absolute_count = mean_results_2[i]
    ax.text(bar.get_x() + bar.get_width() / 2, yval / 2, f'{absolute_count}', 
            rotation=90, ha='center', va='center', fontsize=22, color='black')

# Add percentage labels on top of each bar
for i, bar in enumerate(bars):
    yval = bar.get_height()
    percentage = percentages[i]
    ax.text(bar.get_x() + bar.get_width() / 2, yval + max(percentages) * 0.02, f'{percentage:.1f}%', 
            ha='center', va='bottom', fontsize=22, color='black')

# Set the title and labels
#plt.title(f'Greedy Sampled Number of Invalid SMILES           ', fontsize=22)
plt.ylabel('Percentage of Invalid Molecules (%)', fontsize=22)
plt.xticks(positions, labels, ha='center', fontsize=22)
ax.set_xticklabels(labels, rotation=45, ha='center', fontsize=22)

# Adding grid lines for better readability
ax.grid(axis='y', linestyle='--', alpha=0.7)

# Set y-limit slightly higher than max for percentage labels on top
ax.set_ylim(0, max(percentages) * 1.15)
ax.tick_params(axis='both', which='major', labelsize=22)

# Style the spines to match your other plots
for spine in ax.spines.values():
    spine.set_edgecolor('gray')
    spine.set_alpha(0.8)

# Set aspect ratio
ax.set_aspect(aspect='auto')

# Adjust layout
plt.tight_layout()

# Save the figure
save_path = os.path.abspath('./_FIGURES/0.0_Invalid_molecules_Epoch_7_v8.png')
plt.savefig(save_path, format='png', dpi=300, bbox_inches='tight')

# Show plot
plt.show()

print(f"\nPlot saved to: {save_path}")

# Print summary
print("\nSummary Statistics:")
print("="*60)
for i, label in enumerate(labels):
    print(f"{label:15s}: {percentages[i]:.1f}% ({mean_results_2[i]:4d}/{total_entries:4d})")

#### Load saved data - 20M trained network

In [ ]:
import pickle
import os
import gc
import numpy as np
import statistics
import matplotlib.pyplot as plt

### Exactly 20M molecules seen - File-based loading
results_dict_results = []
prob_dict_results = []

# Define file paths for prob_dict files
prob_dict_file_paths = [
    './past_experiments/ChemXriv/0.0_Experiment_Training_Strategy/20240508_20M_Molecules/0.0_prob_dict_results_0.1M_L_20M_mol.pkl',
    './past_experiments/ChemXriv/0.0_Experiment_Training_Strategy/20240508_20M_Molecules/0.0_prob_dict_results_0.5M_L_20M_mol.pkl', 
 #   './past_experiments/ChemXriv/0.0_Experiment_Training_Strategy/20240508_20M_Molecules/0.0_prob_dict_results_1M_L_20M_mol.pkl',
    './past_experiments/ChemXriv/0.0_Experiment_Training_Strategy/20240508_20M_Molecules/0.0_prob_dict_results_2M_L_20M_mol.pkl',
    './past_experiments/ChemXriv/0.0_Experiment_Training_Strategy/20240508_20M_Molecules/0.0_prob_dict_results_4M_L_20M_mol.pkl'
]

# Load prob_dict files
for i, file_path in enumerate(prob_dict_file_paths):
    print(f"Loading prob_dict file {i+1}/{len(prob_dict_file_paths)}: {os.path.basename(file_path)}")
    
    try:
        full_path = os.path.abspath(file_path)
        with open(full_path, 'rb') as file:
            prob_dict = pickle.load(file)
            prob_dict_results.append(prob_dict)
        print(f"  - Successfully loaded {labels[i]} prob_dict")
    except FileNotFoundError:
        print(f"  - File not found: {file_path}")
    except Exception as e:
        print(f"  - Error loading {file_path}: {e}")

# Define file paths for results_dict files
results_dict_file_paths = [
    './past_experiments/ChemXriv/0.0_Experiment_Training_Strategy/20240508_20M_Molecules/0.1_results_dict_0.1M_L_20M_mol.pkl',
    './past_experiments/ChemXriv/0.0_Experiment_Training_Strategy/20240508_20M_Molecules/0.1_results_dict_0.5M_L_20M_mol.pkl', 
#    './past_experiments/ChemXriv/0.0_Experiment_Training_Strategy/20240508_20M_Molecules/0.1_results_dict_1M_L_20M_mol.pkl',
    './past_experiments/ChemXriv/0.0_Experiment_Training_Strategy/20240508_20M_Molecules/0.1_results_dict_2M_L_20M_mol.pkl',
    './past_experiments/ChemXriv/0.0_Experiment_Training_Strategy/20240508_20M_Molecules/0.1_results_dict_4M_L_20M_mol.pkl'
]

# Load results_dict files
for i, file_path in enumerate(results_dict_file_paths):
    print(f"Loading results_dict file {i+1}/{len(results_dict_file_paths)}: {os.path.basename(file_path)}")
    
    try:
        full_path = os.path.abspath(file_path)
        with open(full_path, 'rb') as file:
            results_dict = pickle.load(file)
            results_dict_results.append(results_dict)
        print(f"  - Successfully loaded {labels[i]} results_dict")
    except FileNotFoundError:
        print(f"  - File not found: {file_path}")
    except Exception as e:
        print(f"  - Error loading {file_path}: {e}")

print(f"\nLoaded {len(prob_dict_results)} prob_dict files and {len(results_dict_results)} results_dict files")


##### Probability of Correct SMILES - Violine Chart

In [ ]:
labels = ['0.1M | 20M', '0.5M | 20M', '2M | 20M', '4M | 20M']

# Assign to original variable names for compatibility
prob_dict_results_2 = prob_dict_results

# Calculate mean and standard deviation for each dictionary
mean_results = []
std_results = []
data_for_violin = []

for prob_dict in prob_dict_results_2:
    mean_prob = np.mean(prob_dict["aggregated_corr_prob_multi"])
    mean_results.append(mean_prob)
    std_value = statistics.stdev(prob_dict["aggregated_corr_prob_multi"])
    std_results.append(std_value)
    data_for_violin.append(prob_dict["aggregated_corr_prob_multi"])

# Create the violin plot
fig, ax = plt.subplots(figsize=(6, 10))
parts = ax.violinplot(data_for_violin, showmeans=True, showmedians=False, showextrema=False)

# Customizing colors
color = "#8CB0FE"  # Use a single color for all the violin plots
for pc in parts['bodies']:
    pc.set_facecolor(color)
    pc.set_edgecolor('black')
    pc.set_alpha(0.7)

# Customizing the axes and labels
#ax.set_title('Correct SMILES Sample Probability', fontsize=22)
ax.set_ylabel('Probability of Correct SMILES', fontsize=22)
ax.set_xticks(np.arange(1, len(labels) + 1))
ax.set_xticklabels(labels, rotation=45, ha='center', fontsize=22)
ax.tick_params(axis='both', which='major', labelsize=22)

mean_values = [np.mean(data) if data is not None else 0 for data in data_for_violin]

# Add mean values as text
for pos, mean_value in zip(np.arange(1, len(mean_values) + 1), mean_values):
    ax.text(pos, mean_value, f'{mean_value:.2f}', ha='center', va='bottom', fontsize=22)

# Add grid and set the limits
ax.grid(axis='y', linestyle='--', alpha=0.7)
ax.set_ylim(0, 1)  # Adjust based on your data's range

plt.tight_layout()
save_path = os.path.abspath('./_FIGURES/0.1_Violin_20M_v8.png')
plt.savefig(save_path, format='png')
plt.show()

print(f"\nPlot saved to: {save_path}")
print("\n20M probability analysis completed!")

##### Tanimoto Plot - Violine Chart

In [ ]:
# Tanimoto Similarity Violin Plot
labels = ['0.1M | 20M', '0.5M | 20M', '2M | 20M', '4M | 20M']
results_dict_2 = results_dict_results  # Use your loaded data

# Prepare data - check for empty datasets
data_for_violin_raw = [d["tanimoto_sim"] for d in results_dict_2]

# Filter out empty datasets
data_for_violin = []
filtered_labels = []
for i, data in enumerate(data_for_violin_raw):
    if len(data) > 0:
        data_for_violin.append(data)
        filtered_labels.append(labels[i])
    else:
        print(f"Warning: {labels[i]} has no Tanimoto data, skipping...")

# Use filtered labels
labels = filtered_labels

# Calculate means for each dataset
mean_values = [np.mean(data) for data in data_for_violin]

# Create the violin plot
fig, ax = plt.subplots(figsize=(6, 10))
parts = ax.violinplot(data_for_violin, showmeans=True, showmedians=False, showextrema=False)

# Customizing colors to all lightseagreen
color = "#8CB0FE"
for pc in parts['bodies']:
    pc.set_facecolor(color)
    pc.set_edgecolor('black')
    pc.set_alpha(0.7)

# Customizing the axes and labels
#ax.set_title('Greedy Sampled Average Tanimoto Similarity', fontsize=22)
ax.set_ylabel('Average Tanimoto Similarity', fontsize=22)
ax.set_xticks(np.arange(1, len(labels) + 1))
ax.set_xticklabels(labels, rotation=45, ha='center', fontsize=22)

# Add grid and set the limits
ax.grid(axis='y', linestyle='--', alpha=0.7)
ax.set_ylim(0, 1)
ax.tick_params(axis='both', which='major', labelsize=22)

# Add mean values as text
for pos, mean_value in zip(np.arange(1, len(mean_values) + 1), mean_values):
    ax.text(pos, mean_value, f'{mean_value:.2f}', ha='center', va='bottom', fontsize=22)

plt.tight_layout()
save_path = os.path.abspath('./_FIGURES/0.1_Tanimoto_Violin_plot_20M_v8.png')
plt.savefig(save_path, format='png')
plt.show()

print(f"\nTanimoto plot saved to: {save_path}")
print("\n20M Tanimoto analysis completed!")

##### Correct Molecules - Bar Chart

In [ ]:
# Initialize lists to store the extracted data
correct_counts = []
total_counts = []
filtered_labels = []

labels = ['0.1M | 20M', '0.5M | 20M', '2M | 20M', '4M | 20M']

print("Processing loaded data to count correct molecules (Tanimoto = 1.0)...")

# Process each loaded results_dict
for i, (results_dict, label) in enumerate(zip(results_dict_results, labels)):
    print(f"Processing dataset {i+1}/{len(results_dict_results)}: {label}")
    
    # Extract Tanimoto similarity data - using the same key as your violin plot
    tanimoto_sim = results_dict.get("tanimoto_sim", [])
    
    # Only process if data exists
    if len(tanimoto_sim) > 0:
        # Count molecules with Tanimoto similarity = 1.0 (perfect matches)
        correct_count = sum(1 for sim in tanimoto_sim if sim == 1.0)
        total_count = len(tanimoto_sim)
        
        correct_counts.append(correct_count)
        total_counts.append(total_count)
        filtered_labels.append(label)
        
        accuracy_percentage = (correct_count / total_count) * 100 if total_count > 0 else 0
        
        print(f"  - Total molecules: {total_count}")
        print(f"  - Correct molecules: {correct_count}")
        print(f"  - Accuracy: {accuracy_percentage:.1f}%")
    else:
        print(f"  - No Tanimoto data found for {label}")

print(f"\nProcessed {len(correct_counts)} datasets with Tanimoto data.")

# Check if we have any data
if not correct_counts:
    print("Error: All datasets are empty. Unable to create bar plot.")
else:
    # Calculate percentages
    accuracy_percentages = [(correct / total) * 100 if total > 0 else 0 
                           for correct, total in zip(correct_counts, total_counts)]
    
    # Define colors and styling to match invalid molecules plot
    color2 = "#8CB0FE"
    bar_width = 0.35
    positions = np.arange(len(filtered_labels))
    
    # Create the plot with same dimensions as other plots
    fig, ax = plt.subplots(figsize=(6, 10))
    
    # Create bars with consistent styling
    bars = ax.bar(positions, accuracy_percentages, align='center', alpha=0.7, 
                  ecolor='black', capsize=10, edgecolor='black', color=color2)
    
    # Add absolute count labels inside each bar (vertical, black, non-bold)
    for i, bar in enumerate(bars):
        yval = bar.get_height()
        absolute_count = correct_counts[i]
        ax.text(bar.get_x() + bar.get_width() / 2, yval / 2, f'{absolute_count}', 
                rotation=90, ha='center', va='center', fontsize=22, color='black')

    # Add percentage labels on top of each bar
    for i, bar in enumerate(bars):
        yval = bar.get_height()
        percentage = accuracy_percentages[i]
        ax.text(bar.get_x() + bar.get_width() / 2, yval + max(accuracy_percentages) * 0.02, f'{percentage:.1f}%', 
                ha='center', va='bottom', fontsize=22, color='black')
    
    # Set the title and labels to match style
    #plt.title(f'Greedy Sampled Accuracy (Tanimoto = 1.0)', fontsize=22)
    plt.ylabel('Accuracy (%) (Tanimoto = 1.0) ', fontsize=22)
    plt.xticks(positions, filtered_labels, ha='center', fontsize=22)
    ax.set_xticklabels(filtered_labels, rotation=45, ha='center', fontsize=22)
    
    # Adding grid lines for better readability
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    
    # Set y-limit slightly higher than max for percentage labels on top
    ax.set_ylim(0, max(accuracy_percentages) * 1.15)
    ax.tick_params(axis='both', which='major', labelsize=22)
    
    # Style the spines to match your other plots
    for spine in ax.spines.values():
        spine.set_edgecolor('gray')
        spine.set_alpha(0.8)
        
    # Set aspect ratio
    ax.set_aspect(aspect='auto')
    
    # Adjust layout
    plt.tight_layout()
    
    # Save the figure
    save_path = os.path.abspath('./_FIGURES/0.1_Training_Strategy_Correct_Molecules_20M_v8.png')
    plt.savefig(save_path, format='png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"\nPlot saved to: {save_path}")

# Print summary statistics
print("\nSummary Statistics:")
print("="*60)
for label, correct, total, percentage in zip(filtered_labels, correct_counts, total_counts, accuracy_percentages):
    print(f"{label:15s}: {correct:4d}/{total:4d} ({percentage:5.1f}%)")

print("\nTraining strategy correct molecules analysis completed!")

##### Invalid Molecules - Bar Chart

In [ ]:
# Invalid Molecules Bar Plot
labels = ['0.1M | 20M', '0.5M | 20M', '2M | 20M', '4M | 20M']
results_dict_2 = results_dict_results  # Use your loaded data

# Prepare data
mean_results_2 = [len(d["failed"]) for d in results_dict_2]  # Assuming "failed" is a key in each dictionary
total_entries = len(results_dict_2[0]["gen_conv_SMI_list"])  # Total for percentage calculation

# Calculate percentages for each dataset
percentages = [(num_failed / total_entries) * 100 for num_failed in mean_results_2]

# Define colors for the bars
color2 = "#8CB0FE"

# Set bar width
bar_width = 0.35
positions = np.arange(len(labels))

# Create the plot
fig, ax = plt.subplots(figsize=(6, 10))

# Plotting the bars
bars = ax.bar(positions, percentages, align='center', alpha=0.7, 
              ecolor='black', capsize=10, edgecolor='black', color=color2)

# Add absolute count labels inside each bar (vertical, black, non-bold)
for i, bar in enumerate(bars):
    yval = bar.get_height()
    absolute_count = mean_results_2[i]
    ax.text(bar.get_x() + bar.get_width() / 2, yval / 2, f'{absolute_count}', 
            rotation=90, ha='center', va='center', fontsize=22, color='black')

# Add percentage labels on top of each bar
for i, bar in enumerate(bars):
    yval = bar.get_height()
    percentage = percentages[i]
    ax.text(bar.get_x() + bar.get_width() / 2, yval + max(percentages) * 0.02, f'{percentage:.1f}%', 
            ha='center', va='bottom', fontsize=22, color='black')

# Set the title and labels
#plt.title(f'Greedy Sampled Number of Invalid SMILES', fontsize=22)
plt.ylabel('Percentage of Invalid Molecules (%)', fontsize=22)
plt.xticks(positions, labels, ha='center', fontsize=22)
ax.set_xticklabels(labels, rotation=45, ha='center', fontsize=22)

# Adding grid lines for better readability
ax.grid(axis='y', linestyle='--', alpha=0.7)

# Set y-limit slightly higher than max for percentage labels on top
ax.set_ylim(0, max(percentages) * 1.15)
ax.tick_params(axis='both', which='major', labelsize=22)

# Style the spines to match your other plots
for spine in ax.spines.values():
    spine.set_edgecolor('gray')
    spine.set_alpha(0.8)
    
# Set aspect ratio
ax.set_aspect(aspect='auto')

# Adjust layout
plt.tight_layout()

# Save the figure
save_path = os.path.abspath('./_FIGURES/0.1_Invalid_molecules_20M_v8.png')
plt.savefig(save_path, format='png', dpi=300, bbox_inches='tight')

# Show plot
plt.show()

print(f"\nPlot saved to: {save_path}")

# Print summary
print("\nSummary Statistics:")
print("="*60)
for i, label in enumerate(labels):
    print(f"{label:15s}: {percentages[i]:.1f}% ({mean_results_2[i]:4d}/{total_entries:4d})")

print("\n20M invalid molecules analysis completed!")

#### Model Size  & Training Time


In [ ]:
"""
# V8i Raw 1M Large
config.num_encoder_layers = 6
config.num_decoder_layers = 6
config.num_heads = 16

# V8i Raw 1M
config.checkpoint_path = os.path.abspath("./models/mmst/experiment_models/different_trainings/0.0_V8i_MMTi_1Mio-epoch=19-loss=0.04.ckpt")
model_MMT = mrtf.load_MMT_model(config)
prob_dict_results_0a, results_dict_0a = mrtf.run_model_analysis(config, model_MMT, val_dataloader_multi, stoi, itos)

import pickle
# Save the data to a file
file_prob_dict_path = os.path.abspath('./past_experiments/ChemXriv/0.0_Experiment_Training_Strategy/20240430_Epoch_20/0.0_prob_dict_results_1M_L_epoch_20.pkl')  
with open(file_prob_dict_path, 'wb') as file:
    pickle.dump(prob_dict_results_0a, file)
    

# Save the data to a file
file_results_dict_path = os.path.abspath('./past_experiments/ChemXriv/0.0_Experiment_Training_Strategy/20240430_Epoch_20/0.1_results_dict_1M_Lepoch_20.pkl') 
with open(file_results_dict_path, 'wb') as file:
    pickle.dump(results_dict_0a, file)"""

In [ ]:
"""
# V8i Raw 1M Medium
config.num_encoder_layers = 3
config.num_decoder_layers = 3
config.num_heads = 8


### 20 M molecules seen
config.checkpoint_path = os.path.abspath("./models/mmst/experiment_models/different_trainings/0.2_V8i_MMTi_1Mio_M-epoch=19-loss=0.09.ckpt")

model_MMT = mrtf.load_MMT_model(config)
prob_dict_results_0b, results_dict_0b = mrtf.run_model_analysis(config, model_MMT, val_dataloader_multi, stoi, itos)

# Save the data to a file
file_prob_dict_path = os.path.abspath('./past_experiments/ChemXriv/0.0_Experiment_Training_Strategy/20240430_Epoch_20/0.0_prob_dict_results_1M_M_20M_mol.pkl') 
with open(file_prob_dict_path, 'wb') as file:
    pickle.dump(prob_dict_results_0b, file)
    

# Save the data to a file
file_results_dict_path = os.path.abspath('./past_experiments/ChemXriv/0.0_Experiment_Training_Strategy/20240430_Epoch_20/0.1_results_dict_1M_M_20M_mol.pkl') 
with open(file_results_dict_path, 'wb') as file:
    pickle.dump(results_dict_0b, file)"""

In [ ]:
"""
# V8i Raw 1M Small
config.num_encoder_layers = 2
config.num_decoder_layers = 2
config.num_heads = 4

### 20 M molecules seen
config.checkpoint_path = os.path.abspath("./models/mmst/experiment_models/different_trainings/0.2_V8i_MMTi_1Mio_S-epoch=19-loss=0.11.ckpt")

model_MMT = mrtf.load_MMT_model(config)
prob_dict_results_0c, results_dict_0c = mrtf.run_model_analysis(config, model_MMT, val_dataloader_multi, stoi, itos)

# Save the data to a file
file_prob_dict_path = os.path.abspath('./past_experiments/ChemXriv/0.0_Experiment_Training_Strategy/20240430_Epoch_20/0.0_prob_dict_results_1M_S_20M_mol.pkl') 
with open(file_prob_dict_path, 'wb') as file:
    pickle.dump(prob_dict_results_0c, file)
    

# Save the data to a file
file_results_dict_path = os.path.abspath('./past_experiments/ChemXriv/0.0_Experiment_Training_Strategy/20240430_Epoch_20/0.1_results_dict_1M_S_20M_mol.pkl') 
with open(file_results_dict_path, 'wb') as file:
    pickle.dump(results_dict_0c, file)"""

In [ ]:
import pickle
import os
import gc
import numpy as np
import statistics
import matplotlib.pyplot as plt

### Load data for S vs M vs L comparison at 1M parameters
results_dict_results = []
prob_dict_results = []

# Define labels for the comparison (S, M, L order)
labels = ['1M S', '1M M', '1M L']

# Define file paths for prob_dict files (1M S, M, L comparison)
# Note: Order is S, M, L as requested
prob_dict_file_paths = [
    './past_experiments/ChemXriv/0.0_Experiment_Training_Strategy/20240430_Epoch_20/0.0_prob_dict_results_1M_S_epoch_20.pkl',  # Added missing comma
    './past_experiments/ChemXriv/0.0_Experiment_Training_Strategy/20240430_Epoch_20/0.0_prob_dict_results_1M_M_epoch_20.pkl',
    './past_experiments/ChemXriv/0.0_Experiment_Training_Strategy/20240430_Epoch_20/0.0_prob_dict_results_1M_L_epoch_20.pkl',
]

# Load prob_dict files
for i, file_path in enumerate(prob_dict_file_paths):
    print(f"Loading prob_dict file {i+1}/{len(prob_dict_file_paths)}: {os.path.basename(file_path)}")
    
    try:
        full_path = os.path.abspath(file_path)
        with open(full_path, 'rb') as file:
            prob_dict = pickle.load(file)
            prob_dict_results.append(prob_dict)
        print(f"  - Successfully loaded {labels[i]} prob_dict")
    except FileNotFoundError:
        print(f"  - File not found: {file_path}")
        print(f"  - Full path attempted: {os.path.abspath(file_path)}")
    except Exception as e:
        print(f"  - Error loading {file_path}: {e}")

# Define file paths for results_dict files (1M S, M, L comparison)
# Note: Order is S, M, L as requested
results_dict_file_paths = [
    './past_experiments/ChemXriv/0.0_Experiment_Training_Strategy/20240430_Epoch_20/0.1_results_dict_1M_S_epoch_20.pkl',  # Added missing comma
    './past_experiments/ChemXriv/0.0_Experiment_Training_Strategy/20240430_Epoch_20/0.1_results_dict_1M_M_epoch_20.pkl',
    './past_experiments/ChemXriv/0.0_Experiment_Training_Strategy/20240430_Epoch_20/0.1_results_dict_1M_L_epoch_20.pkl',
]

# Load results_dict files
for i, file_path in enumerate(results_dict_file_paths):
    print(f"Loading results_dict file {i+1}/{len(results_dict_file_paths)}: {os.path.basename(file_path)}")
    
    try:
        full_path = os.path.abspath(file_path)
        with open(full_path, 'rb') as file:
            results_dict = pickle.load(file)
            results_dict_results.append(results_dict)
        print(f"  - Successfully loaded {labels[i]} results_dict")
    except FileNotFoundError:
        print(f"  - File not found: {file_path}")
        print(f"  - Full path attempted: {os.path.abspath(file_path)}")
    except Exception as e:
        print(f"  - Error loading {file_path}: {e}")

print(f"\nLoaded {len(prob_dict_results)} prob_dict files and {len(results_dict_results)} results_dict files")
print("Ready for S vs M vs L comparison plots!")

# Debug: Check if directory exists
directory = './past_experiments/ChemXriv/0.0_Experiment_Training_Strategy/20240430_Epoch_20/'
print(f"\nDebug - Directory exists: {os.path.exists(directory)}")
if os.path.exists(directory):
    print("Files in directory:")
    for file in os.listdir(directory):
        print(f"  - {file}")
else:
    print("Directory not found!")
    print(f"Current working directory: {os.getcwd()}")

##### Probability of Correct SMILES - Violin Chart


In [ ]:
labels = ['1M S', '1M M', '1M L']

# Assign to original variable names for compatibility
prob_dict_results_2 = prob_dict_results

# Calculate mean and standard deviation for each dictionary
mean_results = []
std_results = []
data_for_violin = []

for prob_dict in prob_dict_results_2:
    mean_prob = np.mean(prob_dict["aggregated_corr_prob_multi"])
    mean_results.append(mean_prob)
    std_value = statistics.stdev(prob_dict["aggregated_corr_prob_multi"])
    std_results.append(std_value)
    data_for_violin.append(prob_dict["aggregated_corr_prob_multi"])

# Create the violin plot
fig, ax = plt.subplots(figsize=(6, 10))
parts = ax.violinplot(data_for_violin, showmeans=True, showmedians=False, showextrema=False)

# Customizing colors
color = "#8CB0FE"  # Use a single color for all the violin plots
for pc in parts['bodies']:
    pc.set_facecolor(color)
    pc.set_edgecolor('black')
    pc.set_alpha(0.7)

# Customizing the axes and labels
#ax.set_title('Correct SMILES Sample Probability', fontsize=22)
ax.set_ylabel('Probability of Correct SMILES', fontsize=22)
ax.set_xticks(np.arange(1, len(labels) + 1))
ax.set_xticklabels(labels, rotation=45, ha='center', fontsize=22)
ax.tick_params(axis='both', which='major', labelsize=22)

mean_values = [np.mean(data) if data is not None else 0 for data in data_for_violin]

# Add mean values as text
for pos, mean_value in zip(np.arange(1, len(mean_values) + 1), mean_values):
    ax.text(pos, mean_value, f'{mean_value:.2f}', ha='center', va='bottom', fontsize=22)

# Add grid and set the limits
ax.grid(axis='y', linestyle='--', alpha=0.7)
ax.set_ylim(0, 1)  # Adjust based on your data's range

plt.tight_layout()
save_path = os.path.abspath('./_FIGURES/0.3_Violin_Size_20M_v8.png')
plt.savefig(save_path, format='png')
plt.show()

print(f"\nPlot saved to: {save_path}")
print("\nSize comparison probability analysis completed!")

##### Correct Molecules - Bar Chart

In [ ]:
# Initialize lists to store the extracted data
correct_counts = []
total_counts = []
filtered_labels = []

labels = ['1M S', '1M M', '1M L']

print("Processing loaded data to count correct molecules (Tanimoto = 1.0)...")

# Process each loaded results_dict
for i, (results_dict, label) in enumerate(zip(results_dict_results, labels)):
    print(f"Processing dataset {i+1}/{len(results_dict_results)}: {label}")
    
    # Extract Tanimoto similarity data - using the same key as your violin plot
    tanimoto_sim = results_dict.get("tanimoto_sim", [])
    
    # Only process if data exists
    if len(tanimoto_sim) > 0:
        # Count molecules with Tanimoto similarity = 1.0 (perfect matches)
        correct_count = sum(1 for sim in tanimoto_sim if sim == 1.0)
        total_count = len(tanimoto_sim)
        
        correct_counts.append(correct_count)
        total_counts.append(total_count)
        filtered_labels.append(label)
        
        accuracy_percentage = (correct_count / total_count) * 100 if total_count > 0 else 0
        
        print(f"  - Total molecules: {total_count}")
        print(f"  - Correct molecules: {correct_count}")
        print(f"  - Accuracy: {accuracy_percentage:.1f}%")
    else:
        print(f"  - No Tanimoto data found for {label}")

print(f"\nProcessed {len(correct_counts)} datasets with Tanimoto data.")

# Check if we have any data
if not correct_counts:
    print("Error: All datasets are empty. Unable to create bar plot.")
else:
    # Calculate percentages
    accuracy_percentages = [(correct / total) * 100 if total > 0 else 0 
                           for correct, total in zip(correct_counts, total_counts)]
    
    # Define colors and styling to match invalid molecules plot
    color2 = "#8CB0FE"
    bar_width = 0.35
    positions = np.arange(len(filtered_labels))
    
    # Create the plot with same dimensions as other plots
    fig, ax = plt.subplots(figsize=(6, 10))
    
    # Create bars with consistent styling
    bars = ax.bar(positions, accuracy_percentages, align='center', alpha=0.7, 
                  ecolor='black', capsize=10, edgecolor='black', color=color2)
    
    # Add absolute count labels inside each bar (vertical, black, non-bold)
    for i, bar in enumerate(bars):
        yval = bar.get_height()
        absolute_count = correct_counts[i]
        ax.text(bar.get_x() + bar.get_width() / 2, yval / 2, f'{absolute_count}', 
                rotation=90, ha='center', va='center', fontsize=22, color='black')

    # Add percentage labels on top of each bar
    for i, bar in enumerate(bars):
        yval = bar.get_height()
        percentage = accuracy_percentages[i]
        ax.text(bar.get_x() + bar.get_width() / 2, yval + max(accuracy_percentages) * 0.02, f'{percentage:.1f}%', 
                ha='center', va='bottom', fontsize=22, color='black')
    
    # Set the title and labels to match style
    #plt.title(f'Greedy Sampled Accuracy (Tanimoto = 1.0)', fontsize=22)
    plt.ylabel('Accuracy (%) (Tanimoto = 1.0) ', fontsize=22)
    plt.xticks(positions, filtered_labels, ha='center', fontsize=22)
    ax.set_xticklabels(filtered_labels, rotation=45, ha='center', fontsize=22)
    
    # Adding grid lines for better readability
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    
    # Set y-limit slightly higher than max for percentage labels on top
    ax.set_ylim(0, max(accuracy_percentages) * 1.15)
    ax.tick_params(axis='both', which='major', labelsize=22)
    
    # Style the spines to match your other plots
    for spine in ax.spines.values():
        spine.set_edgecolor('gray')
        spine.set_alpha(0.8)
        
    # Set aspect ratio
    ax.set_aspect(aspect='auto')
    
    # Adjust layout
    plt.tight_layout()
    
    # Save the figure
    save_path = os.path.abspath('./_FIGURES/0.3_Model_Size_Correct_Molecules_20M_v8.png')
    plt.savefig(save_path, format='png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"\nPlot saved to: {save_path}")

# Print summary statistics
print("\nSummary Statistics:")
print("="*60)
for label, correct, total, percentage in zip(filtered_labels, correct_counts, total_counts, accuracy_percentages):
    print(f"{label:15s}: {correct:4d}/{total:4d} ({percentage:5.1f}%)")

print("\nModel size correct molecules analysis completed!")

##### Tanimoto Chart - Violine Chart

In [ ]:
# Tanimoto Similarity Violin Plot
labels = ['1M S', '1M M', '1M L']
results_dict_2 = results_dict_results  # Use your loaded data

# Prepare data - check for empty datasets
data_for_violin_raw = [d["tanimoto_sim"] for d in results_dict_2]

# Filter out empty datasets
data_for_violin = []
filtered_labels = []
for i, data in enumerate(data_for_violin_raw):
    if len(data) > 0:
        data_for_violin.append(data)
        filtered_labels.append(labels[i])
    else:
        print(f"Warning: {labels[i]} has no Tanimoto data, skipping...")

# Use filtered labels
labels = filtered_labels

# Calculate means for each dataset
mean_values = [np.mean(data) for data in data_for_violin]

# Create the violin plot
fig, ax = plt.subplots(figsize=(6, 10))
parts = ax.violinplot(data_for_violin, showmeans=True, showmedians=False, showextrema=False)

# Customizing colors to all lightseagreen
color = "#8CB0FE"
for pc in parts['bodies']:
    pc.set_facecolor(color)
    pc.set_edgecolor('black')
    pc.set_alpha(0.7)

# Customizing the axes and labels
#ax.set_title('Greedy Sampled Average Tanimoto Similarity', fontsize=22)
ax.set_ylabel('Average Tanimoto Similarity', fontsize=22)
ax.set_xticks(np.arange(1, len(labels) + 1))
ax.set_xticklabels(labels, rotation=45, ha='center', fontsize=22)

# Add grid and set the limits
ax.grid(axis='y', linestyle='--', alpha=0.7)
ax.set_ylim(0, 1)
ax.tick_params(axis='both', which='major', labelsize=22)

# Add mean values as text
for pos, mean_value in zip(np.arange(1, len(mean_values) + 1), mean_values):
    ax.text(pos, mean_value, f'{mean_value:.2f}', ha='center', va='bottom', fontsize=22)

plt.tight_layout()
save_path = os.path.abspath('./_FIGURES/0.3_Tanimoto_Violin_plot_Size_20M_v8.png')
plt.savefig(save_path, format='png')
plt.show()

print(f"\nTanimoto plot saved to: {save_path}")
print("\nSize comparison Tanimoto analysis completed!")

##### Invalid Molecules - Bar Chart

In [ ]:
# Invalid Molecules Bar Plot
labels = ['1M S', '1M M', '1M L']
results_dict_2 = results_dict_results  # Use your loaded data

# Prepare data
mean_results_2 = [len(d["failed"]) for d in results_dict_2]  # Assuming "failed" is a key in each dictionary
total_entries = len(results_dict_2[0]["gen_conv_SMI_list"])  # Total for percentage calculation

# Calculate percentages for each dataset
percentages = [(num_failed / total_entries) * 100 for num_failed in mean_results_2]

# Define colors for the bars
color2 = "#8CB0FE"

# Set bar width
bar_width = 0.35
positions = np.arange(len(labels))

# Create the plot
fig, ax = plt.subplots(figsize=(6, 10))

# Plotting the bars
bars = ax.bar(positions, percentages, align='center', alpha=0.7, 
              ecolor='black', capsize=10, edgecolor='black', color=color2)

# Add absolute count labels inside each bar (vertical, black, non-bold)
for i, bar in enumerate(bars):
    yval = bar.get_height()
    absolute_count = mean_results_2[i]
    ax.text(bar.get_x() + bar.get_width() / 2, yval / 2, f'{absolute_count}', 
            rotation=90, ha='center', va='center', fontsize=22, color='black')

# Add percentage labels on top of each bar
for i, bar in enumerate(bars):
    yval = bar.get_height()
    percentage = percentages[i]
    ax.text(bar.get_x() + bar.get_width() / 2, yval + max(percentages) * 0.02, f'{percentage:.1f}%', 
            ha='center', va='bottom', fontsize=22, color='black')

# Set the title and labels
#plt.title(f'Greedy Sampled Number of Invalid SMILES', fontsize=22)
plt.ylabel('Percentage of Invalid Molecules (%)', fontsize=22)
plt.xticks(positions, labels, ha='center', fontsize=22)
ax.set_xticklabels(labels, rotation=45, ha='center', fontsize=22)

# Adding grid lines for better readability
ax.grid(axis='y', linestyle='--', alpha=0.7)

# Set y-limit slightly higher than max for percentage labels on top
ax.set_ylim(0, max(percentages) * 1.15)
ax.tick_params(axis='both', which='major', labelsize=22)

# Style the spines to match your other plots
for spine in ax.spines.values():
    spine.set_edgecolor('gray')
    spine.set_alpha(0.8)
    
# Set aspect ratio
ax.set_aspect(aspect='auto')

# Adjust layout
plt.tight_layout()

# Save the figure
save_path = os.path.abspath('./_FIGURES/0.3_Invalid_molecules_Size_20M_v8.png')
plt.savefig(save_path, format='png', dpi=300, bbox_inches='tight')

# Show plot
plt.show()

print(f"\nPlot saved to: {save_path}")

# Print summary
print("\nSummary Statistics:")
print("="*60)
for i, label in enumerate(labels):
    print(f"{label:15s}: {percentages[i]:.1f}% ({mean_results_2[i]:4d}/{total_entries:4d})")

print("\nSize comparison invalid molecules analysis completed!")

### 1.0 Test different models for paper


#### Run calculations


In [ ]:
"""
config.IR_data_folder=os.path.abspath("./data/ZINK_dataset/IR_spectra_NN")
config.data_size = 489993 #int(1000*data_fraction)
config.IR_data_folder = ""
config.training_mode = "1H_13C_HSQC_COSY_IR_MF_MW" # it ignors IR because no folder provided and puts zeros in for IR
config.multinom_runs = 1
config.batch_size = 1024 #int(1000*data_fraction)

config.csv_path_val =   os.path.abspath('./data/ZINK_dataset/ML_NMR_5M_XL_1H_comb_test_V8.csv')
config.pickle_file_path = os.path.abspath('./data/ZINK_dataset/ML_NMR_5M_XL_1H_comb_test_V8_355655.pkl')

val_dataloader_multi = mrtf.load_data(config, stoi, stoi_MF, single=False, mode="val")
"""

In [ ]:
"""
# V8i Raw
config.checkpoint_path = os.path.abspath("./models/mmst/base_models/1_0_V8i_MMTi_RAW_Loss_0.088.ckpt")
model_MMT = mrtf.load_MMT_model(config)
prob_dict_results_1bi, results_dict_1bi = mrtf.run_model_analysis(config, model_MMT, val_dataloader_multi, stoi, itos)
print(np.mean(results_dict_1bi["tanimoto_sim"]))


# Save the data to a file
file_prob_dict_path = os.path.abspath('./past_experiments/ChemXriv/1.0_Experiment_Trainings_Experiments/1.0_prob_dict_results_1bi.pkl')
with open(file_prob_dict_path, 'wb') as file:
    pickle.dump(prob_dict_results_1bi, file)
    

# Save the data to a file
file_results_dict_path = os.path.abspath('./past_experiments/ChemXriv/1.0_Experiment_Trainings_Experiments/1.1_results_dict_1bi.pkl')  
with open(file_results_dict_path, 'wb') as file:
    pickle.dump(results_dict_1bi, file)"""


In [ ]:
"""
# V8i Raw + Drop
config.checkpoint_path = os.path.abspath("./models/mmst/base_models/1_0_V8i_MMTi_RAW_DROP_Loss_0.112.ckpt")
model_MMT = mrtf.load_MMT_model(config)
prob_dict_results_1ci, results_dict_1ci = mrtf.run_model_analysis(config, model_MMT, val_dataloader_multi, stoi, itos)
print(np.mean(results_dict_1ci["tanimoto_sim"]))


# Save the data to a file
file_prob_dict_path = os.path.abspath('./past_experiments/ChemXriv/1.0_Experiment_Trainings_Experiments/1.0_prob_dict_results_1ci.pkl')  
with open(file_prob_dict_path, 'wb') as file:
    pickle.dump(prob_dict_results_1ci, file)
        

# Save the data to a file
file_results_dict_path = os.path.abspath('./past_experiments/ChemXriv/1.0_Experiment_Trainings_Experiments/1.1_results_dict_1ci.pkl')  
with open(file_results_dict_path, 'wb') as file:
    pickle.dump(results_dict_1ci, file)"""

In [ ]:
import pickle
import os
import gc
import numpy as np
import statistics
import matplotlib.pyplot as plt

### Load data for 1bi vs 1ci comparison
results_dict_results = []
prob_dict_results = []

# Define labels for the comparison
labels = ['1bi', '1ci']

# Define file paths for prob_dict files (1bi, 1ci comparison)
prob_dict_file_paths = [
    './past_experiments/ChemXriv/1.0_Experiment_Trainings_Experiments/1.0_prob_dict_results_1bi.pkl',
    './past_experiments/ChemXriv/1.0_Experiment_Trainings_Experiments/1.0_prob_dict_results_1ci.pkl'
]

# Load prob_dict files
for i, file_path in enumerate(prob_dict_file_paths):
    print(f"Loading prob_dict file {i+1}/{len(prob_dict_file_paths)}: {os.path.basename(file_path)}")
    
    try:
        full_path = os.path.abspath(file_path)
        with open(full_path, 'rb') as file:
            prob_dict = pickle.load(file)
            prob_dict_results.append(prob_dict)
        print(f"  - Successfully loaded {labels[i]} prob_dict")
    except FileNotFoundError:
        print(f"  - File not found: {file_path}")
    except Exception as e:
        print(f"  - Error loading {file_path}: {e}")

# Define file paths for results_dict files (1bi, 1ci comparison)
results_dict_file_paths = [
    './past_experiments/ChemXriv/1.0_Experiment_Trainings_Experiments/1.1_results_dict_1bi.pkl',
    './past_experiments/ChemXriv/1.0_Experiment_Trainings_Experiments/1.1_results_dict_1ci.pkl'
]

# Load results_dict files
for i, file_path in enumerate(results_dict_file_paths):
    print(f"Loading results_dict file {i+1}/{len(results_dict_file_paths)}: {os.path.basename(file_path)}")
    
    try:
        full_path = os.path.abspath(file_path)
        with open(full_path, 'rb') as file:
            results_dict = pickle.load(file)
            results_dict_results.append(results_dict)
        print(f"  - Successfully loaded {labels[i]} results_dict")
    except FileNotFoundError:
        print(f"  - File not found: {file_path}")
    except Exception as e:
        print(f"  - Error loading {file_path}: {e}")

print(f"\nLoaded {len(prob_dict_results)} prob_dict files and {len(results_dict_results)} results_dict files")
print("Ready for 1bi vs 1ci comparison plots!")

# Debug: Check if directory exists
directory = './past_experiments/ChemXriv/1.0_Experiment_Trainings_Experiments/'
print(f"\nDebug - Directory exists: {os.path.exists(directory)}")
if os.path.exists(directory):
    print("Files in directory:")
    for file in os.listdir(directory):
        if '1bi' in file or '1ci' in file:
            print(f"  - {file}")
else:
    print("Directory not found!")
    print(f"Current working directory: {os.getcwd()}")

#### Probability of Correct SMILES - Violin Plot

In [ ]:
# Dropout comparison violin plot
labels = ['Base', 'Base +\nDropout']

# Assign to original variable names for compatibility
prob_dict_results_2 = prob_dict_results

# Calculate mean and standard deviation for each dictionary
mean_results = []
std_results = []
data_for_violin = []

for prob_dict in prob_dict_results_2:
    mean_prob = np.mean(prob_dict["aggregated_corr_prob_multi"])
    mean_results.append(mean_prob)
    std_value = statistics.stdev(prob_dict["aggregated_corr_prob_multi"])
    std_results.append(std_value)
    data_for_violin.append(prob_dict["aggregated_corr_prob_multi"])

# Create the violin plot
fig, ax = plt.subplots(figsize=(6, 10))
parts = ax.violinplot(data_for_violin, showmeans=True, showmedians=False, showextrema=False)

# Customizing colors
color = "#8CB0FE"  # Use a single color for all the violin plots
for pc in parts['bodies']:
    pc.set_facecolor(color)
    pc.set_edgecolor('black')
    pc.set_alpha(0.7)

# Customizing the axes and labels
#ax.set_title('Correct SMILES Sample Probability', fontsize=22)
ax.set_ylabel('Probability of Correct SMILES', fontsize=22)
ax.set_xticks(np.arange(1, len(labels) + 1))
ax.set_xticklabels(labels, rotation=45, ha='center', fontsize=22)
ax.tick_params(axis='both', which='major', labelsize=22)

mean_values = [np.mean(data) if data is not None else 0 for data in data_for_violin]

# Add mean values as text
for pos, mean_value in zip(np.arange(1, len(mean_values) + 1), mean_values):
    ax.text(pos, mean_value, f'{mean_value:.2f}', ha='center', va='bottom', fontsize=22)

# Add grid and set the limits
ax.grid(axis='y', linestyle='--', alpha=0.7)
ax.set_ylim(0, 1)  # Adjust based on your data's range

plt.tight_layout()

save_path = os.path.abspath('./_FIGURES/1.0_Dropout_Violin_v8.png')
plt.savefig(save_path, format='png')
plt.show()

print(f"\nPlot saved to: {save_path}")
print("\nDropout probability analysis completed!")

#### Tanimoto Similarity - Violine Chart

In [ ]:
# Tanimoto Similarity Violin Plot
labels = ['Base', 'Base +\nDropout']
results_dict_2 = results_dict_results  # Use your loaded data

# Prepare data - check for empty datasets
data_for_violin_raw = [d["tanimoto_sim"] for d in results_dict_2]

# Filter out empty datasets
data_for_violin = []
filtered_labels = []
for i, data in enumerate(data_for_violin_raw):
    if len(data) > 0:
        data_for_violin.append(data)
        filtered_labels.append(labels[i])
    else:
        print(f"Warning: {labels[i]} has no Tanimoto data, skipping...")

# Use filtered labels
labels = filtered_labels

# Calculate means for each dataset
mean_values = [np.mean(data) for data in data_for_violin]

# Create the violin plot
fig, ax = plt.subplots(figsize=(6, 10))
parts = ax.violinplot(data_for_violin, showmeans=True, showmedians=False, showextrema=False)

# Customizing colors to all lightseagreen
color = "#8CB0FE"
for pc in parts['bodies']:
    pc.set_facecolor(color)
    pc.set_edgecolor('black')
    pc.set_alpha(0.7)

# Customizing the axes and labels
#ax.set_title('Greedy Sampled Average Tanimoto Similarity', fontsize=22)
ax.set_ylabel('Average Tanimoto Similarity', fontsize=22)
ax.set_xticks(np.arange(1, len(labels) + 1))
ax.set_xticklabels(labels, rotation=45, ha='center', fontsize=22)

# Add grid and set the limits
ax.grid(axis='y', linestyle='--', alpha=0.7)
ax.set_ylim(0, 1)
ax.tick_params(axis='both', which='major', labelsize=22)

# Add mean values as text
for pos, mean_value in zip(np.arange(1, len(mean_values) + 1), mean_values):
    ax.text(pos, mean_value, f'{mean_value:.2f}', ha='center', va='bottom', fontsize=22)

plt.tight_layout()
save_path = os.path.abspath('./_FIGURES/1.0_Dropout_Tanimoto_Violin_v8.png')
plt.savefig(save_path, format='png')
plt.show()

print(f"\nTanimoto plot saved to: {save_path}")
print("\nDropout Tanimoto analysis completed!")

#### Invalid Molecules - Bar Chart

In [ ]:
# Invalid Molecules Bar Plot
labels = ['Base', 'Base +\nDropout']
results_dict_2 = results_dict_results  # Use your loaded data

# Prepare data
mean_results_2 = [len(d["failed"]) for d in results_dict_2]  # Assuming "failed" is a key in each dictionary
total_entries = len(results_dict_2[0]["gen_conv_SMI_list"])  # Total for percentage calculation

# Calculate percentages for each dataset
percentages = [(num_failed / total_entries) * 100 for num_failed in mean_results_2]

# Define colors for the bars
color2 = "#8CB0FE"

# Set bar width
bar_width = 0.35
positions = np.arange(len(labels))

# Create the plot
fig, ax = plt.subplots(figsize=(6, 10))

# Plotting the bars
bars = ax.bar(positions, percentages, align='center', alpha=0.7, 
              ecolor='black', capsize=10, edgecolor='black', color=color2)

# Add absolute count labels inside each bar (vertical, black, non-bold)
for i, bar in enumerate(bars):
    yval = bar.get_height()
    absolute_count = mean_results_2[i]
    ax.text(bar.get_x() + bar.get_width() / 2, yval / 2, f'{absolute_count}', 
            rotation=90, ha='center', va='center', fontsize=22, color='black')

# Add percentage labels on top of each bar
for i, bar in enumerate(bars):
    yval = bar.get_height()
    percentage = percentages[i]
    ax.text(bar.get_x() + bar.get_width() / 2, yval + max(percentages) * 0.02, f'{percentage:.1f}%', 
            ha='center', va='bottom', fontsize=22, color='black')

# Set the title and labels
#plt.title(f'Greedy Sampled Number of Invalid SMILES', fontsize=22)
plt.ylabel('Percentage of Invalid Molecules (%)', fontsize=22)
plt.xticks(positions, labels, ha='center', fontsize=22)
ax.set_xticklabels(labels, rotation=45, ha='center', fontsize=22)

# Adding grid lines for better readability
ax.grid(axis='y', linestyle='--', alpha=0.7)

# Set y-limit slightly higher than max for percentage labels on top
ax.set_ylim(0, max(percentages) * 1.15)
ax.tick_params(axis='both', which='major', labelsize=22)

# Style the spines to match your other plots
for spine in ax.spines.values():
    spine.set_edgecolor('gray')
    spine.set_alpha(0.8)
    
# Set aspect ratio
ax.set_aspect(aspect='auto')

# Adjust layout
plt.tight_layout()

# Save the figure
save_path = os.path.abspath('./_FIGURES/1.0_Dropout_Invalid_molecules_v8.png')
plt.savefig(save_path, format='png', dpi=300, bbox_inches='tight')

# Show plot
plt.show()

print(f"\nPlot saved to: {save_path}")

# Print summary
print("\nSummary Statistics:")
print("="*60)
for i, label in enumerate(labels):
    print(f"{label:15s}: {percentages[i]:.1f}% ({mean_results_2[i]:4d}/{total_entries:4d})")

print("\nDropout invalid molecules analysis completed!")

#### Correct Molecules - Bar Chart

In [ ]:
# Initialize lists to store the extracted data
correct_counts = []
total_counts = []
filtered_labels = []

labels = ['Base', 'Base +\nDropout']

print("Processing loaded data to count correct molecules (Tanimoto = 1.0)...")

# Process each loaded results_dict
for i, (results_dict, label) in enumerate(zip(results_dict_results, labels)):
    print(f"Processing dataset {i+1}/{len(results_dict_results)}: {label}")
    
    # Extract Tanimoto similarity data - using the same key as your violin plot
    tanimoto_sim = results_dict.get("tanimoto_sim", [])
    
    # Only process if data exists
    if len(tanimoto_sim) > 0:
        # Count molecules with Tanimoto similarity = 1.0 (perfect matches)
        correct_count = sum(1 for sim in tanimoto_sim if sim == 1.0)
        total_count = len(tanimoto_sim)
        
        correct_counts.append(correct_count)
        total_counts.append(total_count)
        filtered_labels.append(label)
        
        accuracy_percentage = (correct_count / total_count) * 100 if total_count > 0 else 0
        
        print(f"  - Total molecules: {total_count}")
        print(f"  - Correct molecules: {correct_count}")
        print(f"  - Accuracy: {accuracy_percentage:.1f}%")
    else:
        print(f"  - No Tanimoto data found for {label}")

print(f"\nProcessed {len(correct_counts)} datasets with Tanimoto data.")

# Check if we have any data
if not correct_counts:
    print("Error: All datasets are empty. Unable to create bar plot.")
else:
    # Calculate percentages
    accuracy_percentages = [(correct / total) * 100 if total > 0 else 0 
                           for correct, total in zip(correct_counts, total_counts)]
    
    # Define colors and styling to match invalid molecules plot
    color2 = "#8CB0FE"
    bar_width = 0.35
    positions = np.arange(len(filtered_labels))
    
    # Create the plot with same dimensions as other plots
    fig, ax = plt.subplots(figsize=(6, 10))
    
    # Create bars with consistent styling
    bars = ax.bar(positions, accuracy_percentages, align='center', alpha=0.7, 
                  ecolor='black', capsize=10, edgecolor='black', color=color2)
    
    # Add absolute count labels inside each bar (vertical, black, non-bold)
    for i, bar in enumerate(bars):
        yval = bar.get_height()
        absolute_count = correct_counts[i]
        ax.text(bar.get_x() + bar.get_width() / 2, yval / 2, f'{absolute_count}', 
                rotation=90, ha='center', va='center', fontsize=22, color='black')

    # Add percentage labels on top of each bar
    for i, bar in enumerate(bars):
        yval = bar.get_height()
        percentage = accuracy_percentages[i]
        ax.text(bar.get_x() + bar.get_width() / 2, yval + max(accuracy_percentages) * 0.02, f'{percentage:.1f}%', 
                ha='center', va='bottom', fontsize=22, color='black')
    
    # Set the title and labels to match style
    #plt.title(f'Greedy Sampled Accuracy (Tanimoto = 1.0)', fontsize=22)
    plt.ylabel('Accuracy (%) (Tanimoto = 1.0) ', fontsize=22)
    plt.xticks(positions, filtered_labels, ha='center', fontsize=22)
    ax.set_xticklabels(filtered_labels, rotation=45, ha='center', fontsize=22)
    
    # Adding grid lines for better readability
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    
    # Set y-limit slightly higher than max for percentage labels on top
    ax.set_ylim(0, max(accuracy_percentages) * 1.15)
    ax.tick_params(axis='both', which='major', labelsize=22)
    
    # Style the spines to match your other plots
    for spine in ax.spines.values():
        spine.set_edgecolor('gray')
        spine.set_alpha(0.8)
        
    # Set aspect ratio
    ax.set_aspect(aspect='auto')
    
    # Adjust layout
    plt.tight_layout()
    
    # Save the figure
    save_path = os.path.abspath('./_FIGURES/1.0_Dropout_Correct_Molecules_v8.png')
    plt.savefig(save_path, format='png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"\nPlot saved to: {save_path}")

# Print summary statistics
print("\nSummary Statistics:")
print("="*60)
for label, correct, total, percentage in zip(filtered_labels, correct_counts, total_counts, accuracy_percentages):
    print(f"{label:15s}: {correct:4d}/{total:4d} ({percentage:5.1f}%)")

print("\nDropout correct molecules analysis completed!")

In [ ]:
import pickle
import os
import numpy as np
import matplotlib.pyplot as plt

# Load the data files
file_path_1bi = os.path.abspath("./past_experiments/ChemXriv/1.0_Experiment_Trainings_Experiments/1.1_results_dict_1bi.pkl")
with open(file_path_1bi, 'rb') as file:
    results_dict_1bi = pickle.load(file)
    
file_path_1ci = os.path.abspath("./past_experiments/ChemXriv/1.0_Experiment_Trainings_Experiments/1.1_results_dict_1ci.pkl")
with open(file_path_1ci, 'rb') as file:
    results_dict_1ci = pickle.load(file)

print("Loading Raw and Raw Dropout experiments...")

# Extract Tanimoto data and calculate correct molecules (Tanimoto = 1.0)
datasets = [results_dict_1bi, results_dict_1ci]
labels = ['Raw', 'Raw\nDropout']

correct_counts = []
total_counts = []
accuracy_percentages = []

for i, (dataset, label) in enumerate(zip(datasets, labels)):
    print(f"Processing {label}...")
    
    # Get Tanimoto similarity data
    tanimoto_sim = dataset.get("tanimoto_sim", [])
    
    if len(tanimoto_sim) > 0:
        # Count molecules with Tanimoto similarity = 1.0 (perfect matches)
        correct_count = sum(1 for sim in tanimoto_sim if sim == 1.0)
        total_count = len(tanimoto_sim)
        accuracy_percentage = (correct_count / total_count) * 100 if total_count > 0 else 0
        
        correct_counts.append(correct_count)
        total_counts.append(total_count)
        accuracy_percentages.append(accuracy_percentage)
        
        print(f"  - Total molecules: {total_count}")
        print(f"  - Correct molecules: {correct_count}")
        print(f"  - Accuracy: {accuracy_percentage:.1f}%")
    else:
        print(f"  - No Tanimoto data found for {label}")

# Create the bar plot
fig, ax = plt.subplots(figsize=(8, 10))

# Define positions for the bars
positions = np.arange(len(labels))

# Use blue color for both bars
color = "#8CB0FE"

# Create bars with percentages
bars = ax.bar(positions, accuracy_percentages, color=color, alpha=0.7, edgecolor='black', linewidth=1)

# Add absolute count labels inside the bars - black, non-bold, vertical
for i, (pos, percentage, count, total) in enumerate(zip(positions, accuracy_percentages, correct_counts, total_counts)):
    ax.text(pos, percentage / 2, f'{count}', 
            rotation=90, ha='center', va='center', fontsize=20, color='black')

# Add percentage labels on top of bars
for i, (pos, percentage) in enumerate(zip(positions, accuracy_percentages)):
    ax.text(pos, percentage + max(accuracy_percentages) * 0.02, f'{percentage:.1f}%', 
            ha='center', va='bottom', fontsize=20, color='black')

# Customizing the axes and labels
ax.set_ylabel('Accuracy (%)\n(Tanimoto = 1.0)', fontsize=22)
ax.set_title('Correct Molecules: Raw vs Raw Dropout', fontsize=22)

# Set x-ticks
ax.set_xticks(positions)
ax.set_xticklabels(labels, ha='center', fontsize=22)
ax.tick_params(axis='both', which='major', labelsize=22)

# Add grid
ax.grid(axis='y', linestyle='--', alpha=0.7)
ax.set_ylim(0, max(accuracy_percentages) * 1.1)  # Add some space at the top

plt.tight_layout()

# Save the figure
save_path = os.path.abspath('./_FIGURES/1.0_Raw_vs_Dropout_Correct_Molecules.png')
plt.savefig(save_path, format='png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nPlot saved to: {save_path}")

# Print summary statistics
print("\nSummary Statistics:")
print("="*50)
for label, correct, total, percentage in zip(labels, correct_counts, total_counts, accuracy_percentages):
    print(f"{label:15s}: {correct:4d}/{total:4d} ({percentage:5.1f}%)")

print("\nCorrect molecules comparison completed!")

### 1.1 Run IBM Benchmarking

In [ ]:
config.csv_1H_path_SGNN = os.path.abspath('./data/IBM_dataset/IBM_data_1H_837154_test.csv') 
df = pd.read_csv(config.csv_1H_path_SGNN )
df

In [ ]:
# IBM Data
config.csv_1H_path_SGNN = os.path.abspath('./data/IBM_dataset/IBM_data_1H_837154_test.csv') 
config.csv_13C_path_SGNN = os.path.abspath('./data/IBM_dataset/IBM_data_13C_837154_test.csv')    
config.csv_HSQC_path_SGNN = os.path.abspath('./data/IBM_dataset/IBM_data_HSQC_837154_test.csv') 
config.csv_COSY_path_SGNN = os.path.abspath('./data/IBM_dataset/IBM_data_COSY_837154_test.csv') 
config.csv_path_val = os.path.abspath('./data/IBM_dataset/IBM_data_1H_837154_test.csv') 
config.pickle_file_path = ""
config.IR_data_folder = os.path.abspath('./data/IBM_dataset/IBM_ir_sim') 
config.data_size = 63357 

config.checkpoint_path = os.path.abspath("/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/1_old_models/models_v2/MMST_all_IBM_v2/model-epoch=19-loss=0.0534.ckpt")
config.training_mode = "1H_13C_HSQC_COSY_IR_MF_MW" # it ignors IR because no folder provided and puts zeros in for IR
config.multinom_runs = 1
config.batch_size = 1024 #int(1000*data_fraction)

val_dataloader_multi = mrtf.load_data(config, stoi, stoi_MF, single=False, mode="val")


In [ ]:
model_MMT = mrtf.load_MMT_model(config)
prob_dict_results_1bi, results_dict_1bi = mrtf.run_model_analysis(config, model_MMT, val_dataloader_multi, stoi, itos)
print(np.mean(results_dict_1bi["tanimoto_sim"]))

# Save the data to a file
file_prob_dict_path = os.path.abspath('/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/___FIGURES_PAPERS/Figures_Paper_2/precomputed_raw_data/20250612_IBM_comparison/1.0_prob_dict_results_IBM.pkl')
with open(file_prob_dict_path, 'wb') as file:
    pickle.dump(prob_dict_results_1bi, file)
    
# Save the data to a file
file_results_dict_path = os.path.abspath('/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/___FIGURES_PAPERS/Figures_Paper_2/precomputed_raw_data/20250612_IBM_comparison/1.1_results_dict_IBM.pkl')  
with open(file_results_dict_path, 'wb') as file:
    pickle.dump(results_dict_1bi, file)

In [ ]:
import os
import pickle
import numpy as np
import torch
import matplotlib.pyplot as plt
from tqdm import tqdm
from collections import defaultdict
from utils_MMT.mmt_result_test_functions_15_4 import run_precentage_calculation_v2

def calc_percentage_and_count_top_x_correct_greedy_ibm(results_dict):
    # Check if results_dict is a dictionary
    if not isinstance(results_dict, dict):
        print(f"Warning: results_dict_greedy is not a dictionary, it's a {type(results_dict)}")
        return 0, 0, 0
        
    count_yes = 0
    count_no = 0
    
    for idx in results_dict.keys():
        for result in results_dict[idx]:
            if isinstance(result, dict) and "tanimoto_sim" in result:
                tanimoto_values = result["tanimoto_sim"]
                # Convert tensor to numpy if needed
                if torch.is_tensor(tanimoto_values):
                    tanimoto_values = tanimoto_values.cpu().numpy()
                
                # Handle both single values and arrays
                if hasattr(tanimoto_values, '__iter__'):
                    for i in tanimoto_values:
                        if i == 1:
                            count_yes += 1
                        else:
                            count_no += 1
                else:
                    # Handle single value
                    if tanimoto_values == 1:
                        count_yes += 1
                    else:
                        count_no += 1

    total = count_yes + count_no
    percentage = (count_yes / total) * 100 if total > 0 else 0
    return percentage, count_yes, total

def calc_percentage_and_count_top_x_correct_ibm(results_dict, top_x):
    # Check if results_dict is a dictionary
    if not isinstance(results_dict, dict):
        print(f"Warning: results_dict is not a dictionary, it's a {type(results_dict)}")
        return 0, 0, 0
        
    count_yes = 0
    count_no = 0
    
    for idx in results_dict.keys():
        for result in results_dict[idx]:
            if isinstance(result, dict) and "tanimoto_sim" in result:
                tanimoto_sim = result["tanimoto_sim"]
                # Convert tensor to numpy if needed
                if torch.is_tensor(tanimoto_sim):
                    tanimoto_sim = tanimoto_sim.cpu().numpy()
                
                # Handle both single values and arrays
                if hasattr(tanimoto_sim, '__iter__'):
                    # Take only top_x elements
                    tanimoto_sim = tanimoto_sim[:top_x]
                    
                    if 1 in tanimoto_sim:
                        count_yes += 1
                    else:
                        count_no += 1
                else:
                    # Handle single value
                    if tanimoto_sim == 1:
                        count_yes += 1
                    else:
                        count_no += 1
                    
    total = count_yes + count_no
    percentage = (count_yes / total) * 100 if total > 0 else 0
    return percentage, count_yes, total

def prepare_ibm_data(results_dict_mns_10, results_dict_greedy):
    # Check the structure of results before processing
    print(f"Type of results_dict_mns_10: {type(results_dict_mns_10)}")
    print(f"Type of results_dict_greedy: {type(results_dict_greedy)}")
    
    # For greedy results, we need to handle differently based on what run_precentage_calculation_v2 returns
    if isinstance(results_dict_greedy, dict):
        percentage_greedy, count_greedy, total_greedy = calc_percentage_and_count_top_x_correct_greedy_ibm(results_dict_greedy)
    else:
        # If it's a float (which seems to be the case from the error), use it directly
        percentage_greedy = float(results_dict_greedy) * 100 if results_dict_greedy is not None else 0
        count_greedy = 0  # We don't have count information in this case
        total_greedy = 0
    
    # For multinomial results
    if isinstance(results_dict_mns_10, dict):
        percentage_top_1, count_top_1, total_1 = calc_percentage_and_count_top_x_correct_ibm(results_dict_mns_10, 1)
        percentage_top_3, count_top_3, total_3 = calc_percentage_and_count_top_x_correct_ibm(results_dict_mns_10, 3)
        percentage_top_5, count_top_5, total_5 = calc_percentage_and_count_top_x_correct_ibm(results_dict_mns_10, 5)
        percentage_top_10, count_top_10, total_10 = calc_percentage_and_count_top_x_correct_ibm(results_dict_mns_10, 10)
    else:
        # If it's not a dictionary, use percentage_collection values directly
        percentage_top_1 = percentage_collection_IBM_MW[1] * 100 if percentage_collection_IBM_MW is not None and len(percentage_collection_IBM_MW) > 1 else 0
        percentage_top_3 = percentage_collection_IBM_MW[2] * 100 if percentage_collection_IBM_MW is not None and len(percentage_collection_IBM_MW) > 2 else 0
        percentage_top_5 = percentage_collection_IBM_MW[3] * 100 if percentage_collection_IBM_MW is not None and len(percentage_collection_IBM_MW) > 3 else 0
        percentage_top_10 = percentage_collection_IBM_MW[4] * 100 if percentage_collection_IBM_MW is not None and len(percentage_collection_IBM_MW) > 4 else 0
        count_top_1 = count_top_3 = count_top_5 = count_top_10 = 0
        total_1 = total_3 = total_5 = total_10 = 0
    
    ibm_data = [(percentage_greedy, count_greedy, total_greedy), 
                (percentage_top_1, count_top_1, total_1),
                (percentage_top_3, count_top_3, total_3), 
                (percentage_top_5, count_top_5, total_5),
                (percentage_top_10, count_top_10, total_10)]
    
    return ibm_data



def plot_ibm_results(ibm_data, filter_name="MW Filter"):
    labels = ['Greedy', '1 Sample', '3 Samples', '5 Samples', '10 Samples']
    x = np.arange(len(labels))
    width = 0.6  # Wider bar since we only have one category
    color = '#FFB381'  # Using the MW filter color from your original code
    
    fig, ax = plt.subplots(figsize=(15, 8))
    
    # Plot bars
    rects = ax.bar(x, [d[0] for d in ibm_data], width, label=f'IBM with {filter_name}', color=color)

    # Set labels and title with larger font sizes
    ax.set_ylabel('Percentage', fontsize=30)
    ax.set_title(f'IBM Model Performance with {filter_name}', fontsize=30)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=30)
    ax.legend(fontsize=30)
    ax.tick_params(axis='y', labelsize=30)
    ax.set_ylim(0, 100)  # Full scale from 0-100%

    # Function to add annotations to the bars
    def autolabel(rects, data):
        for rect, (percentage, count, total) in zip(rects, data):
            height = rect.get_height()
            
            # Add percentage on top of the bar
            ax.text(rect.get_x() + rect.get_width() / 2, height + 2, f'{percentage:.1f}%',
                    ha='center', va='bottom', fontsize=20, rotation=0)
            
            # Add count/total in the middle of the bar
            ax.text(rect.get_x() + rect.get_width() / 2, height / 2, f'{count}/{total}',
                    ha='center', va='center', fontsize=20, rotation=0)

    # Add annotations to each bar
    autolabel(rects, ibm_data)

    fig.tight_layout()
    plt.savefig(os.path.abspath(f'./IBM_experiment_results/IBM_performance_{filter_name.replace(" ", "_").lower()}.png'))
    plt.show()

def plot_violin_probability(results_dict, filter_name="MW Filter"):
    fig, ax = plt.subplots(figsize=(8, 10))
    data_for_violin = []

    # Process data for violin plot with tensor handling
    for idx in results_dict.keys():
        for result in results_dict[idx]:
            if "aggregated_corr_prob_multi" in result:
                # Handle tensor if present
                if torch.is_tensor(result["aggregated_corr_prob_multi"]):
                    data_for_violin.append(result["aggregated_corr_prob_multi"].cpu().numpy())
                else:
                    data_for_violin.append(result["aggregated_corr_prob_multi"])
            elif "prob_list" in result:
                # Handle tensor if present
                if torch.is_tensor(result["prob_list"]):
                    data_for_violin.append(result["prob_list"].cpu().numpy())
                else:
                    data_for_violin.append(result["prob_list"])

    parts = ax.violinplot(data_for_violin, showmeans=True, showmedians=False, showextrema=False)

    # Customizing colors
    color = "#8CB0FE"
    for pc in parts['bodies']:
        pc.set_facecolor(color)
        pc.set_edgecolor('black')
        pc.set_alpha(0.7)

    # Customizing the axes and labels
    ax.set_title(f'IBM: Correct SMILES Sample Probability ({filter_name})', fontsize=22)
    ax.set_ylabel('Probability of Correct SMILES', fontsize=22)
    ax.set_xticks([1])
    ax.set_xticklabels(['IBM Model'], fontsize=22)
    ax.tick_params(axis='both', which='major', labelsize=22)

    # Calculate mean safely
    mean_values = []
    for data in data_for_violin:
        if len(data) > 0:
            mean_values.append(np.mean(data))
            
    if mean_values:
        mean_value = np.mean(mean_values)
        ax.text(1, mean_value, f'{mean_value:.2f}', ha='center', va='bottom', fontsize=22)

    # Add grid and set the limits
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    ax.set_ylim(0, 1)

    plt.tight_layout()
    save_path = os.path.join(results_dir, f'IBM_Violin_Prob_{filter_name.replace(" ", "_")}.png')
    plt.savefig(save_path)
    plt.show()

def plot_violin_tanimoto(results_dict, filter_name="MW Filter"):
    fig, ax = plt.subplots(figsize=(8, 10))
    data_for_violin = []

    # Process tanimoto similarity data with tensor handling
    for idx in results_dict.keys():
        for result in results_dict[idx]:
            if "tanimoto_sim" in result:
                # Handle tensor if present
                if torch.is_tensor(result["tanimoto_sim"]):
                    data_for_violin.append(result["tanimoto_sim"].cpu().numpy())
                else:
                    data_for_violin.append(result["tanimoto_sim"])

    parts = ax.violinplot(data_for_violin, showmeans=True, showmedians=False, showextrema=False)

    # Customizing colors
    color = "#8CB0FE"
    for pc in parts['bodies']:
        pc.set_facecolor(color)
        pc.set_edgecolor('black')
        pc.set_alpha(0.7)

    # Customizing the axes and labels
    ax.set_title(f'IBM: Greedy Sampled Average Tanimoto Similarity ({filter_name})', fontsize=22)
    ax.set_ylabel('Average Tanimoto Similarity', fontsize=22)
    ax.set_xticks([1])
    ax.set_xticklabels(['IBM Model'], fontsize=22)
    ax.tick_params(axis='both', which='major', labelsize=22)

    # Calculate mean safely
    mean_values = []
    for data in data_for_violin:
        if len(data) > 0:
            mean_values.append(np.mean(data))
            
    if mean_values:
        mean_value = np.mean(mean_values)
        ax.text(1, mean_value, f'{mean_value:.2f}', ha='center', va='bottom', fontsize=22)

    # Add grid and set the limits
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    ax.set_ylim(0, 1)

    plt.tight_layout()
    save_path = os.path.join(results_dir, f'IBM_Tanimoto_Violin_{filter_name.replace(" ", "_")}.png')
    plt.savefig(save_path)
    plt.show()

def plot_invalid_smiles(results_dict, filter_name="MW Filter"):
    fig, ax = plt.subplots(figsize=(8, 10))
    failed_count = 0
    total_entries = 0

    # Count failed and total entries
    for idx in results_dict.keys():
        for result in results_dict[idx]:
            if "failed" in result and "gen_conv_SMI_list" in result:
                # Handle tensor if present
                if torch.is_tensor(result["failed"]):
                    failed_count += len(result["failed"].cpu().numpy())
                else:
                    failed_count += len(result["failed"])
                    
                # Handle tensor if present
                if torch.is_tensor(result["gen_conv_SMI_list"]):
                    total_entries += len(result["gen_conv_SMI_list"].cpu().numpy())
                else:
                    total_entries += len(result["gen_conv_SMI_list"])

    percentage = (failed_count / total_entries) * 100 if total_entries > 0 else 0
    color = "#8CB0FE"

    # Plotting the bar
    bar = ax.bar([0], [failed_count], width=0.35, color=color, edgecolor='black')

    # Adding value labels inside and percentage on top of each bar
    ax.text(0, failed_count / 2, f'{percentage:.1f}%', 
            rotation=90, ha='center', va='center', fontsize=22)

    # Set the title and labels
    ax.set_title(f'IBM: Greedy Sampled Number of Invalid SMILES ({filter_name})', fontsize=22)
    ax.set_ylabel('Invalid Molecules', fontsize=22)
    ax.set_xticks([0])
    ax.set_xticklabels(['IBM Model'], fontsize=22)
    ax.tick_params(axis='both', which='major', labelsize=22)

    # Adding grid lines for better readability
    ax.grid(axis='y', linestyle='--', alpha=0.7)

    # Set y-limit slightly higher than max for label visibility
    ax.set_ylim(0, failed_count * 1.25 if failed_count > 0 else 1)

    plt.tight_layout()
    save_path = os.path.join(results_dir, f'IBM_Invalid_molecules_{filter_name.replace(" ", "_")}.png')
    plt.savefig(save_path)
    plt.show()
    
def run_ibm_experiment(mw_filter=True, mf_filter=False, data_size=10):
    # Configure paths for IBM dataset
    config.csv_1H_path_SGNN = os.path.abspath('./data/IBM_dataset/IBM_data_1H_837154_test.csv') 
    config.csv_13C_path_SGNN = os.path.abspath('./data/IBM_dataset/IBM_data_13C_837154_test.csv')    
    config.csv_HSQC_path_SGNN = os.path.abspath('./data/IBM_dataset/IBM_data_HSQC_837154_test.csv') 
    config.csv_COSY_path_SGNN = os.path.abspath('./data/IBM_dataset/IBM_data_COSY_837154_test.csv') 
    config.csv_path_val = os.path.abspath('./data/IBM_dataset/IBM_data_1H_837154_test.csv') 
    config.pickle_file_path = ""
    config.IR_data_folder = os.path.abspath('./data/IBM_dataset/IBM_ir_sim') 
    config.data_size = data_size
    
    config.checkpoint_path = os.path.abspath("/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/1_old_models/models_v2/MMST_all_IBM_v2/model-epoch=19-loss=0.0534.ckpt")
    config.training_mode = "1H_13C_HSQC_COSY_IR_MF_MW" 
    config.temperature = 1
    config.multinom_runs = 10
    
    # Create filter string for file naming
    mw_str = "TRUE" if mw_filter else "FALSE"
    mf_str = "TRUE" if mf_filter else "FALSE"
    filter_str = f"{mw_str}_{mf_str}"
    
    # Create results directory
    results_dir = os.path.abspath('./IBM_experiment_results')
    os.makedirs(results_dir, exist_ok=True)
    
    # Determine filter description for plot titles
    if mw_filter and mf_filter:
        filter_desc = "MW+MF Filters"
    elif mw_filter:
        filter_desc = "MW Filter"
    elif mf_filter:
        filter_desc = "MF Filter"
    else:
        filter_desc = "No Filters"
    
    # Run the experiment
    print(f"Running IBM experiment with MW_filter={mw_filter}, MF_filter={mf_filter}")
    percentage_collection, results_dict_mns_10, results_dict_greedy = run_precentage_calculation_v2(
        config, itos, stoi, stoi_MF, MW_filter=mw_filter, MF_filter=mf_filter
    )
    
    # Print the types of the returned values to help debug
    print(f"Type of percentage_collection: {type(percentage_collection)}")
    print(f"Type of results_dict_mns_10: {type(results_dict_mns_10)}")
    print(f"Type of results_dict_greedy: {type(results_dict_greedy)}")
    
    # Save the results
    file_path = os.path.join(results_dir, f'IBM_results_dict_greedy_{filter_str}.pkl')
    with open(file_path, 'wb') as file:
        pickle.dump(results_dict_greedy, file)
    
    file_path = os.path.join(results_dir, f'IBM_results_dict_mns_10_{filter_str}.pkl')
    with open(file_path, 'wb') as file:
        pickle.dump(results_dict_mns_10, file)
    
    file_path = os.path.join(results_dir, f'IBM_percentage_collection_{filter_str}.pkl')
    with open(file_path, 'wb') as file:
        pickle.dump(percentage_collection, file)
    
    # Print results
    print(f"Experiment Results ({filter_desc}):")
    print(f"Greedy sampling accuracy: {percentage_collection[0]:.4f}")
    print(f"Top-1 accuracy: {percentage_collection[1]:.4f}")
    print(f"Top-3 accuracy: {percentage_collection[2]:.4f}")
    print(f"Top-5 accuracy: {percentage_collection[3]:.4f}")
    print(f"Top-10 accuracy: {percentage_collection[4]:.4f}")
    
    # Create bar chart visualization using percentage_collection directly
    labels = ['Greedy', '1 Sample', '3 Samples', '5 Samples', '10 Samples']
    x = np.arange(len(labels))
    width = 0.6
    color = '#FFB381'
    
    fig, ax = plt.subplots(figsize=(15, 8))
    
    # Plot bars using percentage_collection values directly
    percentages = [p * 100 for p in percentage_collection]  # Convert to percentages
    rects = ax.bar(x, percentages, width, label=f'IBM with {filter_desc}', color=color)

    # Set labels and title with larger font sizes
    ax.set_ylabel('Percentage', fontsize=30)
    ax.set_title(f'IBM Model Performance with {filter_desc}', fontsize=30)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=30)
    ax.legend(fontsize=30)
    ax.tick_params(axis='y', labelsize=30)
    ax.set_ylim(0, 100)

    # Add percentage annotations
    for i, rect in enumerate(rects):
        height = rect.get_height()
        ax.text(rect.get_x() + rect.get_width() / 2, height + 2, 
                f'{percentages[i]:.1f}%',
                ha='center', va='bottom', fontsize=20)

    fig.tight_layout()
    plt.savefig(os.path.abspath(f'./IBM_experiment_results/IBM_performance_{filter_desc.replace(" ", "_").lower()}.png'))
    plt.show()
    """
    # Skip the other visualizations if we don't have the right data structure
    if isinstance(results_dict_mns_10, dict):
        # Violin plot for correct SMILES sample probability
        plot_violin_probability(results_dict_mns_10, filter_desc)
    
    if isinstance(results_dict_greedy, dict):
        # Violin plot for Tanimoto similarity
        plot_violin_tanimoto(results_dict_greedy, filter_desc)
        
        # Bar chart for invalid SMILES count
        plot_invalid_smiles(results_dict_greedy, filter_desc)
    """
    return percentage_collection, results_dict_mns_10, results_dict_greedy

# Execute the experiment with MW filter enabled
percentage_collection_IBM_MW, results_dict_mns_10_IBM_MW, results_dict_greedy_IBM_MW = run_ibm_experiment(
    mw_filter=True,
    mf_filter=False,
    data_size=63357  # Set to your desired data size (10 for testing, larger for full experiment)
)

# If you want to run with different filter settings, uncomment the relevant lines:
# Run with no filters
# percentage_collection_IBM_NONE, results_dict_mns_10_IBM_NONE, results_dict_greedy_IBM_NONE = run_ibm_experiment(
#     mw_filter=False,
#     mf_filter=False,
#     data_size=10
# )

# Run with MF filter only
# percentage_collection_IBM_MF, results_dict_mns_10_IBM_MF, results_dict_greedy_IBM_MF = run_ibm_experiment(
#     mw_filter=False,
#     mf_filter=True,
#     data_size=10
# )

# Run with both MW and MF filters
# percentage_collection_IBM_BOTH, results_dict_mns_10_IBM_BOTH, results_dict_greedy_IBM_BOTH = run_ibm_experiment(
#     mw_filter=True,
#     mf_filter=True,
#     data_size=10
# )

In [ ]:
import os
import pickle
import numpy as np
import torch
import matplotlib.pyplot as plt

# Define the results directory
results_dir = "/projects/cc/se_users/knlr326/1_NMR_project/2_Notebooks/MultiModalSpectralTransformer_cleaned/IBM_experiment_results"

# Load the pickle files
def load_ibm_results():
    # Load the files
    with open(os.path.join(results_dir, 'IBM_percentage_collection_TRUE_FALSE.pkl'), 'rb') as f:
        percentage_collection = pickle.load(f)
    
    with open(os.path.join(results_dir, 'IBM_results_dict_greedy_TRUE_FALSE.pkl'), 'rb') as f:
        results_dict_greedy = pickle.load(f)
    
    with open(os.path.join(results_dir, 'IBM_results_dict_mns_10_TRUE_FALSE.pkl'), 'rb') as f:
        results_dict_mns_10 = pickle.load(f)
    
    print(f"Loaded percentage_collection: {type(percentage_collection)}")
    print(f"Loaded results_dict_greedy: {type(results_dict_greedy)}")
    print(f"Loaded results_dict_mns_10: {type(results_dict_mns_10)}")
    
    return percentage_collection, results_dict_greedy, results_dict_mns_10

# Load the data
percentage_collection, results_dict_greedy, results_dict_mns_10 = load_ibm_results()


In [ ]:
empty_gen_count = 0
total_entries = 0

for key in results_dict_mns_10.keys():
   for result in results_dict_mns_10[key]:
       total_entries += 1
       if 'gen_conv_SMI_list' in result and len(result['gen_conv_SMI_list']) == 0:
           empty_gen_count += 1

print(f"Entries with empty gen_conv_SMI_list: {empty_gen_count}")
print(f"Total entries: {total_entries}")
print(f"Percentage with empty gen_conv_SMI_list: {(empty_gen_count/total_entries)*100:.2f}%")

In [ ]:
def extract_smiles_with_ids(results_dict):
    """
    Extract all SMILES with target molecule first, then generated molecules.
    
    Args:
        results_dict: Dictionary containing the results data
        
    Returns:
        dict: Dictionary with 'id' and 'smiles' columns
    """
    extracted_data = {
        'sample-id': [],
        'SMILES': []
    }
    
    for main_key in results_dict.keys():
        # Get the list of results for this key
        results_list = results_dict[main_key]
        
        for result_dict in results_list:
            # First add the target molecule as idx_0
            if 'trg_conv_SMI_list' in result_dict:
                target_smiles_list = result_dict['trg_conv_SMI_list']
                if target_smiles_list:  # Check if list is not empty
                    # Take the first target molecule as idx_0
                    target_smile = target_smiles_list[0]
                    unique_id = f"{main_key}_0"
                    
                    extracted_data['sample-id'].append(unique_id)
                    extracted_data['SMILES'].append(target_smile)
            
            # Then add all generated molecules starting from idx_1
            if 'gen_conv_SMI_list' in result_dict:
                smiles_list = result_dict['gen_conv_SMI_list']
                
                # Create ID and extract SMILES for each molecule in the list
                for mol_index, smile in enumerate(smiles_list):
                    # Create unique identifier: main_key_(mol_index + 1) since 0 is reserved for target
                    unique_id = f"{main_key}_{mol_index + 1}"
                    
                    extracted_data['sample-id'].append(unique_id)
                    extracted_data['SMILES'].append(smile)
    
    return extracted_data

# Example usage with your data
# Assuming your data is stored in a variable called results_dict_mns_10 or similar

# Extract the SMILES with IDs
smiles_data = extract_smiles_with_ids(results_dict_mns_10)

# Display the results
print("Extracted SMILES data:")
print(f"Total molecules: {len(smiles_data['sample-id'])}")
print("\nFirst 10 entries:")
for i in range(min(10, len(smiles_data['sample-id']))):
    print(f"ID: {smiles_data['sample-id'][i]}, SMILES: {smiles_data['SMILES'][i]}")

# Convert to pandas DataFrame for easier handling (optional)
import pandas as pd
df_smiles = pd.DataFrame(smiles_data)
print(f"\nDataFrame shape: {df_smiles.shape}")
print(df_smiles.head(10))

# Set the save path
save_dir = "/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/___FIGURES_PAPERS/Figures_Paper_2/precomputed_raw_data/20250612_IBM_comparison/Experiment_ranking"
save_path = os.path.join(save_dir, 'extracted_smiles.csv')

# Save to CSV if needed
df_smiles.to_csv(save_path, index=False)


In [ ]:
import pandas as pd
import os
from utils_MMT.sgnn_code_pl_v15_4 import load_std_mean, main_execute

def generate_nmr_from_csv(csv_path, output_dir):
    """
    Generate NMR spectra from a CSV file containing SMILES.
    
    Args:
        csv_path (str): Path to CSV file with SMILES
        output_dir (str): Directory to save SDF files with NMR data
    
    Returns:
        pd.DataFrame: DataFrame with processed molecules and paths to SDF files
    """
    # Read CSV file
    df = pd.read_csv(csv_path)
    
    # Ensure required columns exist
    if 'SMILES' not in df.columns:
        raise ValueError("CSV must contain a 'SMILES' column")
    
    # Add sample-id if not present
    if 'sample-id' not in df.columns:
        df['sample-id'] = [f"{i}" for i in range(len(df))]
    
    # Select only required columns
    df_smiles = df[['sample-id', 'SMILES']]
    
    # Load SGNN means and stds the same way as in run_sgnn
    graph_representation = "sparsified"
    target = "13C"
    train_y_mean_C, train_y_std_C = load_std_mean(target, graph_representation)
    target = "1H"
    train_y_mean_H, train_y_std_H = load_std_mean(target, graph_representation)
    sgnn_means_stds = (train_y_mean_C, train_y_std_C, train_y_mean_H, train_y_std_H)
    
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # Process SMILES in batches
    print(f"Processing {len(df_smiles)} molecules...")
    batch_data, failed_ids = main_execute(df_smiles, sgnn_means_stds, output_dir, batch_size=10)
    
    # Process failed molecules with smaller batch size
    if failed_ids:
        print(f"Retrying {len(failed_ids)} failed molecules with batch size 1...")
        df_failed = df_smiles[df_smiles['sample-id'].isin(failed_ids)]
        batch_data_add, remaining_failed = main_execute(df_failed, sgnn_means_stds, output_dir, batch_size=1)
        
        # Combine results
        if len(batch_data_add) > 0:
            batch_data = pd.concat([batch_data, batch_data_add], ignore_index=True)
        
        print(f"Final results: {len(batch_data)} successful, {len(remaining_failed)} failed")
    
    return batch_data

# Example usage
if __name__ == "__main__":
    csv_path = "/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/___FIGURES_PAPERS/Figures_Paper_2/precomputed_raw_data/20250612_IBM_comparison/Experiment_ranking/extracted_smiles.csv"
    output_dir = "/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/___FIGURES_PAPERS/Figures_Paper_2/precomputed_raw_data/20250612_IBM_comparison/Experiment_ranking"
    
    results = generate_nmr_from_csv(csv_path, output_dir)
    print(f"Generated NMR spectra for {len(results)} molecules")
    print(f"SDF files saved to {output_dir}")

In [ ]:
from utils_MMT.data_generation_v15_4 import main_run_data_generation

# Set the required paths
config.SGNN_csv_gen_smi = "/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/___FIGURES_PAPERS/Figures_Paper_2/precomputed_raw_data/20250612_IBM_comparison/Experiment_ranking/extracted_smiles.csv"
config.SGNN_gen_folder_path = "/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/___FIGURES_PAPERS/Figures_Paper_2/precomputed_raw_data/20250612_IBM_comparison/Experiment_ranking"

# Run the SGNN simulation
combined_df, data_1H, data_13C, data_COSY, data_HSQC, csv_1H_path, csv_13C_path, csv_COSY_path, csv_HSQC_path = main_run_data_generation(config)


In [ ]:
import pickle
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ast
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem
# Assuming utils_MMT is in your Python path or the same directory
import utils_MMT.functions_HSQC_sim_v15_4 as hsqc_sim

def calculate_tanimoto(smiles1, smiles2):
    """Calculate Tanimoto similarity between two SMILES strings"""
    try:
        mol1 = Chem.MolFromSmiles(smiles1)
        mol2 = Chem.MolFromSmiles(smiles2)
        if mol1 is None or mol2 is None:
            return 0.0
            
        fp1 = AllChem.GetMorganFingerprintAsBitVect(mol1, 2, nBits=2048)
        fp2 = AllChem.GetMorganFingerprintAsBitVect(mol2, 2, nBits=2048)
        
        return DataStructs.TanimotoSimilarity(fp1, fp2)
    except Exception as e:
        print(f"Error calculating Tanimoto: {e}")
        return 0.0

def process_hsqc_data(csv_path):
    """Process the CSV file and calculate HSQC errors"""
    # Load and parse CSV data
    df = pd.read_csv(csv_path)
    df['shifts'] = df['shifts'].apply(ast.literal_eval)
    
    results_dict = {}
    
    # Group by first part of the sample-id (before underscore)
    for group_id, group_df in df.groupby(df['sample-id'].str.split('_').str[0]):
        # Find the target molecule
        target_df = group_df[group_df['sample-id'].str.endswith('_0')]
        if len(target_df) == 0:
            print(f"No target molecule found for group {group_id}")
            continue
            
        target_row = target_df.iloc[0]
        target_smile = target_row['SMILES']
        target_shifts = target_row['shifts']
        
        # Convert target shifts to required format
        target_hsqc_df = pd.DataFrame(target_shifts, columns=['F2 (ppm)', 'F1 (ppm)'])
        
        # Get generated molecules
        generated_rows = group_df[~group_df['sample-id'].str.endswith('_0')]
        
        # Calculate HSQC errors and store results
        results = []
        for _, gen_row in generated_rows.iterrows():
            gen_id = gen_row['sample-id']
            gen_smile = gen_row['SMILES']
            gen_shifts = gen_row['shifts']
            
            # Convert generated shifts to required format
            gen_hsqc_df = pd.DataFrame(gen_shifts, columns=['F2 (ppm)', 'F1 (ppm)'])
            
            # Calculate HSQC error using Hungarian distance with nearest neighbor
            hsqc_error, _, _ = hsqc_sim.similarity_calculations(
                target_hsqc_df, 
                gen_hsqc_df, 
                mode="hung_dist_nn",
                similarity_type="euclidean", 
                error="avg", 
                assignment_plot=False
            )
            
            # Calculate Tanimoto similarity
            tanimoto = calculate_tanimoto(target_smile, gen_smile)
            
            results.append({
                'generated_id': gen_id,
                'generated_smile': gen_smile,
                'hsqc_error': float(hsqc_error),
                'tanimoto': tanimoto
            })
        
        # Sort by HSQC error (lowest first)
        results = sorted(results, key=lambda x: x['hsqc_error'])
        
        # Store in the results dictionary
        results_dict[group_id] = {
            'target_smile': target_smile,
            'results': results
        }
    
    return results_dict

def calculate_top_n_accuracy(results_dict, top_n):
    """Calculate top-N accuracy and return both counts and percentage"""
    count_yes = 0
    count_total = len(results_dict)  # Total number of groups/samples
    
    for group_id, group_data in results_dict.items():
        # Get the top N results sorted by HSQC error
        top_results = group_data['results'][:top_n]
        
        # Check if any of the top results has Tanimoto=1
        has_match = any(result['tanimoto'] == 1.0 for result in top_results)
        
        if has_match:
            count_yes += 1
    
    # Calculate accuracy percentage
    percentage = (count_yes / count_total) * 100 if count_total > 0 else 0.0
    
    return count_yes, count_total, percentage

def load_ibm_results(ibm_results_dir):
    """Load IBM experiment results from pickle files"""
    
    # Load the IBM experiment results
    with open(os.path.join(ibm_results_dir, 'IBM_percentage_collection_TRUE_FALSE.pkl'), 'rb') as f:
        ibm_percentage_collection = pickle.load(f)
    
    with open(os.path.join(ibm_results_dir, 'IBM_results_dict_greedy_TRUE_FALSE.pkl'), 'rb') as f:
        results_dict_greedy = pickle.load(f)
    
    with open(os.path.join(ibm_results_dir, 'IBM_results_dict_mns_10_TRUE_FALSE.pkl'), 'rb') as f:
        results_dict_mns_10 = pickle.load(f)
    
    return ibm_percentage_collection, results_dict_greedy, results_dict_mns_10

def get_ibm_counts(results_dict_mns_10, results_dict_greedy, ibm_percentage_collection):
    """Calculate counts from IBM results"""
    
    # Calculate total samples
    mns_total = 0
    for idx in results_dict_mns_10:
        mns_total += len(results_dict_mns_10[idx])
    
    # Calculate counts
    greedy_count = int(round(ibm_percentage_collection[0] * mns_total))
    top_1_count = int(round(ibm_percentage_collection[1] * mns_total))
    top_3_count = int(round(ibm_percentage_collection[2] * mns_total))
    top_5_count = int(round(ibm_percentage_collection[3] * mns_total))
    top_10_count = int(round(ibm_percentage_collection[4] * mns_total))
    
    return greedy_count, top_1_count, top_3_count, top_5_count, top_10_count, mns_total

def plot_combined_comparison(csv_path, ibm_results_dir, save_path):
    """Load all data and create combined comparison plot"""
    
    # Load IBM results
    print("Loading IBM experiment results...")
    ibm_percentage_collection, results_dict_greedy, results_dict_mns_10 = load_ibm_results(ibm_results_dir)
    
    # Get counts and total from the IBM results
    print("Calculating IBM counts...")
    greedy_count, top_1_count, top_3_count, top_5_count, top_10_count, mns_total = get_ibm_counts(
        results_dict_mns_10, 
        results_dict_greedy, 
        ibm_percentage_collection
    )
    
    # Process the HSQC data
    print("Processing HSQC data...")
    hsqc_results_dict = process_hsqc_data(csv_path)
    
    # Calculate top-N accuracies for HSQC
    print("Calculating HSQC accuracies...")
    hsqc_top_1_count, hsqc_total, hsqc_top1_pct = calculate_top_n_accuracy(hsqc_results_dict, 1)
    hsqc_top_3_count, _, hsqc_top3_pct = calculate_top_n_accuracy(hsqc_results_dict, 3)
    hsqc_top_5_count, _, hsqc_top5_pct = calculate_top_n_accuracy(hsqc_results_dict, 5)
    hsqc_top_10_count, _, hsqc_top10_pct = calculate_top_n_accuracy(hsqc_results_dict, 10)
    
    # Extract IBM percentages and convert to percentages
    ibm_greedy = ibm_percentage_collection[0] * 100
    ibm_top1 = ibm_percentage_collection[1] * 100
    ibm_top3 = ibm_percentage_collection[2] * 100
    ibm_top5 = ibm_percentage_collection[3] * 100
    ibm_top10 = ibm_percentage_collection[4] * 100
    
    # Reference data from IR + MF paper (Alberts et al.)
    ref_top1 = 45.33
    ref_top3 = 0.0  # Not provided in reference
    ref_top5 = 72.21
    ref_top10 = 78.50
    
    # Calculate reference counts based on percentages and MNS total
    ref_counts = []
    # Note: Removed top-3 from the reference data as it's 0.0
    for ref_pct in [0.0, 0.0, ref_top1, ref_top5, ref_top10]: 
        if ref_pct > 0:
            ref_counts.append(int(round((ref_pct / 100) * mns_total)))
        else:
            ref_counts.append(0)
    
    # Print results
    print("\nHSQC Results:")
    print(f"Top-1: {hsqc_top1_pct:.1f}% ({hsqc_top_1_count:,}/{hsqc_total:,})")
    print(f"Top-3: {hsqc_top3_pct:.1f}% ({hsqc_top_3_count:,}/{hsqc_total:,})")
    print(f"Top-5: {hsqc_top5_pct:.1f}% ({hsqc_top_5_count:,}/{hsqc_total:,})")
    print(f"Top-10: {hsqc_top10_pct:.1f}% ({hsqc_top_10_count:,}/{hsqc_total:,})")
    
    print("\nMMST Results:")
    print(f"Greedy: {ibm_greedy:.1f}% ({greedy_count:,}/{mns_total:,})")
    print(f"Top-1: {ibm_top1:.1f}% ({top_1_count:,}/{mns_total:,})")
    print(f"Top-3: {ibm_top3:.1f}% ({top_3_count:,}/{mns_total:,})")
    print(f"Top-5: {ibm_top5:.1f}% ({top_5_count:,}/{mns_total:,})")
    print(f"Top-10: {ibm_top10:.1f}% ({top_10_count:,}/{mns_total:,})")
    
    print("\nIR + MF Reference:")
    print(f"Top-1: {ref_top1:.1f}%")
    print(f"Top-5: {ref_top5:.1f}%")
    print(f"Top-10: {ref_top10:.1f}%")
    
    # Define labels and data (excluding Top-3 as reference is not available)
    labels = ['MMST-Top-10\nHSQC ranked', 'Greedy', 'MNS\nTop-1', 'MNS\nTop-5', 'MNS\nTop-10']
    mmst_values = [hsqc_top1_pct, ibm_greedy, ibm_top1, ibm_top5, ibm_top10]
    ref_values = [0.0, 0.0, ref_top1, ref_top5, ref_top10]
    
    # Count data for display inside bars
    mmst_display_counts = [hsqc_top_1_count, 
                           greedy_count,
                           top_1_count,
                           top_5_count,
                           top_10_count]
    
    # Colors
    mmst_color = '#8CB0FE'  # Blue for MMST
    ref_color = '#FFA07A'   # Orange for IR+MF
    hsqc_color = '#90EE90'  # Light green for HSQC
    
    # Create the figure with 16:10 aspect ratio
    fig, ax = plt.subplots(figsize=(16, 10))
    
    # Set up bar positions
    x = np.arange(len(labels))
    width = 0.35
    
    # Create custom colors for MMST bars
    mmst_colors = [hsqc_color if i == 0 else mmst_color for i in range(len(labels))]
    
    # Create bars - center single bars for first two categories
    bars1 = []
    bars2 = []
    
    for i in range(len(labels)):
        if i <= 1:  # MMST-Top-10 HSQC ranked and Greedy - center the bars
            bar1 = ax.bar(x[i], mmst_values[i], width, color=mmst_colors[i], alpha=0.7, edgecolor='black')
            bar2 = ax.bar(x[i], 0, width, color=ref_color, alpha=0.7, edgecolor='black')  # Empty bar for reference
        else:  # Other categories - keep side by side
            bar1 = ax.bar(x[i] - width/2, mmst_values[i], width, color=mmst_colors[i], alpha=0.7, edgecolor='black')
            bar2 = ax.bar(x[i] + width/2, ref_values[i], width, color=ref_color, alpha=0.7, edgecolor='black')
        
        bars1.append(bar1)
        bars2.append(bar2)
    
    # --- UPDATED SECTION ---
    # Add value labels on bars
    for i, (bar_container1, bar_container2) in enumerate(zip(bars1, bars2)):
        # MMST bars - percentage above, count inside
        bar_patch1 = bar_container1[0]
        height1 = bar_patch1.get_height()
        
        if height1 > 0:
            # Percentage above bar (font size increased to 20)
            ax.text(bar_patch1.get_x() + bar_patch1.get_width()/2, height1 + 1,
                    f'{mmst_values[i]:.1f}%',
                    ha='center', va='bottom', fontsize=20)
            # Count inside bar (formatted with comma)
            ax.text(bar_patch1.get_x() + bar_patch1.get_width()/2, height1/2,
                    f'{mmst_display_counts[i]:,}',
                    ha='center', va='center', fontsize=20, color='black', rotation=90)
        
        # Reference bars - percentage above, count inside if available
        bar_patch2 = bar_container2[0]
        height2 = bar_patch2.get_height()
        
        if height2 > 0:
            # Percentage above bar (font size increased to 20)
            ax.text(bar_patch2.get_x() + bar_patch2.get_width()/2, height2 + 1,
                    f'{ref_values[i]:.1f}%',
                    ha='center', va='bottom', fontsize=20)
            # Count inside bar (formatted with comma)
            ax.text(bar_patch2.get_x() + bar_patch2.get_width()/2, height2/2,
                    f'{ref_counts[i]:,}',
                    ha='center', va='center', fontsize=20, color='black', rotation=90)
    
    # Customize the plot
    ax.set_ylabel('Accuracy (%)', fontsize=22)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=20)
    ax.tick_params(axis='y', labelsize=20)
    ax.set_ylim(0, max(max(mmst_values), max(ref_values)) * 1.25) # Increased upper limit slightly for new font size
    
    # Add legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor=hsqc_color, label='MMST-Top-10 HSQC ranked', alpha=0.7, edgecolor='black'),
        Patch(facecolor=mmst_color, label='MMST Greedy/MNS', alpha=0.7, edgecolor='black'),
        Patch(facecolor=ref_color, label='IR + MF (Alberts et al.)', alpha=0.7, edgecolor='black')
    ]
    # Legend fontsize increased to 20
    ax.legend(handles=legend_elements, fontsize=20, loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=3)
    
    # Add grid
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    
    plt.tight_layout(rect=[0, 0.05, 1, 1]) # Adjust layout to make space for the larger legend
    
    # Create results directory if it doesn't exist
    os.makedirs(save_path, exist_ok=True)
    
    # Save the figure
    plt.savefig(os.path.join(save_path, 'combined_mmst_vs_ir_mf_comparison.png'), dpi=300, bbox_inches='tight')
    plt.savefig(os.path.join(save_path, 'combined_mmst_vs_ir_mf_comparison.pdf'), bbox_inches='tight')
    plt.show()
    
    print(f"\nPlot saved to: {save_path}")
    
    return fig

def main():
    """Main function to run the complete analysis"""
    
    # Set the path to your CSV file
    csv_path = "/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/___FIGURES_PAPERS/Figures_Paper_2/precomputed_raw_data/20250612_IBM_comparison/Experiment_ranking/data_HSQC_245260.csv"
    
    # Set the path to the IBM results directory
    ibm_results_dir = "/projects/cc/se_users/knlr326/1_NMR_project/2_Notebooks/MultiModalSpectralTransformer_cleaned/IBM_experiment_results"
    
    # Set the save path
    save_path = "/projects/cc/se_users/knlr326/1_NMR_project/2_Notebooks/MultiModalSpectralTransformer_cleaned/_FIGURES"
    
    # Create the combined plot
    fig = plot_combined_comparison(csv_path, ibm_results_dir, save_path)
    
    return fig

if __name__ == "__main__":
    try:
        fig = main()
    except FileNotFoundError as e:
        print(f"Error: A required file or directory was not found.")
        print(f"Details: {e}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

#### Dataset Comparison ZINC vs Pubchem (IR)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors
from rdkit.Chem.Fingerprints import FingerprintMols
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import pairwise_distances
import os
import warnings
warnings.filterwarnings('ignore')

def load_smiles_from_csv(file_path, smiles_column='SMILES', sample_size=None):
    """Load SMILES from CSV file with optional sampling"""
    print(f"Loading data from {file_path}")
    df = pd.read_csv(file_path)
    
    if sample_size and len(df) > sample_size:
        df = df.sample(n=sample_size, random_state=42)
        print(f"Sampled {sample_size} compounds from {len(pd.read_csv(file_path))} total")
    
    smiles_list = df[smiles_column].dropna().tolist()
    print(f"Loaded {len(smiles_list)} valid SMILES")
    return smiles_list

def calculate_molecular_weights(smiles_list):
    """Calculate molecular weights from SMILES"""
    weights = []
    valid_smiles = []
    
    for smi in smiles_list:
        try:
            mol = Chem.MolFromSmiles(smi)
            if mol is not None:
                mw = Descriptors.MolWt(mol)
                weights.append(mw)
                valid_smiles.append(smi)
        except:
            continue
    
    print(f"Calculated molecular weights for {len(weights)} compounds")
    return weights, valid_smiles

def calculate_tanimoto_fingerprints(smiles_list):
    """Calculate Morgan fingerprints for Tanimoto similarity"""
    fingerprints = []
    valid_smiles = []
    
    for smi in smiles_list:
        try:
            mol = Chem.MolFromSmiles(smi)
            if mol is not None:
                # Use Morgan fingerprints (ECFP4)
                fp = rdMolDescriptors.GetMorganFingerprintAsBitVect(mol, 2, nBits=1024)
                fingerprints.append(np.array(fp))
                valid_smiles.append(smi)
        except:
            continue
    
    if fingerprints:
        fingerprints = np.array(fingerprints)
        print(f"Calculated fingerprints for {len(fingerprints)} compounds")
        return fingerprints, valid_smiles
    else:
        return None, []

def plot_molecular_weight_distribution(zinc_weights, pubchem_weights, save_path):
    """Create histogram comparing molecular weight distributions"""
    plt.figure(figsize=(10, 8))
    
    # Filter weights to the desired range
    zinc_weights_filtered = [w for w in zinc_weights if 70 <= w <= 370]
    pubchem_weights_filtered = [w for w in pubchem_weights if 70 <= w <= 370]
    
    # Create histogram
    bins = np.linspace(70, 370, 40)
    
    plt.hist(zinc_weights_filtered, bins=bins, alpha=0.7, label='ZINC', 
             color='steelblue', density=False)
    plt.hist(pubchem_weights_filtered, bins=bins, alpha=0.7, label='PubChem (IR)', 
             color='coral', density=False)
    
    plt.xlabel('Molecular Weight (Da)', fontsize=22)
    plt.ylabel('Count', fontsize=22)
    plt.legend(fontsize=22)
    plt.grid(True, alpha=0.3)
    plt.xlim(70, 370)
    
    # Add statistics
    zinc_mean = np.mean(zinc_weights_filtered)
    pubchem_mean = np.mean(pubchem_weights_filtered)
    
    plt.axvline(zinc_mean, color='steelblue', linestyle='--', alpha=0.8, 
                label=f'ZINC mean: {zinc_mean:.1f} Da')
    plt.axvline(pubchem_mean, color='coral', linestyle='--', alpha=0.8, 
                label=f'PubChem (IR) mean: {pubchem_mean:.1f} Da')
    
    plt.legend(fontsize=22)
    plt.xticks(fontsize=20)
    plt.yticks(fontsize=20)
    plt.tight_layout()
    
    # Save plot
    plt.savefig(os.path.join(save_path, 'molecular_weight_distribution_comparison.png'), 
                dpi=300, bbox_inches='tight')
    plt.savefig(os.path.join(save_path, 'molecular_weight_distribution_comparison.pdf'), 
                bbox_inches='tight')
    plt.show()
    
    print(f"Molecular weight statistics (70-370 Da range):")
    print(f"ZINC: Mean = {zinc_mean:.1f} Da, Std = {np.std(zinc_weights_filtered):.1f} Da")
    print(f"PubChem (IR): Mean = {pubchem_mean:.1f} Da, Std = {np.std(pubchem_weights_filtered):.1f} Da")

def plot_tsne_comparison(zinc_fp, pubchem_fp, save_path):
    """Create t-SNE plot comparing chemical spaces"""
    print("Preparing data for t-SNE...")
    
    # Combine fingerprints
    combined_fp = np.vstack((zinc_fp, pubchem_fp))
    labels = np.array(['ZINC'] * len(zinc_fp) + ['PubChem (IR)'] * len(pubchem_fp))
    
    print("Computing t-SNE embedding...")
    # Calculate Tanimoto distance matrix
    tanimoto_distances = pairwise_distances(combined_fp, metric='jaccard')
    
    # Perform t-SNE
    tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000, 
                metric='precomputed', init='random')
    tsne_result = tsne.fit_transform(tanimoto_distances)
    
    # Create the plot
    plt.figure(figsize=(10, 8))
    
    # Plot ZINC points
    zinc_mask = labels == 'ZINC'
    plt.scatter(tsne_result[zinc_mask, 0], tsne_result[zinc_mask, 1], 
                c='steelblue', alpha=0.6, s=20, label=f'ZINC (n={np.sum(zinc_mask)})')
    
    # Plot PubChem points
    pubchem_mask = labels == 'PubChem (IR)'
    plt.scatter(tsne_result[pubchem_mask, 0], tsne_result[pubchem_mask, 1], 
                c='coral', alpha=0.7, s=20, label=f'PubChem (IR) (n={np.sum(pubchem_mask)})')
    
    plt.xlabel('t-SNE Dimension 1', fontsize=22)
    plt.ylabel('t-SNE Dimension 2', fontsize=22)
    plt.legend(fontsize=22)
    plt.grid(True, alpha=0.3)
    plt.xticks(fontsize=20)
    plt.yticks(fontsize=20)
    
    plt.tight_layout()
    
    # Save plot
    plt.savefig(os.path.join(save_path, 'tsne_chemical_space_comparison.png'), 
                dpi=300, bbox_inches='tight')
    plt.savefig(os.path.join(save_path, 'tsne_chemical_space_comparison.pdf'), 
                bbox_inches='tight')
    plt.show()

def main():
    # Set random seed for reproducibility
    np.random.seed(42)
    
    # File paths
    zinc_path = "/projects/cc/se_users/knlr326/1_NMR_project/2_Notebooks/MultiModalSpectralTransformer_cleaned/data/ZINK_dataset/ML_NMR_5M_XL_1H_comb_test_V8.csv"
    pubchem_path = "/projects/cc/se_users/knlr326/1_NMR_project/2_Notebooks/MultiModalSpectralTransformer_cleaned/data/IBM_dataset/IBM_data_1H_837154_train.csv"
    save_folder = "/projects/cc/se_users/knlr326/1_NMR_project/2_Notebooks/MultiModalSpectralTransformer_cleaned/_FIGURES"
    
    # Create save folder if it doesn't exist
    os.makedirs(save_folder, exist_ok=True)
    
    # Load SMILES data (sample for computational efficiency)
    print("Loading ZINC dataset...")
    zinc_smiles = load_smiles_from_csv(zinc_path, sample_size=5000)
    
    print("Loading PubChem (IBM) dataset...")
    pubchem_smiles = load_smiles_from_csv(pubchem_path, sample_size=5000)
    
    # Calculate molecular weights
    print("\nCalculating molecular weights...")
    zinc_weights, zinc_smiles_valid = calculate_molecular_weights(zinc_smiles)
    pubchem_weights, pubchem_smiles_valid = calculate_molecular_weights(pubchem_smiles)
    
    # Plot molecular weight distribution
    print("\nGenerating molecular weight distribution plot...")
    plot_molecular_weight_distribution(zinc_weights, pubchem_weights, save_folder)
    
    # Calculate fingerprints for t-SNE (use smaller sample for computational efficiency)
    print("\nCalculating fingerprints for t-SNE...")
    zinc_fp, _ = calculate_tanimoto_fingerprints(zinc_smiles_valid[:1000])
    pubchem_fp, _ = calculate_tanimoto_fingerprints(pubchem_smiles_valid[:1000])
    
    if zinc_fp is not None and pubchem_fp is not None:
        # Plot t-SNE comparison
        print("\nGenerating t-SNE chemical space comparison...")
        plot_tsne_comparison(zinc_fp, pubchem_fp, save_folder)
    else:
        print("Error: Could not calculate fingerprints for t-SNE analysis")
    
    print(f"\nAll plots saved to: {save_folder}")
    print("Analysis complete!")

if __name__ == "__main__":
    main()

### 2.0 Ablation Study fine-tuned

#### Run Experiment

In [ ]:
'''config.multinom_runs = 1
config.training_mode = "13C_HSQC_COSY_IR_MF_MW"
config.data_size = 489993

# No 1H
config.checkpoint_path = os.path.abspath("./models/mmst/experiment_models/ablation_models/2_0_V8i_MMTi_no_1H-epoch=00-loss=0.03.ckpt")
model_MMT = mrtf.load_MMT_model(config)
val_dataloader_multi = mrtf.load_data(config, stoi, stoi_MF, single=False, mode="val")
prob_dict_results_2_1a, results_dict_2_1a = mrtf.run_model_analysis(config, model_MMT, val_dataloader_multi, stoi, itos)



# Save the data to a file
file_prob_dict_path = os.path.abspath('./past_experiments/ChemXriv/2.0_Experiment_Ablation_Study/2.1_prob_dict_results_2_1a.pkl') 
with open(file_prob_dict_path, 'wb') as file:
    pickle.dump(prob_dict_results_2_1a, file)

# Save the data to a file
file_results_dict_path = os.path.abspath('./past_experiments/ChemXriv/2.0_Experiment_Ablation_Study/2.1_results_dict_2_1a.pkl') 
with open(file_results_dict_path, 'wb') as file:
    pickle.dump(results_dict_2_1a, file)

In [ ]:
'''config.multinom_runs = 1
config.training_mode = "1H_HSQC_COSY_IR_MF_MW"
config.data_size = 489993

# No 13C
config.checkpoint_path = os.path.abspath("./models/mmst/experiment_models/ablation_models/2_0_V8i_MMTi_no_13C-epoch=00-loss=0.03.ckpt")
model_MMT = mrtf.load_MMT_model(config)
val_dataloader_multi = mrtf.load_data(config, stoi, stoi_MF, single=False, mode="val")
prob_dict_results_2_1b, results_dict_2_1b = mrtf.run_model_analysis(config, model_MMT, val_dataloader_multi, stoi, itos)


# Save the data to a file
file_prob_dict_path = os.path.abspath('./past_experiments/ChemXriv/2.0_Experiment_Ablation_Study/2.1_prob_dict_results_2_1b.pkl')  
with open(file_prob_dict_path, 'wb') as file:
    pickle.dump(prob_dict_results_2_1b, file)

# Save the data to a file
file_results_dict_path = os.path.abspath('./past_experiments/ChemXriv/2.0_Experiment_Ablation_Study/2.1_results_dict_2_1b.pkl')  
with open(file_results_dict_path, 'wb') as file:
    pickle.dump(results_dict_2_1b, file)

In [ ]:
'''config.multinom_runs = 1
config.training_mode = "1H_13C_COSY_IR_MF_MW"
config.data_size = 489993

# No HSQC
config.checkpoint_path = os.path.abspath("./models/mmst/experiment_models/ablation_models/2_0_V8i_MMTi_no_HSQC-epoch=00-loss=0.03.ckpt")
model_MMT = mrtf.load_MMT_model(config)
val_dataloader_multi = mrtf.load_data(config, stoi, stoi_MF, single=False, mode="val")
prob_dict_results_2_1c, results_dict_2_1c = mrtf.run_model_analysis(config, model_MMT, val_dataloader_multi, stoi, itos)



# Save the data to a file
file_prob_dict_path =   os.path.abspath('./past_experiments/ChemXriv/2.0_Experiment_Ablation_Study/2.1_prob_dict_results_2_1c.pkl') 
with open(file_prob_dict_path, 'wb') as file:
    pickle.dump(prob_dict_results_2_1c, file)

# Save the data to a file
file_results_dict_path = os.path.abspath('./past_experiments/ChemXriv/2.0_Experiment_Ablation_Study/2.1_results_dict_2_1c.pkl') 
with open(file_results_dict_path, 'wb') as file:
    pickle.dump(results_dict_2_1c, file)

In [ ]:
'''config.multinom_runs = 1
config.training_mode = "1H_13C_HSQC_IR_MF_MW"
config.data_size = 489993

# No COSY
config.checkpoint_path = os.path.abspath("./models/mmst/experiment_models/ablation_models/2_0_V8i_MMTi_no_COSY-epoch=00-loss=0.03.ckpt")
model_MMT = mrtf.load_MMT_model(config)
val_dataloader_multi = mrtf.load_data(config, stoi, stoi_MF, single=False, mode="val")
prob_dict_results_2_1d, results_dict_2_1d = mrtf.run_model_analysis(config, model_MMT, val_dataloader_multi, stoi, itos)


# Save the data to a file
file_prob_dict_path = os.path.abspath('./past_experiments/ChemXriv/2.0_Experiment_Ablation_Study/2.1_prob_dict_results_2_1d.pkl') 
with open(file_prob_dict_path, 'wb') as file:
    pickle.dump(prob_dict_results_2_1d, file)

# Save the data to a file
file_results_dict_path = os.path.abspath('./past_experiments/ChemXriv/2.0_Experiment_Ablation_Study/2.1_results_dict_2_1d.pkl') 
with open(file_results_dict_path, 'wb') as file:
    pickle.dump(results_dict_2_1d, file)

In [ ]:
'''config.multinom_runs = 1
config.training_mode = "1H_13C_HSQC_COSY_MF_MW"
config.data_size = 489993

# No IR
config.checkpoint_path = os.path.abspath("./models/mmst/experiment_models/ablation_models/2_0_V8i_MMTi_no_IR-epoch=00-loss=0.02.ckpt")
model_MMT = mrtf.load_MMT_model(config)
val_dataloader_multi = mrtf.load_data(config, stoi, stoi_MF, single=False, mode="val")
prob_dict_results_2_1e, results_dict_2_1e = mrtf.run_model_analysis(config, model_MMT, val_dataloader_multi, stoi, itos)


# Save the data to a file
file_prob_dict_path =   os.path.abspath('./past_experiments/ChemXriv/2.0_Experiment_Ablation_Study/2.1_prob_dict_results_2_1e.pkl') 
with open(file_prob_dict_path, 'wb') as file:
    pickle.dump(prob_dict_results_2_1e, file)

# Save the data to a file
file_prob_dict_path = os.path.abspath('./past_experiments/ChemXriv/2.0_Experiment_Ablation_Study/2.1_results_dict_2_1e.pkl') 
with open(file_results_dict_path, 'wb') as file:
    pickle.dump(results_dict_2_1e, file)

In [ ]:
"""# All small Data
config.csv_1H_path_SGNN = os.path.abspath('./data/ZINK_dataset/ML_NMR_5M_XL_1H_comb_test_V8.csv') 
config.csv_13C_path_SGNN = os.path.abspath('./data/ZINK_dataset/ML_NMR_5M_XL_13C_test.csv')    
config.csv_HSQC_path_SGNN = os.path.abspath('./data/ZINK_dataset/ML_NMR_5M_XL_HSQC_test.csv') 
config.csv_COSY_path_SGNN = os.path.abspath('./data/ZINK_dataset/ML_NMR_5M_XL_13C_test.csv') 
config.csv_path_val = os.path.abspath('./data/ZINK_dataset/ML_NMR_5M_XL_1H_comb_test_V8.csv') 
config.pickle_file_path = ""
config.IR_data_folder = os.path.abspath('./data/ZINK_dataset/IR_spectra_NN') 
config.data_size = 48999 

config.checkpoint_path = os.path.abspath("./models/mmst/base_models/1_0_V8i_MMTi_RAW_DROP_Loss_0.112.ckpt")
config.training_mode = "1H_13C_HSQC_COSY_IR_MF_MW" # it ignors IR because no folder provided and puts zeros in for IR
config.multinom_runs = 1
config.batch_size = 1024 #int(1000*data_fraction)

val_dataloader_multi = mrtf.load_data(config, stoi, stoi_MF, single=False, mode="val")
"""

In [ ]:
"""
# all small
config.checkpoint_path = os.path.abspath("./models/mmst/base_models/1_0_V8i_MMTi_RAW_DROP_Loss_0.112.ckpt")
model_MMT = mrtf.load_MMT_model(config)
val_dataloader_multi = mrtf.load_data(config, stoi, stoi_MF, single=False, mode="val")
prob_dict_results_2_1e, results_dict_2_1e = mrtf.run_model_analysis(config, model_MMT, val_dataloader_multi, stoi, itos)


# Save the data to a file
file_prob_dict_path =   os.path.abspath('./past_experiments/ChemXriv/2.0_Experiment_Ablation_Study/2.1_prob_dict_results_2_all_small.pkl') 
with open(file_prob_dict_path, 'wb') as file:
    pickle.dump(prob_dict_results_2_1e, file)

# Save the data to a file
file_results_dict_path = os.path.abspath('./past_experiments/ChemXriv/2.0_Experiment_Ablation_Study/2.1_results_dict_2_all_small.pkl') 
with open(file_results_dict_path, 'wb') as file:
    pickle.dump(results_dict_2_1e, file)"""

In [ ]:
from utils_MMT.data_generation_v15_4 import main_run_data_generation

# Set the required paths
config.SGNN_csv_gen_smi = '/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/15_ZINC270M/ML_NMR_5M_XL_1H_comb_test_V8_4000.csv'
config.SGNN_gen_folder_path = "/projects/cc/se_users/knlr326/1_NMR_project/2_Notebooks/MultiModalSpectralTransformer_cleaned/data/ZINC_4000/NMR"

# Run the SGNN simulation
combined_df, data_1H, data_13C, data_COSY, data_HSQC, csv_1H_path, csv_13C_path, csv_COSY_path, csv_HSQC_path = main_run_data_generation(config)


In [ ]:
config.SGNN_csv_gen_smi = '/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/15_ZINC270M/ML_NMR_5M_XL_1H_comb_test_V8_4000.csv'


In [ ]:
import os
import pickle

config.multinom_runs = 1
config.training_mode = "1H_13C_HSQC_COSY_IR_MF_MW"
config.data_size = 4000

# Updated paths for ZINC_4000 dataset
config.csv_1H_path_SGNN = os.path.abspath('./data/ZINC_4000/data_1H_4953177.csv') 
config.csv_13C_path_SGNN = os.path.abspath('./data/ZINC_4000/data_13C_4953177.csv')    
config.csv_HSQC_path_SGNN = os.path.abspath('./data/ZINC_4000/data_HSQC_4953177.csv') 
config.csv_COSY_path_SGNN = os.path.abspath('./data/ZINC_4000/data_COSY_4953177.csv') 
config.csv_path_val = os.path.abspath('./data/ZINC_4000/data_1H_4953177.csv') 
config.pickle_file_path = ""
config.IR_data_folder = os.path.abspath('./data/ZINK_dataset/IR_spectra_NN') 
   
# all tiny
config.checkpoint_path = os.path.abspath("./models/mmst/base_models/1_0_V8i_MMTi_RAW_DROP_Loss_0.112.ckpt")
model_MMT = mrtf.load_MMT_model(config)
val_dataloader_multi = mrtf.load_data(config, stoi, stoi_MF, single=False, mode="val")
prob_dict_results_2_1e, results_dict_2_1e = mrtf.run_model_analysis(config, model_MMT, val_dataloader_multi, stoi, itos)

# Save the data to a file
file_prob_dict_path = os.path.abspath('./past_experiments/ChemXriv/2.0_Experiment_Ablation_Study/2.1_prob_dict_results_2_ZINC_4000.pkl') 
with open(file_prob_dict_path, 'wb') as file:
   pickle.dump(prob_dict_results_2_1e, file)

# Save the data to a file
file_results_dict_path = os.path.abspath('./past_experiments/ChemXriv/2.0_Experiment_Ablation_Study/2.1_results_dict_2_ZINC_4000.pkl') 
with open(file_results_dict_path, 'wb') as file:
   pickle.dump(results_dict_2_1e, file)

In [ ]:
"""# without MF_MW
# need to use a different model_MMT.py  !!!
# Rename this one: utils_MMT/models_MMT_v15_4_MW_MF.py -> models_MMT_v15_4 
# then run the experiment (reason - manually zeroing the MW and MF inputs
config.multinom_runs = 1
config.training_mode = "1H_13C_HSQC_COSY_IR_MF_MW"
config.data_size = 489993

# all small
config.checkpoint_path = os.path.abspath("/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/1_old_models/models_v2/MMST_MW&MF_ablation/MultimodalTransformer_time_1749700099.1328886_Loss_0.218.ckpt")
model_MMT = mrtf.load_MMT_model(config)
val_dataloader_multi = mrtf.load_data(config, stoi, stoi_MF, single=False, mode="val")
prob_dict_results_2_1e, results_dict_2_1e = mrtf.run_model_analysis(config, model_MMT, val_dataloader_multi, stoi, itos)

# Save the data to a file
file_prob_dict_path =   os.path.abspath('./past_experiments/ChemXriv/2.0_Experiment_Ablation_Study/2.1_prob_dict_results_2_wo_MW_MF_small.pkl') 
with open(file_prob_dict_path, 'wb') as file:
    pickle.dump(prob_dict_results_2_1e, file)

# Save the data to a file
file_prob_dict_path = os.path.abspath('./past_experiments/ChemXriv/2.0_Experiment_Ablation_Study/2.1_results_dict_2_wo_MW_MF_small.pkl') 
with open(file_results_dict_path, 'wb') as file:
    pickle.dump(results_dict_2_1e, file)"""

#### Load data

#### Probability of Correct SMILES - Violin Chart

In [ ]:
import pickle
import os
import gc  # For garbage collection
import numpy as np
import matplotlib.pyplot as plt

# List of file paths for prob_dict_results (not results_dict)
prob_dict_file_paths = [
    './past_experiments/ChemXriv/2.0_Experiment_Ablation_Study/2.0_prob_dict_results_2ai.pkl',
    './past_experiments/ChemXriv/2.0_Experiment_Ablation_Study/2.1_prob_dict_results_2_1a.pkl',
    './past_experiments/ChemXriv/2.0_Experiment_Ablation_Study/2.1_prob_dict_results_2_1b.pkl',
    './past_experiments/ChemXriv/2.0_Experiment_Ablation_Study/2.1_prob_dict_results_2_1c.pkl',
    './past_experiments/ChemXriv/2.0_Experiment_Ablation_Study/2.1_prob_dict_results_2_1d.pkl',
    './past_experiments/ChemXriv/2.0_Experiment_Ablation_Study/2.1_prob_dict_results_2_1e.pkl',
    './past_experiments/ChemXriv/2.0_Experiment_Ablation_Study/2.1_prob_dict_results_2_ZINC_4000.pkl'
]

labels = ['All', 'no $^1$H', 'no $^{13}$C', 
          'no HSQC', 'no COSY', 'no IR',
          'All (1% data)']

# Initialize lists to store the extracted data
data_for_violin = []
filtered_labels = []

print("Loading prob_dict files one by one to extract aggregated_corr_prob_multi data...")

# Load each file individually, extract data, then release memory
for i, (file_path, label) in enumerate(zip(prob_dict_file_paths, labels)):
    print(f"Loading file {i+1}/{len(prob_dict_file_paths)}: {os.path.basename(file_path)} ({label})")
    
    try:
        # Load the file
        full_path = os.path.abspath(file_path)
        with open(full_path, 'rb') as file:
            prob_dict = pickle.load(file)
        
        # Extract the aggregated_corr_prob_multi data
        prob_data = prob_dict.get("aggregated_corr_prob_multi", [])
        
        # Only add if data exists
        if len(prob_data) > 0:
            data_for_violin.append(prob_data)
            filtered_labels.append(label)
            print(f"  - Probability data points: {len(prob_data)}")
            print(f"  - Mean probability: {np.mean(prob_data):.3f}")
        else:
            print(f"  - No aggregated_corr_prob_multi data found for {label}")
        
        # Clear the variable and force garbage collection
        del prob_dict
        gc.collect()
        
    except FileNotFoundError:
        print(f"  - File not found: {file_path}")
    except Exception as e:
        print(f"  - Error loading {file_path}: {e}")

print(f"\nProcessed {len(data_for_violin)} datasets with probability data.")

# Check if we have any data
if not data_for_violin:
    print("Error: All datasets are empty. Unable to create violin plot.")
else:
    # Create the violin plot
    fig, ax = plt.subplots(figsize=(12, 10))
    parts = ax.violinplot(data_for_violin, showmeans=True, showmedians=False, showextrema=False)
    
    # Customizing colors - different colors for the new datasets
    colors = []
    for label in filtered_labels:
        if '1%' in label:
            colors.append("#FFA07A")  # Orange for small
        else:
            colors.append("#8CB0FE")  # Blue for original ablation study
    
    for i, pc in enumerate(parts['bodies']):
        pc.set_facecolor(colors[i])
        pc.set_edgecolor('black')
        pc.set_alpha(0.7)
        
    mean_values = [np.mean(data) if data is not None else 0 for data in data_for_violin]
    
    # Add mean values as text
    for pos, mean_value in zip(np.arange(1, len(mean_values) + 1), mean_values):
        ax.text(pos, mean_value, f'{mean_value:.2f}', ha='center', va='bottom', fontsize=20)
    
    # Customizing the axes and labels
    ax.set_ylabel('Probability of Correct SMILES', fontsize=22)
    ax.set_xticks(np.arange(1, len(filtered_labels) + 1))
    ax.set_xticklabels(filtered_labels, rotation=45, ha='center', fontsize=20)
    ax.tick_params(axis='both', which='major', labelsize=20)
    
    # Add grid and set the limits
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    ax.set_ylim(0, 1)
    
    # Add vertical line to separate ablation study from dataset size comparison if we have both types
    original_count = sum(1 for label in filtered_labels if 'Small' not in label and 'Tiny' not in label)
    if original_count < len(filtered_labels):
        ax.axvline(x=original_count + 0.5, color='gray', linestyle='--', alpha=0.5, linewidth=2)
        
        # Add text annotations to clarify sections
        if original_count > 1:
            ax.text(original_count/2, 0.95, 'Ablation Study', ha='center', fontsize=16, style='italic', 
                    bbox=dict(boxstyle="round,pad=0.3", facecolor="lightblue", alpha=0.7))
        if len(filtered_labels) > original_count:
            remaining_pos = original_count + (len(filtered_labels) - original_count)/2
            ax.text(remaining_pos, 0.95, 'Dataset Size', ha='center', fontsize=16, style='italic',
                    bbox=dict(boxstyle="round,pad=0.3", facecolor="lightcoral", alpha=0.7))
    for spine in ax.spines.values():
        spine.set_edgecolor('gray')
        spine.set_alpha(0.8) # Make it visible but not dominant

    plt.tight_layout()
    save_path = os.path.abspath('./_FIGURES/2.1_Ablation_Violin_chart_with_size_comparison_v3.png')
    plt.savefig(save_path, format='png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"\nPlot saved to: {save_path}")

# Print the data for debugging
print("\nDebugging information:")
print(f"Number of datasets with data: {len(data_for_violin)}")
print("Filtered labels:", filtered_labels)
print("Data lengths:", [len(data) for data in data_for_violin])

print("Memory efficient probability violin plotting completed!")

#### Invalid Molecules - Bar Chart

In [ ]:
import pickle
import os
import gc  # For garbage collection
import matplotlib.pyplot as plt

# List of file paths and labels
file_paths = [
    './past_experiments/ChemXriv/2.0_Experiment_Ablation_Study/2.1_results_dict_2ai.pkl',
    './past_experiments/ChemXriv/2.0_Experiment_Ablation_Study/2.1_results_dict_2_1a.pkl',
    './past_experiments/ChemXriv/2.0_Experiment_Ablation_Study/2.1_results_dict_2_1b.pkl',
    './past_experiments/ChemXriv/2.0_Experiment_Ablation_Study/2.1_results_dict_2_1c.pkl',
    './past_experiments/ChemXriv/2.0_Experiment_Ablation_Study/2.1_results_dict_2_1d.pkl',
    './past_experiments/ChemXriv/2.0_Experiment_Ablation_Study/2.1_results_dict_2_1e.pkl',
    './past_experiments/ChemXriv/2.0_Experiment_Ablation_Study/2.1_results_dict_2_ZINC_4000.pkl'
]

labels = ['All', 'no $^1$H', 'no $^{13}$C', 
          'no HSQC', 'no COSY', 'no IR',
          'All (1% data)']

# Initialize lists to store the extracted data
mean_results_2 = []
test_data_sizes = []

print("Loading files one by one to extract failed results...")

# Load each file individually, extract data, then release memory
for i, file_path in enumerate(file_paths):
    print(f"Loading file {i+1}/{len(file_paths)}: {os.path.basename(file_path)}")
    
    # Load the file
    full_path = os.path.abspath(file_path)
    with open(full_path, 'rb') as file:
        results_dict = pickle.load(file)
    
    # Extract only the needed data
    num_failed = len(results_dict["failed"])
    data_size = len(results_dict["trg_conv_SMI_list"])  # or another size indicator
    
    # Store the extracted data
    mean_results_2.append(num_failed)
    test_data_sizes.append(data_size)
    
    # Clear the variable and force garbage collection
    del results_dict
    gc.collect()
    
    print(f"  - Failed results: {num_failed}")
    print(f"  - Data size: {data_size}")

print("\nAll files processed. Creating plot...")

# Calculate percentages for each dataset
percentages = [(num_failed / data_size) * 100 for num_failed, data_size in zip(mean_results_2, test_data_sizes)]

# Define colors for the bars - different colors for different dataset sizes
colors = ["#8CB0FE", "#8CB0FE", "#8CB0FE", "#8CB0FE", "#8CB0FE", "#8CB0FE", "#FFA07A", "#98FB98"]

# Plotting the bar chart with percentages
fig, ax = plt.subplots(figsize=(12, 10))
bars = ax.bar(range(len(percentages)), percentages, align='center', alpha=0.7, 
              ecolor='black', capsize=10, edgecolor='black', color=colors)

# Adding absolute number labels inside each bar (vertical, black, non-bold)
for i, bar in enumerate(bars):
    yval = bar.get_height()
    absolute_num = mean_results_2[i]
    ax.text(bar.get_x() + bar.get_width() / 2, yval / 2, f'{absolute_num}', 
            rotation=90, ha='center', va='center', fontsize=20, color='black')

# Adding percentage labels on top of each bar
for i, bar in enumerate(bars):
    yval = bar.get_height()
    percentage = percentages[i]
    ax.text(bar.get_x() + bar.get_width() / 2, yval + max(percentages) * 0.02, f'{percentage:.1f}%', 
            ha='center', va='bottom', fontsize=20, color='black')

# Set the title and labels
ax.set_ylabel('Percentage of Invalid Molecules (%)', fontsize=22)

# Set x-ticks
ax.set_xticks(range(len(percentages)))
ax.set_xticklabels(labels, rotation=45, ha='center', fontsize=20)
ax.tick_params(axis='both', which='major', labelsize=20)

# Adding grid lines for better readability
ax.grid(axis='y', linestyle='--', alpha=0.7)
ax.set_ylim(0, max(percentages) * 1.15)  # Add extra space for percentage labels on top

# Add a vertical line to separate original ablations from dataset size comparisons
ax.axvline(x=5.5, color='gray', linestyle='--', alpha=0.5, linewidth=2)

for spine in ax.spines.values():
    spine.set_edgecolor('gray')
    spine.set_alpha(0.8) # Make it visible but not dominant

# Adjust layout
plt.tight_layout()

# Save the plot
save_path = os.path.abspath('./_FIGURES/2.1_Ablation_Invalid_molecules_percentage_scale_v3.png')  
plt.savefig(save_path, format='png', dpi=300, bbox_inches='tight')

# Show plot
plt.show()

print(f"\nPlot saved to: {save_path}")

# Print summary
print("\nSummary Statistics:")
print("="*60)
for i, label in enumerate(labels):
    print(f"{label:15s}: {percentages[i]:.1f}% ({mean_results_2[i]:4d}/{test_data_sizes[i]:4d})")

print("\nPercentage-based plot completed!")

#### Tanimoto comparison - Violin Chart

In [ ]:
import pickle
import os
import gc  # For garbage collection
import numpy as np
import matplotlib.pyplot as plt

# List of file paths and labels
file_paths = [
    './past_experiments/ChemXriv/2.0_Experiment_Ablation_Study/2.1_results_dict_2ai.pkl',
    './past_experiments/ChemXriv/2.0_Experiment_Ablation_Study/2.1_results_dict_2_1a.pkl',
    './past_experiments/ChemXriv/2.0_Experiment_Ablation_Study/2.1_results_dict_2_1b.pkl',
    './past_experiments/ChemXriv/2.0_Experiment_Ablation_Study/2.1_results_dict_2_1c.pkl',
    './past_experiments/ChemXriv/2.0_Experiment_Ablation_Study/2.1_results_dict_2_1d.pkl',
    './past_experiments/ChemXriv/2.0_Experiment_Ablation_Study/2.1_results_dict_2_1e.pkl',
    './past_experiments/ChemXriv/2.0_Experiment_Ablation_Study/2.1_results_dict_2_ZINC_4000.pkl'
]

labels = ['All', 'no $^1$H', 'no $^{13}$C', 
          'no HSQC', 'no COSY', 'no IR',
          'All (1% data)']

# Initialize lists to store the extracted data
data_for_violin = []
filtered_labels = []

print("Loading files one by one to extract Tanimoto similarity data...")

# Load each file individually, extract data, then release memory
for i, (file_path, label) in enumerate(zip(file_paths, labels)):
    print(f"Loading file {i+1}/{len(file_paths)}: {os.path.basename(file_path)} ({label})")
    
    # Load the file
    full_path = os.path.abspath(file_path)
    with open(full_path, 'rb') as file:
        results_dict = pickle.load(file)
    
    # Extract the appropriate Tanimoto similarity data
    if label in ['All']:
        tanimoto_sim = results_dict.get("tanimoto_sim", [])
    elif label in ['All (1% data)']:
        # For small and tiny datasets, look for keys starting with "tanimoto_scores_"
        tanimoto_sim = []
        for key in results_dict.keys():
            if key.startswith("tanimoto_scores_"):
                tanimoto_sim = results_dict[key]
                print(f"  - Found Tanimoto data in key: {key}")
                break
        if not tanimoto_sim:
            print(f"  - No 'tanimoto_scores_*' key found, checking 'tanimoto_sim'")
            tanimoto_sim = results_dict.get("tanimoto_sim", [])
    else:
        tanimoto_sim = results_dict.get("tanimoto_scores_all", [])
    
    # Only add if data exists
    if len(tanimoto_sim) > 0:
        data_for_violin.append(tanimoto_sim)
        filtered_labels.append(label)
        print(f"  - Tanimoto data points: {len(tanimoto_sim)}")
        print(f"  - Mean Tanimoto: {np.mean(tanimoto_sim):.3f}")
    else:
        print(f"  - No Tanimoto data found for {label}")
    
    # Clear the variable and force garbage collection
    del results_dict
    gc.collect()

print(f"\nProcessed {len(data_for_violin)} datasets with Tanimoto data.")

# Check if we have any data
if not data_for_violin:
    print("Error: All datasets are empty. Unable to create violin plot.")
else:
    # Create the violin plot
    fig, ax = plt.subplots(figsize=(12, 10))
    
    # Define positions for the violins
    positions = np.arange(1, len(filtered_labels) + 1)
    
    # Plot violins
    parts = ax.violinplot(data_for_violin, positions=positions, showmeans=True, showmedians=False, showextrema=False)
    
    # Customizing colors - different colors for different dataset sizes
    colors = []
    for label in filtered_labels:
        if '1%' in label:
            colors.append("#FFA07A")  # Orange for small
        else:
            colors.append("#8CB0FE")  # Blue for original ablation study
    
    for i, pc in enumerate(parts['bodies']):
        pc.set_facecolor(colors[i])
        pc.set_edgecolor('black')
        pc.set_alpha(0.8)
    
    # Customizing the axes and labels
    ax.set_ylabel('Tanimoto Similarity', fontsize=22)
    
    # Set x-ticks in the middle of each group
    ax.set_xticks(positions)
    ax.set_xticklabels(filtered_labels, rotation=45, ha='center', fontsize=20)
    ax.tick_params(axis='both', which='major', labelsize=20)
    
    # Add grid and set the limits
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    ax.set_ylim(0, 1)
    
    # Calculate and add mean values as text
    for pos, data in zip(positions, data_for_violin):
        mean_value = np.mean(data)
        ax.text(pos, mean_value, f'{mean_value:.2f}', ha='center', va='bottom', fontsize=20)
    
    # Add vertical line to separate ablation study from dataset size comparison if we have both types
    original_count = sum(1 for label in filtered_labels if 'Small' not in label and 'Tiny' not in label)
    if original_count < len(filtered_labels):
        ax.axvline(x=original_count + 0.5, color='gray', linestyle='--', alpha=0.5, linewidth=2)
        
        # Add text annotations to clarify sections
        if original_count > 1:
            ax.text(original_count/2, 0.95, 'Ablation Study', ha='center', fontsize=16, style='italic', 
                    bbox=dict(boxstyle="round,pad=0.3", facecolor="lightblue", alpha=0.7))
        if len(filtered_labels) > original_count:
            remaining_pos = original_count + (len(filtered_labels) - original_count)/2
            ax.text(remaining_pos, 0.95, 'Dataset Size', ha='center', fontsize=16, style='italic',
                    bbox=dict(boxstyle="round,pad=0.3", facecolor="lightcoral", alpha=0.7))
    for spine in ax.spines.values():
        spine.set_edgecolor('gray')
        spine.set_alpha(0.8) # Make it visible but not dominant
        
    plt.tight_layout()
    
    # Save the figure
    save_path = os.path.abspath('./_FIGURES/2.1_Ablation_Tanimoto_with_size_comparison_v3.png')
    plt.savefig(save_path, format='png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"\nPlot saved to: {save_path}")

# Print the data for debugging
print("\nDebugging information:")
print(f"Number of datasets with data: {len(data_for_violin)}")
print("Filtered labels:", filtered_labels)
print("Data lengths:", [len(data) for data in data_for_violin])

print("Memory efficient Tanimoto plotting completed!")

#### Correct molecules - Bar Chart

In [ ]:
import pickle
import os
import gc  # For garbage collection
import numpy as np
import matplotlib.pyplot as plt

# List of file paths and labels
file_paths = [
    './past_experiments/ChemXriv/2.0_Experiment_Ablation_Study/2.1_results_dict_2ai.pkl',
    './past_experiments/ChemXriv/2.0_Experiment_Ablation_Study/2.1_results_dict_2_1a.pkl',
    './past_experiments/ChemXriv/2.0_Experiment_Ablation_Study/2.1_results_dict_2_1b.pkl',
    './past_experiments/ChemXriv/2.0_Experiment_Ablation_Study/2.1_results_dict_2_1c.pkl',
    './past_experiments/ChemXriv/2.0_Experiment_Ablation_Study/2.1_results_dict_2_1d.pkl',
    './past_experiments/ChemXriv/2.0_Experiment_Ablation_Study/2.1_results_dict_2_1e.pkl',
    './past_experiments/ChemXriv/2.0_Experiment_Ablation_Study/2.1_results_dict_2_ZINC_4000.pkl'
]

labels = ['All', 'no $^1$H', 'no $^{13}$C', 
          'no HSQC', 'no COSY', 'no IR',
           'All (1% data)']

# Initialize lists to store the extracted data
correct_counts = []
total_counts = []
filtered_labels = []

print("Loading files one by one to count correct molecules (Tanimoto = 1.0)...")

# Load each file individually, extract data, then release memory
for i, (file_path, label) in enumerate(zip(file_paths, labels)):
    print(f"Loading file {i+1}/{len(file_paths)}: {os.path.basename(file_path)} ({label})")
    
    try:
        # Load the file
        full_path = os.path.abspath(file_path)
        with open(full_path, 'rb') as file:
            results_dict = pickle.load(file)
        
        # Extract the appropriate Tanimoto similarity data
        if label in ['All']:
            tanimoto_sim = results_dict.get("tanimoto_sim", [])
        elif label in ['All (1% data)']:
            # For small and tiny datasets, look for keys starting with "tanimoto_scores_"
            tanimoto_sim = []
            for key in results_dict.keys():
                if key.startswith("tanimoto_scores_"):
                    tanimoto_sim = results_dict[key]
                    print(f"  - Found Tanimoto data in key: {key}")
                    break
            if not tanimoto_sim:
                print(f"  - No 'tanimoto_scores_*' key found, checking 'tanimoto_sim'")
                tanimoto_sim = results_dict.get("tanimoto_sim", [])
        else:
            tanimoto_sim = results_dict.get("tanimoto_scores_all", [])
        
        # Only process if data exists
        if len(tanimoto_sim) > 0:
            # Count molecules with Tanimoto similarity = 1.0 (perfect matches)
            correct_count = sum(1 for sim in tanimoto_sim if sim == 1.0)
            total_count = len(tanimoto_sim)
            
            correct_counts.append(correct_count)
            total_counts.append(total_count)
            filtered_labels.append(label)
            
            accuracy_percentage = (correct_count / total_count) * 100 if total_count > 0 else 0
            
            print(f"  - Total molecules: {total_count}")
            print(f"  - Correct molecules: {correct_count}")
            print(f"  - Accuracy: {accuracy_percentage:.1f}%")
        else:
            print(f"  - No Tanimoto data found for {label}")
        
        # Clear the variable and force garbage collection
        del results_dict
        gc.collect()
        
    except FileNotFoundError:
        print(f"  - File not found: {file_path}")
    except Exception as e:
        print(f"  - Error loading {file_path}: {e}")

print(f"\nProcessed {len(correct_counts)} datasets with Tanimoto data.")

# Check if we have any data
if not correct_counts:
    print("Error: All datasets are empty. Unable to create bar plot.")
else:
    # Calculate percentages
    accuracy_percentages = [(correct / total) * 100 if total > 0 else 0 
                           for correct, total in zip(correct_counts, total_counts)]
    
    # Create the bar plot
    fig, ax = plt.subplots(figsize=(12, 10))
    
    # Define positions for the bars
    positions = np.arange(len(filtered_labels))
    
    # Define colors based on dataset type
    colors = []
    for label in filtered_labels:
        if '1%' in label:
            colors.append("#FFA07A")  # Orange for small
        else:
            colors.append("#8CB0FE")  # Blue for original ablation study
    
    # Create bars with percentages - changed edgecolor from 'black' to 'gray'
    bars = ax.bar(positions, accuracy_percentages, color=colors, alpha=0.7, edgecolor='gray', linewidth=1)
    
    # Add absolute count labels inside the bars - black, non-bold, vertical
    for i, (pos, percentage, count, total) in enumerate(zip(positions, accuracy_percentages, correct_counts, total_counts)):
        ax.text(pos, percentage / 2, f'{count}', 
                rotation=90, ha='center', va='center', fontsize=20, color='black')
    
    # Add percentage labels on top of each bar
    for i, (pos, percentage) in enumerate(zip(positions, accuracy_percentages)):
        ax.text(pos, percentage + max(accuracy_percentages) * 0.02, f'{percentage:.1f}%', 
                ha='center', va='bottom', fontsize=20, color='black')
    
    # Customizing the axes and labels
    ax.set_ylabel('Accuracy (%) (Tanimoto = 1.0)', fontsize=22)
    
    # Set x-ticks
    ax.set_xticks(positions)
    ax.set_xticklabels(filtered_labels, rotation=45, ha='center', fontsize=20)
    ax.tick_params(axis='both', which='major', labelsize=20)
    
    # Add grid
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    ax.set_ylim(0, max(accuracy_percentages) * 1.1)  # Add some space at the top
    
    # Add vertical line to separate ablation study from dataset size comparison if we have both types
    original_count = sum(1 for label in filtered_labels if 'Small' not in label and 'Tiny' not in label)
    if original_count < len(filtered_labels):
        ax.axvline(x=original_count - 0.5, color='gray', linestyle='--', alpha=0.5, linewidth=2)
        
        # Add text annotations to clarify sections
        if original_count > 1:
            ax.text((original_count - 1)/2, max(accuracy_percentages) * 0.9, 'Ablation Study', 
                    ha='center', fontsize=16, style='italic', 
                    bbox=dict(boxstyle="round,pad=0.3", facecolor="lightblue", alpha=0.7))
        if len(filtered_labels) > original_count:
            remaining_pos = original_count + (len(filtered_labels) - original_count - 1)/2
            ax.text(remaining_pos, max(accuracy_percentages) * 0.9, 'Dataset Size', 
                    ha='center', fontsize=16, style='italic',
                    bbox=dict(boxstyle="round,pad=0.3", facecolor="lightcoral", alpha=0.7))
    
    for spine in ax.spines.values():
        spine.set_edgecolor('gray')
        spine.set_alpha(0.8) # Make it visible but not dominant
        
    plt.tight_layout()
    
    # Save the figure
    save_path = os.path.abspath('./_FIGURES/2.1_Ablation_Correct_Molecules_Percentage_v3.png')
    plt.savefig(save_path, format='png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"\nPlot saved to: {save_path}")

# Print summary statistics
print("\nSummary Statistics:")
print("="*50)
for label, correct, total, percentage in zip(filtered_labels, correct_counts, total_counts, accuracy_percentages):
    print(f"{label:15s}: {correct:4d}/{total:4d} ({percentage:5.1f}%)")

print("\nMemory efficient correct molecules counting completed!")

In [ ]:
import pickle

# Load the file
file_path = './past_experiments/ChemXriv/2.0_Experiment_Ablation_Study/2.1_results_dict_2_wo_MW_MF.pkl'
with open(file_path, 'rb') as f:
   data = pickle.load(f)

# Check different possible keys for Tanimoto data
tanimoto_data = None
if 'tanimoto_sim' in data:
   tanimoto_data = data['tanimoto_sim']
   key_used = 'tanimoto_sim'
elif 'tanimoto_scores_all' in data:
   tanimoto_data = data['tanimoto_scores_all']
   key_used = 'tanimoto_scores_all'
else:
   # Look for any key starting with 'tanimoto_scores_'
   for key in data.keys():
       if key.startswith('tanimoto_scores_'):
           tanimoto_data = data[key]
           key_used = key
           break

if tanimoto_data:
   correct_count = sum(1 for sim in tanimoto_data if sim == 1.0)
   total_count = len(tanimoto_data)
   accuracy = (correct_count / total_count) * 100 if total_count > 0 else 0
   
   print(f"Key used: {key_used}")
   print(f"Total molecules: {total_count}")
   print(f"Correct molecules (Tanimoto = 1.0): {correct_count}")
   print(f"Accuracy: {accuracy:.1f}%")
else:
   print("No Tanimoto data found")
   print("Available keys:", list(data.keys()))

In [ ]:
data.keys()

In [ ]:
data["trg_conv_SMI_list"]

In [ ]:
data

### 3.0 Percentage Calculations V8 and V8i with TRUE or FALSE

#### Generate a table with Greedy/Top 1/3/5/10 accuracy


#### 3.2 Load/process and plot data

In [ ]:

# Relative paths from the notebook location:
config.csv_1H_path_SGNN = './data/ZINK_dataset/ML_NMR_5M_XL_1H_comb_test_V8.csv'
config.csv_13C_path_SGNN = './data/ZINK_dataset/ML_NMR_5M_XL_13C.csv'    
config.csv_HSQC_path_SGNN = './data/ZINK_dataset/ML_NMR_5M_XL_HSQC.csv'    
config.csv_COSY_path_SGNN = './data/ZINK_dataset/ML_NMR_5M_XL_COSY.csv'
config.csv_path_val = './data/ZINK_dataset/ML_NMR_5M_XL_1H_comb_test_V8.csv'
config.pickle_file_path = './data/ZINK_dataset/ML_NMR_5M_XL_1H_comb_test_V8_355655.pkl'

In [ ]:
# config.training_mode = "1H_13C_HSQC_COSY_IR_MF_MW"
# config.IR_data_folder= os.path.abspath("./data/ZINK_dataset/IR_spectra_NN")
# config.checkpoint_path = os.path.abspath("./models/mmst/base_models/1_0_V8i_MMTi_RAW_DROP_Loss_0.112.ckpt")
# config.temperature = 1
# config.multinom_runs = 10
# config.data_size = 10000
# MW_filter = False
# MF_filter = False

# percentage_collection_FALSE_FALSE, results_dict_mns_10_FALSE_FALSE, results_dict_greedy_FALSE_FALSE = run_precentage_calculation_v2(config, itos, stoi, stoi_MF, MW_filter, MF_filter)

# file_path_c = os.path.abspath('./past_experiments/ChemXriv/3.0_Experiment_MW_MF_filtering/3.0.1_results_dict_greedy_FALSE_FALSE_10000.pkl')
# with open(file_path_c, 'wb') as file:
#     pickle.dump(results_dict_greedy_FALSE_FALSE, file)

# file_path_c = os.path.abspath('./past_experiments/ChemXriv/3.0_Experiment_MW_MF_filtering/3.0.2_results_dict_mns_10_FALSE_FALSE_10000.pkl')
# with open(file_path_c, 'wb') as file:
#     pickle.dump(results_dict_mns_10_FALSE_FALSE, file)

# file_path_c = os.path.abspath('./past_experiments/ChemXriv/3.0_Experiment_MW_MF_filtering/3.0.3_percentage_collection_FALSE_FALSE_10000.pkl')
# with open(file_path_c, 'wb') as file:
#     pickle.dump(percentage_collection_FALSE_FALSE, file)


In [ ]:

# config.training_mode = "1H_13C_HSQC_COSY_IR_MF_MW"
# config.IR_data_folder=os.path.abspath("./data/ZINK_dataset/IR_spectra_NN")
# config.checkpoint_path = os.path.abspath("./models/mmst/base_models/1_0_V8i_MMTi_RAW_DROP_Loss_0.112.ckpt")
# config.temperature = 1
# config.multinom_runs = 10
# config.data_size = 10000
# MW_filter = True
# MF_filter = False

# percentage_collection_TRUE_FALSE, results_dict_mns_10_TRUE_FALSE, results_dict_greedy_TRUE_FALSE = run_precentage_calculation_v2(config, itos, stoi, stoi_MF, MW_filter, MF_filter)

# file_path_c = os.path.abspath('./past_experiments/ChemXriv/3.0_Experiment_MW_MF_filtering/3.0.1_results_dict_greedy_TRUE_FALSE_10000.pkl')
# with open(file_path_c, 'wb') as file:
#     pickle.dump(results_dict_greedy_TRUE_FALSE, file)

# file_path_c = os.path.abspath('./past_experiments/ChemXriv/3.0_Experiment_MW_MF_filtering/3.0.2_results_dict_mns_10_TRUE_FALSE_10000.pkl')
# with open(file_path_c, 'wb') as file:
#     pickle.dump(results_dict_mns_10_TRUE_FALSE, file)

# file_path_c = os.path.abspath('./past_experiments/ChemXriv/3.0_Experiment_MW_MF_filtering/3.0.3_percentage_collection_TRUE_FALSE_10000.pkl')
# with open(file_path_c, 'wb') as file:
#     pickle.dump(percentage_collection_TRUE_FALSE, file)


In [ ]:
# config.training_mode = "1H_13C_HSQC_COSY_IR_MF_MW"
# config.IR_data_folder=os.path.abspath('./data/ZINK_dataset/IR_spectra_NN')
# config.checkpoint_path = os.path.abspath('./models/mmst/base_models/1_0_V8i_MMTi_RAW_DROP_Loss_0.112.ckpt')
# config.temperature = 1
# config.multinom_runs = 10
# config.data_size = 10000
# MW_filter = False
# MF_filter = True

# percentage_collection_FALSE_TRUE, results_dict_mns_10_FALSE_TRUE, results_dict_greedy_FALSE_TRUE = run_precentage_calculation_v2(config, itos, stoi, stoi_MF, MW_filter, MF_filter)

# file_path_c = os.path.abspath('./past_experiments/ChemXriv/3.0_Experiment_MW_MF_filtering/3.0.1_results_dict_greedy_FALSE_TRUE_10000.pkl')
# with open(file_path_c, 'wb') as file:
#     pickle.dump(results_dict_greedy_FALSE_TRUE, file)

# file_path_c = os.path.abspath('./past_experiments/ChemXriv/3.0_Experiment_MW_MF_filtering/3.0.2_results_dict_mns_10_FALSE_TRUE_10000.pkl')
# with open(file_path_c, 'wb') as file:
#     pickle.dump(results_dict_mns_10_FALSE_TRUE, file)

# file_path_c = os.path.abspath('./past_experiments/ChemXriv/3.0_Experiment_MW_MF_filtering/3.0.3_percentage_collection_FALSE_TRUE_10000.pkl')
# with open(file_path_c, 'wb') as file:
#     pickle.dump(percentage_collection_FALSE_TRUE, file)


In [ ]:

# config.training_mode = "1H_13C_HSQC_COSY_IR_MF_MW"
# config.IR_data_folder=os.path.abspath("./data/ZINK_dataset/IR_spectra_NN")
# config.checkpoint_path = os.path.abspath("./models/mmst/base_models/1_0_V8i_MMTi_RAW_DROP_Loss_0.112.ckpt")
# config.temperature = 1
# config.multinom_runs = 10
# config.data_size = 10000
# MW_filter = True
# MF_filter = True

# percentage_collection_TRUE_TRUE, results_dict_mns_10_TRUE_TRUE, results_dict_greedy_TRUE_TRUE = run_precentage_calculation_v2(config, itos, stoi, stoi_MF, MW_filter, MF_filter)

# file_path_c = os.path.abspath('./past_experiments/ChemXriv/3.0_Experiment_MW_MF_filtering/3.0.1_results_dict_greedy_TRUE_TRUE_10000.pkl')
# with open(file_path_c, 'wb') as file:
#     pickle.dump(results_dict_greedy_TRUE_TRUE, file)

# file_path_c = os.path.abspath('./past_experiments/ChemXriv/3.0_Experiment_MW_MF_filtering/3.0.2_results_dict_mns_10_TRUE_TRUE_10000.pkl')
# with open(file_path_c, 'wb') as file:
    
# file_path_c = os.path.abspath('./past_experiments/ChemXriv/3.0_Experiment_MW_MF_filtering/3.0.3_percentage_collection_TRUE_TRUE_10000.pkl')
# with open(file_path_c, 'wb') as file:
#     pickle.dump(percentage_collection_TRUE_TRUE, file)


##### Plot Data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
import os

def filter_first_n_examples(results_dict, n=4000):
    """Filter the first n examples from each results dictionary"""
    if isinstance(results_dict, dict):
        if "tanimoto_sim" in results_dict:
            # For greedy results (single level dict)
            filtered_dict = {}
            for key, value in results_dict.items():
                if isinstance(value, list):
                    filtered_dict[key] = value[:n]
                else:
                    filtered_dict[key] = value
            return filtered_dict
        else:
            # For MNS results (nested dict)
            filtered_dict = {}
            count = 0
            for idx in sorted(results_dict.keys()):
                if count >= n:
                    break
                remaining_slots = n - count
                samples_to_take = min(len(results_dict[idx]), remaining_slots)
                if samples_to_take > 0:
                    filtered_dict[idx] = results_dict[idx][:samples_to_take]
                    count += samples_to_take
            return filtered_dict
    return results_dict

def calc_percentage_and_count_top_x_correct_greedy(results_dict):
    count_yes = 0
    count_no = 0

    for i in results_dict["tanimoto_sim"]:
        if i == 1:
            count_yes += 1
        else:
            count_no += 1

    total = count_yes + count_no
    percentage = (count_yes / total) * 100 if total > 0 else 0
    return percentage, count_yes, total

def calc_percentage_and_count_top_x_correct(results_dict, top_x):
    count_yes = 0
    count_no = 0
    for idx in results_dict.keys():
        for result in results_dict[idx]:
            tanimoto_sim = result["tanimoto_sim"][:top_x]
            if 1 in tanimoto_sim:
                count_yes += 1
            else:
                count_no += 1
    total = count_yes + count_no
    percentage = (count_yes / total) * 100 if total > 0 else 0
    return percentage, count_yes, total

def prepare_data(results_dict_FALSE_FALSE, results_dict_FALSE_TRUE, 
                 results_dict_TRUE_FALSE, results_dict_TRUE_TRUE, 
                 results_dict_greedy, n_examples=4000):
    
    # Filter all datasets to first n examples
    print(f"Filtering datasets to first {n_examples} examples...")
    
    filtered_FALSE_FALSE = filter_first_n_examples(results_dict_FALSE_FALSE, n_examples)
    filtered_FALSE_TRUE = filter_first_n_examples(results_dict_FALSE_TRUE, n_examples)
    filtered_TRUE_FALSE = filter_first_n_examples(results_dict_TRUE_FALSE, n_examples)
    filtered_TRUE_TRUE = filter_first_n_examples(results_dict_TRUE_TRUE, n_examples)
    filtered_greedy = filter_first_n_examples(results_dict_greedy, n_examples)
    
    # Calculate percentages for each filter combination
    def calc_percentages(results_dict, greedy_dict):
        percentage_greedy, count_greedy, total_greedy = calc_percentage_and_count_top_x_correct_greedy(greedy_dict)
        percentage_top_1, count_top_1, total_1 = calc_percentage_and_count_top_x_correct(results_dict, 1)
        percentage_top_3, count_top_3, total_3 = calc_percentage_and_count_top_x_correct(results_dict, 3)
        percentage_top_5, count_top_5, total_5 = calc_percentage_and_count_top_x_correct(results_dict, 5)
        percentage_top_10, count_top_10, total_10 = calc_percentage_and_count_top_x_correct(results_dict, 10)
        
        print(f"  Greedy: {count_greedy}/{total_greedy} = {percentage_greedy:.1f}%")
        print(f"  Top-1: {count_top_1}/{total_1} = {percentage_top_1:.1f}%")
        print(f"  Top-3: {count_top_3}/{total_3} = {percentage_top_3:.1f}%")
        print(f"  Top-5: {count_top_5}/{total_5} = {percentage_top_5:.1f}%")
        print(f"  Top-10: {count_top_10}/{total_10} = {percentage_top_10:.1f}%")
        
        return [(percentage_greedy, count_greedy), (percentage_top_1, count_top_1),
                (percentage_top_3, count_top_3), (percentage_top_5, count_top_5),
                (percentage_top_10, count_top_10)]

    print("\nNo Filter:")
    no_filter = calc_percentages(filtered_FALSE_FALSE, filtered_greedy)
    
    print("\nMF Filter:")
    mf_filter = calc_percentages(filtered_FALSE_TRUE, filtered_greedy)
    
    print("\nMW Filter:")
    mw_filter = calc_percentages(filtered_TRUE_FALSE, filtered_greedy)
    
    print("\nMW + MF Filter:")
    mw_mf_filter = calc_percentages(filtered_TRUE_TRUE, filtered_greedy)

    return no_filter, mf_filter, mw_filter, mw_mf_filter

def plot_results(no_filter, mf_filter, mw_filter, mw_mf_filter, n_examples=4000):
    labels = ['Greedy', '1 Sample', '3 Samples', '5 Samples', '10 Samples']
    x = np.arange(len(labels))
    width = 0.25  # Increased width to accommodate 3 bars
    colors = ['#A1C8F3', '#FFB381', '#8BE5A0', '#FF9D9A']
    
    fig, ax = plt.subplots(figsize=(15, 8))
    
    # Plot bars in the desired order - only 3 filters
    rects1 = ax.bar(x - width, [d[0] for d in no_filter], width, label='No Filter', color=colors[0])
    rects2 = ax.bar(x, [d[0] for d in mw_filter], width, label='MW Filter', color=colors[1])
    rects3 = ax.bar(x + width, [d[0] for d in mw_mf_filter], width, label='MW + MF Filter', color=colors[2])

    # Set labels and title with larger font sizes
    ax.set_ylabel('Percentage', fontsize=30)
    ax.set_title(f'Performance Comparison with Different Filters ({n_examples} examples)', fontsize=30)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=30)
    ax.legend(fontsize=24)
    ax.tick_params(axis='y', labelsize=30)
    ax.set_ylim(40, 100)

    # Function to add annotations to the bars
    def autolabel(rects, data):
        for rect, (percentage, count) in zip(rects, data):
            height = rect.get_height()
            visible_height = height - 40  # Adjust for the 40% lower limit
            
            # Add rotated percentage on top of the bar
            ax.text(rect.get_x() + rect.get_width() / 2, height + 2, f'{percentage:.1f}%',
                    ha='center', va='bottom', fontsize=18, rotation=90)
            
            # Add count at the middle of the visible part of the bar
            ax.text(rect.get_x() + rect.get_width() / 2, 40 + visible_height / 2, f'{count}',
                    ha='center', va='center', fontsize=18, rotation=90)

    # Add annotations to each bar - only 3 filters
    autolabel(rects1, no_filter)
    autolabel(rects2, mw_filter)
    autolabel(rects3, mw_mf_filter)

    fig.tight_layout()
    
    # Create output directory if it doesn't exist
    os.makedirs('./_FIGURES', exist_ok=True)
    plt.savefig(os.path.abspath(f'./_FIGURES/performance_comparison_with_counts_all_filters_{n_examples}_examples.png'), dpi=300, bbox_inches='tight')
    plt.show()

def main():
    """Main function to run the analysis with 4000 examples"""
    
    # Load your data
    file_path_base = os.path.abspath('./past_experiments/ChemXriv/3.0_Experiment_MW_MF_filtering/')
    
    print("Loading data files...")
    with open(file_path_base + '/3.0.2_results_dict_mns_10_FALSE_FALSE_10000.pkl', 'rb') as file:
        results_dict_FALSE_FALSE = pickle.load(file)
    with open(file_path_base + '/3.0.2_results_dict_mns_10_FALSE_TRUE_10000.pkl', 'rb') as file:
        results_dict_FALSE_TRUE = pickle.load(file)
    with open(file_path_base + '/3.0.2_results_dict_mns_10_TRUE_FALSE_10000.pkl', 'rb') as file:
        results_dict_TRUE_FALSE = pickle.load(file)
    with open(file_path_base + '/3.0.2_results_dict_mns_10_TRUE_TRUE_10000.pkl', 'rb') as file:
        results_dict_TRUE_TRUE = pickle.load(file)
    with open(file_path_base + '/3.0.1_results_dict_greedy_FALSE_FALSE_10000.pkl', 'rb') as file:
        results_dict_greedy = pickle.load(file)
    
    # Prepare and plot data with 4000 examples
    no_filter, mf_filter, mw_filter, mw_mf_filter = prepare_data(
        results_dict_FALSE_FALSE, 
        results_dict_FALSE_TRUE,
        results_dict_TRUE_FALSE, 
        results_dict_TRUE_TRUE,
        results_dict_greedy,
        n_examples=4000
    )
    
    plot_results(no_filter, mf_filter, mw_filter, mw_mf_filter, n_examples=4000)
    
    return no_filter, mf_filter, mw_filter, mw_mf_filter

if __name__ == "__main__":
    results = main()

### 4.0. Similarity reduction plotting

#### 4.1 Tanimoto Similarity

In [ ]:
import utils_MMT.experiment_function_v15_4 as exp_func
def main():
    np.random.seed(42)  # Set random seed for reproducibility   
    zinc_path = os.path.abspath('./data/ZINK_dataset/ML_NMR_5M_XL_1H_comb_train_V8.csv')
    save_folder = os.path.abspath("./_FIGURES")

    # Convert to absolute paths
    pubchem_paths = [
        os.path.abspath('./past_experiments/ChemXriv/6.0_Dataset_Improvement_Cycle/val_data_0_250_x1000/ML_NMR_2M_XL_1H_V1_test_f_0_250_x1000.csv'),
        os.path.abspath('./past_experiments/ChemXriv/6.0_Dataset_Improvement_Cycle/val_data_250_350_x1000/ML_NMR_2M_XL_1H_V1_test_f_250_350_x1000.csv'),
        os.path.abspath('./past_experiments/ChemXriv/6.0_Dataset_Improvement_Cycle/val_data_350_500_x1000/ML_NMR_2M_XL_1H_V1_test_f_350_500_x1000.csv')
    ]
    
    weight_ranges = ['0-250 Da', '250-350 Da', '350-500 Da']

    print("Loading ZINC data...")
    zinc_smiles = exp_func.load_data_smi_list(zinc_path)
    zinc_smiles = exp_func.load_data_smi_list(zinc_path, sample_size=3000)

    zinc_fp = exp_func.calculate_fingerprints(zinc_smiles)  # Limit to 10000 compounds for performance

    for pubchem_path, weight_range in zip(pubchem_paths, weight_ranges):
        print(f"Processing PubChem data for {weight_range}...")
        pubchem_smiles = exp_func.load_data_smi_list(pubchem_path)
        pubchem_fp = exp_func.calculate_fingerprints(pubchem_smiles[:100])  # Limit to 100 compounds for performance

        combined_fp = np.vstack((zinc_fp, pubchem_fp))
        labels = np.array(['ZINC'] * len(zinc_fp) + ['PubChem'] * len(pubchem_fp))

        print("Performing dimensionality reduction...")
        tsne_result, pca_result, umap_result = exp_func.perform_dimensionality_reduction(combined_fp)

        print("Plotting results...")
        exp_func.plot_tsne_umap_pca([tsne_result, pca_result, umap_result], labels, 
                     f"ZINC vs PubChem ({weight_range})", 
                     ['t-SNE', 'PCA', 'UMAP'],
                     save_folder)

    print("All plots generated successfully!")



In [ ]:
main()
    

#### 4.2 Vector Similarity

4.2.1 Pubchem vectorization

In [ ]:
# First block (commented out)
#config.csv_1H_path_SGNN = os.path.abspath('./past_experiments/ChemXriv/6.0_Dataset_Improvement_Cycle/val_data_350_500_x1000/ML_NMR_2M_XL_1H_V1_test_f_350_500_x1000.csv')
#config.csv_13C_path_SGNN = os.path.abspath('./past_experiments/ChemXriv/6.0_Dataset_Improvement_Cycle/val_data_350_500_x1000/ML_NMR_2M_XL_13C_V1_test_350_500_x1000.csv')    
#config.csv_HSQC_path_SGNN = os.path.abspath('./past_experiments/ChemXriv/6.0_Dataset_Improvement_Cycle/val_data_350_500_x1000/ML_NMR_2M_XL_HSQC_V1_test_350_500_x1000.csv')    
#config.csv_COSY_path_SGNN = os.path.abspath('./past_experiments/ChemXriv/6.0_Dataset_Improvement_Cycle/val_data_350_500_x1000/ML_NMR_2M_XL_COSY_V1_test_350_500_x1000.csv')   
#config.csv_path_val = os.path.abspath('./past_experiments/ChemXriv/6.0_Dataset_Improvement_Cycle/val_data_350_500_x1000/ML_NMR_2M_XL_1H_V1_test_f_350_500_x1000.csv')
#config.pickle_file_path = ''

# Second block (commented out)
#config.csv_1H_path_SGNN = os.path.abspath('./past_experiments/ChemXriv/6.0_Dataset_Improvement_Cycle/val_data_0_250_x1000/ML_NMR_2M_XL_1H_V1_test_f_0_250_x1000.csv')
#config.csv_13C_path_SGNN = os.path.abspath('./past_experiments/ChemXriv/6.0_Dataset_Improvement_Cycle/val_data_0_250_x1000/ML_NMR_2M_XL_13C_V1_test_0_250_x1000.csv')
#config.csv_HSQC_path_SGNN = os.path.abspath('./past_experiments/ChemXriv/6.0_Dataset_Improvement_Cycle/val_data_0_250_x1000/ML_NMR_2M_XL_HSQC_V1_test_0_250_x1000.csv')    
#config.csv_COSY_path_SGNN = os.path.abspath('./past_experiments/ChemXriv/6.0_Dataset_Improvement_Cycle/val_data_0_250_x1000/ML_NMR_2M_XL_COSY_V1_test_0_250_x1000.csv')   
#config.csv_path_val = os.path.abspath('./past_experiments/ChemXriv/6.0_Dataset_Improvement_Cycle/val_data_0_250_x1000/ML_NMR_2M_XL_1H_V1_test_f_0_250_x1000.csv')
#config.pickle_file_path = ''

# Third block (active)
config.csv_1H_path_SGNN = os.path.abspath('./past_experiments/ChemXriv/6.0_Dataset_Improvement_Cycle/val_data_250_350_x1000/ML_NMR_2M_XL_1H_V1_test_f_250_350_x1000.csv')
config.csv_13C_path_SGNN = os.path.abspath('./past_experiments/ChemXriv/6.0_Dataset_Improvement_Cycle/val_data_250_350_x1000/ML_NMR_2M_XL_13C_V1_test_250_350_x1000.csv')    
config.csv_HSQC_path_SGNN = os.path.abspath('./past_experiments/ChemXriv/6.0_Dataset_Improvement_Cycle/val_data_250_350_x1000/ML_NMR_2M_XL_HSQC_V1_test_250_350_x1000.csv')    
config.csv_COSY_path_SGNN = os.path.abspath('./past_experiments/ChemXriv/6.0_Dataset_Improvement_Cycle/val_data_250_350_x1000/ML_NMR_2M_XL_COSY_V1_test_250_350_x1000.csv')   
config.csv_path_val = os.path.abspath('./past_experiments/ChemXriv/6.0_Dataset_Improvement_Cycle/val_data_250_350_x1000/ML_NMR_2M_XL_1H_V1_test_f_250_350_x1000.csv')
config.pickle_file_path = ''

# IR data folder
# config.IR_data_folder="./data/ZINK_dataset/IR_spectra_NN"
config.IR_data_folder=os.path.abspath("./data/PubChem_dataset/IR_data")



In [ ]:
# Test Data
config.csv_1H_path_SGNN = os.path.abspath('./past_experiments/ChemXriv/6.0_Dataset_Improvement_Cycle/val_data_250_350_x1000/ML_NMR_2M_XL_1H_V1_test_f_250_350_x1000.csv')
config.csv_13C_path_SGNN = os.path.abspath('./past_experiments/ChemXriv/6.0_Dataset_Improvement_Cycle/val_data_250_350_x1000/ML_NMR_2M_XL_13C_V1_test_250_350_x1000.csv')    
config.csv_HSQC_path_SGNN = os.path.abspath('./past_experiments/ChemXriv/6.0_Dataset_Improvement_Cycle/val_data_250_350_x1000/ML_NMR_2M_XL_HSQC_V1_test_250_350_x1000.csv')    
config.csv_COSY_path_SGNN = os.path.abspath('./past_experiments/ChemXriv/6.0_Dataset_Improvement_Cycle/val_data_250_350_x1000/ML_NMR_2M_XL_COSY_V1_test_250_350_x1000.csv')   
config.csv_path_val = os.path.abspath('./past_experiments/ChemXriv/6.0_Dataset_Improvement_Cycle/val_data_250_350_x1000/ML_NMR_2M_XL_1H_V1_test_f_250_350_x1000.csv')
config.pickle_file_path = ""

config.IR_data_folder=os.path.abspath("./data/PubChem_dataset/IR_data")
config.data_size = 1000

config.checkpoint_path = os.path.abspath("./models/mmst/base_models/1_0_V8i_MMTi_RAW_DROP_Loss_0.112.ckpt")
config.vector_db = os.path.abspath('./past_experiments/ChemXriv/6.0_Dataset_Improvement_Cycle/vectors/ML_NMR_2M_XL_1H_V1_test_f_250_350_x1000_Vectors.csv')

In [ ]:
### Data needs to be loaded in csv_1H_path_SGNN 13C ...etc
# path to store db
config = exp_func.vectorize_db(config, stoi, stoi_MF, "db", "all")

In [ ]:
"""
vector_db = config.vector_db
# Load CSV data
df = pd.read_csv(vector_db)

# Convert 'Fingerprints' to tensors
tqdm.pandas()
df['Fingerprints'] = df['Fingerprints'].progress_apply(ast.literal_eval)
df['Fingerprints'] = df['Fingerprints'].progress_apply(lambda x: torch.tensor(x, dtype=torch.float32))

# Save the DataFrame to Pickle
pickle_file = vector_db.replace('.csv', '_v2.pkl')
with open(pickle_file, 'wb') as f:
    pickle.dump(df, f)

print(f"Data saved to {pickle_file}")
"""

In [ ]:
config.csv_1H_path_SGNN = os.path.abspath('./past_experiments/ChemXriv/6.0_Dataset_Improvement_Cycle/val_data_0_250_x1000/ML_NMR_2M_XL_1H_V1_test_f_0_250_x1000.csv')
config.csv_13C_path_SGNN = os.path.abspath('./past_experiments/ChemXriv/6.0_Dataset_Improvement_Cycle/val_data_0_250_x1000/ML_NMR_2M_XL_13C_V1_test_0_250_x1000.csv')
config.csv_HSQC_path_SGNN = os.path.abspath('./past_experiments/ChemXriv/6.0_Dataset_Improvement_Cycle/val_data_0_250_x1000/ML_NMR_2M_XL_HSQC_V1_test_0_250_x1000.csv')    
config.csv_COSY_path_SGNN = os.path.abspath('./past_experiments/ChemXriv/6.0_Dataset_Improvement_Cycle/val_data_0_250_x1000/ML_NMR_2M_XL_COSY_V1_test_0_250_x1000.csv')   
config.csv_path_val = os.path.abspath('./past_experiments/ChemXriv/6.0_Dataset_Improvement_Cycle/val_data_0_250_x1000/ML_NMR_2M_XL_1H_V1_test_f_0_250_x1000.csv')
config.pickle_file_path = ''

config.IR_data_folder=os.path.abspath("./data/PubChem_dataset/IR_data")
config.data_size = 1000

config.checkpoint_path = os.path.abspath("./models/mmst/base_models/1_0_V8i_MMTi_RAW_DROP_Loss_0.112.ckpt")
config.vector_db = os.path.abspath('./past_experiments/ChemXriv/6.0_Dataset_Improvement_Cycle/vectors/ML_NMR_2M_XL_1H_V1_test_f_0_250_x1000_Vectors.csv')

In [ ]:
### Data needs to be loaded in csv_1H_path_SGNN 13C ...etc
# path to store db
config = exp_func.vectorize_db(config, stoi, stoi_MF, "db", "all")

In [ ]:
"""
vector_db = config.vector_db
# Load CSV data
df = pd.read_csv(vector_db)

# Convert 'Fingerprints' to tensors
tqdm.pandas()
df['Fingerprints'] = df['Fingerprints'].progress_apply(ast.literal_eval)
df['Fingerprints'] = df['Fingerprints'].progress_apply(lambda x: torch.tensor(x, dtype=torch.float32))

# Save the DataFrame to Pickle
pickle_file = vector_db.replace('.csv', '_v2.pkl')
with open(pickle_file, 'wb') as f:
    pickle.dump(df, f)

print(f"Data saved to {pickle_file}")
"""

##### UMAP, tSNE, PCA Set 1-3

In [ ]:
# Load test data
test_files = [
    os.path.abspath('./past_experiments/ChemXriv/6.0_Dataset_Improvement_Cycle/vectors/ML_NMR_2M_XL_1H_V1_test_f_0_250_x1000_Vectors_v2.pkl'),
    os.path.abspath('./past_experiments/ChemXriv/6.0_Dataset_Improvement_Cycle/vectors/ML_NMR_2M_XL_1H_V1_test_f_250_350_x1000_Vectors_v2.pkl'),
    os.path.abspath('./past_experiments/ChemXriv/6.0_Dataset_Improvement_Cycle/vectors/ML_NMR_2M_XL_1H_V1_test_f_350_500_x1000_Vectors_v2.pkl')
]

weight_ranges = ['0-250 Da', '250-350 Da', '350-500 Da']
weight_ranges = ["Set 1", "Set 2", "Set 3"]
save_folder = os.path.abspath("./_FIGURES")

# Load training data
print("Loading training data...")
train_vectors, train_smiles = exp_func.load_pickle_data(os.path.abspath('./past_experiments/ChemXriv/6.0_Dataset_Improvement_Cycle/vectors/vector_db_train_5Mod_4M_v2_v2_100k.pkl'), sample_size=3000)

for test_file, weight_range in zip(test_files, weight_ranges):
    print(f"Processing test data for {weight_range}...")
    test_vectors, test_smiles = exp_func.load_pickle_data(test_file)
    
    # Use only the first 100 test vectors
    test_vectors_sample = test_vectors[:300]
    
    combined_vectors = np.vstack((train_vectors, test_vectors_sample))
    labels = ['Train'] * len(train_vectors) + ['Test'] * len(test_vectors_sample)
    
    print(f"Shape of combined vectors: {combined_vectors.shape}")
    print(f"Number of labels: {len(labels)}")
    
    print("Performing dimensionality reduction...")
    tsne_result, pca_result, umap_result = exp_func.perform_dimensionality_reduction(combined_vectors)
    
    print("Plotting results...")
    exp_func.plot_tsne_umap_pca_train_test([tsne_result, pca_result, umap_result], labels, 
                 f"Train vs Test Vectors ({weight_range})", 
                 ['t-SNE', 'PCA', 'UMAP'],
                 save_folder)

print("All plots generated successfully!")

#### 4.3 1000 Calculations

##### PC and ZINC Latent space comparison for correct and incorrect molecules (FAIL now)
- calculations done: /projects/cc/se_users/knlr326/1_NMR_project/2_Notebooks/MultiModalSpectralTransformer/scripts

In [ ]:
"""
def main(data_configs, train_data_path, output_folder, ranking_method):
    os.makedirs(output_folder, exist_ok=True)

    print("Loading training data...")
    train_vectors, train_smiles = exp_func.load_pickle_data(train_data_path, sample_size=3000)

    for data_config in data_configs:
        exp_func.process_dataset_(data_config, train_vectors, train_smiles, output_folder, ranking_method)

if __name__ == "__main__":
    data_configs = [
        {
            'weight_range': 'PC_0-250',
            'pkl_folder': os.path.abspath('./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/PC_0_250'),
            'file_path': os.path.abspath('./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/PubChem_vectors/sim_mol_0_250/experiment_results_0_250.pkl')
        },
        {
            'weight_range': 'PC_250-350',
            'pkl_folder': os.path.abspath('./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/PC_250_350'),
            'file_path': os.path.abspath('./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/PubChem_vectors/sim_mol_250_350/experiment_results_250_350.pkl')
        },
        {
            'weight_range': 'PC_350-500',
            'pkl_folder': os.path.abspath('./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/PC_350_500'),
            'file_path': os.path.abspath('./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/PubChem_vectors/sim_mol_350_500/experiment_results_350_500.pkl')
        },
        {
            'weight_range': 'ZINC_250-350',
            'pkl_folder': os.path.abspath('./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/ZINC_250_350'),
            'file_path': os.path.abspath('./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/ZINC_1000_data/ZINC_1000_vectors_v2.pkl.pkl')
        }
    ]

    train_data_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Dataset_Improvement_Cycle/vectors/vector_db_train_5Mod_4M_v2_v2_100k.pkl')
    output_folder = os.path.abspath('./_FIGURES')
    ranking_method = 'HSQC'

    main(data_configs, train_data_path, output_folder, ranking_method)"""

#### 4.4 Histograms for NN

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import random
import string

def main_csv_generation(data_configs, output_folder, ranking_method):
    os.makedirs(output_folder, exist_ok=True)

    for data_config in data_configs:
        print(f"Processing {data_config['weight_range']}...")
        pos_csv_path, neg_csv_path = exp_func.process_dataset_and_save_csv(data_config, output_folder, ranking_method)
        print(f"Positive examples saved to: {pos_csv_path}")
        print(f"Negative examples saved to: {neg_csv_path}")

if __name__ == "__main__":
    data_configs = [
        {
            'weight_range': 'PC_0-250',
            'pkl_folder': os.path.abspath('./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/PC_0_250'),
            'file_path': os.path.abspath('./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/PubChem_vectors/ML_NMR_2M_XL_1H_V1_test_f_0_250_x1000_Vectors_v2.pkl')
        },
        {
            'weight_range': 'PC_250-350',
            'pkl_folder': os.path.abspath('./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/PC_250_350'),
            'file_path': os.path.abspath('./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/PubChem_vectors/ML_NMR_2M_XL_1H_V1_test_f_250_350_x1000_Vectors_v2.pkl')
        },
        {
            'weight_range': 'PC_350-500',
            'pkl_folder': os.path.abspath('./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/PC_350_500'),
            'file_path': os.path.abspath('./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/PubChem_vectors/ML_NMR_2M_XL_1H_V1_test_f_350_500_x1000_Vectors_v2.pkl')
        },
        {
            'weight_range': 'ZINC_250-350',
            'pkl_folder': os.path.abspath('./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/ZINC_250_350'),
            'file_path': os.path.abspath('./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/ZINC_1000_data/ZINC_1000_vectors_v2.pkl')
        },
        {
            'weight_range': 'ZINC_250-350_4000',
            'pkl_folder': os.path.abspath('./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/ZINC_250_350_4000'),
            'file_path': os.path.abspath('./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/ZINC_1000_data/ZINC_1000_data/ZINC_1000_vectors_v2.pkl')
        } ,       
    ]
    
    data_configs = [            
        {
            'weight_range': 'ZINC_250-350_4000',
            'pkl_folder': os.path.abspath('./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/ZINC_250_350_4000'),
            'file_path': os.path.abspath('./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/ZINC_1000_data/ZINC_1000_vectors_v2.pkl')
        }  
    ]
    output_folder = os.path.abspath('./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/CSV_examples_2')
    ranking_method = 'HSQC'

    #main_csv_generation(data_configs, output_folder, ranking_method)


#### After Fine-tuning for 100 neg compounds

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import random
import string

def main_csv_generation(data_configs, output_folder, ranking_method):
    os.makedirs(output_folder, exist_ok=True)

    for data_config in data_configs:
        print(f"Processing {data_config['weight_range']}...")
        pos_csv_path, neg_csv_path = exp_func.process_dataset_and_save_csv(data_config, output_folder, ranking_method)
        print(f"Positive examples saved to: {pos_csv_path}")
        print(f"Negative examples saved to: {neg_csv_path}")

if __name__ == "__main__":
    data_configs = [
        {
            'weight_range': 'PC_0-250_neg_100',
            'pkl_folder': os.path.abspath('./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/PC_0_250'),
            'file_path': os.path.abspath('./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/PubChem_vectors/ML_NMR_2M_XL_1H_V1_test_f_0_250_x1000_Vectors_v2.pkl')
        },
        {
            'weight_range': 'PC_250-350_neg_100',
            'pkl_folder': os.path.abspath('./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/PC_250_350'),
            'file_path': os.path.abspath('./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/PubChem_vectors/ML_NMR_2M_XL_1H_V1_test_f_250_350_x1000_Vectors_v2.pkl')
        },
        {
            'weight_range': 'PC_350-500_neg_100',
            'pkl_folder': os.path.abspath('./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/PC_350_500'),
            'file_path': os.path.abspath('./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/PubChem_vectors/ML_NMR_2M_XL_1H_V1_test_f_350_500_x1000_Vectors_v2.pkl')
        },
        {
            'weight_range': 'ZINC_250-350_neg_100',
            'pkl_folder': os.path.abspath('./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/ZINC_250_350'),
            'file_path': os.path.abspath('./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/ZINC_1000_data/ZINC_1000_vectors_v2.pkl')
        },
     
    ]

    
    output_folder = os.path.abspath('/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/47_Anna_paper_data/Experiment_baseline_PC_ZINC/CSV_examples_2')
    ranking_method = 'HSQC'

    main_csv_generation(data_configs, output_folder, ranking_method)


In [ ]:
import pandas as pd
import os

# List of file paths
file_paths = [
    os.path.abspath("./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/CSV_examples/PC_0-250_negative.csv"),
    os.path.abspath("./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/CSV_examples/PC_0-250_positive.csv"),
    os.path.abspath("./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/CSV_examples/PC_250-350_negative.csv"),
    os.path.abspath("./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/CSV_examples/PC_250-350_positive.csv"),
    os.path.abspath("./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/CSV_examples/PC_350-500_negative.csv"),
    os.path.abspath("./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/CSV_examples/PC_350-500_positive.csv"),
    os.path.abspath("./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/CSV_examples/ZINC_250-350_negative.csv"),
    os.path.abspath("./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/CSV_examples/ZINC_250-350_positive.csv")
]

# Process each file
for file_path in file_paths:
    if os.path.exists(file_path):
        exp_func.rename_column_in_csv(file_path, "sample_id", "sample-id")
    else:
        print(f"File not found: {file_path}")

print("All files processed.")

In [ ]:
config.SGNN_csv_gen_smi = os.path.abspath("./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/CSV_examples_2/PC_0-250_negative.csv")
config.SGNN_csv_gen_smi = os.path.abspath("./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/CSV_examples_2/PC_0-250_positive.csv")
config.SGNN_csv_gen_smi = os.path.abspath("./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/CSV_examples_2/PC_250-350_negative.csv")
config.SGNN_csv_gen_smi = os.path.abspath("./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/CSV_examples_2/PC_250-350_positive.csv")
config.SGNN_csv_gen_smi = os.path.abspath("./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/CSV_examples_2/PC_350-500_negative.csv")
config.SGNN_csv_gen_smi = os.path.abspath("./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/CSV_examples_2/PC_350-500_positive.csv")
config.SGNN_csv_gen_smi = os.path.abspath("./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/CSV_examples_2/ZINC_250-350_negative.csv")


config.SGNN_gen_folder_path = os.path.abspath("./experiments/SGNN_gen_folder")
config = ex.gen_sim_aug_data(config, IR_config)
config.csv_path_val = config.csv_1H_path_SGNN
#config = ex.filter_invalid_criteria(config.csv_1H_path_SGNN)

In [ ]:
import pandas as pd
import pickle
import torch
from tqdm import tqdm
import ast

config.pickle_file_path = ""
config.data_size = 1000

config.checkpoint_path = os.path.abspath("./models/mmst/base_models/1_0_V8i_MMTi_RAW_DROP_Loss_0.112.ckpt"
config.vector_db = config.SGNN_csv_gen_smi[:-4] + "vector.csv"
config = vectorize_db(config, stoi, stoi_MF, "db", "all")

vector_db = config.vector_db
# Load CSV data
df = pd.read_csv(vector_db)

# Convert 'Fingerprints' to tensors
tqdm.pandas()
df['Fingerprints'] = df['Fingerprints'].progress_apply(ast.literal_eval)
df['Fingerprints'] = df['Fingerprints'].progress_apply(lambda x: torch.tensor(x, dtype=torch.float32))

# Save the DataFrame to Pickle
pickle_file = vector_db.replace('.csv', '_v2.pkl')
with open(pickle_file, 'wb') as f:
    pickle.dump(df, f)

print(f"Data saved to {pickle_file}")


In [ ]:
PC_0_250_pkl_pos = os.path.abspath("./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/CSV_examples_2/PC_0-250_positivevector_v2.pkl")
PC_250_350_pkl_pos = os.path.abspath("./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/CSV_examples_2/PC_250-350_positivevector_v2.pkl")
PC_350_500_pkl_pos = os.path.abspath("./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/CSV_examples_2/PC_350-500_positivevector_v2.pkl")
ZINC_250_350_pkl_pos = os.path.abspath("./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/CSV_examples_2/ZINC_250-350_positivevector_v2.pkl")

PC_0_250_pkl_neg = os.path.abspath("./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/CSV_examples_2/PC_0-250_negativevector_v2.pkl")
PC_250_350_pkl_neg = os.path.abspath("./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/CSV_examples_2/PC_250-350_negativevector_v2.pkl")
PC_350_500_pkl_neg = os.path.abspath("./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/CSV_examples_2/PC_350-500_negativevector_v2.pkl")
ZINC_250_350_pkl_neg = os.path.abspath("./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/CSV_examples_2/ZINC_250-350_negativevector_v2.pkl")

# big_vector_db = os.path.abspath("/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/15_ZINC270M/vector_db_train_5Mod_4M_v2_v2.pkl")
# output_folder = os.path.abspath("/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/47_Anna_paper_data/Experiment_baseline_PC_ZINC/CSV_examples")
# database_nr = 3975764


##### Chart PubChem, ZINC

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os

# File paths
PC_0_250_pos = os.path.abspath("./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/CSV_examples_2/PC_0-250_positive.csv")
PC_250_350_pos = os.path.abspath("./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/CSV_examples_2/PC_250-350_positive.csv")
PC_350_500_pos = os.path.abspath("./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/CSV_examples_2/PC_350-500_positive.csv")
ZINC_250_350_pos = os.path.abspath("./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/CSV_examples_2/ZINC_250-350_positive.csv")
PC_0_250_neg = os.path.abspath("./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/CSV_examples_2/PC_0-250_negative.csv")
PC_250_350_neg = os.path.abspath("./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/CSV_examples_2/PC_250-350_negative.csv")
PC_350_500_neg = os.path.abspath("./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/CSV_examples_2/PC_350-500_negative.csv")
ZINC_250_350_neg = os.path.abspath("./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/CSV_examples_2/ZINC_250-350_negative.csv")

# Save location for the figure
save_dir = "./_FIGURES"

#def load_pickle(file_path):
#    with open(file_path, 'rb') as file:
#        return pickle.load(file)
def load_pickle(file_path):
    return pd.read_csv(file_path)

def process_data(pos_df, neg_df):
    total_compounds = 1000  # Assuming 1000 compounds per range
    pos_count = len(pos_df)
    neg_count = len(neg_df)
    failed_count = total_compounds - (pos_count + neg_count)
    
    return {
        'Correct': (pos_count / total_compounds) * 100,
        'Incorrect': (neg_count / total_compounds) * 100,
        'Failed': (failed_count / total_compounds) * 100
    }

# Load DataFrames
PC_0_250_pos_df = pd.read_csv(PC_0_250_pos)
PC_0_250_neg_df = pd.read_csv(PC_0_250_neg)
PC_250_350_pos_df = pd.read_csv(PC_250_350_pos)
PC_250_350_neg_df = pd.read_csv(PC_250_350_neg)
PC_350_500_pos_df = pd.read_csv(PC_350_500_pos)
PC_350_500_neg_df = pd.read_csv(PC_350_500_neg)
ZINC_250_350_pos_df = pd.read_csv(ZINC_250_350_pos)
ZINC_250_350_neg_df = pd.read_csv(ZINC_250_350_neg)

# Process data
data_0_250 = process_data(PC_0_250_pos_df, PC_0_250_neg_df)
data_250_350 = process_data(PC_250_350_pos_df, PC_250_350_neg_df)
data_350_500 = process_data(PC_350_500_pos_df, PC_350_500_neg_df)
data_ZINC_250_350 = process_data(ZINC_250_350_pos_df, ZINC_250_350_neg_df)

# Prepare data for plotting
plot_data = {
    'Correct': [data_0_250['Correct'], data_250_350['Correct'], data_350_500['Correct'], data_ZINC_250_350['Correct']],
    'Incorrect': [data_0_250['Incorrect'], data_250_350['Incorrect'], data_350_500['Incorrect'], data_ZINC_250_350['Incorrect']],
    'Failed': [data_0_250['Failed'], data_250_350['Failed'], data_350_500['Failed'], data_ZINC_250_350['Failed']]
}

def create_stacked_bar_chart(data, categories, title, save_path):
    fontsize = 22
    colors = [
    '#8BE5A0',  # mint green
    '#A1C8F3',  # light blue/periwinkle
    '#FFB381',  # salmon/peach
    '#FF9D9A',  # coral pink
    '#D1B9FE',  # lavender
    '#DEBA9A',  # beige/tan
    '#FCAEE3',  # pink
    '#CFCECE',  # light gray
    '#FEFDA2',  # pale yellow
    '#B8F1EF',  # light cyan/aqua
]
    fig, ax = plt.subplots(figsize=(8, 8))
    
    bottom = np.zeros(4)
    bars = []
    for i, category in enumerate(categories):
        values = data[category]
        bar = ax.bar(range(4), values, bottom=bottom, label=category, color=colors[i], edgecolor='black')
        # Set black edges for each bar patch
        for patch in bar:
            patch.set_edgecolor('black')
        bottom += values
        bars.append(bar)
    
    ax.set_title(title, fontsize=fontsize, pad=20)
    ax.set_xticks(range(4))
    ax.set_xticklabels(['PC (0-250 Da)', 'PC (250-350 Da)', 'PC (350-500 Da)', 'ZINC (250-350 Da)'], fontsize=12)
    ax.set_xticklabels(['Set 1', 'Set 2', 'Set 3', 'ZINC'], fontsize=fontsize)
    ax.set_ylabel('Percentage', fontsize=fontsize)
    ax.set_ylim(0, 100)
    
    ax.tick_params(axis='y', labelsize=fontsize)
    
    ax.legend(loc='lower right', fontsize=fontsize-4)
    
    for i in ax.containers:
        ax.bar_label(i, fmt='%.1f%%', label_type='center', fontsize=fontsize)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(save_path)
    return fig

categories = ['Correct', 'Incorrect', 'Failed']

# Create save path
if not os.path.exists(save_dir):
    os.makedirs(save_dir)
save_path = os.path.join(save_dir, 'PubChem_ZINC_Dataset_Results_v2.png')

fig = create_stacked_bar_chart(plot_data, categories, 'PubChem and ZINC Dataset Results', save_path)
"""
# Print the percentages for each range
for range_name, data in zip(['PC 0-250', 'PC 250-350', 'PC 350-500', 'ZINC 250-350'], 
                            [data_0_250, data_250_350, data_350_500, data_ZINC_250_350]):
    print(f"\nPercentages for {range_name} Da range:")
    for category, percentage in data.items():
        print(f"{category}: {percentage:.1f}%")

# Print DataFrame information
for name, df in [
    ("PC_0_250_pos", PC_0_250_pos_df),
    ("PC_0_250_neg", PC_0_250_neg_df),
    ("PC_250_350_pos", PC_250_350_pos_df),
    ("PC_250_350_neg", PC_250_350_neg_df),
    ("PC_350_500_pos", PC_350_500_pos_df),
    ("PC_350_500_neg", PC_350_500_neg_df),
    ("ZINC_250_350_pos", ZINC_250_350_pos_df),
    ("ZINC_250_350_neg", ZINC_250_350_neg_df)
]:
    print(f"\n{name} DataFrame:")
    print(df.info())
    print(df.head())

print(f"\nFigure saved to: {save_path}")"""

##### Save 100 neg molecules

In [ ]:
import pickle
import pandas as pd
import os

# File paths
negative_files = {
    'PC_0_250': os.path.abspath("./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/CSV_examples/PC_0-250_negativevector_v2.pkl"),
    'PC_250_350': os.path.abspath("./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/CSV_examples/PC_250-350_negativevector_v2.pkl"),
    'PC_350_500': os.path.abspath("./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/CSV_examples/PC_350-500_negativevector_v2.pkl"),
    'ZINC_250_350': os.path.abspath("./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/CSV_examples/ZINC_250-350_negativevector_v2.pkl")
}

def load_pickle(file_path):
    with open(file_path, 'rb') as file:
        return pickle.load(file)

def sample_and_save(file_path, sample_size=100):
    # Load the DataFrame
    df = load_pickle(file_path)
    
    # Sample 100 rows
    sampled_df = df.sample(n=sample_size, random_state=42)
    
    # Create the new file name
    dir_path = os.path.dirname(file_path)
    file_name = os.path.basename(file_path)
    new_file_name = file_name.replace('.pkl', '_100_neg.csv')
    new_file_path = os.path.join(dir_path, new_file_name)
    
    # Save the sampled DataFrame as CSV
    sampled_df.to_csv(new_file_path, index=False)
    
    print(f"Saved {sample_size} samples to: {new_file_path}")

# Process each negative file
for name, file_path in negative_files.items():
    print(f"Processing {name}...")
    sample_and_save(file_path)

print("All files processed and saved.")

In [ ]:
import pandas as pd
import os

# File paths
csv_files = [
    os.path.abspath("./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/CSV_examples_2/PC_0-250_negativevector_v2_100_neg.csv"),
    os.path.abspath("./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/CSV_examples_2/PC_250-350_negativevector_v2_100_neg.csv"),
    os.path.abspath("./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/CSV_examples_2/PC_350-500_negativevector_v2_100_neg.csv"),
    os.path.abspath("./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/CSV_examples_2/ZINC_250-350_negativevector_v2_100_neg.csv")
]

def add_sample_id(file_path):
    # Load the CSV file
    df = pd.read_csv(file_path)
    
    # Create the sample-id column
    df['sample-id'] = [f'NEG{i:05d}' for i in range(1, len(df) + 1)]
    
    # Move the sample-id column to the first position
    cols = df.columns.tolist()
    cols = ['sample-id'] + [col for col in cols if col != 'sample-id']
    df = df[cols]
    
    # Save the updated DataFrame back to CSV
    df.to_csv(file_path, index=False)
    
    print(f"Added sample-id column to: {file_path}")
    print(f"First 5 rows of the updated file:")
    print(df.head())
    print("\n")

# Process each CSV file
for file_path in csv_files:
    add_sample_id(file_path)

print("All files processed and updated with sample-id column.")

#### 4.5 Use 100 of each failed categories to do the improvment cycle

In [ ]:
def should_skip_chunk(base_path, chunk_idx):
    """
    Check if a chunk folder exists and contains sufficient processed data,
    ignoring timestamps in folder names.
    
    Args:
        base_path (str): Base directory path
        chunk_idx (int): Index of the chunk to check
        
    Returns:
        bool: True if the chunk should be skipped (already processed), False otherwise
    """
    import os
    
    # Format the chunk prefix to match (ensuring 3 digits with leading zeros)
    chunk_prefix = f"chunk_{chunk_idx:03d}_"
    
    # Get all directories in base_path
    try:
        all_dirs = [d for d in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, d))]
        
        # Find matching chunk folder (ignoring timestamp)
        matching_chunks = [d for d in all_dirs if d.startswith(chunk_prefix)]
        
        if not matching_chunks:
            return False
            
        # Use the first matching chunk folder found
        chunk_folder = matching_chunks[0]
        chunk_path = os.path.join(base_path, chunk_folder)
        
        # Check if there are any run folders inside the chunk folder
        run_folders = [d for d in os.listdir(chunk_path) if os.path.isdir(os.path.join(chunk_path, d))]
        if not run_folders:
            return False
        
        # Check if at least one run folder contains more than 5 files
        for run_folder in run_folders:
            run_path = os.path.join(chunk_path, run_folder)
            files = [f for f in os.listdir(run_path) if os.path.isfile(os.path.join(run_path, f))]
            if len(files) > 5:
                return True
                
        return False
        
    except Exception as e:
        print(f"Error checking chunk {chunk_idx}: {str(e)}")
        return False


def main_IC_neg(chunk_size, config, IR_config, stoi, itos, stoi_MF, itos_MF, num_training_runs=3):
    chunks = icne.split_dataset(config, chunk_size)
    config.model_save_dir = config.pkl_save_folder
    model_save_dir_backup = config.model_save_dir
    original_checkpoint_path = config.checkpoint_path  # Store the original checkpoint path

    for chunk_idx, chunk in enumerate(chunks):
        print(f"Processing chunk {chunk_idx+1} of {len(chunks)}")
        
        if should_skip_chunk(config.pkl_save_folder, chunk_idx):
            print(f"Skipping chunk {chunk_idx+1} as it appears to be already processed")
            continue
            
        # If we reach here, either the chunk doesn't exist or is incomplete
        # Find and delete any existing incomplete chunk folder
        chunk_prefix = f"chunk_{chunk_idx:03d}_"
        try:
            all_dirs = [d for d in os.listdir(config.pkl_save_folder) if os.path.isdir(os.path.join(config.pkl_save_folder, d))]
            matching_chunks = [d for d in all_dirs if d.startswith(chunk_prefix)]
            if matching_chunks:
                chunk_to_delete = os.path.join(config.pkl_save_folder, matching_chunks[0])
                print(f"Removing incomplete chunk folder: {matching_chunks[0]}")
                shutil.rmtree(chunk_to_delete)
        except Exception as e:
            print(f"Error while trying to delete incomplete chunk folder: {str(e)}")
            
            
        chunk_folder = icne.create_chunk_folder(config, chunk_idx)
        config.current_chunk_folder = chunk_folder
            
        config.blank_percentage = 0
        config = icne.test_pretrained_model_on_sim_data_before(config, IR_config, stoi, itos, stoi_MF, itos_MF, chunk, f"{chunk_idx}_{0}")
        print(config.csv_1H_path_SGNN)
        for run_idx in range(num_training_runs):
            print(f"Starting training run {run_idx+1} of {num_training_runs}")
            
            run_folder = icne.create_run_folder(config.current_chunk_folder, f"{chunk_idx}_{run_idx}")
            config.current_run_folder = run_folder
            config.model_save_dir = run_folder

            config.blank_percentage = 50
            config, aug_mol_df, all_gen_smis = icne.fine_tune_model_aug_mol(config, IR_config, stoi, stoi_MF, chunk, f"{chunk_idx}_{run_idx}")
            #import IPython; IPython.embed();

            ### Retrun the labelling of the smiles to test it on the correct one
            #chunk.rename(columns={'SMILES': 'SMILES_regio_isomers', 'SMILES_orig': 'SMILES'}, inplace=True)

            config.blank_percentage = 0
            config = icne.setup_data_paths(config)
            config = icne.test_model_on_neg_dataset(config, IR_config, stoi, itos, stoi_MF, itos_MF, chunk, f"{chunk_idx}_{run_idx}", aug_mol_df, all_gen_smis)

            config.checkpoint_path = original_checkpoint_path
        
        print(f"Chunk {chunk_idx+1} completed. All training runs finished.")
        config.model_save_dir = model_save_dir_backup


In [ ]:
### RUN 
config.checkpoint_path = os.path.abspath("./models/mmst/base_models/1_0_V8i_MMTi_RAW_DROP_Loss_0.112.ckpt")
config.pkl_save_folder = os.path.abspath("./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/IC_of_neg_examples/PC_350_500_neg")
config.SGNN_csv_gen_smi = os.path.abspath("./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/CSV_examples/PC_350-500_negativevector_v2_100_neg.csv")

config.MF_generations = 30 #50
config.MF_delta_weight = 100
config.max_scaffold_generations = 300
config.blank_percentage = 50
config.weight_MW = 100
config.lr_pretraining = 3e-4
config.tr_te_split = 0.9
config.batch_size = 64
config.num_epochs = 30
config.temperature = 1
config.multinom_runs = 20 #20
config.train_data_blend = 0

chunk_size = 1

main_IC_neg(chunk_size, config, IR_config, stoi, itos, stoi_MF, itos_MF, 1)

In [ ]:
import os
def analyze_chunks(base_path):
    """
    Analyze all chunk folders to find those that don't meet requirements.
    
    Args:
        base_path (str): Path to the directory containing chunk folders
    """
    incomplete_chunks = []
    try:
        # Get all directories in base_path
        all_dirs = [d for d in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, d))]
        chunk_dirs = [d for d in all_dirs if d.startswith('chunk_')]
        
        for chunk_dir in chunk_dirs:
            chunk_path = os.path.join(base_path, chunk_dir)
            
            # Check run folders
            run_folders = [d for d in os.listdir(chunk_path) if os.path.isdir(os.path.join(chunk_path, d))]
            
            has_sufficient_files = False
            for run_folder in run_folders:
                run_path = os.path.join(chunk_path, run_folder)
                files = [f for f in os.listdir(run_path) if os.path.isfile(os.path.join(run_path, f))]
                if len(files) > 5:
                    has_sufficient_files = True
                    break
            
            if not has_sufficient_files:
                incomplete_chunks.append(chunk_dir)
        
        print(f"\nTotal chunks analyzed: {len(chunk_dirs)}")
        print(f"Number of incomplete chunks: {len(incomplete_chunks)}")
        print("\nIncomplete chunks:")
        for chunk in incomplete_chunks:
            print(f"- {chunk}")
            
    except Exception as e:
        print(f"Error during analysis: {str(e)}")

# Run the analysis
base_path = os.path.abspath("./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/IC_of_neg_examples/ZINC_250_350_neg")
analyze_chunks(base_path)

In [ ]:
# delete before files
import os

def delete_before_files(base_path):
    """
    Delete all files containing 'before' in their name from the directory and its subdirectories.
    
    Args:
        base_path (str): Path to the directory to clean
    """
    count = 0
    try:
        # Walk through all directories and subdirectories
        for root, dirs, files in os.walk(base_path):
            # Find files containing 'before'
            before_files = [f for f in files if 'before' in f.lower()]
            
            # Delete each file
            for file in before_files:
                file_path = os.path.join(root, file)
                try:
                    os.remove(file_path)
                    print(f"Deleted: {file_path}")
                    count += 1
                except Exception as e:
                    print(f"Error deleting {file_path}: {str(e)}")
        
        print(f"\nTotal files deleted: {count}")
        
    except Exception as e:
        print(f"Error walking through directory: {str(e)}")

# Run the deletion
base_path = os.path.abspath("./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/IC_of_neg_examples/ZINC_250_350_neg")
delete_before_files(base_path)

##### Calculate pos/neg after FT

In [ ]:
data_configs = [
        {
            'weight_range': 'PC_0-250_neg_100_IC',
            'pkl_folder': os.path.abspath('./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/IC_of_neg_examples/PC_0_250_neg_2'),
            #'file_path': '/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/47_Anna_paper_data/PubChem_vectors/ML_NMR_2M_XL_1H_V1_test_f_0_250_x1000_Vectors_v2.pkl'
        },
        {
            'weight_range': 'PC_250-350_neg_100_IC',
            'pkl_folder': os.path.abspath('./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/IC_of_neg_examples/PC_250_350_neg'),
            #'file_path': '/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/47_Anna_paper_data/PubChem_vectors/ML_NMR_2M_XL_1H_V1_test_f_0_250_x1000_Vectors_v2.pkl'
        },
        {
            'weight_range': 'PC_350-500_neg_100_IC',
            'pkl_folder': os.path.abspath('./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/IC_of_neg_examples/PC_350_500_neg'),
            #'file_path': '/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/47_Anna_paper_data/PubChem_vectors/ML_NMR_2M_XL_1H_V1_test_f_0_250_x1000_Vectors_v2.pkl'
        },
        {
            'weight_range': 'ZINC_250-350_neg_100_IC',
            'pkl_folder': os.path.abspath('./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/IC_of_neg_examples/ZINC_250_350_neg'),
            #'file_path': '/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/47_Anna_paper_data/PubChem_vectors/ML_NMR_2M_XL_1H_V1_test_f_0_250_x1000_Vectors_v2.pkl'
        },    
    ]

    
output_folder = os.path.abspath('./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/IC_of_neg_examples/CSV_examples_2')
ranking_method = 'HSQC'

#main_csv_generation(data_configs, output_folder, ranking_method)


##### Plot correct, incorrect, failed after IC

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# File paths
base_path = os.path.abspath("./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/IC_of_neg_examples/CSV_examples_2")

def load_and_process_data(weight_range):
    pos_file = f"{base_path}/PC_{weight_range}_neg_100_IC_positive.csv"
    neg_file = f"{base_path}/PC_{weight_range}_neg_100_IC_negative.csv"
    
    if weight_range == "ZINC_250-350":
        pos_file = f"{base_path}/ZINC_250-350_neg_100_IC_positive.csv"
        neg_file = f"{base_path}/ZINC_250-350_neg_100_IC_negative.csv"
    
    pos_df = pd.read_csv(pos_file)
    neg_df = pd.read_csv(neg_file)
    
    total = 100  # Total molecules per set
    correct = len(pos_df)
    incorrect = len(neg_df)
    failed = total - (correct + incorrect)
    
    return {
        'Correct': (correct/total) * 100,
        'Incorrect': (incorrect/total) * 100,
        'Failed': (failed/total) * 100
    }

# Process data for each set
data_0_250 = load_and_process_data("0-250")
data_250_350 = load_and_process_data("250-350")
data_350_500 = load_and_process_data("350-500")
data_zinc = load_and_process_data("ZINC_250-350")

# Prepare plotting data
plot_data = {
    'Correct': [data_0_250['Correct'], data_250_350['Correct'], data_350_500['Correct'], data_zinc['Correct']],
    'Incorrect': [data_0_250['Incorrect'], data_250_350['Incorrect'], data_350_500['Incorrect'], data_zinc['Incorrect']],
    'Failed': [data_0_250['Failed'], data_250_350['Failed'], data_350_500['Failed'], data_zinc['Failed']]
}

def create_stacked_bar_chart(data, categories, title, save_path):
    fontsize = 22
    colors = [
        '#8BE5A0',  # mint green for Failed
        '#A1C8F3',  # light blue for Correct
        '#FFB381',  # salmon for Incorrect
    ]
    
    fig, ax = plt.subplots(figsize=(8, 7))
    bottom = np.zeros(4)
    
    bars = []
    for i, category in enumerate(categories):
        values = data[category]
        bar = ax.bar(range(4), values, bottom=bottom, label=category, color=colors[i], edgecolor='black')
        for patch in bar:
            patch.set_edgecolor('black')
        bottom += values
        bars.append(bar)
    
    ax.set_title(title, fontsize=fontsize, pad=20)
    ax.set_xticks(range(4))
    ax.set_xticklabels(['Set 1', 'Set 2', 'Set 3', 'ZINC'], fontsize=fontsize)
    ax.set_ylabel('Percentage', fontsize=fontsize)
    ax.set_ylim(0, 100)
    
    ax.tick_params(axis='y', labelsize=fontsize)
    
    ax.legend(loc='lower right', fontsize=fontsize)
    
    for i in ax.containers:
        ax.bar_label(i, fmt='%.1f%%', label_type='center', fontsize=fontsize)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    return fig

# Create and save the plot
save_path = "./_FIGURES/IC_Results.png"
categories = ['Correct', 'Incorrect', 'Failed']
fig = create_stacked_bar_chart(plot_data, categories, 'Internal Coordinate Results', save_path)

# Print the percentages for verification
for i, dataset in enumerate(['Set 1 (0-250)', 'Set 2 (250-350)', 'Set 3 (350-500)', 'ZINC']):
    print(f"\n{dataset}:")
    for category in categories:
        print(f"{category}: {plot_data[category][i]:.1f}%")

##### Plot Top 1, 3, 5, 10 Accuracy of IC

In [ ]:
## PC 0-250
import numpy as np
import matplotlib.pyplot as plt

ranking_method = 'HSQC'
pkl_folder= os.path.abspath("./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/IC_of_neg_examples/PC_0_250_neg_2")
all_rankings = exp_func.process_pkl_files_baseline(pkl_folder, ranking_method)

all_rankings, removed_smiles = exp_func.deduplicate_smiles_from_ranking(all_rankings)
all_rankings, filtered_out_rankings = exp_func.filter_rankings_by_molecular_formula(all_rankings)
accuracies = exp_func.calculate_top_k_accuracy(all_rankings)


# Increase default font sizes
plt.rcParams.update({'font.size': 22})  # Base font size
plt.rcParams['axes.titlesize'] = 22
plt.rcParams['axes.labelsize'] = 22
plt.rcParams['xtick.labelsize'] = 22
plt.rcParams['ytick.labelsize'] = 22

# Data
labels = ['Top 1', 'Top 3', 'Top 5', 'Top 10', 'Top 20', "All"]

# Calculate molecules for each accuracy (based on 100 total molecules)
total_molecules = 100
molecules = [int(acc * total_molecules) for acc in accuracies]
all_labels = labels

# Different colors for each bar
colors = ['#A1C8F3',  # light blue/periwinkle
    '#FFB381',  # salmon/peach
    '#8BE5A0',  # mint green
    '#FF9D9A',  # coral pink
    '#D1B9FE',  # lavender
    '#DEBA9A',  # beige/tan
    '#FCAEE3',  # pink
    '#CFCECE',  # light gray
    '#FEFDA2',  # pale yellow
    '#B8F1EF',  # light cyan/aqua
         ]

# Create the bar plot
fig, ax = plt.subplots(figsize=(8, 7))
bars = ax.bar(range(len(molecules)), molecules, color=colors)

# Set black edges for all bars
for bar in bars:
    bar.set_edgecolor('black')
    
# Customize the plot
ax.set_xticks(range(len(all_labels)))
ax.set_xticklabels(all_labels, fontsize=22, rotation=45)
ax.set_ylabel('Number of Molecules', fontsize=22)
ax.set_title('Prediction Accuracy of Set 1 (Molecules: 100)', fontsize=22, pad=20)

# Add value labels on top and inside of each bar
for i, bar in enumerate(bars):
    height = bar.get_height()
    # Percentage on top
    percentage = accuracies[i] * 100
    ax.text(bar.get_x() + bar.get_width()/2., height + 2,
            f'{percentage:.1f}%',
            ha='center', va='bottom', fontsize=22)
    # Number inside bar
    ax.text(bar.get_x() + bar.get_width()/2., height/2,
            f'{int(height)}',
            ha='center', va='center', fontsize=22, color='black')

# Add grid for better readability
ax.grid(True, axis='y', linestyle='--', alpha=0.7)

# Set y-axis limit to accommodate labels
ax.set_ylim(0, 105)

# Adjust layout to prevent label cutoff
plt.tight_layout()

# Save the plot
save_path = os.path.abspath("./_FIGURES/PC_0_250_neg_IC_accuracy_plot.png")
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"Plot saved to: {save_path}")

In [ ]:
## PC 250-350
import numpy as np
import matplotlib.pyplot as plt

ranking_method = 'HSQC'
pkl_folder= os.path.abspath("./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/IC_of_neg_examples/PC_250_350_neg")
all_rankings = exp_func.process_pkl_files_baseline(pkl_folder, ranking_method)

all_rankings, removed_smiles = exp_func.deduplicate_smiles_from_ranking(all_rankings)
all_rankings, filtered_out_rankings = exp_func.filter_rankings_by_molecular_formula(all_rankings)
accuracies = exp_func.calculate_top_k_accuracy(all_rankings)


# Increase default font sizes
plt.rcParams.update({'font.size': 22})  # Base font size
plt.rcParams['axes.titlesize'] = 22
plt.rcParams['axes.labelsize'] = 22
plt.rcParams['xtick.labelsize'] = 22
plt.rcParams['ytick.labelsize'] = 22

# Data
labels = ['Top 1', 'Top 3', 'Top 5', 'Top 10', 'Top 20', "All"]

# Calculate molecules for each accuracy (based on 100 total molecules)
total_molecules = 100
molecules = [int(acc * total_molecules) for acc in accuracies]
all_labels = labels

# Different colors for each bar
colors = ['#A1C8F3',  # light blue/periwinkle
    '#FFB381',  # salmon/peach
    '#8BE5A0',  # mint green
    '#FF9D9A',  # coral pink
    '#D1B9FE',  # lavender
    '#DEBA9A',  # beige/tan
    '#FCAEE3',  # pink
    '#CFCECE',  # light gray
    '#FEFDA2',  # pale yellow
    '#B8F1EF',  # light cyan/aqua
         ]

# Create the bar plot
fig, ax = plt.subplots(figsize=(8, 7))
bars = ax.bar(range(len(molecules)), molecules, color=colors)

# Set black edges for all bars
for bar in bars:
    bar.set_edgecolor('black')
    
# Customize the plot
ax.set_xticks(range(len(all_labels)))
ax.set_xticklabels(all_labels, fontsize=22, rotation=45)
ax.set_ylabel('Number of Molecules', fontsize=22)
ax.set_title('Prediction Accuracy of Set 2 (Molecules: 100)', fontsize=22, pad=20)

# Add value labels on top and inside of each bar
for i, bar in enumerate(bars):
    height = bar.get_height()
    # Percentage on top
    percentage = accuracies[i] * 100
    ax.text(bar.get_x() + bar.get_width()/2., height + 2,
            f'{percentage:.1f}%',
            ha='center', va='bottom', fontsize=22)
    # Number inside bar
    ax.text(bar.get_x() + bar.get_width()/2., height/2,
            f'{int(height)}',
            ha='center', va='center', fontsize=22, color='black')

# Add grid for better readability
ax.grid(True, axis='y', linestyle='--', alpha=0.7)

# Set y-axis limit to accommodate labels
ax.set_ylim(0, 105)

# Adjust layout to prevent label cutoff
plt.tight_layout()

# Save the plot
save_path = os.path.abspath("./_FIGURES/PC_250_350_neg_IC_accuracy_plot.png")
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"Plot saved to: {save_path}")

In [ ]:
all_rankings

In [ ]:
## PC 350-500
import numpy as np
import matplotlib.pyplot as plt

ranking_method = 'HSQC'
pkl_folder= os.path.abspath("./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/IC_of_neg_examples/PC_350_500_neg")
all_rankings = exp_func.process_pkl_files_baseline(pkl_folder, ranking_method)

all_rankings, removed_smiles = exp_func.deduplicate_smiles_from_ranking(all_rankings)
all_rankings, filtered_out_rankings = exp_func.filter_rankings_by_molecular_formula(all_rankings)
accuracies = exp_func.calculate_top_k_accuracy(all_rankings)


# Increase default font sizes
plt.rcParams.update({'font.size': 22})  # Base font size
plt.rcParams['axes.titlesize'] = 22
plt.rcParams['axes.labelsize'] = 22
plt.rcParams['xtick.labelsize'] = 22
plt.rcParams['ytick.labelsize'] = 22

# Data
labels = ['Top 1', 'Top 3', 'Top 5', 'Top 10', 'Top 20', "All"]

# Calculate molecules for each accuracy (based on 100 total molecules)
total_molecules = 100
molecules = [int(acc * total_molecules) for acc in accuracies]
all_labels = labels

# Different colors for each bar
colors = ['#A1C8F3',  # light blue/periwinkle
    '#FFB381',  # salmon/peach
    '#8BE5A0',  # mint green
    '#FF9D9A',  # coral pink
    '#D1B9FE',  # lavender
    '#DEBA9A',  # beige/tan
    '#FCAEE3',  # pink
    '#CFCECE',  # light gray
    '#FEFDA2',  # pale yellow
    '#B8F1EF',  # light cyan/aqua
         ]

# Create the bar plot
fig, ax = plt.subplots(figsize=(8, 7))
bars = ax.bar(range(len(molecules)), molecules, color=colors)

# Set black edges for all bars
for bar in bars:
    bar.set_edgecolor('black')
    
# Customize the plot
ax.set_xticks(range(len(all_labels)))
ax.set_xticklabels(all_labels, fontsize=22, rotation=45)
ax.set_ylabel('Number of Molecules', fontsize=22)
ax.set_title('Prediction Accuracy of Set 3 (Molecules: 100)', fontsize=22, pad=20)

# Add value labels on top and inside of each bar
for i, bar in enumerate(bars):
    height = bar.get_height()
    # Percentage on top
    percentage = accuracies[i] * 100
    ax.text(bar.get_x() + bar.get_width()/2., height + 2,
            f'{percentage:.1f}%',
            ha='center', va='bottom', fontsize=22)
    # Number inside bar
    ax.text(bar.get_x() + bar.get_width()/2., height/2,
            f'{int(height)}',
            ha='center', va='center', fontsize=22, color='black')

# Add grid for better readability
ax.grid(True, axis='y', linestyle='--', alpha=0.7)

# Set y-axis limit to accommodate labels
ax.set_ylim(0, 105)

# Adjust layout to prevent label cutoff
plt.tight_layout()

# Save the plot
save_path = os.path.abspath("./_FIGURES/PC_350_500_neg_IC_accuracy_plot.png")
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"Plot saved to: {save_path}")

In [ ]:
## ZINC 0-250
import numpy as np
import matplotlib.pyplot as plt

ranking_method = 'HSQC'
pkl_folder= os.path.abspath("./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/IC_of_neg_examples/ZINC_250_350_neg")
all_rankings = exp_func.process_pkl_files_baseline(pkl_folder, ranking_method)

all_rankings, removed_smiles = exp_func.deduplicate_smiles_from_ranking(all_rankings)
all_rankings, filtered_out_rankings = exp_func.filter_rankings_by_molecular_formula(all_rankings)
accuracies = exp_func.calculate_top_k_accuracy(all_rankings)


# Increase default font sizes
plt.rcParams.update({'font.size': 22})  # Base font size
plt.rcParams['axes.titlesize'] = 22
plt.rcParams['axes.labelsize'] = 22
plt.rcParams['xtick.labelsize'] = 22
plt.rcParams['ytick.labelsize'] = 22

# Data
labels = ['Top 1', 'Top 3', 'Top 5', 'Top 10', 'Top 20', "All"]

# Calculate molecules for each accuracy (based on 100 total molecules)
total_molecules = 100
molecules = [int(acc * total_molecules) for acc in accuracies]
all_labels = labels

# Different colors for each bar
colors = ['#A1C8F3',  # light blue/periwinkle
    '#FFB381',  # salmon/peach
    '#8BE5A0',  # mint green
    '#FF9D9A',  # coral pink
    '#D1B9FE',  # lavender
    '#DEBA9A',  # beige/tan
    '#FCAEE3',  # pink
    '#CFCECE',  # light gray
    '#FEFDA2',  # pale yellow
    '#B8F1EF',  # light cyan/aqua
         ]

# Create the bar plot
fig, ax = plt.subplots(figsize=(8, 7))
bars = ax.bar(range(len(molecules)), molecules, color=colors)

# Set black edges for all bars
for bar in bars:
    bar.set_edgecolor('black')
    
# Customize the plot
ax.set_xticks(range(len(all_labels)))
ax.set_xticklabels(all_labels, fontsize=22, rotation=45)
ax.set_ylabel('Number of Molecules', fontsize=22)
ax.set_title('Prediction Accuracy of ZINC (Molecules: 100)', fontsize=22, pad=20)

# Add value labels on top and inside of each bar
for i, bar in enumerate(bars):
    height = bar.get_height()
    # Percentage on top
    percentage = accuracies[i] * 100
    ax.text(bar.get_x() + bar.get_width()/2., height + 2,
            f'{percentage:.1f}%',
            ha='center', va='bottom', fontsize=22)
    # Number inside bar
    ax.text(bar.get_x() + bar.get_width()/2., height/2,
            f'{int(height)}',
            ha='center', va='center', fontsize=22, color='black')

# Add grid for better readability
ax.grid(True, axis='y', linestyle='--', alpha=0.7)

# Set y-axis limit to accommodate labels
ax.set_ylim(0, 105)

# Adjust layout to prevent label cutoff
plt.tight_layout()

# Save the plot
save_path = os.path.abspath("./_FIGURES/ZINC_250_350_neg_IC_accuracy_plot.png")
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"Plot saved to: {save_path}")

#### 4.5 Bar chart of ESI 7 HSQC Matching results ZINC 100

In [ ]:

# V8i Raw 
config.csv_train_path = os.path.abspath('./data/ZINK_dataset/ML_NMR_5M_XL_1H_comb_train_V8.csv') 
config.csv_1H_path_SGNN = os.path.abspath('./data/ZINK_dataset/val_data_all_modalities/ML_NMR_1H_combined_ZINC_test_10x100.csv')
config.csv_13C_path_SGNN = os.path.abspath('./data/ZINK_dataset/val_data_all_modalities/ML_NMR_5M_XL_13C_test_10x100.csv')    
config.csv_HSQC_path_SGNN = os.path.abspath('./data/ZINK_dataset/val_data_all_modalities/ML_NMR_5M_XL_HSQC_test_10x100.csv')    
config.csv_COSY_path_SGNN = os.path.abspath('./data/ZINK_dataset/val_data_all_modalities/ML_NMR_5M_XL_COSY_test_10x100.csv')  
config.csv_path_val = os.path.abspath('./data/ZINK_dataset/val_data_all_modalities/ML_NMR_1H_combined_ZINC_test_10x100.csv')
config.pickle_file_path = os.path.abspath('./data/ZINK_dataset/val_data_all_modalities/ML_NMR_1H_combined_ZINC_test_10x100_955629.pkl')
config.IR_data_folder = os.path.abspath('./data/ZINK_dataset/IR_spectra_NN')


config.checkpoint_path = os.path.abspath("./models/mmst/base_models/1_0_V8i_MMTi_RAW_DROP_Loss_0.112.ckpt")
config.multinom_runs = 10
#config.multinom_runs = 3
config.temperature = 1
greedy_full = False
MW_filter = True
config.data_size = 100 # config.test_size # why would I do that? 

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
config.execution_type = "test_performance"
if config.execution_type == "test_performance":
    print("\033[1m\033[31mThis is: test_performance\033[0m")
    # config.csv_path_val = config.csv_SMI_targets  #this already got updated in simulate_syn_data
    
    model_MMT = mrtf.load_MMT_model(config)
    model_CLIP = mrtf.load_CLIP_model(config)
    #model_BLIP = mrtf.load_BLIP_model(config)
    #model_MMT = model_CLIP.MT_model
    val_dataloader = mrtf.load_data(config, stoi, stoi_MF, single=True, mode="val")
    val_dataloader_multi = mrtf.load_data(config, stoi, stoi_MF, single=False, mode="val")

    
    results_dict_bl_ZINC = mrtf.run_test_mns_performance_CLIP_3(config,  
                                                        model_MMT,
                                                        model_CLIP,
                                                        val_dataloader,
                                                        stoi, 
                                                        itos,
                                                        MW_filter)
    results_dict_bl_ZINC, counter = mrtf.filter_invalid_inputs(results_dict_bl_ZINC)
    
    avg_tani_bl_ZINC, html_plot = rbgvm.plot_hist_of_results(results_dict_bl_ZINC)
    
    # Slow because also just takes one at the time
    if greedy_full == True:
        results_dict_greedy_bl_ZINC, failed_bl_ZINC = mrtf.run_test_performance_CLIP_greedy_3(config,  
                                                                stoi, 
                                                                stoi_MF, 
                                                                itos, 
                                                                itos_MF)

        avg_tani_greedy_bl_ZINC, html_plot_greedy = rbgvm.plot_hist_of_results_greedy(results_dict_greedy_bl_ZINC)

    else: 
        config, results_dict_ZINC_greedy_bl = mrtf.run_greedy_sampling(config, model_MMT, val_dataloader_multi, itos, stoi)
        avg_tani_greedy_bl_ZINC = results_dict_ZINC_greedy_bl["tanimoto_mean"]
    
    total_results_bl_ZINC = mrtf.run_test_performance_CLIP_3(config, 
                                                        model_MMT, 
                                                        val_dataloader,
                                                        stoi)
    
    corr_sampleing_prob_bl_ZINC = total_results_bl_ZINC["statistics_multiplication_avg"][0]
    print("avg_tani, avg_tani_greedy, corr_sampleing_prob'")
    print(avg_tani_bl_ZINC, avg_tani_greedy_bl_ZINC, corr_sampleing_prob_bl_ZINC)       

    

In [ ]:

# Save variables to a pickle file
variables_to_save = {
    'avg_tani_bl_ZINC': avg_tani_bl_ZINC,
    'results_dict_greedy_bl_ZINC': results_dict_greedy_bl_ZINC if greedy_full else None,
    'failed_bl_ZINC': failed_bl_ZINC if greedy_full else None,
    'avg_tani_greedy_bl_ZINC': avg_tani_greedy_bl_ZINC,
    'results_dict_ZINC_greedy_bl': results_dict_ZINC_greedy_bl if not greedy_full else None,
    'total_results_bl_ZINC': total_results_bl_ZINC,
    'corr_sampleing_prob_bl_ZINC': corr_sampleing_prob_bl_ZINC,
    'results_dict_bl_ZINC': results_dict_bl_ZINC,
}

with open(os.path.abspath('./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/20240522_ZINC_MNS_HSQC_matching/7.0_imp_cyc_all_100_10_after.pkl'), 'wb') as f:
    pickle.dump(variables_to_save, f)


In [ ]:
file_path = os.path.abspath('./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/20240522_ZINC_MNS_HSQC_matching/7.0_imp_cyc_all_100_10_after.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    variables_to_save = pickle.load(file)  

In [ ]:
results_dict_bl_ZINC = variables_to_save["results_dict_bl_ZINC"]

In [ ]:
# Function to check if the lowest error correctly identifies the correct molecule
def check_lowest_error_correct(results):
    correct_identifications = 0
    total_cases_with_correct_answer = 0
    total_cases_without_correct_answer = 0
    closest_identifications = 0
    incorrect_identification_keys = []

    for key, value in results.items():
        if not isinstance(value, list) or not isinstance(value[0], list):
            continue  # Skip non-list items

        for sublist in value:
            if not isinstance(sublist, list):
                continue  # Skip non-list sublists

            correct_error = None
            lowest_error = float('inf')
            highest_similarity = 0
            closest_error = float('inf')

            for item in sublist:
                if not isinstance(item, list) or len(item) < 5:
                    continue  # Skip invalid items

                similarity = item[3]
                error = item[4]

                if isinstance(similarity, (int, float)) and similarity == 1:
                    correct_error = float(error)
                if isinstance(error, (int, float)) and float(error) < lowest_error:
                    lowest_error = float(error)
                if isinstance(similarity, (int, float)) and similarity > highest_similarity:
                    highest_similarity = similarity
                    closest_error = float(error)

            if correct_error is not None:
                total_cases_with_correct_answer += 1
                if correct_error == lowest_error:
                    correct_identifications += 1
                else:
                    incorrect_identification_keys.append(key)
            else:
                total_cases_without_correct_answer += 1
                if closest_error == lowest_error:
                    closest_identifications += 1

    return (correct_identifications, total_cases_with_correct_answer, 
            closest_identifications, total_cases_without_correct_answer,
            incorrect_identification_keys)

# Run the function
correct_identifications, total_cases_with_correct_answer, closest_identifications, total_cases_without_correct_answer, incorrect_identification_keys = check_lowest_error_correct(results_dict_bl_ZINC)

# Calculate accuracies
accuracy_with_correct = correct_identifications / total_cases_with_correct_answer if total_cases_with_correct_answer > 0 else 0
accuracy_without_correct = closest_identifications / total_cases_without_correct_answer if total_cases_without_correct_answer > 0 else 0

# Print results
print(f"Correct identifications: {correct_identifications}")
print(f"Total cases with correct answer: {total_cases_with_correct_answer}")
print(f"Accuracy with correct answer: {accuracy_with_correct:.2%}")

print(f"Closest identifications: {closest_identifications}")
print(f"Total cases without correct answer: {total_cases_without_correct_answer}")
print(f"Accuracy without correct answer: {accuracy_without_correct:.2%}")

# Print keys of incorrect identifications
print(f"Keys of incorrect identifications: {incorrect_identification_keys}")


In [ ]:
total_cases_greedy

In [ ]:
import matplotlib.pyplot as plt

tani_list = variables_to_save["results_dict_ZINC_greedy_bl"]["tanimoto_sim"]
greedy_correct = tani_list.count(1)
total_cases = len(tani_list)
accuracy_greedy = greedy_correct / total_cases

# Bar chart for identification accuracies
accuracies = {
    'MNS \nSampling': accuracy_with_correct,
    'Greedy \nSampling': accuracy_greedy
}

# Ratios for annotation
ratios = {
    #'MNS': f'{correct_identifications}/{total_cases_with_correct_answer}',
    'MNS': f'{correct_identifications}/{total_cases}',
    'Greedy Sampling': f'{greedy_correct}/{total_cases}'
}

fig, ax = plt.subplots()
bars = ax.bar(accuracies.keys(), accuracies.values(), color=['#A1C8F3',  '#FFB381'])  # LightBlue, LightGreen

ax.set_ylabel('Accuracy')
ax.set_title('Identification Accuracies')
ax.set_ylim(0, 1)

# Annotate bars with accuracy percentages on top of the bar
for bar, (label, ratio) in zip(bars, ratios.items()):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, height,
            f'{height:.2%}',
            ha='center', va='bottom', fontsize=22)

# Annotate bars with ratios inside the bar
for bar, (label, ratio) in zip(bars, ratios.items()):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, height / 2,
            f'{ratio}',
            ha='center', va='center', fontsize=22, color='black')

plt.tight_layout()

# Define the file path for saving the plot
output_path = os.path.abspath('./_FIGURES/7.0_identification_accuracies_new_2.png')

# Save the plot
plt.savefig(output_path, bbox_inches='tight')
plt.show()

#### 4.7 ZINC 4000 MNS Experiment

In [ ]:
ranking_method = 'HSQC'
pkl_folder= os.path.abspath("./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/ZINC_250_350_4000")
all_rankings = exp_func.process_pkl_files_baseline(pkl_folder, ranking_method)

all_rankings, removed_smiles = exp_func.deduplicate_smiles_from_ranking(all_rankings)
all_rankings, filtered_out_rankings = exp_func.filter_rankings_by_molecular_formula(all_rankings)
accuracies = exp_func.calculate_top_k_accuracy(all_rankings)
accuracies = accuracies[:-2]

In [ ]:
accuracies_

In [ ]:
all_rankings

In [ ]:
accuracies

In [ ]:
data["results_dict_bl_ZINC"]

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import torch

def process_pkl_files_multinomial(folder_path):
    """
    Process pickle files and sort by multinomial probability (product of token probabilities)
    """
    pkl_files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) 
                 if f.endswith('.pkl')]
    
    all_molecules_multinomial = defaultdict(list)
    
    for file_path in pkl_files:
        file_data = exp_func.load_data_results(file_path)  
        try:
            molecules = extract_molecules_by_probability(file_data)
            trg_smi = molecules[0][0] if molecules else None
            if trg_smi:
                for molecule in molecules:
                    all_molecules_multinomial[trg_smi].append(molecule)
        except Exception as e:
            print(f"Error processing: {file_path} - {e}")
            continue
    
    print(f"Processed {len(all_molecules_multinomial)} unique molecules (multinomial probability)")
    return all_molecules_multinomial

def extract_molecules_by_probability(file_data):
    """
    Extract molecules and sort by multiplicative token probability (multinomial sampling order)
    """
    molecule_data = []
    
    for trg_smi, value_list in file_data.items():
        try:
            for sublist in value_list[0]:
                gen_smi = sublist[0]
                # Position 3 contains the tensor with token probabilities
                token_probs = sublist[3]  
                tanimoto = sublist[4]
                
                # Convert tensor to numpy if needed and calculate product
                if isinstance(token_probs, torch.Tensor):
                    token_probs_np = token_probs.cpu().numpy()
                else:
                    token_probs_np = np.array(token_probs)
                
                # Calculate multiplicative probability (product of all token probabilities)
                multiplicative_prob = np.prod(token_probs_np)
                
                # Store: (target_smi, generated_smi, tanimoto, multiplicative_probability)
                molecule_data.append((trg_smi, gen_smi, tanimoto, multiplicative_prob))
                
        except Exception as e:
            print(f"Error processing molecule data: {e}")
            pass
    
    # Sort by multiplicative probability (descending - highest probability first)
    # This represents the multinomial sampling order
    molecule_data.sort(key=lambda x: x[3], reverse=True)
    
    return molecule_data

def calculate_multinomial_top1_accuracy(all_molecules_multinomial):
    """
    Calculate top-1 accuracy for multinomial sampling (highest probability molecule)
    """
    total_molecules = len(all_molecules_multinomial)
    correct_count = 0
    
    for trg_smi, molecules in all_molecules_multinomial.items():
        if molecules:  # Check if there are any molecules
            # Take the molecule with highest multiplicative probability (first after sorting)
            top_molecule = molecules[0]
            if top_molecule[2] == 1.0:  # Check if Tanimoto = 1.0
                correct_count += 1
    
    accuracy = correct_count / total_molecules if total_molecules > 0 else 0
    return accuracy

def calculate_multinomial_topk_accuracy(all_molecules_multinomial, k_values=[1, 3, 5, 10]):
    """
    Calculate top-k accuracy for multinomial sampling
    """
    total_molecules = len(all_molecules_multinomial)
    accuracies = []
    
    for k in k_values:
        correct_count = 0
        for trg_smi, molecules in all_molecules_multinomial.items():
            if molecules:
                # Check top-k molecules (highest k probabilities)
                top_k_molecules = molecules[:k]
                if any(mol[2] == 1.0 for mol in top_k_molecules):
                    correct_count += 1
        
        accuracy = correct_count / total_molecules if total_molecules > 0 else 0
        accuracies.append(accuracy)
    
    return accuracies

def create_combined_accuracy_plot(multinomial_accuracies, ranked_accuracies, 
                                total_molecules=4000, processed_molecules=3992):
    """
    Create plot with multinomial sampling top-1 + ranked accuracies
    """
    # Increase default font sizes
    plt.rcParams.update({'font.size': 22})
    plt.rcParams['axes.titlesize'] = 22
    plt.rcParams['axes.labelsize'] = 22
    plt.rcParams['xtick.labelsize'] = 22
    plt.rcParams['ytick.labelsize'] = 22
    
    # Combine accuracies: only multinomial top-1, then ranked accuracies
    all_accuracies = [multinomial_accuracies[0]] + ranked_accuracies[:-2]  # Remove last 2 as in original
    
    # Labels
    labels = ['Multinomial\nTop 1', 'HSQC Ranked\nTop 1', 'HSQC Ranked\nTop 3', 
              'HSQC Ranked\nTop 5', 'HSQC Ranked\nTop 10']
    
    # Calculate molecules for each accuracy
    failed_molecules = total_molecules - processed_molecules
    molecules = [int(acc * processed_molecules) for acc in all_accuracies]
    all_molecules = molecules + [failed_molecules]
    all_labels = labels + ['Failed']
    
    # Different colors for each bar
    colors = ['#FFB366', '#FF9999', '#66B2FF', '#99FF99', '#FFCC99', '#FF99CC']
    
    # Create the bar plot
    fig, ax = plt.subplots(figsize=(14, 9))
    bars = ax.bar(range(len(all_molecules)), all_molecules, color=colors)
    
    # Set black edges for all bars
    for bar in bars:
        bar.set_edgecolor('black')
    
    # Customize the plot
    ax.set_xticks(range(len(all_labels)))
    ax.set_xticklabels(all_labels, fontsize=20, rotation=45, ha='right')
    ax.set_ylabel('Number of Molecules', fontsize=22)
    ax.set_title('Multinomial vs HSQC Ranked Prediction Accuracy (Total: 4000)', fontsize=22, pad=20)
    
    # Add value labels on top and inside of each bar
    for i, bar in enumerate(bars):
        height = bar.get_height()
        if i < len(all_accuracies):  # For accuracy bars
            percentage = all_accuracies[i] * 100
            # Percentage on top
            ax.text(bar.get_x() + bar.get_width()/2., height + 50,
                    f'{percentage:.1f}%',
                    ha='center', va='bottom', fontsize=20)
            # Number inside bar
            ax.text(bar.get_x() + bar.get_width()/2., height/2,
                    f'{int(height)}',
                    ha='center', va='center', fontsize=20, color='black')
        else:  # For failed molecules bar
            percentage = (height/total_molecules) * 100
            # Percentage on top - moved higher for failed bar
            ax.text(bar.get_x() + bar.get_width()/2., height + 200,
                    f'{percentage:.1f}%',
                    ha='center', va='bottom', fontsize=20)
            # Number inside bar - moved higher for failed bar
            ax.text(bar.get_x() + bar.get_width()/2., height + 100,
                    f'{int(height)}',
                    ha='center', va='center', fontsize=20, color='black')
    
    # Add grid for better readability
    ax.grid(True, axis='y', linestyle='--', alpha=0.7)
    # Set y-axis limit to accommodate labels
    ax.set_ylim(0, max(all_molecules) + 400)
    
    # Adjust layout to prevent label cutoff
    plt.tight_layout()
    
    # Save the plot
    save_path = os.path.abspath("./_FIGURES/ZINC4000_multinomial_vs_ranked_accuracy.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Plot saved to: {save_path}")
    
    return all_accuracies

def main_comparison(folder_path):
    """
    Main function to compare multinomial probability vs ranked accuracies
    """
    print("=== CALCULATING MULTINOMIAL PROBABILITY ACCURACIES ===")
    # Calculate multinomial accuracy (sorted by probability)
    all_molecules_multinomial = process_pkl_files_multinomial(folder_path)
    
    # Apply your existing deduplication and filtering if needed
    all_molecules_multinomial, removed_smiles = exp_func.deduplicate_smiles_from_ranking(all_molecules_multinomial)
    all_molecules_multinomial, filtered_out = exp_func.filter_rankings_by_molecular_formula(all_molecules_multinomial)
    
    multinomial_accuracies = calculate_multinomial_topk_accuracy(all_molecules_multinomial, [1, 3, 5, 10])
    
    print("=== CALCULATING HSQC RANKED ACCURACIES ===")
    # Calculate ranked accuracies (your existing code)
    all_rankings = exp_func.process_pkl_files_baseline(folder_path, 'HSQC')  # Your existing function
    all_rankings, removed_smiles = exp_func.deduplicate_smiles_from_ranking(all_rankings)
    all_rankings, filtered_out_rankings = exp_func.filter_rankings_by_molecular_formula(all_rankings)
    ranked_accuracies = exp_func.calculate_top_k_accuracy(all_rankings)  # Your existing function
    
    # Create combined plot
    all_accuracies = create_combined_accuracy_plot(multinomial_accuracies, ranked_accuracies)
    
    print("\n=== RESULTS COMPARISON ===")
    print(f"Multinomial Top-1 Accuracy: {multinomial_accuracies[0]:.3f}")
    print(f"HSQC Ranked Top-1 Accuracy: {ranked_accuracies[0]:.3f}")
    print(f"Top-1 Improvement: {(ranked_accuracies[0] - multinomial_accuracies[0]):.3f}")
    
    print(f"Multinomial Top-10 Accuracy: {multinomial_accuracies[3]:.3f}")
    print(f"HSQC Ranked Top-10 Accuracy: {ranked_accuracies[3]:.3f}")
    print(f"Top-10 Improvement: {(ranked_accuracies[3] - multinomial_accuracies[3]):.3f}")
    
    return multinomial_accuracies, ranked_accuracies

# Example usage:
if __name__ == "__main__":
    folder_path = os.path.abspath("./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/ZINC_250_350_4000")
    multinomial_acc, ranked_acc = main_comparison(folder_path)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pickle

def load_greedy_accuracy():
    """
    Load greedy accuracy from the 1% data file
    """
    file_path = './past_experiments/ChemXriv/2.0_Experiment_Ablation_Study/2.1_results_dict_2_ZINC_4000.pkl'
    
    try:
        full_path = os.path.abspath(file_path)
        with open(full_path, 'rb') as file:
            results_dict = pickle.load(file)
        
        # Extract Tanimoto similarity data - look for keys starting with "tanimoto_scores_"
        tanimoto_sim = []
        for key in results_dict.keys():
            if key.startswith("tanimoto_scores_"):
                tanimoto_sim = results_dict[key]
                print(f"Found Tanimoto data in key: {key}")
                break
        
        if not tanimoto_sim:
            print("No 'tanimoto_scores_*' key found, checking 'tanimoto_sim'")
            tanimoto_sim = results_dict.get("tanimoto_sim", [])
        
        if len(tanimoto_sim) > 0:
            # Count molecules with Tanimoto similarity = 1.0 (perfect matches)
            correct_count = sum(1 for sim in tanimoto_sim if sim == 1.0)
            total_count = len(tanimoto_sim)
            accuracy = correct_count / total_count if total_count > 0 else 0
            
            print(f"Greedy results - Total molecules: {total_count}")
            print(f"Greedy results - Correct molecules: {correct_count}")
            print(f"Greedy results - Accuracy: {accuracy:.3f}")
            
            return accuracy, correct_count, total_count
        else:
            print("No Tanimoto data found in greedy file")
            return 0, 0, 0
            
    except Exception as e:
        print(f"Error loading greedy file: {e}")
        return 0, 0, 0

def create_greedy_vs_ranked_accuracy_plot(greedy_accuracy, ranked_accuracies, 
                                        total_molecules=4000, processed_molecules=3992):
    """
    Create plot with greedy + ranked accuracies
    """
    # Increase default font sizes
    plt.rcParams.update({'font.size': 22})
    plt.rcParams['axes.titlesize'] = 22
    plt.rcParams['axes.labelsize'] = 22
    plt.rcParams['xtick.labelsize'] = 22
    plt.rcParams['ytick.labelsize'] = 22
    
    # Combine accuracies: greedy first, then ranked accuracies (remove last 2 as in original)
    all_accuracies = [greedy_accuracy] + ranked_accuracies[:-2]
    
    # Labels
    labels = ['Greedy\nSampling', 'HSQC Ranked\nTop 1', 'HSQC Ranked\nTop 3', 
              'HSQC Ranked\nTop 5', 'HSQC Ranked\nTop 10']
    
    # Calculate molecules for each accuracy
    failed_molecules = total_molecules - processed_molecules
    molecules = [int(acc * processed_molecules) for acc in all_accuracies]
    all_molecules = molecules + [failed_molecules]
    all_labels = labels + ['Failed']
    
    # Different colors for each bar
    colors = ['#FF6B6B', '#FF9999', '#66B2FF', '#99FF99', '#FFCC99', '#FF99CC']
    
    # Create the bar plot
    fig, ax = plt.subplots(figsize=(14, 9))
    bars = ax.bar(range(len(all_molecules)), all_molecules, color=colors)
    
    # Set black edges for all bars
    for bar in bars:
        bar.set_edgecolor('black')
    
    # Customize the plot
    ax.set_xticks(range(len(all_labels)))
    ax.set_xticklabels(all_labels, fontsize=20, rotation=45, ha='right')
    ax.set_ylabel('Number of Molecules', fontsize=22)
    ax.set_title('Greedy vs HSQC Ranked Prediction Accuracy (Total: 4000)', fontsize=22, pad=20)
    
    # Add value labels on top and inside of each bar
    for i, bar in enumerate(bars):
        height = bar.get_height()
        if i < len(all_accuracies):  # For accuracy bars
            percentage = all_accuracies[i] * 100
            # Percentage on top
            ax.text(bar.get_x() + bar.get_width()/2., height + 50,
                    f'{percentage:.1f}%',
                    ha='center', va='bottom', fontsize=20)
            # Number inside bar
            ax.text(bar.get_x() + bar.get_width()/2., height/2,
                    f'{int(height)}',
                    ha='center', va='center', fontsize=20, color='black')
        else:  # For failed molecules bar
            percentage = (height/total_molecules) * 100
            # Percentage on top - moved higher for failed bar
            ax.text(bar.get_x() + bar.get_width()/2., height + 200,
                    f'{percentage:.1f}%',
                    ha='center', va='bottom', fontsize=20)
            # Number inside bar - moved higher for failed bar
            ax.text(bar.get_x() + bar.get_width()/2., height + 100,
                    f'{int(height)}',
                    ha='center', va='center', fontsize=20, color='black')
    
    # Add grid for better readability
    ax.grid(True, axis='y', linestyle='--', alpha=0.7)
    # Set y-axis limit to accommodate labels
    ax.set_ylim(0, max(all_molecules) + 400)
    
    # Adjust layout to prevent label cutoff
    plt.tight_layout()
    
    # Save the plot
    save_path = os.path.abspath("./_FIGURES/ZINC4000_greedy_vs_ranked_accuracy.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Plot saved to: {save_path}")
    
    return all_accuracies

def main_greedy_comparison(folder_path):
    """
    Main function to compare greedy vs ranked accuracies
    """
    print("=== LOADING GREEDY ACCURACY (1% DATA) ===")
    # Load greedy accuracy from 1% data file
    greedy_accuracy, greedy_correct, greedy_total = load_greedy_accuracy()
    
    print("=== CALCULATING HSQC RANKED ACCURACIES ===")
    # Calculate ranked accuracies (your existing code)
    all_rankings = exp_func.process_pkl_files_baseline(folder_path, 'HSQC')  # Your existing function
    all_rankings, removed_smiles = exp_func.deduplicate_smiles_from_ranking(all_rankings)
    all_rankings, filtered_out_rankings = exp_func.filter_rankings_by_molecular_formula(all_rankings)
    ranked_accuracies = exp_func.calculate_top_k_accuracy(all_rankings)  # Your existing function
    
    # Create combined plot
    all_accuracies = create_greedy_vs_ranked_accuracy_plot(greedy_accuracy, ranked_accuracies)
    
    print("\n=== RESULTS COMPARISON ===")
    print(f"Greedy Sampling Accuracy: {greedy_accuracy:.3f}")
    print(f"HSQC Ranked Top-1 Accuracy: {ranked_accuracies[0]:.3f}")
    print(f"Top-1 Improvement: {(ranked_accuracies[0] - greedy_accuracy):.3f}")
    
    print(f"Greedy Sampling Accuracy: {greedy_accuracy:.3f}")
    print(f"HSQC Ranked Top-10 Accuracy: {ranked_accuracies[3]:.3f}")
    print(f"Top-10 Improvement: {(ranked_accuracies[3] - greedy_accuracy):.3f}")
    
    return greedy_accuracy, ranked_accuracies

# Example usage:
if __name__ == "__main__":
    folder_path = os.path.abspath("./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/ZINC_250_350_4000")
    greedy_acc, ranked_acc = main_greedy_comparison(folder_path)

#### 4.8 Probabilty analysis

##### Data processing

In [ ]:
import os
import pickle
from typing import List, Tuple

def process_pkl_files(folder_path: str) -> Tuple[List[float], List[float]]:
    """
    Process PKL files in the specified folder and separate molecules based on correctness.
    
    Args:
        folder_path (str): Path to the folder containing PKL files
        
    Returns:
        Tuple[List[float], List[float]]: Two lists containing correct and incorrect molecules
    """
    # Lists to store correct and incorrect molecules
    correct_molecules = []
    incorrect_molecules = []
    
    # Get all PKL files in the folder
    pkl_files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) 
                 if f.endswith('.pkl')]
    
    # Process each PKL file
    for idx, pkl_file in enumerate(pkl_files[:]):
        # print( idx, len(correct_molecules), len(incorrect_molecules), pkl_file )
        try:
            with open(pkl_file, 'rb') as f:
                data = pickle.load(f)
            
            # Access the results dictionary
            results_dict = data["results_dict_bl_ZINC"]
            
            # Process each molecule in the results
            for smiles, molecule_data in results_dict.items():
                # Get the first item's data
                first_molecule = molecule_data[0][0]
                # Check if the length is at least 6 to safely access index 4
                if len(first_molecule) >= 6:
                    correctness_value = first_molecule[4]  # Fifth item (index 4)
                    
                    # Store the tensor values based on correctness
                    tensor_values = first_molecule[3]  # Fourth item (index 3)
                    
                    # Convert tensor to list if it's a tensor
                    if hasattr(tensor_values, 'tolist'):
                        tensor_values = tensor_values.tolist()
                    
                    # Classify based on correctness value
                    if correctness_value == 1.0:
                        correct_molecules.append(first_molecule)
                        #print(idx,len(correct_molecules)+len(incorrect_molecules))
                    else:
                        incorrect_molecules.append(first_molecule)
                       # print(idx,len(correct_molecules)+len(incorrect_molecules))
                    break
        except Exception as e:
            print(f"Error processing file {pkl_file}: {str(e)}")
            continue
    
    return correct_molecules, incorrect_molecules

def analyze_results(correct_molecules: List[float], incorrect_molecules: List[float]) -> None:
    """
    Print analysis of the processed molecules.
    
    Args:
        correct_molecules (List[float]): List of correct molecule values
        incorrect_molecules (List[float]): List of incorrect molecule values
    """
    print(f"\nAnalysis Results:")
    print(f"Number of correct molecules: {len(correct_molecules)}")
    print(f"Number of incorrect molecules: {len(incorrect_molecules)}")
    
# Example usage:
if __name__ == "__main__":
    # Example folder path
    folder_path = os.path.abspath("./past_experiments/ChemXriv/4.0_Experiment_Multinomial_Sampling/Experiment_baseline_PC_ZINC/ZINC_250_350_4000")
    
    # Process the files
    correct_molecules, incorrect_molecules = process_pkl_files(folder_path)
    
    # Analyze the results
    analyze_results(correct_molecules, incorrect_molecules)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import os


def extract_minimum_props(molecules_list):
    """
    Extract minimum probability values from tensor data in molecules list.
    
    Args:
        molecules_list (list): List of molecule data where each item contains a tensor
        
    Returns:
        list: List of minimum probability values
    """
    minimum_props = []
    
    for molecule in molecules_list:
        # Get the tensor (fourth item, index 3)
        tensor_data = molecule[3]
        
        # Convert tensor to list if it's not already
        if hasattr(tensor_data, 'tolist'):
            tensor_values = tensor_data.tolist()
        else:
            tensor_values = tensor_data
            
        # Get the minimum value
        min_value = min(tensor_values)
        minimum_props.append(min_value)
    
    return minimum_props

# Example usage:
correct_min_props = extract_minimum_props(correct_molecules)
incorrect_min_props = extract_minimum_props(incorrect_molecules)

# Print some basic statistics
print(f"Number of correct molecules processed: {len(correct_min_props)}")
print(f"Number of incorrect molecules processed: {len(incorrect_min_props)}")


def plot_probability_distribution(correct_props, incorrect_props, max_samples=300):
    """
    Create a box plot with enhanced styling for probability distributions.
    
    Args:
        correct_props (list): List of probability values for correct molecules
        incorrect_props (list): List of probability values for incorrect molecules
        max_samples (int): Maximum number of samples to plot for each category
    """
    # Sample data if needed
    if len(correct_props) > max_samples:
        correct_props = np.random.choice(correct_props, max_samples, replace=False)
    if len(incorrect_props) > max_samples:
        incorrect_props = np.random.choice(incorrect_props, max_samples, replace=False)
    
    # Define colors
    colors = ['#8CB0FE', '#F39878']
    
    # Create figure and axis with larger size
    plt.figure(figsize=(10, 8))
    
    # Create box plot with MUCH lower opacity or no fill
    data = [correct_props, incorrect_props]
    box_plot = plt.boxplot(data, 
                          labels=['Correct', 'Incorrect'], 
                          patch_artist=True,
                          widths=0.7,
                          medianprops=dict(color="black", linewidth=2),
                          flierprops=dict(marker='o', markerfacecolor='gray'),
                          boxprops=dict(linewidth=2))
    
    # Add more padding to the plot
    plt.margins(x=0.2)
    
    # Customize axis limits for more vertical space
    plt.ylim(0, 1.1)
    
    # Customize the plot with larger font sizes
    plt.ylabel('Minimum Token Probability', labelpad=21, fontsize=22)
    
    # Increase tick label sizes
    plt.xticks(fontsize=21)
    plt.yticks(fontsize=21)
    
    # OPTION 1: Make box plots nearly transparent (recommended)
    for box, color in zip(box_plot['boxes'], colors):
        box.set(facecolor=color, alpha=0.15)  # Much lower alpha (was 0.8)
    
    # OPTION 2: Alternative - Remove box fill completely (uncomment if preferred)
    # for box in box_plot['boxes']:
    #     box.set(facecolor='none')  # No fill color
    
    # Add individual points with jitter, matching the box colors
    for i, (data_points, color) in enumerate(zip([correct_props, incorrect_props], colors), 1):
        # Add jitter to x-coordinates
        x = np.random.normal(i, 0.08, size=len(data_points))
        
        # Plot points with edge color
        plt.scatter(x, data_points, 
                   c=[color],  # Fill color
                   edgecolors='#404040',  # Dark gray edge
                   linewidth=1,  # Edge width
                   alpha=0.7,  # Slightly higher alpha for better visibility
                   s=100)  # Slightly larger points
    
    # Add grid for better readability
    plt.grid(True, axis='y', linestyle='--', alpha=0.3)
    
    # Adjust layout to prevent cutting off
    plt.tight_layout()
    
    # Save the plot
    save_path = "/projects/cc/se_users/knlr326/1_NMR_project/2_Notebooks/MultiModalSpectralTransformer_cleaned/_FIGURES/ZINC4000_min_prob_fixed.png"
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"Figure saved to: {save_path}")
    plt.show()
    
    # Print summary statistics
    print("\nSummary Statistics (based on full dataset):")
    print("\nCorrect molecules minimum probabilities:")
    print(f"Mean: {np.mean(correct_props):.4f}")
    print(f"Median: {np.median(correct_props):.4f}")
    print(f"Std Dev: {np.std(correct_props):.4f}")
    print(f"Number of samples plotted: {len(correct_props)}")

    print("\nIncorrect molecules minimum probabilities:")
    print(f"Mean: {np.mean(incorrect_props):.4f}")
    print(f"Median: {np.median(incorrect_props):.4f}")
    print(f"Std Dev: {np.std(incorrect_props):.4f}")
    print(f"Number of samples plotted: {len(incorrect_props)}")

# Example usage:
plot_probability_distribution(correct_min_props, incorrect_min_props, max_samples=300)

In [ ]:
ranking_method = 'HSQC'
all_rankings = exp_func.process_pkl_files_baseline(pkl_folder, ranking_method)

#### 4.9 Dataset of 34 molecules with simulated data vectorization - t-SNE

In [ ]:
# Test Data - Updated to use 8.0_Sim_Data paths
config.csv_1H_path_SGNN = os.path.abspath('./past_experiments/ChemXriv/8.0_Sim_Data/data_1H_924509.csv')
config.csv_13C_path_SGNN = os.path.abspath('./past_experiments/ChemXriv/8.0_Sim_Data/data_13C_924509.csv')    
config.csv_HSQC_path_SGNN = os.path.abspath('./past_experiments/ChemXriv/8.0_Sim_Data/data_HSQC_924509.csv')    
config.csv_COSY_path_SGNN = os.path.abspath('./past_experiments/ChemXriv/8.0_Sim_Data/data_COSY_924509.csv')   
config.csv_path_val = os.path.abspath('./past_experiments/ChemXriv/8.0_Sim_Data/data_1H_924509.csv')
config.pickle_file_path = ""

# IR data folder
config.IR_data_folder = os.path.abspath('./past_experiments/ChemXriv/8.0_Sim_Data/IR_data_924509')
config.data_size = 34  # Updated to match the dataset size
config.checkpoint_path = os.path.abspath('./models/mmst/base_models/1_0_V8i_MMTi_RAW_DROP_Loss_0.112.ckpt')
# Save vectors in the same 8.0_Sim_Data folder
config.vector_db = os.path.abspath('./past_experiments/ChemXriv/8.0_Sim_Data/data_1H_924509_Vectors.csv')

In [ ]:
# path to store db
config = exp_func.vectorize_db(config, stoi, stoi_MF, "db", "all")

In [ ]:

vector_db = config.vector_db
# Load CSV data
df = pd.read_csv(vector_db)

# Convert 'Fingerprints' to tensors
tqdm.pandas()
df['Fingerprints'] = df['Fingerprints'].progress_apply(ast.literal_eval)
df['Fingerprints'] = df['Fingerprints'].progress_apply(lambda x: torch.tensor(x, dtype=torch.float32))

# Save the DataFrame to Pickle
pickle_file = vector_db.replace('.csv', '_v2.pkl')
with open(pickle_file, 'wb') as f:
    pickle.dump(df, f)

print(f"Data saved to {pickle_file}")


In [ ]:


def plot_tsne_umap_pca_train_test(results, labels, title, methods, save_folder):
    plt.rcParams.update({'font.size': 22})  # Set base font size
    fig, axes = plt.subplots(1, 3, figsize=(20, 8))
    fig.suptitle(title, fontsize=22)

    for ax, result, method in zip(axes, results, methods):
        for label, color, alpha in zip(set(labels), ['#FFB381', '#A1C8F3'], [0.5, 0.5]):
            mask = np.array(labels) == label
            ax.scatter(result[mask, 0], result[mask, 1], c=color, label=label, alpha=alpha)
       
        ax.legend(fontsize=22)  # Set legend font size
        ax.set_title(method, fontsize=22, pad=20)  # Set subplot title font size and add padding
        ax.set_xticks([])
        ax.set_yticks([])

    plt.tight_layout()
    #plt.savefig(f"{title.replace(' ', '_')}.png")
    # save_folder = "/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/___FIGURES_PAPERS/Figures_Paper_2"
    save_path = os.path.join(save_folder, f"{title.replace(' ', '_')}.png")
    plt.savefig(save_path)

    plt.close()


In [ ]:
# Load test data - your new 8.0_Sim_Data
test_files = [
    os.path.abspath('./past_experiments/ChemXriv/8.0_Sim_Data/data_1H_924509_Vectors_v2.pkl')
]
weight_ranges = ["Sim Data (34 samples)"]
save_folder = os.path.abspath("./_FIGURES")

# Load training data
print("Loading training data...")
train_vectors, train_smiles = exp_func.load_pickle_data(
    os.path.abspath('./past_experiments/ChemXriv/6.0_Dataset_Improvement_Cycle/vectors/vector_db_train_5Mod_4M_v2_v2_100k.pkl'), 
    sample_size=3000
)

for test_file, weight_range in zip(test_files, weight_ranges):
    print(f"Processing test data for {weight_range}...")
    test_vectors, test_smiles = exp_func.load_pickle_data(test_file)
    
    # Use all your test vectors (should be 34 based on your data_size)
    test_vectors_sample = test_vectors[:34]  # Use all 34 samples
    
    combined_vectors = np.vstack((train_vectors, test_vectors_sample))
    labels = ['Train'] * len(train_vectors) + ['Test'] * len(test_vectors_sample)
    
    print(f"Shape of combined vectors: {combined_vectors.shape}")
    print(f"Number of labels: {len(labels)}")
    print(f"Train samples: {len(train_vectors)}")
    print(f"Test samples: {len(test_vectors_sample)}")
    
    print("Performing dimensionality reduction...")
    tsne_result, pca_result, umap_result = exp_func.perform_dimensionality_reduction(combined_vectors)
    
    print("Plotting results...")
    plot_tsne_umap_pca_train_test([tsne_result, pca_result, umap_result], labels, 
                 f"Train vs Simulated Data Vectors", 
                 ['t-SNE', 'PCA', 'UMAP'],
                 save_folder)

print("All plots generated successfully!")

#### 4.10 Dataset of 34 molecules with Experimental data vectorization - t-SNE

In [ ]:
# Test Data - Updated to use 8.0_Experimental_Data paths
config.csv_1H_path_SGNN = os.path.abspath('./past_experiments/ChemXriv/8.0_Experimenta_Data/real_1H_with_AZ_SMILES_v3.csv')
config.csv_13C_path_SGNN = os.path.abspath('./past_experiments/ChemXriv/8.0_Experimenta_Data/real_13C_with_AZ_SMILES_v3.csv')    
config.csv_HSQC_path_SGNN = os.path.abspath('./past_experiments/ChemXriv/8.0_Experimenta_Data/real_HSQC_with_AZ_SMILES_v3.csv')    
config.csv_COSY_path_SGNN = os.path.abspath('./past_experiments/ChemXriv/8.0_Experimenta_Data/real_COSY_with_AZ_SMILES_v3.csv')   
config.csv_path_val = os.path.abspath('./past_experiments/ChemXriv/8.0_Experimenta_Data/real_1H_with_AZ_SMILES_v3.csv')
config.pickle_file_path = ""
# IR data folder
config.IR_data_folder = os.path.abspath('./past_experiments/ChemXriv/8.0_Experimenta_Data/IR_data')
config.data_size = 34  # Update this to match your actual experimental dataset size
config.checkpoint_path = os.path.abspath('./models/mmst/base_models/1_0_V8i_MMTi_RAW_DROP_Loss_0.112.ckpt')
# Save vectors in the same 8.0_Experimental_Data folder
config.vector_db = os.path.abspath('./past_experiments/ChemXriv/8.0_Experimenta_Data/real_experimental_data_Vectors.csv')

In [ ]:
# path to store db
config = exp_func.vectorize_db(config, stoi, stoi_MF, "db", "all")

In [ ]:

vector_db = config.vector_db
# Load CSV data
df = pd.read_csv(vector_db)

# Convert 'Fingerprints' to tensors
tqdm.pandas()
df['Fingerprints'] = df['Fingerprints'].progress_apply(ast.literal_eval)
df['Fingerprints'] = df['Fingerprints'].progress_apply(lambda x: torch.tensor(x, dtype=torch.float32))

# Save the DataFrame to Pickle
pickle_file = vector_db.replace('.csv', '_v2.pkl')
with open(pickle_file, 'wb') as f:
    pickle.dump(df, f)

print(f"Data saved to {pickle_file}")


In [ ]:
# Load test data - your new 8.0_Experimental_Data
test_files = [
    os.path.abspath('./past_experiments/ChemXriv/8.0_Experimenta_Data/real_experimental_data_Vectors_v2.pkl')
]
weight_ranges = ["Experimental Data (34 samples)"]
save_folder = os.path.abspath("./_FIGURES")

# Load training data
print("Loading training data...")
train_vectors, train_smiles = exp_func.load_pickle_data(
    os.path.abspath('./past_experiments/ChemXriv/6.0_Dataset_Improvement_Cycle/vectors/vector_db_train_5Mod_4M_v2_v2_100k.pkl'), 
    sample_size=3000
)

for test_file, weight_range in zip(test_files, weight_ranges):
    print(f"Processing test data for {weight_range}...")
    test_vectors, test_smiles = exp_func.load_pickle_data(test_file)
    
    # Use all your test vectors (should be 34 based on your data_size)
    test_vectors_sample = test_vectors[:34]  # Use all 34 samples
    
    combined_vectors = np.vstack((train_vectors, test_vectors_sample))
    labels = ['Train'] * len(train_vectors) + ['Test'] * len(test_vectors_sample)
    
    print(f"Shape of combined vectors: {combined_vectors.shape}")
    print(f"Number of labels: {len(labels)}")
    print(f"Train samples: {len(train_vectors)}")
    print(f"Test samples: {len(test_vectors_sample)}")
    
    print("Performing dimensionality reduction...")
    tsne_result, pca_result, umap_result = exp_func.perform_dimensionality_reduction(combined_vectors)
    
    print("Plotting results...")
    plot_tsne_umap_pca_train_test([tsne_result, pca_result, umap_result], labels, 
                 f"Train vs Experimental Data Vectors", 
                 ['t-SNE', 'PCA', 'UMAP'],
                 save_folder)

print("All plots generated successfully!")

### 5.0 Test Improvement cycle 

#### 5.1 Fine Tune on all the molecules that I want to investigate

In [ ]:
config.project = "Improv_Cycle_v3" # Name of the project for wandb monitoring
config.csv_train_path = os.path.abspath('./data/ZINK_dataset/ML_NMR_5M_XL_1H_comb_train_V8.csv') 
config.csv_1H_path_SGNN = os.path.abspath('./data/ZINK_dataset/val_data_all_modalities/ML_NMR_1H_combined_ZINC_test_10x100.csv')
config.csv_13C_path_SGNN = os.path.abspath('./data/ZINK_dataset/val_data_all_modalities/ML_NMR_5M_XL_13C_test_10x100.csv')    
config.csv_HSQC_path_SGNN = os.path.abspath('./data/ZINK_dataset/val_data_all_modalities/ML_NMR_5M_XL_HSQC_test_10x100.csv')    
config.csv_COSY_path_SGNN = os.path.abspath('./data/ZINK_dataset/val_data_all_modalities/ML_NMR_5M_XL_COSY_test_10x100.csv')  
config.csv_path_val = os.path.abspath('./data/ZINK_dataset/val_data_all_modalities/ML_NMR_1H_combined_ZINC_test_10x100.csv')
config.pickle_file_path = os.path.abspath('./data/ZINK_dataset/val_data_all_moldalities/ML_NMR_1H_combined_ZINC_test_10x100_955629.pkl')

#config.csv_1H_path_SGNN = os.path.abspath('/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/26_PubChem_dataset/val_data_350_500_x1000/ML_NMR_2M_XL_1H_V1_test_f_350_500_x1000.csv')
#config.csv_13C_path_SGNN = os.path.abspath('/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/26_PubChem_dataset/val_data_350_500_x1000/ML_NMR_2M_XL_13C_V1_test_350_500_x1000.csv')    
#config.csv_HSQC_path_SGNN = os.path.abspath('/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/26_PubChem_dataset/val_data_350_500_x1000/ML_NMR_2M_XL_HSQC_V1_test_350_500_x1000.csv')    
#config.csv_COSY_path_SGNN = os.path.abspath('/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/26_PubChem_dataset/val_data_350_500_x1000/ML_NMR_2M_XL_COSY_V1_test_350_500_x1000.csv')   
#config.csv_path_val = os.path.abspath('/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/26_PubChem_dataset/val_data_350_500_x1000/ML_NMR_2M_XL_1H_V1_test_f_350_500_x1000.csv')
#config.pickle_file_path = os.path.abspath("/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/26_PubChem_dataset/val_data_350_500_x1000/ML_NMR_2M_XL_1H_V1_test_f_350_500_x1000_185242.pkl")

# V8 Raw 
#config.IR_data_folder = os.path.abspath("/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/26_PubChem_dataset/IR_data")

#config.IR_data_folder = ""
config.IR_data_folder = os.path.abspath("./data/ZINK_dataset/IR_spectra_NN")

#  8Vi Raw Drop
config.checkpoint_path = os.path.abspath("./models/mmst/base_models/1_0_V8i_MMTi_RAW_DROP_Loss_0.112.ckpt")

config.data_size = 100 # config.test_size # why would I do that? 
#config.data_size = 4 # config.test_size # why would I do that? 
config.execution_type = "test_performance"
config.multinom_runs = 1
#config.multinom_runs = 3
config.temperature = 1
greedy_full = False
MW_filter = True
config.MF_generations = 50
config.MF_delta_weight = 20
config.max_scaffold_generations = 30


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
if config.execution_type == "test_performance":
    print("\033[1m\033[31mThis is: test_performance\033[0m")
    # config.csv_path_val = config.csv_SMI_targets  #this already got updated in simulate_syn_data
    
    model_MMT = mrtf.load_MMT_model(config)
    model_CLIP = mrtf.load_CLIP_model(config)
    #model_BLIP = mrtf.load_BLIP_model(config)
    #model_MMT = model_CLIP.MT_model
    val_dataloader = mrtf.load_data(config, stoi, stoi_MF, single=True, mode="val")
    val_dataloader_multi = mrtf.load_data(config, stoi, stoi_MF, single=False, mode="val")

    
    results_dict_bl_ZINC = mrtf.run_test_mns_performance_CLIP_3(config,  
                                                        model_MMT,
                                                        model_CLIP,
                                                        val_dataloader,
                                                        stoi, 
                                                        itos,
                                                        MW_filter)
    results_dict_bl_ZINC, counter = mrtf.filter_invalid_inputs(results_dict_bl_ZINC)
    
    avg_tani_bl_ZINC, html_plot = rbgvm.plot_hist_of_results(results_dict_bl_ZINC)
    
    # Slow because also just takes one at the time
    if greedy_full == True:
        results_dict_greedy_bl_ZINC, failed_bl_ZINC = mrtf.run_test_performance_CLIP_greedy_3(config,  
                                                                stoi, 
                                                                stoi_MF, 
                                                                itos, 
                                                                itos_MF)

        avg_tani_greedy_bl_ZINC, html_plot_greedy = rbgvm.plot_hist_of_results_greedy(results_dict_greedy_bl_ZINC)

    else: 
        config, results_dict_ZINC_greedy_bl = mrtf.run_greedy_sampling(config, model_MMT, val_dataloader_multi, itos, stoi)
        avg_tani_greedy_bl_ZINC = results_dict_ZINC_greedy_bl["tanimoto_mean"]
    
    total_results_bl_ZINC = mrtf.run_test_performance_CLIP_3(config, 
                                                        model_MMT, 
                                                        val_dataloader,
                                                        stoi)
    
    corr_sampleing_prob_bl_ZINC = total_results_bl_ZINC["statistics_multiplication_avg"][0]
    print("avg_tani, avg_tani_greedy, corr_sampleing_prob'")
    print(avg_tani_bl_ZINC, avg_tani_greedy_bl_ZINC, corr_sampleing_prob_bl_ZINC)       

    

In [ ]:
#Select the samples that did not succeed to get right
filtered_results = [key for key, value in results_dict_bl_ZINC.items()]
filtered_results

In [ ]:
# Create a DataFrame
data = {
    "SMILES": filtered_results,
    "sample-id": [f"SOURCE_00000{i+1}" for i in range(len(filtered_results))]
}

df = pd.DataFrame(data)

# Save DataFrame to CSV
csv_file_path = os.path.abspath('./deep-molecular-optimization/deep-molecular-optimization/data/MMP/test_selection_2.csv'
df.to_csv(csv_file_path, index=False)
csv_path_val_backup = config.csv_path_val
pickle_file_path_backup = config.pickle_file_path

print(f"CSV file '{csv_file_path}' created successfully.")

In [ ]:
config.execution_type = "SMI_generation_MF"

if config.execution_type == "SMI_generation_MF":
    config.n_samples = config.data_size
    #if config.execution_type == "SMI_generation_MF":
    print("\033[1m\033[31mThis is: SMI_generation_MF\033[0m")
    config.csv_path_val  = ex.filter_invalid_criteria(config.csv_path_val)#, config.csv_path_val)
    config, results_dict_MF = ex.SMI_generation_MF(config, stoi, stoi_MF, itos, itos_MF)
    #mode = "val"

    # Iterate through the dictionary and remove 'nan' from lists
    results_dict_MF = {key: value for key, value in results_dict_MF.items() if not hf.contains_only_nan(value)}
    for key, value in results_dict_MF.items():
        results_dict_MF[key] = hf.remove_nan_from_list(value)

    combined_list_MF, html_TSNE, html_UMAP, html_PCA = cv.plot_cluster_MF(results_dict_MF, config)
    max_num = 10
    html_plot = pt.plot_molecules_from_list(combined_list_MF, max_num)
    config.execution_type = "combine_MMT_MF"
    #import IPython; IPython.embed();
    print(config.data_size)


In [ ]:
config.execution_type = "combine_MMT_MF"
combined_list_MMT = []
if config.execution_type == "combine_MMT_MF":
    print("\033[1m\033[31mThis is: combine_MMT_MF\033[0m")
    all_gen_smis = combined_list_MMT + combined_list_MF
    combined_list_MF = [smiles for smiles in combined_list_MF if smiles != 'NAN']
    all_gen_smis = [smiles for smiles in all_gen_smis if smiles != 'NAN']
    
    #filter out potential hits from the real test_set
    val_data = pd.read_csv(config.csv_path_val)
    all_gen_smis = mrtf.filter_smiles(val_data, all_gen_smis)
    length_of_list = len(all_gen_smis)   
    random_number_strings = [f"GT_{str(i).zfill(7)}" for i in range(1, length_of_list + 1)]
    aug_mol_df = pd.DataFrame({'SMILES': all_gen_smis, 'sample-id': random_number_strings})
    config.execution_type = "blend_prev_train_data"


In [ ]:
config.train_data_blend = 0
config.execution_type = "blend_prev_train_data"
if config.execution_type == "blend_prev_train_data":
    print("\033[1m\033[31mThis is: blend_prev_train_data\033[0m")
    config, final_df = ex.blend_aug_with_train_data(config, aug_mol_df)
    config.execution_type = "data_generation"
    #import IPython; IPython.embed();


In [ ]:
config.execution_type = "data_generation"
if config.execution_type == "data_generation":
    #config.csv_SMI_targets = config.csv_1H_path_SGNN
    print("\033[1m\033[31mThis is: data_generation\033[0m")
    config = ex.gen_sim_aug_data(config, IR_config)
    config.execution_type = "transformer_improvement"
    sim_data_gen = True
    #import IPython; IPython.embed();

In [ ]:

# Save variables to a pickle file
variables_to_save = {
    'avg_tani_bl_ZINC': avg_tani_bl_ZINC,
    'results_dict_greedy_bl_ZINC': results_dict_greedy_bl_ZINC if greedy_full else None,
    'failed_bl_ZINC': failed_bl_ZINC if greedy_full else None,
    'avg_tani_greedy_bl_ZINC': avg_tani_greedy_bl_ZINC,
    'results_dict_ZINC_greedy_bl': results_dict_ZINC_greedy_bl if not greedy_full else None,
    'total_results_bl_ZINC': total_results_bl_ZINC,
    'corr_sampleing_prob_bl_ZINC': corr_sampleing_prob_bl_ZINC,
    'filtered_results': filtered_results,
    'all_gen_smis': all_gen_smis,
    'aug_mol_df': aug_mol_df,
    'results_dict_bl_ZINC': results_dict_bl_ZINC,
}

with open(os.path.abspath('/projects/cc/se_users/knlr326/1_NMR_project/2_Notebooks/nmr_project/1_Dataexploration/2_paper_code/Experiments_SLURM/20.0_SLURM_MasterTransformer/Figures_Paper_2/precomputed_raw_data/20240516_Improvment_Cycle_v2/6.2_imp_cyc_all_100_50.pkl', 'wb') as f:
    pickle.dump(variables_to_save, f)


In [ ]:
# config.csv_SMI_targets = config.csv_1H_path_SGNN
# data_IR = irs.run_IR_simulation(config, IR_config, "target")
# config.IR_data_folder = data_IR


In [ ]:
config.blank_percentage = 0
config.weight_MW = 0
config.lr_pretraining = 1e-4
config.tr_te_split = 0.9
config.model_save_dir = os.path.abspath("/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/1_old_models/models_v2/imp_cyc_all_100_50_v2") # Folder where networks are saved
data_size = len(pd.read_csv(config.csv_1H_path_SGNN))
config.data_size = data_size
config.gpu_num = 1
config.batch_size = 64
config.num_epochs = 50

In [ ]:
config.execution_type = "transformer_improvement"
sim_data_gen = True

if config.execution_type == "transformer_improvement" and sim_data_gen == True:
    print("\033[1m\033[31mThis is: transformer_improvement, sim_data_gen == TRUE\033[0m")
    config.training_setup = "pretraining"
    mtf.run_MMT(config, stoi, stoi_MF)

    # config.execution_type = "clip_improvement"
    config.execution_type = "update_model"
    # finish_while = True
    #import IPython; IPython.embed();


In [ ]:
config.csv_train_path = os.path.abspath('/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/15_ZINC270M/ML_NMR_5M_XL_1H_comb_train_V8.csv') 
config.csv_1H_path_SGNN = os.path.abspath('/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/15_ZINC270M/val_data_all_modalities/ML_NMR_1H_combined_ZINC_test_10x100.csv')
config.csv_13C_path_SGNN = os.path.abspath('/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/15_ZINC270M/val_data_all_modalities/ML_NMR_5M_XL_13C_test_10x100.csv')    
config.csv_HSQC_path_SGNN = os.path.abspath('/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/15_ZINC270M/val_data_all_modalities/ML_NMR_5M_XL_HSQC_test_10x100.csv')    
config.csv_COSY_path_SGNN = os.path.abspath('/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/15_ZINC270M/val_data_all_modalities/ML_NMR_5M_XL_COSY_test_10x100.csv')  
config.csv_path_val = os.path.abspath('/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/15_ZINC270M/val_data_all_modalities/ML_NMR_1H_combined_ZINC_test_10x100.csv')
config.pickle_file_path = os.path.abspath("/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/15_ZINC270M/val_data_all_modalities/ML_NMR_1H_combined_ZINC_test_10x100_909434.pkl")

#config.csv_1H_path_SGNN = os.path.abspath('/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/26_PubChem_dataset/val_data_350_500_x1000/ML_NMR_2M_XL_1H_V1_test_f_350_500_x1000.csv')
#config.csv_13C_path_SGNN = os.path.abspath('/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/26_PubChem_dataset/val_data_350_500_x1000/ML_NMR_2M_XL_13C_V1_test_350_500_x1000.csv')    
#config.csv_HSQC_path_SGNN = os.path.abspath('/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/26_PubChem_dataset/val_data_350_500_x1000/ML_NMR_2M_XL_HSQC_V1_test_350_500_x1000.csv')    
#config.csv_COSY_path_SGNN = os.path.abspath('/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/26_PubChem_dataset/val_data_350_500_x1000/ML_NMR_2M_XL_COSY_V1_test_350_500_x1000.csv')   
#config.csv_path_val = os.path.abspath('/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/26_PubChem_dataset/val_data_350_500_x1000/ML_NMR_2M_XL_1H_V1_test_f_350_500_x1000.csv')
#config.pickle_file_path = os.path.abspath("/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/26_PubChem_dataset/val_data_350_500_x1000/ML_NMR_2M_XL_1H_V1_test_f_350_500_x1000_185242.pkl")

# V8 Raw 
#config.IR_data_folder = os.path.abspath("/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/26_PubChem_dataset/IR_data")

#config.IR_data_folder = ""
config.IR_data_folder = os.path.abspath("/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/15_ZINC270M/IR_spectra_NN")

# V8i Raw Drop
config.checkpoint_path = os.path.abspath("/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/1_old_models/models_v2/V8i_MMT_Drop4/MultimodalTransformer_time_1710027004.1571195_Loss_0.112.ckpt")

config.data_size = 100 
config.execution_type = "test_performance"
config.multinom_runs = 1
config.temperature = 1
greedy_full = False
MW_filter = True
config.MF_generations = 50


In [ ]:
config = ex.update_model_path(config)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
if config.execution_type == "test_performance":
    print("\033[1m\033[31mThis is: test_performance\033[0m")
    # config.csv_path_val = config.csv_SMI_targets  #this already got updated in simulate_syn_data
    
    model_MMT = mrtf.load_MMT_model(config)
    model_CLIP = mrtf.load_CLIP_model(config)

    val_dataloader = mrtf.load_data(config, stoi, stoi_MF, single=True, mode="val")
    val_dataloader_multi = mrtf.load_data(config, stoi, stoi_MF, single=False, mode="val")

    
    results_dict_bl_ZINC = mrtf.run_test_mns_performance_CLIP_3(config,  
                                                        model_MMT,
                                                        model_CLIP,
                                                        val_dataloader,
                                                        stoi, 
                                                        itos,
                                                        MW_filter)
    results_dict_bl_ZINC, counter = mrtf.filter_invalid_inputs(results_dict_bl_ZINC)
    
    avg_tani_bl_ZINC, html_plot = rbgvm.plot_hist_of_results(results_dict_bl_ZINC)
    
    # Slow because also just takes one at the time
    if greedy_full == True:
        results_dict_greedy_bl_ZINC, failed_bl_ZINC = mrtf.run_test_performance_CLIP_greedy_3(config,  
                                                                stoi, 
                                                                stoi_MF, 
                                                                itos, 
                                                                itos_MF)

        avg_tani_greedy_bl_ZINC, html_plot_greedy = rbgvm.plot_hist_of_results_greedy(results_dict_greedy_bl_ZINC)

    else: 
        config, results_dict_ZINC_greedy_bl = mrtf.run_greedy_sampling(config, model_MMT, val_dataloader_multi, itos, stoi)
        avg_tani_greedy_bl_ZINC = results_dict_ZINC_greedy_bl["tanimoto_mean"]
    
    total_results_bl_ZINC = mrtf.run_test_performance_CLIP_3(config, 
                                                        model_MMT, 
                                                        val_dataloader,
                                                        stoi)
    
    corr_sampleing_prob_bl_ZINC = total_results_bl_ZINC["statistics_multiplication_avg"][0]
    print("avg_tani, avg_tani_greedy, corr_sampleing_prob'")
    print(avg_tani_bl_ZINC, avg_tani_greedy_bl_ZINC, corr_sampleing_prob_bl_ZINC)       

    

In [ ]:
# Select the samples that did not succeed to get right
filtered_results = [key for key, value in results_dict_bl_ZINC.items()]
# filtered_results

In [ ]:

# Save variables to a pickle file
variables_to_save = {
    'avg_tani_bl_ZINC': avg_tani_bl_ZINC,
    'results_dict_greedy_bl_ZINC': results_dict_greedy_bl_ZINC if greedy_full else None,
    'failed_bl_ZINC': failed_bl_ZINC if greedy_full else None,
    'avg_tani_greedy_bl_ZINC': avg_tani_greedy_bl_ZINC,
    'results_dict_ZINC_greedy_bl': results_dict_ZINC_greedy_bl if not greedy_full else None,
    'total_results_bl_ZINC': total_results_bl_ZINC,
    'corr_sampleing_prob_bl_ZINC': corr_sampleing_prob_bl_ZINC,
    'filtered_results': filtered_results,
    'results_dict_bl_ZINC': results_dict_bl_ZINC,
}

with open(os.path.abspath('/projects/cc/se_users/knlr326/1_NMR_project/2_Notebooks/nmr_project/1_Dataexploration/2_paper_code/Experiments_SLURM/20.0_SLURM_MasterTransformer/Figures_Paper_2/precomputed_raw_data/20240516_Improvment_Cycle_v2/6.2_imp_cyc_all_100_50_after.pkl', 'wb') as f:
    pickle.dump(variables_to_save, f)


In [ ]:

config.data_size = 100 
config.execution_type = "test_performance"
config.multinom_runs = 1
config.multinom_runs = 3
config.temperature = 1
greedy_full = False
MW_filter = True



In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
if config.execution_type == "test_performance":
    print("\033[1m\033[31mThis is: test_performance\033[0m")
    # config.csv_path_val = config.csv_SMI_targets  #this already got updated in simulate_syn_data
    
    model_MMT = mrtf.load_MMT_model(config)
    model_CLIP = mrtf.load_CLIP_model(config)

    val_dataloader = mrtf.load_data(config, stoi, stoi_MF, single=True, mode="val")
    val_dataloader_multi = mrtf.load_data(config, stoi, stoi_MF, single=False, mode="val")

    
    results_dict_bl_ZINC = mrtf.run_test_mns_performance_CLIP_3(config,  
                                                        model_MMT,
                                                        model_CLIP,
                                                        val_dataloader,
                                                        stoi, 
                                                        itos,
                                                        MW_filter)
    
    results_dict_bl_ZINC, counter = mrtf.filter_invalid_inputs(results_dict_bl_ZINC)
    
    avg_tani_bl_ZINC, html_plot = rbgvm.plot_hist_of_results(results_dict_bl_ZINC)
    
    # Slow because also just takes one at the time
    if greedy_full == True:
        results_dict_greedy_bl_ZINC, failed_bl_ZINC = mrtf.run_test_performance_CLIP_greedy_3(config,  
                                                                stoi, 
                                                                stoi_MF, 
                                                                itos, 
                                                                itos_MF)

        avg_tani_greedy_bl_ZINC, html_plot_greedy = rbgvm.plot_hist_of_results_greedy(results_dict_greedy_bl_ZINC)

    else: 
        config, results_dict_ZINC_greedy_bl = mrtf.run_greedy_sampling(config, model_MMT, val_dataloader_multi, itos, stoi)
        avg_tani_greedy_bl_ZINC = results_dict_ZINC_greedy_bl["tanimoto_mean"]
    
    total_results_bl_ZINC = mrtf.run_test_performance_CLIP_3(config, 
                                                        model_MMT, 
                                                        val_dataloader,
                                                        stoi)
    
    corr_sampleing_prob_bl_ZINC = total_results_bl_ZINC["statistics_multiplication_avg"][0]
    print("avg_tani, avg_tani_greedy, corr_sampleing_prob'")
    print(avg_tani_bl_ZINC, avg_tani_greedy_bl_ZINC, corr_sampleing_prob_bl_ZINC)       

    

In [ ]:
#Select the samples that did not succeed to get right
filtered_results = [key for key, value in results_dict_bl_ZINC.items() if value[0][0][-2] != 1]
#filtered_results

In [ ]:

# Save variables to a pickle file
variables_to_save = {
    'avg_tani_bl_ZINC': avg_tani_bl_ZINC,
    'results_dict_greedy_bl_ZINC': results_dict_greedy_bl_ZINC if greedy_full else None,
    'failed_bl_ZINC': failed_bl_ZINC if greedy_full else None,
    'avg_tani_greedy_bl_ZINC': avg_tani_greedy_bl_ZINC,
    'results_dict_ZINC_greedy_bl': results_dict_ZINC_greedy_bl if not greedy_full else None,
    'total_results_bl_ZINC': total_results_bl_ZINC,
    'corr_sampleing_prob_bl_ZINC': corr_sampleing_prob_bl_ZINC,
    'filtered_results': filtered_results,
    'results_dict_bl_ZINC': results_dict_bl_ZINC,
}

with open(os.path.abspath('/projects/cc/se_users/knlr326/1_NMR_project/2_Notebooks/nmr_project/1_Dataexploration/2_paper_code/Experiments_SLURM/20.0_SLURM_MasterTransformer/Figures_Paper_2/precomputed_raw_data/20240516_Improvment_Cycle_v2/6.2_imp_cyc_all_100_50_after_MNS_3.pkl', 'wb') as f:
    pickle.dump(variables_to_save, f)


In [ ]:
#config.csv_1H_path_SGNN = os.path.abspath('/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/26_PubChem_dataset/val_data_350_500_x1000/ML_NMR_2M_XL_1H_V1_test_f_350_500_x1000.csv')
#config.csv_13C_path_SGNN = os.path.abspath('/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/26_PubChem_dataset/val_data_350_500_x1000/ML_NMR_2M_XL_13C_V1_test_350_500_x1000.csv')    
#config.csv_HSQC_path_SGNN = os.path.abspath('/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/26_PubChem_dataset/val_data_350_500_x1000/ML_NMR_2M_XL_HSQC_V1_test_350_500_x1000.csv')    
#config.csv_COSY_path_SGNN = os.path.abspath('/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/26_PubChem_dataset/val_data_350_500_x1000/ML_NMR_2M_XL_COSY_V1_test_350_500_x1000.csv')   
#config.csv_path_val = os.path.abspath('/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/26_PubChem_dataset/val_data_350_500_x1000/ML_NMR_2M_XL_1H_V1_test_f_350_500_x1000.csv')
#config.pickle_file_path = os.path.abspath("/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/26_PubChem_dataset/val_data_350_500_x1000/ML_NMR_2M_XL_1H_V1_test_f_350_500_x1000_185242.pkl")

#config.csv_1H_path_SGNN = os.path.abspath('/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/26_PubChem_dataset/val_data_0_250_x1000/ML_NMR_2M_XL_1H_V1_test_f_0_250_x1000.csv')
#config.csv_13C_path_SGNN = os.path.abspath('/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/26_PubChem_dataset/val_data_0_250_x1000/ML_NMR_2M_XL_13C_V1_test_0_250_x1000.csv')
#config.csv_HSQC_path_SGNN = os.path.abspath('/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/26_PubChem_dataset/val_data_0_250_x1000/ML_NMR_2M_XL_HSQC_V1_test_0_250_x1000.csv')    
#config.csv_COSY_path_SGNN = os.path.abspath('/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/26_PubChem_dataset/val_data_0_250_x1000/ML_NMR_2M_XL_COSY_V1_test_0_250_x1000.csv')   
#config.csv_path_val = os.path.abspath('/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/26_PubChem_dataset/val_data_0_250_x1000/ML_NMR_2M_XL_1H_V1_test_f_0_250_x1000.csv')
#config.pickle_file_path = os.path.abspath("/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/26_PubChem_dataset/val_data_0_250_x1000/ML_NMR_2M_XL_1H_V1_test_f_0_250_x1000_933335.pkl")


config.csv_1H_path_SGNN = os.path.abspath('/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/26_PubChem_dataset/val_data_250_350_x1000/ML_NMR_2M_XL_1H_V1_test_f_250_350_x1000.csv')
config.csv_13C_path_SGNN = os.path.abspath('/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/26_PubChem_dataset/val_data_250_350_x1000/ML_NMR_2M_XL_13C_V1_test_250_350_x1000.csv')    
config.csv_HSQC_path_SGNN = os.path.abspath('/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/26_PubChem_dataset/val_data_250_350_x1000/ML_NMR_2M_XL_HSQC_V1_test_250_350_x1000.csv')    
config.csv_COSY_path_SGNN = os.path.abspath('/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/26_PubChem_dataset/val_data_250_350_x1000/ML_NMR_2M_XL_COSY_V1_test_250_350_x1000.csv')   
config.csv_path_val = os.path.abspath('/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/26_PubChem_dataset/val_data_250_350_x1000/ML_NMR_2M_XL_1H_V1_test_f_250_350_x1000.csv')
config.pickle_file_path = os.path.abspath("/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/26_PubChem_dataset/val_data_250_350_x1000/ML_NMR_2M_XL_1H_V1_test_f_250_350_x1000_285005.pkl")

# V8i Raw 
config.IR_data_folder = os.path.abspath("/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/26_PubChem_dataset/IR_data"


In [ ]:

config.data_size = 100 # config.test_size # why would I do that? 
#config.data_size = 4 # config.test_size # why would I do that? 
config.execution_type = "test_performance"
config.multinom_runs = 1
config.multinom_runs = 1
config.temperature = 1
greedy_full = False
MW_filter = True



In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
if config.execution_type == "test_performance":
    print("\033[1m\033[31mThis is: test_performance\033[0m")
    # config.csv_path_val = config.csv_SMI_targets  #this already got updated in simulate_syn_data
    
    model_MMT = mrtf.load_MMT_model(config)
    model_CLIP = mrtf.load_CLIP_model(config)
    #model_BLIP = mrtf.load_BLIP_model(config)
    #model_MMT = model_CLIP.MT_model
    val_dataloader = mrtf.load_data(config, stoi, stoi_MF, single=True, mode="val")
    val_dataloader_multi = mrtf.load_data(config, stoi, stoi_MF, single=False, mode="val")

    
    results_dict_bl_ZINC = mrtf.run_test_mns_performance_CLIP_3(config,  
                                                        model_MMT,
                                                        model_CLIP,
                                                        val_dataloader,
                                                        stoi, 
                                                        itos,
                                                        MW_filter)
    results_dict_bl_ZINC, counter = mrtf.filter_invalid_inputs(results_dict_bl_ZINC)
    
    avg_tani_bl_ZINC, html_plot = rbgvm.plot_hist_of_results(results_dict_bl_ZINC)
    
    # Slow because also just takes one at the time
    if greedy_full == True:
        results_dict_greedy_bl_ZINC, failed_bl_ZINC = mrtf.run_test_performance_CLIP_greedy_3(config,  
                                                                stoi, 
                                                                stoi_MF, 
                                                                itos, 
                                                                itos_MF)

        avg_tani_greedy_bl_ZINC, html_plot_greedy = rbgvm.plot_hist_of_results_greedy(results_dict_greedy_bl_ZINC)

    else: 
        config, results_dict_ZINC_greedy_bl = mrtf.run_greedy_sampling(config, model_MMT, val_dataloader_multi, itos, stoi)
        avg_tani_greedy_bl_ZINC = results_dict_ZINC_greedy_bl["tanimoto_mean"]
    
    total_results_bl_ZINC = mrtf.run_test_performance_CLIP_3(config, 
                                                        model_MMT, 
                                                        val_dataloader,
                                                        stoi)
    
    corr_sampleing_prob_bl_ZINC = total_results_bl_ZINC["statistics_multiplication_avg"][0]
    print("avg_tani, avg_tani_greedy, corr_sampleing_prob'")
    print(avg_tani_bl_ZINC, avg_tani_greedy_bl_ZINC, corr_sampleing_prob_bl_ZINC)       

    

In [ ]:

# Save variables to a pickle file
variables_to_save = {
    'avg_tani_bl_ZINC': avg_tani_bl_ZINC,
    'results_dict_greedy_bl_ZINC': results_dict_greedy_bl_ZINC if greedy_full else None,
    'failed_bl_ZINC': failed_bl_ZINC if greedy_full else None,
    'avg_tani_greedy_bl_ZINC': avg_tani_greedy_bl_ZINC,
    'results_dict_ZINC_greedy_bl': results_dict_ZINC_greedy_bl if not greedy_full else None,
    'total_results_bl_ZINC': total_results_bl_ZINC,
    'corr_sampleing_prob_bl_ZINC': corr_sampleing_prob_bl_ZINC,
    'filtered_results': filtered_results,
    'results_dict_bl_ZINC': results_dict_bl_ZINC,
}

with open(os.path.abspath('/projects/cc/se_users/knlr326/1_NMR_project/2_Notebooks/nmr_project/1_Dataexploration/2_paper_code/Experiments_SLURM/20.0_SLURM_MasterTransformer/Figures_Paper_2/precomputed_raw_data/20240520_Improvment_Cycle_PC_v2/6.2_PC_FT_Model_0_250.pkl', 'wb') as f:
    pickle.dump(variables_to_save, f)

#### 5.2 Load ZINC

In [ ]:
def mns_tani_extraction(data):
    """
    Extracts the 4th item of the first sublist of the value for each key in the dictionary.

    Parameters:
    data (dict): The input dictionary.

    Returns:
    list: A list containing the 4th item of the first sublist for each key.
    """
    # Initialize an empty list to store the results
    result_list = []

    # Iterate over the dictionary items
    for key, value in data.items():
        # Check if the value is not None and has at least one sublist
        if value and value[0]:
            # Get the 4th item of the first sublist
            item = value[0][0][3]
            # Append the item to the result list
            result_list.append(item)

    # Return the result list
    return result_list

In [ ]:
'''
# 100_10
import pickle

file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240517_Improvment_Cycle_v3/6.1_imp_cyc_100_10.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_10_before = pickle.load(file)
    
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240517_Improvment_Cycle_v3/6.1_imp_cyc_100_10_after.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_10_after = pickle.load(file)
    
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240517_Improvment_Cycle_v3/6.1_imp_cyc_100_10_after_MNS_3.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_10_after_mns = pickle.load(file)


# 100_30
import pickle

file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240517_Improvment_Cycle_v3/6.1_imp_cyc_100_30.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_30_before = pickle.load(file)
    
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240517_Improvment_Cycle_v3/6.1_imp_cyc_100_30_after.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_30_after = pickle.load(file)
    
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240517_Improvment_Cycle_v3/6.1_imp_cyc_100_30_after_MNS_3.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_30_after_mns = pickle.load(file) 
        
    
# 100_50
import pickle

file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240517_Improvment_Cycle_v3/6.1_imp_cyc_100_50.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_50_before = pickle.load(file)
    
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240517_Improvment_Cycle_v3/6.1_imp_cyc_100_50_after.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_50_after = pickle.load(file)
    
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240517_Improvment_Cycle_v3/6.1_imp_cyc_100_50_after_MNS_3.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_50_after_mns = pickle.load(file) 
    
# 100_100
import pickle

file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240517_Improvment_Cycle_v3/6.1_imp_cyc_100_100.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_100_before = pickle.load(file)
    
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240517_Improvment_Cycle_v3/6.1_imp_cyc_100_100_after.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_100_after = pickle.load(file)
    
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240517_Improvment_Cycle_v3/6.1_imp_cyc_100_100_after_MNS_3.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_100_after_mns = pickle.load(file)     
'''

##### Load ZINC data experiment

In [ ]:
# 100_10
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240517_Improvment_Cycle_v3/6.2_imp_cyc_all_100_10.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_10_before = pickle.load(file)
    
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240517_Improvment_Cycle_v3/6.2_imp_cyc_all_100_10_after.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_10_after = pickle.load(file)
    
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240517_Improvment_Cycle_v3/6.2_imp_cyc_all_100_10_after_MNS_3.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_10_after_mns = pickle.load(file)


# 100_30
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240517_Improvment_Cycle_v3/6.2_imp_cyc_all_100_30.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_30_before = pickle.load(file)
    
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240517_Improvment_Cycle_v3/6.2_imp_cyc_all_100_30_after.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_30_after = pickle.load(file)
    
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240517_Improvment_Cycle_v3/6.2_imp_cyc_all_100_30_after_MNS_3.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_30_after_mns = pickle.load(file) 
        
    
# 100_50
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240517_Improvment_Cycle_v3/6.2_imp_cyc_all_100_50.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_50_before = pickle.load(file)
    
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240517_Improvment_Cycle_v3/6.2_imp_cyc_all_100_50_after.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_50_after = pickle.load(file)
    
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240517_Improvment_Cycle_v3/6.2_imp_cyc_all_100_50_after_MNS_3.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_50_after_mns = pickle.load(file) 
    
# 100_100
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240517_Improvment_Cycle_v3/6.2_imp_cyc_all_100_100_v2.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_100_before = pickle.load(file)
    
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240517_Improvment_Cycle_v3/6.2_imp_cyc_all_100_100_after_MNS_3_v2.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_100_after = pickle.load(file)
    
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240517_Improvment_Cycle_v3/6.2_imp_cyc_all_100_100_after_MNS_3_v2.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_100_after_mns = pickle.load(file)     

In [ ]:
tani_before_all_100_10 = imp_cyc_all_100_10_before["results_dict_ZINC_greedy_bl"]["tanimoto_sim"]
tani_after_all_100_10 = imp_cyc_all_100_10_after["results_dict_ZINC_greedy_bl"]["tanimoto_sim"]
tani_after_all_100_10_mns = mns_tani_extraction(imp_cyc_all_100_10_after_mns["results_dict_bl_ZINC"])
tani_before_all_100_30 = imp_cyc_all_100_30_before["results_dict_ZINC_greedy_bl"]["tanimoto_sim"]
tani_after_all_100_30 = imp_cyc_all_100_30_after["results_dict_ZINC_greedy_bl"]["tanimoto_sim"]
tani_after_all_100_30_mns = mns_tani_extraction(imp_cyc_all_100_30_after_mns["results_dict_bl_ZINC"])
tani_before_all_100_50 = imp_cyc_all_100_50_before["results_dict_ZINC_greedy_bl"]["tanimoto_sim"]
tani_after_all_100_50 = imp_cyc_all_100_50_after["results_dict_ZINC_greedy_bl"]["tanimoto_sim"]
tani_after_all_100_50_mns = mns_tani_extraction(imp_cyc_all_100_50_after_mns["results_dict_bl_ZINC"])
tani_before_all_100_100 = imp_cyc_all_100_100_before["results_dict_ZINC_greedy_bl"]["tanimoto_sim"]
tani_after_all_100_100 = imp_cyc_all_100_100_after["results_dict_ZINC_greedy_bl"]["tanimoto_sim"]
tani_after_all_100_100_mns = mns_tani_extraction(imp_cyc_all_100_100_after_mns["results_dict_bl_ZINC"])

In [ ]:
imp_cyc_all_100_10_before["results_dict_ZINC_greedy_bl"].keys()

In [ ]:

fontsize = 26
tani_data = [
    (tani_before_all_100_10, 'Greedy Tanomoto Before FT10'), (tani_after_all_100_10, 'Greedy Tanomoto After FT10'), (tani_after_all_100_10_mns, '3 Multinomial Sampling After FT10'),
    (tani_before_all_100_30, 'Greedy Tanomoto Before FT30'), (tani_after_all_100_30, 'Greedy Tanomoto After FT30'), (tani_after_all_100_30_mns, '3 Multinomial Sampling After FT30'),
    (tani_before_all_100_50, 'Greedy Tanomoto Before FT50'), (tani_after_all_100_50, 'Greedy Tanomoto After FT50'), (tani_after_all_100_50_mns, '3 Multinomial Sampling After FT50'), 
    (tani_before_all_100_50, 'Greedy Tanomoto Before FT100'), (tani_after_all_100_100, 'Greedy Tanomoto After FT50'), (tani_after_all_100_100_mns, '3 Multinomial Sampling After FT100')
]

# Extract numerical data (count of perfect matches)
numerical_data = [np.sum(np.array(data) == 1.0) for data, _ in tani_data]

# Labels for each group
group_labels = ['MMST IC-10', 'MMST IC-30', 'MMST IC-50', 'MMST IC-100']
condition_labels = ['Before IC', 'After IC', 'MNS: 3']

# Colors for the bars
colors = ['#A1C8F3', '#FFB381', '#8BE5A0']  # Red, Blue, Green

# Create the plot
fig, ax = plt.subplots(figsize=(17, 9))

# Set the width of each bar and the spacing between groups
bar_width = 0.25
group_spacing = 0.05

# Calculate positions for each bar
num_groups = len(group_labels)
indices = np.arange(num_groups)
positions = [indices + i * (bar_width + group_spacing) for i in range(3)]

# Create grouped bar plot
for i in range(3):  # Three conditions: Before, After, MNS
    ax.bar(positions[i], numerical_data[i::3], 
           width=bar_width, color=colors[i], edgecolor='black', 
           label=condition_labels[i])

# Customize the plot
ax.set_ylabel('Number of Perfect Matches (Tanimoto = 1)', fontsize=fontsize)
ax.set_title('Tanimoto Matches Across Different Conditions: ZINC 250-350 Da', fontsize=fontsize)
ax.set_xlabel('Number of Molecule Analogues for Improvement Cycle', fontsize=fontsize)

# Set x-ticks in the middle of each group
group_centers = indices + bar_width
ax.set_xticks(group_centers)
ax.set_xticklabels(group_labels, fontsize=fontsize)

# Add value labels on top of each bar
for i, v in enumerate(numerical_data):
    ax.text(positions[i % 3][i // 3], v, str(v), ha='center', va='bottom', fontsize=fontsize)

# Add legend
ax.legend(fontsize=fontsize, loc='upper left')
ax.tick_params(axis='y', labelsize=fontsize)
ax.set_ylim(0, max(numerical_data) * 1.1)  # Set y-axis limit to 110% of max value

# Adjust layout and display
plt.tight_layout()

# Specify the path and file name where you want to save the figure.
save_path = './_FIGURES/6.1.2_ZINC_Bar_Chart_Greedy_MNS_Tanimoto_v4.png'
plt.savefig(save_path, format='png', dpi=300)  # Save as PNG with 300 dpi

plt.show()

In [ ]:
# Extract the variables

corr_sampleing_prob_bl_ZINC = imp_cyc_all_100_10_before['corr_sampleing_prob_bl_ZINC']
results_dict_ZINC_greedy_bl = imp_cyc_all_100_10_before['results_dict_ZINC_greedy_bl']
results_dict_greedy_bl_ZINC = imp_cyc_all_100_10_before['results_dict_greedy_bl_ZINC']


#avg_tani_bl_ZINC = imp_cyc_all_100_10_before['avg_tani_bl_ZINC']
#failed_bl_ZINC = imp_cyc_all_100_10_before['failed_bl_ZINC']
#avg_tani_greedy_bl_ZINC = imp_cyc_all_100_10_before['avg_tani_greedy_bl_ZINC']
#total_results_bl_ZINC = imp_cyc_all_100_10_before['total_results_bl_ZINC']
#filtered_results = imp_cyc_all_100_10_before['filtered_results']

In [ ]:
imp_cyc_all_100_10_before.keys()

In [ ]:
corr_sp_before_all_100_10 = imp_cyc_all_100_10_before["corr_sampleing_prob_bl_ZINC"]
corr_sp_after_all_100_10 = imp_cyc_all_100_10_after["corr_sampleing_prob_bl_ZINC"]
corr_sp_before_all_100_30 = imp_cyc_all_100_30_before["corr_sampleing_prob_bl_ZINC"]
corr_sp_after_all_100_30 = imp_cyc_all_100_30_after["corr_sampleing_prob_bl_ZINC"]
corr_sp_before_all_100_50 = imp_cyc_all_100_50_before["corr_sampleing_prob_bl_ZINC"]
corr_sp_after_all_100_50 = imp_cyc_all_100_50_after["corr_sampleing_prob_bl_ZINC"]
corr_sp_before_all_100_100 = imp_cyc_all_100_100_before["corr_sampleing_prob_bl_ZINC"]
corr_sp_after_all_100_100 = imp_cyc_all_100_100_after["corr_sampleing_prob_bl_ZINC"]



In [ ]:

# Data for the bar chart
values = [
    corr_sp_before_all_100_10,  # Before (same for all)
    corr_sp_after_all_100_10,   # After FT10
    corr_sp_after_all_100_30,   # After FT30
    corr_sp_after_all_100_50,   # After FT50
    corr_sp_after_all_100_100   # After FT100
]

# Labels for the bars
labels = [
    'MMST Model', 
    'IC-10', 
    'IC-30', 
    'IC-50', 
    'IC-100'
]

# Colors for the bars (based on provided image)
colors = [
        '#A1C8F3',  # light blue/periwinkle
    '#FFB381',  # salmon/peach
    '#8BE5A0',  # mint green
    '#FF9D9A',  # coral pink
    '#D1B9FE',  # lavender
    '#DEBA9A',  # beige/tan
    '#FCAEE3',  # pink
    '#CFCECE',  # light gray
    '#FEFDA2',  # pale yellow
    '#B8F1EF',  # light cyan/aqua
      ]

# Create a bar chart
fig, ax = plt.subplots(figsize=(17, 9))
bars = ax.bar(labels, values, color=colors, edgecolor='black')

# Add numbers inside the bars
for bar, value in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() / 2, f'{value:.2f}', ha='center', va='center', fontsize=fontsize, color='black')

# Add title and labels
plt.title('Averaged Correct Sample Probability: ZINC 250-350 Da', fontsize=fontsize)
plt.ylabel('Averaged Correct Sample Probability', fontsize=fontsize)
ax.set_xlabel('Number of Molecule Analogues for Improvment Cycle', fontsize=fontsize)

#plt.xlabel('Stages', fontsize=16)

# Increase the size of the labels on the x-axis
ax.set_xticklabels(labels, fontsize=fontsize, rotation=0)

# Increase the size of the ticks on the y-axis
ax.tick_params(axis='y', labelsize=fontsize)

# Save the plot to a specified location
output_path = os.path.abspath("./_FIGURES/6.2.2_ZINC_correct_sample_prob_comparison_v3.png")
plt.savefig(output_path, bbox_inches='tight')

# Show the plot
plt.show()


##### Chemical spaces

In [ ]:
tani_before_all_100_10 = imp_cyc_all_100_10_before["results_dict_ZINC_greedy_bl"]["tanimoto_sim"]
tani_after_all_100_10 = imp_cyc_all_100_10_after["results_dict_ZINC_greedy_bl"]["tanimoto_sim"]
tani_after_all_100_10_mns = mns_tani_extraction(imp_cyc_all_100_10_after_mns["results_dict_bl_ZINC"])
tani_before_all_100_30 = imp_cyc_all_100_30_before["results_dict_ZINC_greedy_bl"]["tanimoto_sim"]
tani_after_all_100_30 = imp_cyc_all_100_30_after["results_dict_ZINC_greedy_bl"]["tanimoto_sim"]
tani_after_all_100_30_mns = mns_tani_extraction(imp_cyc_all_100_30_after_mns["results_dict_bl_ZINC"])
tani_before_all_100_50 = imp_cyc_all_100_50_before["results_dict_ZINC_greedy_bl"]["tanimoto_sim"]
tani_after_all_100_50 = imp_cyc_all_100_50_after["results_dict_ZINC_greedy_bl"]["tanimoto_sim"]
tani_after_all_100_50_mns = mns_tani_extraction(imp_cyc_all_100_50_after_mns["results_dict_bl_ZINC"])
tani_before_all_100_100 = imp_cyc_all_100_100_before["results_dict_ZINC_greedy_bl"]["tanimoto_sim"]
tani_after_all_100_100 = imp_cyc_all_100_100_after["results_dict_ZINC_greedy_bl"]["tanimoto_sim"]
tani_after_all_100_100_mns = mns_tani_extraction(imp_cyc_all_100_100_after_mns["results_dict_bl_ZINC"])

In [ ]:
results_dict_bl_ZINC_before = imp_cyc_all_100_10_before["results_dict_bl_ZINC"]
filtered_results_false_before = [key for key, value in results_dict_bl_ZINC_before.items() if value[0][0][-2] == 1]
len(filtered_results_false_before)

In [ ]:
results_dict_bl_ZINC_after = imp_cyc_all_100_10_after["results_dict_bl_ZINC"]
filtered_results_false_after = [key for key, value in results_dict_bl_ZINC_after.items() if value[0][0][-2] == 1]
len(filtered_results_false_after)

In [ ]:
all_gen_smis = imp_cyc_all_100_10_before["all_gen_smis"]

In [ ]:
# Convert SMILES to fingerprints
def smiles_to_fps(smiles_list):
    fps = []
    for smiles in tqdm(smiles_list):
        mol = Chem.MolFromSmiles(smiles)
        if mol is not None:
            fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=512)
            fps.append(fp)
    return np.array(fps)

fps1 = smiles_to_fps(filtered_results_false_before)
fps2 = smiles_to_fps(all_gen_smis)
fps3 = smiles_to_fps(filtered_results_false_after)

# Concatenate the two fingerprint arrays
all_fps = np.vstack([fps1, fps2, fps3])

# Dimensionality Reduction: t-SNE
tsne = TSNE(n_components=2, random_state=0)
X_tsne = tsne.fit_transform(all_fps)

# Dimensionality Reduction: UMAP
umap_model = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2)
X_umap = umap_model.fit_transform(all_fps)

# Dimensionality Reduction: PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(all_fps)


# Plotting function with the legend below the graph
def plot_2D(X, title, label1='Molecules Correct before', label2='Generated molecules', label3='Molecules Correct after'):
    plt.figure(figsize=(10, 8))
    plt.scatter(X[len(fps1):len(fps1)+len(fps2), 0], X[len(fps1):len(fps1)+len(fps2), 1], c='g', marker='^', label=label2, alpha=0.1)
    plt.scatter(X[:len(fps1), 0], X[:len(fps1), 1], c='b', marker='o', label=label1, alpha=1, s=50)
    plt.scatter(X[len(fps1)+len(fps2):, 0], X[len(fps1)+len(fps2):, 1], c='r', marker='s', label=label3, alpha=1, s=10)
    plt.title(title, fontsize=20)
    plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.05), ncol=3, fontsize=14)  
    output_path = "./_FIGURES/6.1.2_t-SNE_FT10_v1.png"
    plt.savefig(output_path, bbox_inches='tight')
    plt.show()

# Plot t-SNE
plot_2D(X_tsne, 't-SNE Plot: FT10')

# Plot UMAP
#plot_2D(X_umap, 'UMAP Plot')

# Plot PCA
#plot_2D(X_pca, 'PCA Plot')

#### 6.0 Load PubChem

##### Load PubChem 0-250 v1

In [ ]:
# 100_10
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240520_Improvment_Cycle_PC_0_250/6.2_imp_cyc_all_100_10.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_10_before = pickle.load(file)
    
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240520_Improvment_Cycle_PC_0_250/6.2_imp_cyc_all_100_10_after.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_10_after = pickle.load(file)
    
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240520_Improvment_Cycle_PC_0_250/6.2_imp_cyc_all_100_10_after_MNS_3.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_10_after_mns = pickle.load(file)

# 100_30
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240520_Improvment_Cycle_PC_0_250/6.2_imp_cyc_all_100_30.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_30_before = pickle.load(file)
    
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240520_Improvment_Cycle_PC_0_250/6.2_imp_cyc_all_100_30_after.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_30_after = pickle.load(file)
    
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240520_Improvment_Cycle_PC_0_250/6.2_imp_cyc_all_100_30_after_MNS_3.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_30_after_mns = pickle.load(file) 
        
# 100_50
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240520_Improvment_Cycle_PC_0_250/6.2_imp_cyc_all_100_50.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_50_before = pickle.load(file)
    
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240520_Improvment_Cycle_PC_0_250/6.2_imp_cyc_all_100_50_after.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_50_after = pickle.load(file)
    
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240520_Improvment_Cycle_PC_0_250/6.2_imp_cyc_all_100_50_after_MNS_3.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_50_after_mns = pickle.load(file) 
    
# 100_100
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240520_Improvment_Cycle_PC_0_250/6.2_imp_cyc_all_100_100.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_100_before = pickle.load(file)
    
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240520_Improvment_Cycle_PC_0_250/6.2_imp_cyc_all_100_100_after.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_100_after = pickle.load(file)
    
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240520_Improvment_Cycle_PC_0_250/6.2_imp_cyc_all_100_100_after_MNS_3.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_100_after_mns = pickle.load(file)     

In [ ]:
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240520_Improvment_Cycle_PC_250_350/6.2_PC_FT_Model_0_250.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    PC_0_250_FT = pickle.load(file)     

In [ ]:
tani_before_all_100_10 = imp_cyc_all_100_10_before["results_dict_ZINC_greedy_bl"]["tanimoto_sim"]
tani_after_all_100_10 = imp_cyc_all_100_10_after["results_dict_ZINC_greedy_bl"]["tanimoto_sim"]
tani_after_all_100_10_mns = mns_tani_extraction(imp_cyc_all_100_10_after_mns["results_dict_bl_ZINC"])
tani_before_all_100_30 = imp_cyc_all_100_30_before["results_dict_ZINC_greedy_bl"]["tanimoto_sim"]
tani_after_all_100_30 = imp_cyc_all_100_30_after["results_dict_ZINC_greedy_bl"]["tanimoto_sim"]
tani_after_all_100_30_mns = mns_tani_extraction(imp_cyc_all_100_30_after_mns["results_dict_bl_ZINC"])
tani_before_all_100_50 = imp_cyc_all_100_50_before["results_dict_ZINC_greedy_bl"]["tanimoto_sim"]
tani_after_all_100_50 = imp_cyc_all_100_50_after["results_dict_ZINC_greedy_bl"]["tanimoto_sim"]
tani_after_all_100_50_mns = mns_tani_extraction(imp_cyc_all_100_50_after_mns["results_dict_bl_ZINC"])
tani_before_all_100_100 = imp_cyc_all_100_100_before["results_dict_ZINC_greedy_bl"]["tanimoto_sim"]
tani_after_all_100_100 = imp_cyc_all_100_100_after["results_dict_ZINC_greedy_bl"]["tanimoto_sim"]
tani_after_all_100_100_mns = mns_tani_extraction(imp_cyc_all_100_100_after_mns["results_dict_bl_ZINC"])
tani_PC_0_250_FT = PC_0_250_FT["results_dict_ZINC_greedy_bl"]["tanimoto_sim"]


In [ ]:

# Data and labels
tani_data = [
    (tani_before_all_100_10, 'FT10 Before'), (tani_after_all_100_10, 'FT10 After'), (tani_after_all_100_10_mns, 'FT10 MNS'),
    (tani_before_all_100_30, 'FT30 Before'), (tani_after_all_100_30, 'FT30 After'), (tani_after_all_100_30_mns, 'FT30 MNS'),
    (tani_before_all_100_50, 'FT50 Before'), (tani_after_all_100_50, 'FT50 After'), (tani_after_all_100_50_mns, 'FT50 MNS'), 
    (tani_before_all_100_100, 'FT100 Before'), (tani_after_all_100_100, 'FT100 After'), (tani_after_all_100_100_mns, 'FT100 MNS')
]

# Add the PubChem Fine-Tuned data
tani_PC_0_250_FT_list = [tani_PC_0_250_FT for _ in range(4)]  # Repeat the data for each group

# Count perfect matches (Tanimoto = 1) for each condition
perfect_matches = [np.sum(np.array(data) == 1) for data, _ in tani_data]
perfect_matches_pc = [np.sum(np.array(data) == 1) for data in tani_PC_0_250_FT_list]

# Combine all perfect matches
all_perfect_matches = []
for i in range(4):  # For each FT group (10, 30, 50, 100)
    all_perfect_matches.extend([perfect_matches_pc[i]] + perfect_matches[i*3:(i+1)*3])

# Colors for the bars
colors = [
    '#A1C8F3',  # light blue/periwinkle
    '#FFB381',  # salmon/peach
    '#8BE5A0',  # mint green
    '#FF9D9A',  # coral pink
    '#D1B9FE',  # lavender
    '#DEBA9A',  # beige/tan
    '#FCAEE3',  # pink
    '#CFCECE',  # light gray
    '#FEFDA2',  # pale yellow
    '#B8F1EF',  # light cyan/aqua
]

# Labels
group_labels = ['MMST IC-10', 'MMST IC-30', 'MMST IC-50', 'MMST IC-100']
condition_labels = ['PubChem FT', 'Before IC', 'After IC', 'MNS: 3']

# Create the plot
fig, ax = plt.subplots(figsize=(17, 9))

# Set the width of each bar and the spacing between groups
bar_width = 0.2
group_spacing = 0.03

# Calculate positions for each bar
num_groups = len(group_labels)
indices = np.arange(num_groups)
positions = [indices + i * (bar_width + group_spacing) for i in range(4)]

# Create grouped bar plot
for i in range(4):  # Four conditions: PubChem FT, Before, After, MNS
    ax.bar(positions[i], all_perfect_matches[i::4], 
           width=bar_width, color=colors[i], edgecolor='black', 
           label=condition_labels[i])

# Customize the plot
ax.set_ylabel('Number of Perfect Matches (Tanimoto = 1)', fontsize=fontsize)
ax.set_title('Tanimoto Matches Across Different Conditions: PubChem 0-250 Da', fontsize=fontsize)
ax.set_xlabel('Number of Molecule Analogues for Improvment Cycle', fontsize=fontsize)

# Set x-ticks in the middle of each group
group_centers = indices + 1.5 * bar_width + 0.5 * group_spacing
ax.set_xticks(group_centers)
ax.set_xticklabels(group_labels, fontsize=fontsize)

# Add value labels on top of each bar
for i, v in enumerate(all_perfect_matches):
    ax.text(positions[i % 4][i // 4], v, str(v), ha='center', va='bottom', fontsize=fontsize)

# Add legend
ax.legend(fontsize=fontsize, loc='upper left')
ax.tick_params(axis='y', labelsize=fontsize)
#ax.set_ylim(0, 105)

# Adjust layout and display
plt.tight_layout()
save_path = os.path.abspath('./_FIGURES/6.2.2_PC_Bar_Chart_Greedy_MNS_Tanimoto_0_250_v4.png')  # Update this path as needed

# Save the figure
plt.savefig(save_path, format='png', dpi=300)  # Save as PNG with 300 dpi

plt.show()

# Uncomment to save the plot
# plt.savefig("/path/to/save/grouped_perfect_matches_bar_plot.png", dpi=300, bbox_inches='tight')

In [ ]:
corr_sp_before_all_100_10 = imp_cyc_all_100_10_before["corr_sampleing_prob_bl_ZINC"]
corr_sp_after_all_100_10 = imp_cyc_all_100_10_after["corr_sampleing_prob_bl_ZINC"]
corr_sp_before_all_100_30 = imp_cyc_all_100_30_before["corr_sampleing_prob_bl_ZINC"]
corr_sp_after_all_100_30 = imp_cyc_all_100_30_after["corr_sampleing_prob_bl_ZINC"]
corr_sp_before_all_100_50 = imp_cyc_all_100_50_before["corr_sampleing_prob_bl_ZINC"]
corr_sp_after_all_100_50 = imp_cyc_all_100_50_after["corr_sampleing_prob_bl_ZINC"]
corr_sp_before_all_100_100 = imp_cyc_all_100_100_before["corr_sampleing_prob_bl_ZINC"]
corr_sp_after_all_100_100 = imp_cyc_all_100_100_after["corr_sampleing_prob_bl_ZINC"]
corr_PC_0_250_FT = PC_0_250_FT["corr_sampleing_prob_bl_ZINC"]


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Data for the bar chart
values = [
    corr_sp_before_all_100_10,  # Before (same for all)
    corr_PC_0_250_FT,         # PC FT
    corr_sp_after_all_100_10,   # After FT10
    corr_sp_after_all_100_30,   # After FT30
    corr_sp_after_all_100_50,   # After FT50
    corr_sp_after_all_100_100   # After FT100
]

# Labels for the bars
labels = [
    'MMST Model', 
    'PC FT', 
    'IC-10', 
    'IC-30', 
    'IC-50', 
    'IC-100'
]

# Colors for the bars (based on provided image)
colors = [
    '#A1C8F3',  # light blue/periwinkle
    '#B8F1EF',  # light cyan/aqua
    '#FFB381',  # salmon/peach
    '#8BE5A0',  # mint green
    '#FF9D9A',  # coral pink
    '#D1B9FE',  # lavender
    '#DEBA9A',  # beige/tan
    '#FCAEE3',  # pink
    '#CFCECE',  # light gray
    '#FEFDA2',  # pale yellow
]

# Create a bar chart
fig, ax = plt.subplots(figsize=(17, 9))
bars = ax.bar(labels, values, color=colors, edgecolor='black')

# Add numbers inside the bars
for bar, value in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() / 2, f'{value:.2f}', ha='center', va='center', fontsize=fontsize, color='black')

# Add title and labels
plt.title('Averaged Correct Sample Probability: PubChem 0-250 Da', fontsize=fontsize)
plt.ylabel('Averaged Correct Sample Probability', fontsize=fontsize)
ax.set_xlabel('Number of Molecule Analogues for Improvment Cycle', fontsize=fontsize)

# Increase the size of the labels on the x-axis
ax.set_xticklabels(labels, fontsize=fontsize, rotation=0)

# Increase the size of the ticks on the y-axis
ax.tick_params(axis='y', labelsize=21)

# Save the plot to a specified location
output_path = os.path.abspath("./_FIGURES/6.2.2_PC_correct_sample_prob_comparison_0_250_v3.png")
plt.savefig(output_path, bbox_inches='tight')

# Show the plot
plt.show()


##### Load PubChem 250-350 v2

In [ ]:
# 100_10
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240520_Improvment_Cycle_PC_250_350/6.2_imp_cyc_all_100_10.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_10_before = pickle.load(file)
    
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240520_Improvment_Cycle_PC_250_350/6.2_imp_cyc_all_100_10_after.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_10_after = pickle.load(file)
    
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240520_Improvment_Cycle_PC_250_350/6.2_imp_cyc_all_100_10_after_MNS_3.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_10_after_mns = pickle.load(file)


# 100_30
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240520_Improvment_Cycle_PC_250_350/6.2_imp_cyc_all_100_30.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_30_before = pickle.load(file)
    
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240520_Improvment_Cycle_PC_250_350/6.2_imp_cyc_all_100_30_after.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_30_after = pickle.load(file)
    
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240520_Improvment_Cycle_PC_250_350/6.2_imp_cyc_all_100_30_after_MNS_3.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_30_after_mns = pickle.load(file) 
        
    
# 100_50
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240520_Improvment_Cycle_PC_250_350/6.2_imp_cyc_all_100_50.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_50_before = pickle.load(file)
    
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240520_Improvment_Cycle_PC_250_350/6.2_imp_cyc_all_100_50_after.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_50_after = pickle.load(file)
    
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240520_Improvment_Cycle_PC_250_350/6.2_imp_cyc_all_100_50_after_MNS_3.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_50_after_mns = pickle.load(file) 
    
# 100_100
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240520_Improvment_Cycle_PC_250_350/6.2_imp_cyc_all_100_100.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_100_before = pickle.load(file)
    
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240520_Improvment_Cycle_PC_250_350/6.2_imp_cyc_all_100_100_after.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_100_after = pickle.load(file)
    
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240520_Improvment_Cycle_PC_250_350/6.2_imp_cyc_all_100_100_after_MNS_3.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_100_after_mns = pickle.load(file)     

In [ ]:
    
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240520_Improvment_Cycle_PC_250_350/6.2_PC_FT_Model_250_350.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    PC_250_350_FT = pickle.load(file)     

In [ ]:
tani_before_all_100_10 = imp_cyc_all_100_10_before["results_dict_ZINC_greedy_bl"]["tanimoto_sim"]
tani_after_all_100_10 = imp_cyc_all_100_10_after["results_dict_ZINC_greedy_bl"]["tanimoto_sim"]
tani_after_all_100_10_mns = mns_tani_extraction(imp_cyc_all_100_10_after_mns["results_dict_bl_ZINC"])
tani_before_all_100_30 = imp_cyc_all_100_30_before["results_dict_ZINC_greedy_bl"]["tanimoto_sim"]
tani_after_all_100_30 = imp_cyc_all_100_30_after["results_dict_ZINC_greedy_bl"]["tanimoto_sim"]
tani_after_all_100_30_mns = mns_tani_extraction(imp_cyc_all_100_30_after_mns["results_dict_bl_ZINC"])
tani_before_all_100_50 = imp_cyc_all_100_50_before["results_dict_ZINC_greedy_bl"]["tanimoto_sim"]
tani_after_all_100_50 = imp_cyc_all_100_50_after["results_dict_ZINC_greedy_bl"]["tanimoto_sim"]
tani_after_all_100_50_mns = mns_tani_extraction(imp_cyc_all_100_50_after_mns["results_dict_bl_ZINC"])
tani_before_all_100_100 = imp_cyc_all_100_100_before["results_dict_ZINC_greedy_bl"]["tanimoto_sim"]
tani_after_all_100_100 = imp_cyc_all_100_100_after["results_dict_ZINC_greedy_bl"]["tanimoto_sim"]
tani_after_all_100_100_mns = mns_tani_extraction(imp_cyc_all_100_100_after_mns["results_dict_bl_ZINC"])
tani_PC_250_350_FT = PC_250_350_FT["results_dict_ZINC_greedy_bl"]["tanimoto_sim"]


In [ ]:

fontsize = 26
# Data and labels
tani_data = [
    (tani_before_all_100_10, 'FT10 Before'), (tani_after_all_100_10, 'FT10 After'), (tani_after_all_100_10_mns, 'FT10 MNS'),
    (tani_before_all_100_30, 'FT30 Before'), (tani_after_all_100_30, 'FT30 After'), (tani_after_all_100_30_mns, 'FT30 MNS'),
    (tani_before_all_100_50, 'FT50 Before'), (tani_after_all_100_50, 'FT50 After'), (tani_after_all_100_50_mns, 'FT50 MNS'), 
    (tani_before_all_100_100, 'FT100 Before'), (tani_after_all_100_100, 'FT100 After'), (tani_after_all_100_100_mns, 'FT100 MNS')
]

# Add the PubChem Fine-Tuned data
tani_PC_250_350_FT_list = [tani_PC_250_350_FT for _ in range(4)]  # Repeat the data for each group

# Count perfect matches (Tanimoto = 1) for each condition
perfect_matches = [np.sum(np.array(data) == 1) for data, _ in tani_data]
perfect_matches_pc = [np.sum(np.array(data) == 1) for data in tani_PC_250_350_FT_list]

# Combine all perfect matches
all_perfect_matches = []
for i in range(4):  # For each FT group (10, 30, 50, 100)
    all_perfect_matches.extend([perfect_matches_pc[i]] + perfect_matches[i*3:(i+1)*3])

# Colors for the bars
condition_labels = ['PubChem FT', 'Before IC', 'After IC', 'MNS: 3']

# Labels
group_labels = ['MMST IC-10', 'MMST IC-30', 'MMST IC-50', 'MMST IC-100']
colors = [ '#A1C8F3',  # light blue/periwinkle
    '#FFB381',  # salmon/peach
    '#8BE5A0',  # mint green
    '#FF9D9A',  # coral pink
    '#D1B9FE',  # lavender
    '#DEBA9A',  # beige/tan
    '#FCAEE3',  # pink
    '#CFCECE',  # light gray
    '#FEFDA2',  # pale yellow
    '#B8F1EF',  # light cyan/aqua
]

# Create the plot
fig, ax = plt.subplots(figsize=(17, 9))

# Set the width of each bar and the spacing between groups
bar_width = 0.2
group_spacing = 0.03

# Calculate positions for each bar
num_groups = len(group_labels)
indices = np.arange(num_groups)
positions = [indices + i * (bar_width + group_spacing) for i in range(4)]

# Create grouped bar plot
for i in range(4):  # Four conditions: PubChem FT, Before, After, MNS
    ax.bar(positions[i], all_perfect_matches[i::4], 
           width=bar_width, color=colors[i], edgecolor='black', 
           label=condition_labels[i])

# Customize the plot
ax.set_ylabel('Number of Perfect Matches (Tanimoto = 1)', fontsize=fontsize)
ax.set_xlabel('Number of Molecule Analogues for Improvement Cycle', fontsize=fontsize)
ax.set_title('Tanimoto Matches Across Different Conditions: PubChem 250-350 Da', fontsize=fontsize)

# Set x-ticks in the middle of each group
group_centers = indices + 1.5 * bar_width + 0.5 * group_spacing
ax.set_xticks(group_centers)
ax.set_xticklabels(group_labels, fontsize=fontsize)

# Set y-axis tick label font size
ax.tick_params(axis='y', labelsize=fontsize)

# Add value labels on top of each bar
for i, v in enumerate(all_perfect_matches):
    ax.text(positions[i % 4][i // 4], v, str(v), ha='center', va='bottom', fontsize=fontsize)

# Add legend
ax.legend(fontsize=22, loc='upper left')

# Adjust layout and display
plt.tight_layout()

# Save the plot to a specified location
output_path = './_FIGURES/6.2.2_PC_Bar_Chart_Greedy_MNS_Tanimoto_250_350_v3.png'
plt.savefig(output_path, dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
corr_sp_before_all_100_10 = imp_cyc_all_100_10_before["corr_sampleing_prob_bl_ZINC"]
corr_sp_after_all_100_10 = imp_cyc_all_100_10_after["corr_sampleing_prob_bl_ZINC"]
corr_sp_before_all_100_30 = imp_cyc_all_100_30_before["corr_sampleing_prob_bl_ZINC"]
corr_sp_after_all_100_30 = imp_cyc_all_100_30_after["corr_sampleing_prob_bl_ZINC"]
corr_sp_before_all_100_50 = imp_cyc_all_100_50_before["corr_sampleing_prob_bl_ZINC"]
corr_sp_after_all_100_50 = imp_cyc_all_100_50_after["corr_sampleing_prob_bl_ZINC"]
corr_sp_before_all_100_100 = imp_cyc_all_100_100_before["corr_sampleing_prob_bl_ZINC"]
corr_sp_after_all_100_100 = imp_cyc_all_100_100_after["corr_sampleing_prob_bl_ZINC"]
corr_PC_250_350_FT = PC_250_350_FT["corr_sampleing_prob_bl_ZINC"]


In [ ]:

# Data for the bar chart
values = [
    corr_sp_before_all_100_10,  # Before (same for all)
    corr_PC_250_350_FT,         # PC FT
    corr_sp_after_all_100_10,   # After FT10
    corr_sp_after_all_100_30,   # After FT30
    corr_sp_after_all_100_50,   # After FT50
    corr_sp_after_all_100_100   # After FT100
]

# Labels for the bars
labels = [
    'MMST Model', 
    'PC FT', 
    'IC-10', 
    'IC-30', 
    'IC-50', 
    'IC-100'
]

# Colors for the bars (based on provided image)
colors = [
    '#A1C8F3',  # light blue/periwinkle
    '#B8F1EF',  # light cyan/aqua
    '#FFB381',  # salmon/peach
    '#8BE5A0',  # mint green
    '#FF9D9A',  # coral pink
    '#D1B9FE',  # lavender
    '#DEBA9A',  # beige/tan
    '#FCAEE3',  # pink
    '#CFCECE',  # light gray
    '#FEFDA2',  # pale yellow
]

# Create a bar chart
fig, ax = plt.subplots(figsize=(17, 9))
bars = ax.bar(labels, values, color=colors, edgecolor='black')

# Add numbers inside the bars
for bar, value in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() / 2, f'{value:.2f}', ha='center', va='center', fontsize=fontsize, color='black')

# Add title and labels
plt.title('Averaged Correct Sample Probability: PubChem 250-350 Da', fontsize=fontsize)
plt.ylabel('Averaged Correct Sample Probability', fontsize=fontsize)
ax.set_xlabel('Number of Molecule Analogues for Improvment Cycle', fontsize=fontsize)

# Increase the size of the labels on the x-axis
ax.set_xticklabels(labels, fontsize=fontsize, rotation=0)

# Increase the size of the ticks on the y-axis
ax.tick_params(axis='y', labelsize=fontsize)

# Save the plot to a specified location
output_path = os.path.abspath("./_FIGURES/6.2.2_PC_correct_sample_prob_comparison_250_350_v3.png")
plt.savefig(output_path, bbox_inches='tight')

# Show the plot
plt.show()


##### Load PubChem 350-500

In [ ]:
# 100_10
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240517_Improvment_Cycle_PC_350_500/6.2_imp_cyc_all_100_10.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_10_before = pickle.load(file)
    
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240517_Improvment_Cycle_PC_350_500/6.2_imp_cyc_all_100_10_after.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_10_after = pickle.load(file)
    
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240517_Improvment_Cycle_PC_350_500/6.2_imp_cyc_all_100_30_after_MNS_3.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_10_after_mns = pickle.load(file)


# 100_30
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240517_Improvment_Cycle_PC_350_500/6.2_imp_cyc_all_100_30.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_30_before = pickle.load(file)
    
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240517_Improvment_Cycle_PC_350_500/6.2_imp_cyc_all_100_30_after.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_30_after = pickle.load(file)
    
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240517_Improvment_Cycle_PC_350_500/6.2_imp_cyc_all_100_30_after_MNS_3.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_30_after_mns = pickle.load(file) 
        
    
# 100_50
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240517_Improvment_Cycle_PC_350_500/6.2_imp_cyc_all_100_50.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_50_before = pickle.load(file)
    
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240517_Improvment_Cycle_PC_350_500/6.2_imp_cyc_all_100_50_after.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_50_after = pickle.load(file)
    
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240517_Improvment_Cycle_PC_350_500/6.2_imp_cyc_all_100_50_after_MNS_3.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_50_after_mns = pickle.load(file) 
    
# 100_100
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240517_Improvment_Cycle_PC_350_500/6.2_imp_cyc_all_100_100.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_100_before = pickle.load(file)
    
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240517_Improvment_Cycle_PC_350_500/6.2_imp_cyc_all_100_100_after.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_100_after = pickle.load(file)
    
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240517_Improvment_Cycle_PC_350_500/6.2_imp_cyc_all_100_100_after_MNS_3.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_100_after_mns = pickle.load(file)     

In [ ]:
    
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240520_Improvment_Cycle_PC_250_350/6.2_PC_FT_Model_350_500.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    PC_350_500_FT = pickle.load(file)     

In [ ]:
# 100_10 second fine-tuning step
    
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240522_PC_Imp_Cycle_2_from_350_500/6.2_imp_cyc_all_100_10_after_epoch41.pkl')
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240522_PC_Imp_Cycle_2_from_350_500/6.2_imp_cyc_all_100_10_after_MNS_3.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_10_after_2 = pickle.load(file)
    
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240522_PC_Imp_Cycle_2_from_350_500/6.2_imp_cyc_all_100_10_after_epoch41.pkl')
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240522_PC_Imp_Cycle_2_from_350_500/6.2_imp_cyc_all_100_10_after_MNS_3.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_10_after_mns_2 = pickle.load(file)


In [ ]:
tani_before_all_100_10 = imp_cyc_all_100_10_before["results_dict_ZINC_greedy_bl"]["tanimoto_sim"]
tani_after_all_100_10 = imp_cyc_all_100_10_after["results_dict_ZINC_greedy_bl"]["tanimoto_sim"]
tani_after_all_100_10_mns = mns_tani_extraction(imp_cyc_all_100_10_after_mns["results_dict_bl_ZINC"])

tani_after_all_100_10_2 = imp_cyc_all_100_10_after_2["results_dict_ZINC_greedy_bl"]["tanimoto_sim"]
tani_after_all_100_10_mns_2 = mns_tani_extraction(imp_cyc_all_100_10_after_mns_2["results_dict_bl_ZINC"])


tani_before_all_100_30 = imp_cyc_all_100_30_before["results_dict_ZINC_greedy_bl"]["tanimoto_sim"]
tani_after_all_100_30 = imp_cyc_all_100_30_after["results_dict_ZINC_greedy_bl"]["tanimoto_sim"]
tani_after_all_100_30_mns = mns_tani_extraction(imp_cyc_all_100_30_after_mns["results_dict_bl_ZINC"])
tani_before_all_100_50 = imp_cyc_all_100_50_before["results_dict_ZINC_greedy_bl"]["tanimoto_sim"]
tani_after_all_100_50 = imp_cyc_all_100_50_after["results_dict_ZINC_greedy_bl"]["tanimoto_sim"]
tani_after_all_100_50_mns = mns_tani_extraction(imp_cyc_all_100_50_after_mns["results_dict_bl_ZINC"])
tani_before_all_100_100 = imp_cyc_all_100_100_before["results_dict_ZINC_greedy_bl"]["tanimoto_sim"]
tani_after_all_100_100 = imp_cyc_all_100_100_after["results_dict_ZINC_greedy_bl"]["tanimoto_sim"]
tani_after_all_100_100_mns = mns_tani_extraction(imp_cyc_all_100_100_after_mns["results_dict_bl_ZINC"])
tani_PC_350_500_FT = PC_350_500_FT["results_dict_ZINC_greedy_bl"]["tanimoto_sim"]


In [ ]:

# Data and labels
tani_data = [
    (tani_before_all_100_10, 'FT10 Before'), (tani_after_all_100_10, 'FT10 After'), (tani_after_all_100_10_mns, 'FT10 MNS'),
    #(tani_before_all_100_10, 'FT10 2x Before'), (tani_after_all_100_10_2, 'FT10 2x After'), (tani_after_all_100_10_mns_2, 'FT10 2x MNS'),
    (tani_before_all_100_30, 'FT30 Before'), (tani_after_all_100_30, 'FT30 After'), (tani_after_all_100_30_mns, 'FT30 MNS'),
    (tani_before_all_100_50, 'FT50 Before'), (tani_after_all_100_50, 'FT50 After'), (tani_after_all_100_50_mns, 'FT50 MNS'), 
    (tani_before_all_100_100, 'FT100 Before'), (tani_after_all_100_100, 'FT100 After'), (tani_after_all_100_100_mns, 'FT100 MNS')
]

# Add the PubChem Fine-Tuned data
tani_PC_350_500_FT_list = [tani_PC_350_500_FT for _ in range(5)]  # Repeat the data for each group

# Count perfect matches (Tanimoto = 1) for each condition
perfect_matches = [np.sum(np.array(data) == 1) for data, _ in tani_data]
perfect_matches_pc = [np.sum(np.array(data) == 1) for data in tani_PC_350_500_FT_list]

# Combine all perfect matches
all_perfect_matches = []
for i in range(4):  # For each FT group (10, 10 2x, 30, 50, 100)
    all_perfect_matches.extend([perfect_matches_pc[i]] + perfect_matches[i*3:(i+1)*3])

# Colors for the bars
colors = [    '#A1C8F3',  # light blue/periwinkle
    '#FFB381',  # salmon/peach
    '#8BE5A0',  # mint green
    '#FF9D9A',  # coral pink
    '#D1B9FE',  # lavender
    '#DEBA9A',  # beige/tan
    '#FCAEE3',  # pink
    '#CFCECE',  # light gray
    '#FEFDA2',  # pale yellow
    ]

# Labels
group_labels = ['MMST IC-10',  'MMST IC-30', 'MMST IC-50', 'MMST IC-100'] #'FT10 2x',
condition_labels = ['PubChem FT', 'Before IC', 'After IC', 'MNS: 3']

# Create the plot
fig, ax = plt.subplots(figsize=(17, 9))

# Set the width of each bar and the spacing between groups
bar_width = 0.2
group_spacing = 0.03

# Calculate positions for each bar
num_groups = len(group_labels)
indices = np.arange(num_groups)
positions = [indices + i * (bar_width + group_spacing) for i in range(4)]

# Create grouped bar plot
for i in range(4):  # Four conditions: PubChem FT, Before, After, MNS
    ax.bar(positions[i], all_perfect_matches[i::4], 
           width=bar_width, color=colors[i], edgecolor='black', 
           label=condition_labels[i])

# Customize the plot
ax.set_ylabel('Number of Perfect Matches (Tanimoto = 1)', fontsize=fontsize)
ax.set_xlabel('Number of Molecule Analogues for Improvement Cycle', fontsize=fontsize)
ax.set_title('Tanimoto Matches Across Different Conditions: PubChem 350-500 Da', fontsize=fontsize)

# Set x-ticks in the middle of each group
group_centers = indices + 1.5 * bar_width + 0.5 * group_spacing
ax.set_xticks(group_centers)
ax.set_xticklabels(group_labels, fontsize=fontsize)

# Set y-axis tick label font size
ax.tick_params(axis='y', labelsize=fontsize)

# Add value labels on top of each bar
for i, v in enumerate(all_perfect_matches):
    ax.text(positions[i % 4][i // 4], v, str(v), ha='center', va='bottom', fontsize=fontsize)

# Add legend
ax.legend(fontsize=fontsize, loc='upper left')

# Adjust layout and display
plt.tight_layout()

# Save the plot to a specified location
output_path = "./_FIGURES/6.2.2_PC_Bar_Chart_Greedy_MNS_Tanimoto_350_500_v3.png"
plt.savefig(output_path, dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
# Extract the variables
"""
corr_sampleing_prob_bl_ZINC = imp_cyc_all_100_10_before['corr_sampleing_prob_bl_ZINC']
results_dict_ZINC_greedy_bl = imp_cyc_all_100_10_before['results_dict_ZINC_greedy_bl']
results_dict_greedy_bl_ZINC = imp_cyc_all_100_10_before['results_dict_greedy_bl_ZINC']
tani_PC_350_500_FT = PC_350_500_FT["results_dict_ZINC_greedy_bl"]["tanimoto_sim"]
"""

#avg_tani_bl_ZINC = imp_cyc_all_100_10_before['avg_tani_bl_ZINC']
#failed_bl_ZINC = imp_cyc_all_100_10_before['failed_bl_ZINC']
#avg_tani_greedy_bl_ZINC = imp_cyc_all_100_10_before['avg_tani_greedy_bl_ZINC']
#total_results_bl_ZINC = imp_cyc_all_100_10_before['total_results_bl_ZINC']
#filtered_results = imp_cyc_all_100_10_before['filtered_results']

In [ ]:
corr_sp_before_all_100_10 = imp_cyc_all_100_10_before["corr_sampleing_prob_bl_ZINC"]
corr_sp_after_all_100_10 = imp_cyc_all_100_10_after["corr_sampleing_prob_bl_ZINC"]
corr_sp_after_all_100_10_2 = imp_cyc_all_100_10_after_2["corr_sampleing_prob_bl_ZINC"]
corr_sp_before_all_100_30 = imp_cyc_all_100_30_before["corr_sampleing_prob_bl_ZINC"]
corr_sp_after_all_100_30 = imp_cyc_all_100_30_after["corr_sampleing_prob_bl_ZINC"]
corr_sp_before_all_100_50 = imp_cyc_all_100_50_before["corr_sampleing_prob_bl_ZINC"]
corr_sp_after_all_100_50 = imp_cyc_all_100_50_after["corr_sampleing_prob_bl_ZINC"]
corr_sp_before_all_100_100 = imp_cyc_all_100_100_before["corr_sampleing_prob_bl_ZINC"]
corr_sp_after_all_100_100 = imp_cyc_all_100_100_after["corr_sampleing_prob_bl_ZINC"]
corr_PC_350_500_FT = PC_350_500_FT["corr_sampleing_prob_bl_ZINC"]


In [ ]:

# Data for the bar chart
values = [
    corr_sp_before_all_100_10,  # Before (same for all)
    corr_PC_350_500_FT,         # PC FT
    corr_sp_after_all_100_10,   # After FT10
  #  corr_sp_after_all_100_10_2,  # After 2x FT10
    corr_sp_after_all_100_30,   # After FT30
    corr_sp_after_all_100_50,   # After FT50
    corr_sp_after_all_100_100   # After FT100
]

# Labels for the bars
labels = [
    'MMST Model', 
    'PC FT', 
    'IC-10', 
    'IC-30', 
    'IC-50', 
    'IC-100'
]

# Colors for the bars (based on provided image)
colors = [
    '#A1C8F3',  # light blue/periwinkle
    '#B8F1EF',  # light cyan/aqua
    '#FFB381',  # salmon/peach
    '#8BE5A0',  # mint green
    '#FF9D9A',  # coral pink
    '#D1B9FE',  # lavender
    '#DEBA9A',  # beige/tan
    '#FCAEE3',  # pink
    '#CFCECE',  # light gray
    '#FEFDA2',  # pale yellow
]

# Create a bar chart
fig, ax = plt.subplots(figsize=(17, 9))
bars = ax.bar(labels, values, color=colors, edgecolor='black')

# Add numbers inside the bars
for bar, value in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() / 2, f'{value:.2f}', ha='center', va='center', fontsize=fontsize, color='black')

# Add title and labels
plt.title('Averaged Correct Sample Probability: PubChem 350-500 Da', fontsize=fontsize)
plt.ylabel('Averaged Correct Sample Probability', fontsize=21)
ax.set_xlabel('Number of Molecule Analogues for Improvment Cycle', fontsize=fontsize)

# Increase the size of the labels on the x-axis
ax.set_xticklabels(labels, fontsize=fontsize, rotation=0)

# Increase the size of the ticks on the y-axis
ax.tick_params(axis='y', labelsize=fontsize)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x:.2f}'))

# Save the plot to a specified location
output_path = os.path.abspath("./_FIGURES/6.2.2_PC_correct_sample_prob_comparison_350_500_v3.png")
plt.savefig(output_path, bbox_inches='tight')

# Show the plot
plt.show()


#### PubChem Second round training

In [ ]:
# 100_10 0_250
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240520_Improvment_Cycle_PC_0_250/6.2_imp_cyc_all_100_10.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_10_before_0_250 = pickle.load(file)
    
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240520_Improvment_Cycle_PC_0_250/6.2_imp_cyc_all_100_10_after.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_10_after_0_250 = pickle.load(file)

#maybe need to take the first one    
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240523_PC_Imp_Cycle_2_from_250_350/6.2_imp_cyc_all_100_10_after.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_10_after_2_0_250 = pickle.load(file)


In [ ]:
# 100_10 250_350
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240520_Improvment_Cycle_PC_250_350/6.2_imp_cyc_all_100_10.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_10_before_250_350 = pickle.load(file)
    
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240520_Improvment_Cycle_PC_250_350/6.2_imp_cyc_all_100_10_after.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_10_after_250_350 = pickle.load(file)
    
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240523_PC_Imp_Cycle_2_from_250_350/6.2_imp_cyc_all_100_10_after.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_10_after_2_250_350 = pickle.load(file)

In [ ]:
# 100_10 350_500
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240517_Improvment_Cycle_PC_350_500/6.2_imp_cyc_all_100_10.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_10_before_350_500 = pickle.load(file)
    
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240517_Improvment_Cycle_PC_350_500/6.2_imp_cyc_all_100_10_after.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_10_after_350_500 = pickle.load(file)
    
file_path = os.path.abspath('./past_experiments/ChemXriv/6.0_Experiment_IC/20240522_PC_Imp_Cycle_2_from_350_500/6.2_imp_cyc_all_100_10_after_v3_second_round.pkl')
# Loading results_list_b
with open(file_path, 'rb') as file:
    imp_cyc_all_100_10_after_2_350_500 = pickle.load(file)

In [ ]:

tani_before_0_250 = imp_cyc_all_100_10_before_0_250["results_dict_ZINC_greedy_bl"]["tanimoto_sim"]
tani_after_0_250 = imp_cyc_all_100_10_after_0_250["results_dict_ZINC_greedy_bl"]["tanimoto_sim"]
tani_after_2_0_250 = imp_cyc_all_100_10_after_2_0_250["results_dict_ZINC_greedy_bl"]["tanimoto_sim"]
tani_before_250_350 = imp_cyc_all_100_10_before_250_350["results_dict_ZINC_greedy_bl"]["tanimoto_sim"]
tani_after_250_350 = imp_cyc_all_100_10_after_250_350["results_dict_ZINC_greedy_bl"]["tanimoto_sim"]
tani_after_2_250_350 = imp_cyc_all_100_10_after_2_250_350["results_dict_ZINC_greedy_bl"]["tanimoto_sim"]
tani_before_350_500 = imp_cyc_all_100_10_before_350_500["results_dict_ZINC_greedy_bl"]["tanimoto_sim"]
tani_after_350_500 = imp_cyc_all_100_10_after_350_500["results_dict_ZINC_greedy_bl"]["tanimoto_sim"]
tani_after_2_350_500 = imp_cyc_all_100_10_after_2_350_500["results_dict_ZINC_greedy_bl"]["tanimoto_sim"]


In [ ]:

# Data and labels
tani_data = [
    (tani_before_0_250, 'FT0-250 Before'), (tani_after_0_250, 'FT0-250 After'), (tani_after_2_0_250, 'FT0-250 2nd Round'),
    (tani_before_250_350, 'FT250-350 Before'), (tani_after_250_350, 'FT250-350 After'), (tani_after_2_250_350, 'FT250-350 2nd Round'),
    (tani_before_350_500, 'FT350-500 Before'), (tani_after_350_500, 'FT350-500 After'), (tani_after_2_350_500, 'FT350-500 2nd Round')
]

# Count perfect matches (Tanimoto = 1) for each condition
perfect_matches = [np.sum(np.array(data) == 1) for data, _ in tani_data]

# Colors for the bars
colors = [    '#A1C8F3',  # light blue/periwinkle
    '#FFB381',  # salmon/peach
    '#8BE5A0',  # mint green
    '#FF9D9A',  # coral pink
    '#D1B9FE',  # lavender
    '#DEBA9A',  # beige/tan
    '#FCAEE3',  # pink
    '#CFCECE',  # light gray
    '#FEFDA2',  # pale yellow
    '#B8F1EF',  # light cyan/aqua
        ]

# Labels
group_labels = ['PubChem: 0-250', 'PubChem: 250-350', 'PubChem: 350-500']
condition_labels = ['Before IC', 'After IC', 'After 2x IC']

# Create the plot
fig, ax = plt.subplots(figsize=(17, 9))

# Set the width of each bar and the spacing between groups
bar_width = 0.25
group_spacing = 0.05

# Calculate positions for each bar
num_groups = len(group_labels)
indices = np.arange(num_groups)
positions = [indices + i * (bar_width + group_spacing) for i in range(3)]

# Create grouped bar plot
for i in range(3):  # Three conditions: Before, After, 2nd Round
    ax.bar(positions[i], perfect_matches[i::3], 
           width=bar_width, color=colors[i], edgecolor='black', 
           label=condition_labels[i])

# Customize the plot
ax.set_ylabel('Number of Perfect Matches (Tanimoto = 1)', fontsize=fontsize)
ax.set_xlabel('Molecular Weight Ranges', fontsize=fontsize)
ax.set_title('Tanimoto Matches Across Different Conditions: PubChem Data', fontsize=fontsize)

# Set x-ticks in the middle of each group
group_centers = indices + bar_width
ax.set_xticks(group_centers)
ax.set_xticklabels(group_labels, fontsize=fontsize)

# Set y-axis tick label font size
ax.tick_params(axis='y', labelsize=fontsize)

# Add value labels on top of each bar
for i, v in enumerate(perfect_matches):
    ax.text(positions[i % 3][i // 3], v, str(v), ha='center', va='bottom', fontsize=fontsize)

# Add legend to the upper right corner
ax.legend(fontsize=fontsize, loc='upper left')

# Adjust layout and display
plt.tight_layout()

# Save the plot to a specified location
output_path = os.path.abspath("./_FIGURES/6.2.2_PC_Bar_Chart_Greedy_second_round_v3.png")
plt.savefig(output_path, dpi=300, bbox_inches='tight')

plt.show()

In [ ]:

corr_sp_before_0_250 = imp_cyc_all_100_10_before_0_250["corr_sampleing_prob_bl_ZINC"]
corr_sp_after_0_250 = imp_cyc_all_100_10_after_0_250["corr_sampleing_prob_bl_ZINC"]
corr_sp_after_2_0_250 = imp_cyc_all_100_10_after_2_0_250["corr_sampleing_prob_bl_ZINC"]
corr_sp_before_250_350 = imp_cyc_all_100_10_before_250_350["corr_sampleing_prob_bl_ZINC"]
corr_sp_after_250_350 = imp_cyc_all_100_10_after_250_350["corr_sampleing_prob_bl_ZINC"]
corr_sp_after_2_250_350 = imp_cyc_all_100_10_after_2_250_350["corr_sampleing_prob_bl_ZINC"]
corr_sp_before_350_500 = imp_cyc_all_100_10_before_350_500["corr_sampleing_prob_bl_ZINC"]
corr_sp_after_350_500 = imp_cyc_all_100_10_after_350_500["corr_sampleing_prob_bl_ZINC"]
corr_sp_after_2_350_500 = imp_cyc_all_100_10_after_2_350_500["corr_sampleing_prob_bl_ZINC"]


In [ ]:

# Data for the bar chart
values = [
    np.mean(corr_sp_before_0_250), np.mean(corr_sp_after_0_250), np.mean(corr_sp_after_2_0_250),
    np.mean(corr_sp_before_250_350), np.mean(corr_sp_after_250_350), np.mean(corr_sp_after_2_250_350),
    np.mean(corr_sp_before_350_500), np.mean(corr_sp_after_350_500), np.mean(corr_sp_after_2_350_500)
]

# Colors for the bars
colors = ['#E57373', '#A2A37E', '#64B689']
colors = [    '#A1C8F3',  # light blue/periwinkle
    '#FFB381',  # salmon/peach
    '#8BE5A0',  # mint green
    '#FF9D9A',]  # coral pink]  # Red, Blue, Orange, Green

# Create a bar chart
fig, ax = plt.subplots(figsize=(17, 9))

# Set the width of each bar and the spacing between groups
bar_width = 0.25
group_spacing = 0.05

# Calculate positions for each bar
indices = np.arange(3)
positions = [indices + i * (bar_width + group_spacing) for i in range(3)]

# Create grouped bar plot
for i in range(3):  # Three conditions: Before, After, 2nd Round
    bars = ax.bar(positions[i], values[i::3], 
                  width=bar_width, color=colors[i], edgecolor='black', 
                  label=['Before IC', 'After IC', 'After 2x IC'][i])
    
    # Add numbers inside the bars
    for bar, value in zip(bars, values[i::3]):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() / 2, 
                f'{value:.2f}', ha='center', va='center', fontsize=fontsize, color='black')

# Customize the plot
ax.set_ylabel('Averaged Correct Sample Probability', fontsize=fontsize)
ax.set_title('Averaged Correct Sample Probability: PubChem Data', fontsize=fontsize)
ax.set_xlabel('Molecular Weight Ranges', fontsize=fontsize)

# Set x-ticks in the middle of each group
group_centers = indices + bar_width
ax.set_xticks(group_centers)
ax.set_xticklabels(['PubChem 0-250', 'PubChem 250-350', 'PubChem 350-500'], fontsize=fontsize)

# Set y-axis to display two decimal places
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x:.2f}'))

# Increase the size of the ticks on the y-axis
ax.tick_params(axis='y', labelsize=fontsize)

# Add legend
ax.legend(fontsize=fontsize, loc='upper left')

# Adjust layout
plt.tight_layout()

# Save the plot to a specified location
output_path = os.path.abspath("./_FIGURES/6.2.2_PC_correct_sample_prob_comparison_second_ro_v3.png")
plt.savefig(output_path, dpi=300, bbox_inches='tight')

# Show the plot
plt.show()

In [ ]:
# Data for the bar chart
values = [
    np.mean(corr_sp_before_0_250),  # Before (0-250)
    np.mean(corr_sp_after_0_250),   # After 0-250
    np.mean(corr_sp_after_2_0_250), # 2nd Round After 0-250
    np.mean(corr_sp_before_250_350),  # Before (250-350)
    np.mean(corr_sp_after_250_350),   # After 250-350
    np.mean(corr_sp_after_2_250_350), # 2nd Round After 250-350
    np.mean(corr_sp_before_350_500),  # Before (350-500)
    np.mean(corr_sp_after_350_500),   # After 350-500
    np.mean(corr_sp_after_2_350_500)  # 2nd Round After 350-500
]

# Labels for the bars
labels = [
    'Before IC 0-250', 
    'After IC 0-250', 
    'After 2x IC 0-250',
    'Before IC 250-350', 
    'After IC 250-350', 
    'After 2x IC 250-350',
    'Before IC 350-500', 
    'After IC 350-500', 
    'After 2x IC 350-500'
]

# Colors for the bars (based on provided image)
colors = [
    '#A1C8F3',  # light blue/periwinkle
    '#FFB381',  # salmon/peach
    '#8BE5A0',  # mint green
    '#FF9D9A',  # coral pink
    '#D1B9FE',  # lavender
    '#DEBA9A',  # beige/tan
    '#FCAEE3',  # pink
    '#CFCECE',  # light gray
    '#FEFDA2',  # pale yellow
    '#B8F1EF',  # light cyan/aqua
]

# Create a bar chart
fig, ax = plt.subplots(figsize=(14, 7))
bars = ax.bar(labels, values, color=colors, edgecolor='black')

# Add numbers inside the bars
for bar, value in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() / 2, f'{value:.2f}', ha='center', va='center', fontsize=fontsize, color='black')

# Add title and labels
plt.title('Averaged Correct Sample Probability', fontsize=fontsize)
plt.ylabel('Averaged Correct Sample Probability', fontsize=fontsize)
#plt.xlabel('Stages', fontsize=16)

# Increase the size of the labels on the x-axis
ax.set_xticklabels(labels, fontsize=fontsize, rotation=45, ha='right')

# Increase the size of the ticks on the y-axis
ax.tick_params(axis='y', labelsize=fontsize)

# Save the plot to a specified location
output_path =    os.path.abspath("./_FIGURES/6.2.2_PC_correct_sample_prob_comparison_second_ro_v3.png")
plt.savefig(output_path, bbox_inches='tight')

# Show the plot
plt.show()


### 7.0 Experimental vs Simulated Data

In [ ]:
def plot_comparison_bars(data_dict, colors, figsize=(12, 6), save_path=None):
    """
    Create a bar plot comparing different metrics across all categories.
    Shows accuracy as percentages and adds rotated value labels above bars.
    
    Parameters:
    -----------
    data_dict : dict
        Dictionary with categories and their accuracy values (0-1 scale)
    colors : list
        List of colors for the bars
    figsize : tuple
        Figure size as (width, height)
    save_path : str, optional
        Full path where to save the figure
    """
    
    # Create the figure with specified figsize
    fig = plt.figure(figsize=figsize)
    ax = fig.add_subplot(111)
    
    # Calculate positions for bars
    categories = list(data_dict.keys())
    n_categories = len(categories)
    n_metrics = len(list(data_dict.values())[0])
    
    # Set width of bars and positions of the bars
    total_width = 0.8
    width = total_width / n_metrics
    x = np.arange(n_categories)
    
    # Create bars for each metric
    metric_names = ['top-1', 'top-3', 'top-10']
    
    # Set y-axis limit first to properly position labels
    ax.set_ylim(0, 1.1)  # Set y-axis limit from 0 to 110%
    
    for i in range(n_metrics):
        values = [data_dict[cat][i] for cat in categories]
        offset = width * i - (total_width/2) + (width/2)
        bars = ax.bar(x + offset, values, width, label=metric_names[i], color=colors[i], edgecolor='black')
        
        # Add value labels above bars with more spacing
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2, 
                   height + 0.03,  # Add more space between bar and label
                   f'{height*100:.0f}%',
                   ha='center', 
                   va='bottom',
                   rotation=90,
                   fontsize=10)
    
    # Customize the plot
    ax.set_xlabel('')
    ax.set_ylabel('Accuracy (%)', fontsize=14)
    ax.set_title('Comparison Across Categories', fontsize=16, pad=20)
    
    # Set x-ticks
    ax.set_xticks(x)
    ax.set_xticklabels(categories, rotation=45, ha='right', fontsize=12)
    
    # Set y-ticks and convert to percentages
    ax.tick_params(axis='y', labelsize=12)
    y_ticks = np.arange(0, 1.2, 0.2)  # Create ticks at 0%, 20%, 40%, 60%, 80%, 100%
    ax.set_yticks(y_ticks)
    ax.set_yticklabels([f'{x*100:.0f}%' for x in y_ticks])
    
    # Add grid
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    
    # Add legend
    ax.legend(fontsize=12, bbox_to_anchor=(1.05, 1), loc='upper left')
    
    # Adjust layout to prevent label cutoff
    plt.tight_layout()
    
    # Save if path is provided
    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=300)
        print(f"Plot saved to: {save_path}")
    
    plt.show()

# Example usage
if __name__ == "__main__":
    colors = [
        '#A1C8F3',  # light blue/periwinkle
        '#B8F1EF',  # light cyan/aqua
        '#FFB381',  # salmon/peach
        #'#8BE5A0',  # mint green
        #'#D1B9FE',  # lavender
    ]
    
    # Example save path
    save_path = os.path.abspath("./_FIGURES/7.0_IC_comparison_exp_sim_aug_plot.png")
    
    # Call with dummy data
    #plot_comparison_bars({}, colors, save_path=save_path)

In [ ]:

def plot_comparison_bars(data_dict, colors, figsize=(12, 6), save_path=None):
    """
    Create a bar plot comparing different metrics across all categories.
    Shows accuracy as percentages with labels inside bars (except for BL Experimental).
    X-axis labels are split into two rows.
    Axis labels: font size 18
    All other text: font size 20
    """
    
    # Create the figure with specified figsize
    fig = plt.figure(figsize=figsize)
    ax = fig.add_subplot(111)
    
    # Calculate positions for bars
    categories = list(data_dict.keys())
    n_categories = len(categories)
    n_metrics = len(list(data_dict.values())[0])
    
    # Set width of bars and positions of the bars
    total_width = 0.8
    width = total_width / n_metrics
    x = np.arange(n_categories)
    
    # Create bars for each metric
    metric_names = ['top-1', 'top-3', 'top-10']
    
    # Set y-axis limit first to properly position labels
    ax.set_ylim(0, 1.1)
    
    for i in range(n_metrics):
        values = [data_dict[cat][i] for cat in categories]
        offset = width * i - (total_width/2) + (width/2)
        bars = ax.bar(x + offset, values, width, label=metric_names[i], color=colors[i], edgecolor='black')
        
        # Add value labels
        for j, bar in enumerate(bars):
            height = bar.get_height()
            
            # Special case for BL Experimental (target) where height is very small
            if categories[j] == 'BL Experimental (target)' and height < 0.1:
                # Place label above the bar
                ax.text(bar.get_x() + bar.get_width()/2, 
                       height + 0.03,
                       f'{height*100:.0f}%',
                       ha='center', 
                       va='bottom',
                       rotation=90,
                       fontsize=20)
            else:
                # Place label inside the bar
                ax.text(bar.get_x() + bar.get_width()/2, 
                       height/2,  # Center vertically inside bar
                       f'{height*100:.0f}%',
                       ha='center', 
                       va='center',
                       rotation=90,
                       fontsize=20)
    
    # Customize the plot
    ax.set_xlabel('')
    ax.set_ylabel('Accuracy (%)', fontsize=18)  # Changed to 18
    ax.set_title('Comparison Across Categories', fontsize=20, pad=20)
    
    # Create two-line labels
    def split_category(category):
        if '(target)' in category or '(analogue)' in category:
            main_part = category.split(' (')[0]
            suffix = f'({category.split("(")[1]}'
            return f'{main_part}\n{suffix}'
        return category
    
    # Set x-ticks
    ax.set_xticks(x)
    ax.set_xticklabels([split_category(cat) for cat in categories], 
                       ha='center',  # Center align
                       fontsize=18)  # Changed to 18
    
    # Set y-ticks and convert to percentages
    ax.tick_params(axis='y', labelsize=18)  # Changed to 18
    y_ticks = np.arange(0, 1.2, 0.2)
    ax.set_yticks(y_ticks)
    ax.set_yticklabels([f'{x*100:.0f}%' for x in y_ticks])
    
    # Add grid
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    
    # Add legend inside the plot in the upper right corner
    ax.legend(fontsize=20, loc='upper right')
    
    # Adjust layout to prevent label cutoff
    plt.tight_layout()
    
    # Save if path is provided
    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=300)
        print(f"Plot saved to: {save_path}")
    
    plt.show()

    
# Example usage
if __name__ == "__main__":
    colors = [
        '#A1C8F3',  # light blue/periwinkle
        '#B8F1EF',  # light cyan/aqua
        '#FFB381',  # salmon/peach
    ]
    
    # Example save path
    save_path = os.path.abspath("./_FIGURES/7.0_IC_comparison_exp_sim_aug_plot.png")
    
    # Call with dummy data
    #plot_comparison_bars({}, colors, save_path=save_path)

#### Extracting the data for the plotting

In [ ]:

def process_pkl_files_new(folder_path, file_type, ranking_method):
    pkl_files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) 
                 if f.endswith('.pkl') and file_type in f]
    all_rankings = defaultdict(list)
    for file_path in pkl_files:
        file_data = load_data(file_path)
        
        for trg_smi, value_list in file_data.items():
            for sublist in value_list[0]:
                
                gen_smi = sublist[0]
                tanimoto = sublist[4]
                errors = sublist[5]
                if errors == 9:  # Check if both errors are 9
                    errors = [9, 9]  # Keep it as is
                try:
                    all_rankings[trg_smi].append((trg_smi, gen_smi, tanimoto, 0, errors[0], errors[1]))
                except:
                    import IPython; IPython.embed();           
    all_rankings = rank_all_molecules(all_rankings, ranking_method)
    
    return all_rankings


def rank_all_molecules(all_rankings, ranking_method):
    new_rankings = defaultdict(list)
    
    for trg_smi, molecule_list in all_rankings.items():
        molecule_data = []
        seen_gen_smiles = set()
        
        for molecule in molecule_list:
            gen_smi = molecule[1]
            if gen_smi in seen_gen_smiles:
                continue
            seen_gen_smiles.add(gen_smi)
            
            tanimoto = molecule[2]
            errors = molecule[4:6]
            if errors == (9, 9):  # Check if both errors are 9
                errors = [9, 9]  # Keep it as is
            molecule_data.append((gen_smi, errors[0], errors[1], tanimoto))
        
        if not molecule_data:
            continue
        
        molecule_array = np.array(molecule_data, dtype=[('gen_smi', 'U100'), 
                                                        ('error1', float), ('error2', float), 
                                                        ('tanimoto', float)])
        
        if ranking_method == 'HSQC & COSY':
            rank1 = molecule_array.argsort(order='error1')
            rank2 = molecule_array.argsort(order='error2')
            average_ranks = (np.arange(len(rank1))[np.argsort(rank1)] + 
                             np.arange(len(rank2))[np.argsort(rank2)]) / 2
            sorted_indices = average_ranks.argsort()
            
        elif ranking_method == 'HSQC':
            sorted_indices = molecule_array.argsort(order='error1')
        elif ranking_method == 'COSY':
            sorted_indices = molecule_array.argsort(order='error2')
        else:
            raise ValueError("Invalid ranking method. Choose 'HSQC & COSY', 'HSQC', or 'COSY'.")
        
        for new_rank, i in enumerate(sorted_indices):
            gen_smi = molecule_array['gen_smi'][i]
            tanimoto = molecule_array['tanimoto'][i]
            error1 = molecule_array['error1'][i]
            error2 = molecule_array['error2'][i]
            
            new_rankings[trg_smi].append((trg_smi, gen_smi, tanimoto, new_rank, error1, error2))
    
    return new_rankings

def load_data(file_path):
    with open(file_path, 'rb') as f:
        data = pickle.load(f)
    return data["results_dict_bl_ZINC"]

def process_experimental_data(base_folder_path, ic_folder_path, analogue_folder_path):
    """
    Process experimental data from different sources and prepare it for plotting.
    
    Parameters:
    -----------
    base_folder_path : str
        Path to the folder containing base experiment data (without IC)
    ic_folder_path : str
        Path to the folder containing IC experiment data
    analogue_folder_path : str
        Path to the folder containing analogue experiment data
    
    Returns:
    --------
    dict
        Dictionary with processed data ready for plotting
    """
    # Define constants
    ranking_method = "HSQC"
    file_types = ["exp_sim_data", "sim_sim_data"]
    
    plot_data = {}
    
    # Process IC experiments
    for file_type in file_types:
        IC_rankings = process_pkl_files_new(ic_folder_path, file_type, ranking_method)
        IC_rankings, _ = exp_func.deduplicate_smiles_from_ranking(IC_rankings)
        IC_rankings, _ = exp_func.filter_rankings_by_molecular_formula(IC_rankings)
        accuracies = exp_func.calculate_top_k_accuracy(IC_rankings, k_range=[1, 3, 10])
        accuracies = [round(acc, 3) for acc in accuracies][:3]

        if file_type == "exp_sim_data":
            plot_data['IC Experimental (target)'] = accuracies
        else:
            plot_data['IC Simulated (target)'] = accuracies
            
    # Process base experiments (without IC)
    for file_type in file_types:
        if file_type == "exp_sim_data":
            input_dict = {"exp_sim_data" : base_pkl_dicts["exp_sim_data"]}
            rankings = exp_func.process_pkl_files_BL(input_dict, ranking_method)
            accuracies = exp_func.calculate_top_k_accuracy_BL(rankings["exp_sim_data"],k_range=[1, 3, 10])
            accuracies = [round(acc, 3) for acc in accuracies][:3]
            plot_data['BL Experimental (target)'] = accuracies

        elif file_type == "sim_sim_data":
            input_dict = {"sim_sim_data" : base_pkl_dicts["sim_sim_data"]}
            rankings = exp_func.process_pkl_files_BL(input_dict, ranking_method)
            accuracies = exp_func.calculate_top_k_accuracy_BL(rankings["sim_sim_data"],k_range=[1, 3, 10])       
            accuracies = [round(acc, 3) for acc in accuracies][:3]
            plot_data['BL Simulated (target)'] = accuracies

            
    # Process analogue experiments
    for file_type in file_types:
        analogue_rankings = process_pkl_files_new(analogue_folder_path, file_type, ranking_method)
        analogue_rankings, _ = exp_func.deduplicate_smiles_from_ranking(analogue_rankings)
        analogue_rankings, _ = exp_func.filter_rankings_by_molecular_formula(analogue_rankings)
        accuracies = exp_func.calculate_top_k_accuracy(analogue_rankings, k_range=[1, 3, 10])
        accuracies = [round(acc, 3) for acc in accuracies][:3]
        
        if file_type == "exp_sim_data":
            plot_data['IC Experimental (analogue)'] = accuracies
        else:
            plot_data['IC Simulated (analogue)'] = accuracies
            
    return plot_data

def reorder_data_dict(data_dict):
    """
    Reorders the data dictionary to group simulated and experimental results together.
    """
    # Define the desired order
    desired_order = [
        'BL Simulated (target)',
        'IC Simulated (target)',
        'IC Simulated (analogue)',
        'BL Experimental (target)',
        'IC Experimental (target)',
        'IC Experimental (analogue)'
    ]
    
    # Create new ordered dictionary
    ordered_dict = {key: data_dict[key] for key in desired_order if key in data_dict}
    
    return ordered_dict

# Helper functions remain the same as in previous version

In [ ]:
# Define your folder paths
base_folder = ""
ic_folder = os.path.abspath("./past_experiments/ChemXriv/7.0_Experiment/20240724_IC_batched_MMT_a1/")
base_pkl_dicts = {
        "sim_sim_data": os.path.abspath("./past_experiments/ChemXriv/7.0_Experiment/20240724_IC_batched_MMT_a1_baseline/8.0_sim_real_data_before_FT_MMTi_v0.pkl"),
        "exp_sim_data": os.path.abspath("./past_experiments/ChemXriv/7.0_Experiment/20240724_IC_batched_MMT_a1_baseline/8.3_real_data_before_FT_MMTi_v0.pkl")
        }    
analogue_folder = os.path.abspath("./past_experiments/ChemXriv/7.0_Experiment/Sim_ACD_Exp_aug/test_on_aug_data_34_v3/")

#file_paths_dict = {
#    "sim_sim_data": os.path.abspath("/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/___FIGURES_PAPERS/Figures_Paper_2/precomputed_raw_data/20240724_IC_batched_MMT_d1_baseline/8.0_sim_real_data_before_FT_MMT_v0.pkl"),
#    "ACD_sim_data": os.path.abspath("/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/___FIGURES_PAPERS/Figures_Paper_2/precomputed_raw_data/20240724_IC_batched_MMT_d1_baseline/8.2_simACD_real_data_before_FT_MMT_v0.pkl"),
#    "exp_sim_data": os.path.abspath("/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/___FIGURES_PAPERS/Figures_Paper_2/precomputed_raw_data/20240724_IC_batched_MMT_d1_baseline/8.3_real_data_before_FT_MMT_v0.pkl")
#}    

# Process the data
data = process_experimental_data(base_pkl_dicts, ic_folder, analogue_folder)
data = reorder_data_dict(data)
# Plot with your color scheme
colors = [
    '#A1C8F3',  # light blue/periwinkle
    '#FFB381',  # salmon/peach
    '#8BE5A0',  # mint green
    #'#FF9D9A',  # coral pink
    #'#D1B9FE',  # lavender
    #'#DEBA9A',  # beige/tan
]
save_path = os.path.abspath("./_FIGURES/7.0_IC_comparison_exp_sim_aug_plot.png")

plot_comparison_bars( data_dict=data,     colors=colors,     save_path=save_path,    figsize=(12, 6))

#### Extract Top 1 and top 3 correct Molecules

In [ ]:
def extract_correct_smiles_analogues(analogue_folder_path, file_type="exp_sim_data", ranking_method="HSQC"):
    """
    Extract SMILES of molecules that are correct matches in top 1 and top 3 rankings
    for experimental analogues.
    
    Parameters:
    -----------
    analogue_folder_path : str
        Path to the folder containing analogue experiment data
    file_type : str
        Type of file to process ('exp_sim_data' or 'sim_sim_data')
    ranking_method : str
        Method used for ranking ('HSQC', 'COSY', or 'HSQC & COSY')
        
    Returns:
    --------
    tuple
        (correct_top1_pairs, correct_top3_pairs) where each is a list of tuples containing
        (target_smiles, generated_smiles, rank) where rank indicates which position (1,2,3)
        had the correct match
    """
    # Process the rankings
    analogue_rankings = process_pkl_files_new(analogue_folder_path, file_type, ranking_method)
    
    # Deduplicate and filter rankings
    analogue_rankings, _ = exp_func.deduplicate_smiles_from_ranking(analogue_rankings)
    analogue_rankings, _ = exp_func.filter_rankings_by_molecular_formula(analogue_rankings)
    
    correct_top1_pairs = []
    correct_top3_pairs = []
    
    # For each target molecule
    for target_smi, rankings in analogue_rankings.items():
        # Sort rankings by rank
        sorted_rankings = sorted(rankings, key=lambda x: x[3])  # x[3] is the rank
        
        # Check top 1 match
        if sorted_rankings:
            top1_gen_smi = sorted_rankings[0][1]  # x[1] is generated SMILES
            if top1_gen_smi == target_smi:  # Only append if it's a correct match
                correct_top1_pairs.append((target_smi, top1_gen_smi, 1))
        
        # Check each of the top 3 matches individually
        for idx, ranking in enumerate(sorted_rankings[:3], 1):  # idx will be 1, 2, or 3
            gen_smi = ranking[1]
            if gen_smi == target_smi:  # If we find a match at any position
                correct_top3_pairs.append((target_smi, gen_smi, idx))
    
    return correct_top1_pairs, correct_top3_pairs

def save_correct_smiles_to_file(smiles_pairs, output_path, include_rank=False):
    """
    Save correct SMILES pairs to a text file.
    
    Parameters:
    -----------
    smiles_pairs : list
        List of (target_smiles, generated_smiles, rank) tuples that are correct matches
    output_path : str
        Path where to save the file
    include_rank : bool
        If True, includes which position (1,2,3) had the correct match
    """
    with open(output_path, 'w') as f:
        if include_rank:
            f.write("SMILES\tRank\n")
            for target_smi, _, rank in smiles_pairs:  # We can use either SMILES since they're identical
                f.write(f"{target_smi}\t{rank}\n")
        else:
            f.write("SMILES\n")
            for target_smi, _, _ in smiles_pairs:
                f.write(f"{target_smi}\n")


In [ ]:
### with Analogues
# Example usage:

analogue_folder = os.path.abspath("./past_experiments/ChemXriv/7.0_Experiment/test_on_aug_data_34_v3/")

# Extract only correct SMILES matches
correct_top1_pairs, correct_top3_pairs = extract_correct_smiles_analogues(
    analogue_folder,
    file_type="exp_sim_data",
    ranking_method="HSQC"
)

# Save to files (only correct matches)
# For top 1, we don't need the rank
save_correct_smiles_to_file(correct_top1_pairs, "correct_top1_experimental_analogues.txt", include_rank=False)
# For top 3, we might want to know which position had the match
save_correct_smiles_to_file(correct_top3_pairs, "correct_top3_experimental_analogues.txt", include_rank=True)

# Optional: Print some statistics
print(f"Number of correct top 1 matches: {len(correct_top1_pairs)}")
print(f"Number of correct top 3 matches: {len(correct_top3_pairs)}")
print("\nBreakdown of top 3 matches by position:")
for rank in [1, 2, 3]:
    count = sum(1 for _, _, r in correct_top3_pairs if r == rank)
    print(f"Position {rank}: {count} matches")


In [ ]:
analogue_folder = os.path.abspath("./past_experiments/ChemXriv/7.0_Experiment/20240724_IC_batched_MMT_a1/")

# Extract only correct SMILES matches
correct_top1_pairs, correct_top3_pairs = extract_correct_smiles_analogues(
    analogue_folder,
    file_type="exp_sim_data",
    ranking_method="HSQC"
)

# Save to files (only correct matches)
# For top 1, we don't need the rank
save_correct_smiles_to_file(correct_top1_pairs, "correct_top1_experimental_analogues.txt", include_rank=False)
# For top 3, we might want to know which position had the match
save_correct_smiles_to_file(correct_top3_pairs, "correct_top3_experimental_analogues.txt", include_rank=True)


# Optional: Print some statistics
print(f"Number of correct top 1 matches: {len(correct_top1_pairs)}")
print(f"Number of correct top 3 matches: {len(correct_top3_pairs)}")
print("\nBreakdown of top 3 matches by position:")
for rank in [1, 2, 3]:
    count = sum(1 for _, _, r in correct_top3_pairs if r == rank)
    print(f"Position {rank}: {count} matches")


### 8.0 Simulated, Experimental results

#### Plotting

In [ ]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from rdkit import Chem
from rdkit.Chem import AllChem, Draw, Descriptors
import pandas as pd
import random
import string
import rdkit
from rdkit import Chem

colors = [
    '#E57373', '#A2A37E', '#64B689', '#4DA6A9', '#5C95CC', '#9574D0', '#EB6CC2'
]

def load_data(file_path):
    with open(file_path, 'rb') as f:
        data = pickle.load(f)
    return data["results_dict_bl_ZINC"]


def process_pkl_files_new(folder_path, file_type, ranking_method):
    pkl_files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) 
                 if f.endswith('.pkl') and file_type in f]
    all_rankings = defaultdict(list)
    for file_path in pkl_files:
        file_data = load_data(file_path)
        
        for trg_smi, value_list in file_data.items():
            for sublist in value_list[0]:
                
                gen_smi = sublist[0]
                tanimoto = sublist[4]
                errors = sublist[5]
                if errors == 9:  # Check if both errors are 9
                    errors = [9, 9]  # Keep it as is
                try:
                    all_rankings[trg_smi].append((trg_smi, gen_smi, tanimoto, 0, errors[0], errors[1]))
                except:
                    import IPython; IPython.embed();           
    all_rankings = rank_all_molecules(all_rankings, ranking_method)
    
    return all_rankings


def calculate_top_k_accuracy(all_rankings, k_range=[1, 3, 5, 10, 20]):
    accuracies = []
    total_molecules = 34 #len(all_rankings)
    for k in k_range:
        correct_count = sum(
            any(molecule[2] == 1 for molecule in rankings[:k])
            for rankings in all_rankings.values()
        )
        accuracy = correct_count / total_molecules
        accuracies.append(accuracy)
    correct_count = sum(
        any(molecule[2] == 1 for molecule in rankings[:])
        for rankings in all_rankings.values()
    )
    accuracy = correct_count / total_molecules
    accuracies.append(accuracy)    
    return accuracies


def plot_top_k_accuracy(accuracies, data_type, model_type, ranking_method, sim_rank_one_count, save_path=None, total_samples=34):
    fontsize = 30
    k_values = [1, 3, 5, 10, 20]
    labels = [f'Top {k}' for k in k_values] + ['Total']
    
    plt.figure(figsize=(12, 8))  # Slightly wider to accommodate longer labels
    bars = plt.bar(range(len(labels)), accuracies, color=colors[:len(labels)])
    
    plt.ylabel('Accuracy', fontsize=fontsize)
    plt.title(f'{data_type} Data and {ranking_method} Ranking', fontsize=fontsize+4)
    
    plt.xticks(range(len(labels)), labels, fontsize=fontsize, rotation=0, ha='center')
    plt.yticks(fontsize=fontsize)
    plt.ylim(0, 1.1)
    
    for bar in bars:
        height = bar.get_height()
        correct_samples = int(height * total_samples)
        
        plt.text(bar.get_x() + bar.get_width()/2., height,
                 f'{height:.2f}',
                 ha='center', va='bottom', fontsize=fontsize)
        
        plt.text(bar.get_x() + bar.get_width()/2., height/2,
                 f'{correct_samples}',
                 ha='center', va='center', fontsize=fontsize, color='black')
    
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Plot saved to {save_path}")
    
    plt.close()
    

def main_normal():
    folder_path = os.path.abspath("./past_experiments/ChemXriv/7.0_Experiment/20240724_IC_batched_MMT_a1/")
    model_type = "MMT"
    ranking_methods = [ "HSQC"]
    file_types = ["exp_sim_data", "sim_sim_data"]

    data_type_map = {
        "exp_sim_data": "Experimental",
        "sim_sim_data": "Our Simulated",
    }
    filtered_out = []
    all_rankings_list = []
    for ranking_method in ranking_methods:
        for file_type in file_types:
            all_rankings = process_pkl_files_new(folder_path, file_type, ranking_method)
            #import IPython; IPython.embed();
            all_rankings, removed_smiles = exp_func.deduplicate_smiles_from_ranking(all_rankings)
            all_rankings, filtered_out_rankings = exp_func.filter_rankings_by_molecular_formula(all_rankings)
            #filtered_out.append(filtered_out_rankings)
            all_rankings_list.append(all_rankings)
            print("second_breakout")
            #import IPython; IPython.embed();

            accuracies = calculate_top_k_accuracy(all_rankings)
            sim_rank_one_count = exp_func.count_molecules_with_sim_rank_one(all_rankings)

            data_type = data_type_map[file_type]
            save_path = f"./_FIGURES/top_k_accuracy_MMTi_{file_type}_{ranking_method}_va_FINAL_2.png"

            plot_top_k_accuracy(accuracies, data_type, model_type, ranking_method, sim_rank_one_count, save_path, total_samples=len(all_rankings.keys()))

            print(f"Completed plot for {ranking_method} - {data_type}")
    return all_rankings_list, filtered_out


In [ ]:

def load_data(file_path):
    with open(file_path, 'rb') as f:
        data = pickle.load(f)
    return data["results_dict_bl_ZINC"]


def rank_molecules_in_file(file_data, ranking_method):
    molecule_data = []
    for trg_smi, value_list in file_data.items():
        for sublist in value_list[0]:
            gen_smi = sublist[0]
            tanimoto = sublist[4]
            errors = sublist[5]
            if errors == 9:
                errors = [9,9] ### Necessary if it doesn't manage to calculate HSQC or COSY to calculate the errors
                                ## Basically put it last then
            molecule_data.append((trg_smi, gen_smi, errors[0], errors[1], tanimoto))

    if not molecule_data:
        return []

    molecule_array = np.array(molecule_data, dtype=[('trg_smi', 'U100'), ('gen_smi', 'U100'), 
                                                    ('error1', float), ('error2', float), 
                                                    ('tanimoto', float)])
    
    if ranking_method == 'HSQC & COSY':
        rank1 = molecule_array.argsort(order='error1')
        rank2 = molecule_array.argsort(order='error2')
        average_ranks = (np.arange(len(rank1))[np.argsort(rank1)] + 
                         np.arange(len(rank2))[np.argsort(rank2)]) / 2
        sorted_indices = average_ranks.argsort()
        
    elif ranking_method == 'HSQC':
        sorted_indices = molecule_array.argsort(order='error1')
    elif ranking_method == 'COSY':
        sorted_indices = molecule_array.argsort(order='error2')
    else:
        raise ValueError("Invalid ranking method. Choose 'HSQC & COSY', 'HSQC', or 'COSY'.")

    sorted_molecules = [(molecule_array['trg_smi'][i], 
                         molecule_array['gen_smi'][i],
                         molecule_array["tanimoto"][i],
                         new_rank,  # Use index as rank
                         molecule_array['error1'][i], 
                         molecule_array['error2'][i]) 
                        for new_rank, i  in enumerate(sorted_indices)]
    return sorted_molecules


def rank_all_molecules(all_rankings, ranking_method):
    new_rankings = defaultdict(list)
    
    for trg_smi, molecule_list in all_rankings.items():
        molecule_data = []
        seen_gen_smiles = set()
        
        for molecule in molecule_list:
            gen_smi = molecule[1]
            if gen_smi in seen_gen_smiles:
                continue
            seen_gen_smiles.add(gen_smi)
            
            tanimoto = molecule[2]
            errors = molecule[4:6]
            if errors == (9, 9):  # Check if both errors are 9
                errors = [9, 9]  # Keep it as is
            molecule_data.append((gen_smi, errors[0], errors[1], tanimoto))
        
        if not molecule_data:
            continue
        
        molecule_array = np.array(molecule_data, dtype=[('gen_smi', 'U100'), 
                                                        ('error1', float), ('error2', float), 
                                                        ('tanimoto', float)])
        
        if ranking_method == 'HSQC & COSY':
            rank1 = molecule_array.argsort(order='error1')
            rank2 = molecule_array.argsort(order='error2')
            average_ranks = (np.arange(len(rank1))[np.argsort(rank1)] + 
                             np.arange(len(rank2))[np.argsort(rank2)]) / 2
            sorted_indices = average_ranks.argsort()
            
        elif ranking_method == 'HSQC':
            sorted_indices = molecule_array.argsort(order='error1')
        elif ranking_method == 'COSY':
            sorted_indices = molecule_array.argsort(order='error2')
        else:
            raise ValueError("Invalid ranking method. Choose 'HSQC & COSY', 'HSQC', or 'COSY'.")
        
        for new_rank, i in enumerate(sorted_indices):
            gen_smi = molecule_array['gen_smi'][i]
            tanimoto = molecule_array['tanimoto'][i]
            error1 = molecule_array['error1'][i]
            error2 = molecule_array['error2'][i]
            
            new_rankings[trg_smi].append((trg_smi, gen_smi, tanimoto, new_rank, error1, error2))
    
    return new_rankings


def prepare_mol(smiles):
    mol = Chem.MolFromSmiles(smiles)
    AllChem.Compute2DCoords(mol)
    return mol

def get_mol_weight(mol):
    return round(Descriptors.ExactMolWt(mol), 2)


In [ ]:
all_rankings_list, filtered_out = main_normal()


#### Plot baseline performance

In [ ]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

colors = [
    '#E57373', '#A2A37E', '#64B689', '#4DA6A9', '#5C95CC', '#9574D0', '#EB6CC2'
]

def load_data(file_path):
    with open(file_path, 'rb') as f:
        data = pickle.load(f)
    return data["results_dict_bl_ZINC"]

def rank_molecules(file_data, ranking_method):
    all_rankings = {}
    for trg_smi, value_list in file_data.items():
        molecule_data = []
        for sublist in value_list[0]:
            try:
                gen_smi = sublist[0]
                tanimoto = sublist[4]
                errors = sublist[5]
                molecule_data.append((gen_smi, errors[0], errors[1], tanimoto))
            except:
                print(f"Error processing sublist for {trg_smi}: {sublist}")
                continue

        if not molecule_data:
            continue

        molecule_array = np.array(molecule_data, dtype=[('gen_smi', 'U100'), 
                                                        ('error1', float), ('error2', float), 
                                                        ('tanimoto', float)])
        
        if ranking_method == 'HSQC & COSY':
            rank1 = molecule_array.argsort(order='error1')
            rank2 = molecule_array.argsort(order='error2')
            average_ranks = (np.arange(len(rank1))[np.argsort(rank1)] + 
                             np.arange(len(rank2))[np.argsort(rank2)]) / 2
            sorted_indices = average_ranks.argsort()
        elif ranking_method == 'HSQC':
            sorted_indices = molecule_array.argsort(order='error1')
        elif ranking_method == 'COSY':
            sorted_indices = molecule_array.argsort(order='error2')
        else:
            raise ValueError("Invalid ranking method. Choose 'HSQC & COSY', 'HSQC', or 'COSY'.")

        sorted_molecules = [(trg_smi,
                             molecule_array['gen_smi'][i],
                             molecule_array["tanimoto"][i],
                             new_rank,  # Use index as rank
                             molecule_array['error1'][i], 
                             molecule_array['error2'][i]) 
                            for  new_rank, i in enumerate(sorted_indices)]
        
        all_rankings[trg_smi] = sorted_molecules
    
    return all_rankings


def plot_top_k_accuracy(accuracies, data_type, model_type, ranking_method, sim_rank_one_count, save_path=None, total_samples=34):
    fontsize = 30
    k_values = [1, 3, 5, 10, 20]
    labels = [f'Top {k}' for k in k_values] + ['Total']
    
    
    plt.figure(figsize=(12, 8))  # Slightly wider to accommodate longer labels
    # print(range(len(labels)), all_values, colors[:len(labels)])
    # bars = plt.bar(range(len(labels)), all_values, color=colors[:len(labels)])
    #bars = plt.bar(range(len(labels)), accuracies + [sim_rank_one_count / total_samples], color=colors[:len(labels)])
    bars = plt.bar(range(len(labels)), accuracies, color=colors[:len(labels)])
    
    plt.ylabel('Accuracy', fontsize=fontsize)
    plt.title(f'{data_type} Data and {ranking_method} Ranking', fontsize=fontsize+4)
    
    plt.xticks(range(len(labels)), labels, fontsize=fontsize, rotation=0, ha='center')
    plt.yticks(fontsize=fontsize)
    plt.ylim(0, 1.1)
    
    for bar in bars:
        height = bar.get_height()
        correct_samples = int(height * total_samples)
        
        plt.text(bar.get_x() + bar.get_width()/2., height,
                 f'{height:.2f}',
                 ha='center', va='bottom', fontsize=fontsize)
        
        plt.text(bar.get_x() + bar.get_width()/2., height/2,
                 f'{correct_samples}',
                 ha='center', va='center', fontsize=fontsize, color='black')
    
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Plot saved to {save_path}")
    
    plt.close()
    
def main_BL():
    #folder_path = "/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/___FIGURES_PAPERS/Figures_Paper_2/precomputed_raw_data/20240724_IC_batched_MMT_a1_baseline/"
    model_type = "MMT"
    ranking_methods = ["HSQC"] #, "HSQC & COSY"
    file_types = ["sim_sim_data", "exp_sim_data"]
    
    
    file_paths_dict = {
        "sim_sim_data": os.path.abspath("./past_experiments/ChemXriv/7.0_Experiment/20240724_IC_batched_MMT_a1_baseline/8.0_sim_real_data_before_FT_MMTi_v0.pkl"),
        "exp_sim_data": os.path.abspath("./past_experiments/ChemXriv/7.0_Experiment/20240724_IC_batched_MMT_a1_baseline/8.3_real_data_before_FT_MMTi_v0.pkl")
    }    
    
    
    data_type_map = {
        "sim_sim_data": "Our Simulated",
        "exp_sim_data": "Experimental"
    }
    for ranking_method in ranking_methods:
        all_rankings = exp_func.process_pkl_files_BL(file_paths_dict, ranking_method)
        
        for data_type, rankings in all_rankings.items():
            #import IPython; IPython.embed();
            accuracies = calculate_top_k_accuracy(rankings)
            sim_rank_one_count = exp_func.count_molecules_with_sim_rank_one(rankings)
            
            save_path = os.path.abspath(f"./_FIGURES/top_k_accuracy_MMTi_baseline_{data_type}_{ranking_method}_FINAL_2.png")
            #save_path = f"/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/___FIGURES_PAPERS/Figures_Paper_2/top_k_accuracy_MMT_baseline_{data_type}_{ranking_method}_v8.png"
            plot_top_k_accuracy(accuracies, data_type_map[data_type], model_type, ranking_method, sim_rank_one_count, save_path, total_samples=34)

            print(f"Completed plot for {ranking_method} - {data_type_map[data_type]}")



In [ ]:
main_BL()

#### Plot all molecules

In [ ]:

def plot_molecules_from_df(df, smiles_col, id_col, mols_per_row=5, output_folder='molecule_images'):
    """
    Plot molecules from a DataFrame with SMILES and sample ID columns and save images to a folder.
    
    Args:
    df (pd.DataFrame): DataFrame containing SMILES and sample ID columns
    smiles_col (str): Name of the column containing SMILES strings
    id_col (str): Name of the column containing sample IDs
    mols_per_row (int): Number of molecules to display per row (default: 5)
    output_folder (str): Name of the folder to save images (default: 'molecule_images')
    """
    mols = []
    legends = []
    
    # Create output folder if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)
    
    for _, row in df.iterrows():
        mol = Chem.MolFromSmiles(row[smiles_col])
        if mol is not None:
            mols.append(mol)
            mol_weight = Descriptors.ExactMolWt(mol)
            legend = f"{row[id_col]} | MW: {mol_weight:.2f}"
            legends.append(legend)
    
    n_rows = ceil(len(mols) / mols_per_row)
    
    for i in range(n_rows):
        start_idx = i * mols_per_row
        end_idx = min((i + 1) * mols_per_row, len(mols))
        
        img = Draw.MolsToGridImage(
            mols[start_idx:end_idx],
            molsPerRow=mols_per_row,
            subImgSize=(300, 300),
            legends=[legends[j] for j in range(start_idx, end_idx)],
            useSVG=True
        )
        
        # Save the SVG image
        filename = f'molecules_row_aug_{i+1}.svg'
        filepath = os.path.join(output_folder, filename)
        with open(filepath, 'w') as f:
            f.write(img.data)
        
        print(f"Saved {filename}")
        
        # Display the image (optional, comment out if not needed)
        display(SVG(img.data))


from rdkit.Chem.Draw import IPythonConsole
IPythonConsole.drawOptions.addAtomIndices = False 

#df = pd.read_csv("/projects/cc/se_users/knlr326/1_NMR_project/1_NMR_data_AZ/37_Richard_ACD_sim_data/ACD_1H_with_SN_filtered_v3.csv")
df = pd.read_csv(os.path.abspath("./past_experiments/ChemXriv/7.0_Experiment/ACD_1H_with_SN_filtered_v3_regio_aug.csv"))
output_folder = os.path.abspath("./_FIGURES")

plot_molecules_from_df(df, 'SMILES_regio_isomers', 'sample-id', output_folder=output_folder)
#plot_molecules_from_df(df, 'SMILES', 'sample-id', output_folder=output_folder)


In [ ]:

def find_duplicate_smiles(df, smiles_col, id_col):
    """
    Find duplicate SMILES in a DataFrame after canonicalization.
    
    Args:
    df (pd.DataFrame): DataFrame containing SMILES and sample ID columns
    smiles_col (str): Name of the column containing SMILES strings
    id_col (str): Name of the column containing sample IDs
    
    Returns:
    pd.DataFrame: A DataFrame containing duplicate SMILES, their IDs, and count
    """
    # Create a dictionary to store canonical SMILES and their corresponding IDs
    canonical_smiles_dict = {}
    
    for _, row in df.iterrows():
        smiles = row[smiles_col]
        mol_id = row[id_col]
        
        # Generate canonical SMILES
        mol = Chem.MolFromSmiles(smiles)
        if mol is not None:
            canonical_smiles = Chem.MolToSmiles(mol, canonical=True)
            
            # Add to dictionary
            if canonical_smiles in canonical_smiles_dict:
                canonical_smiles_dict[canonical_smiles].append((mol_id, smiles))
            else:
                canonical_smiles_dict[canonical_smiles] = [(mol_id, smiles)]
    
    # Filter for duplicates and create a list of results
    duplicates = []
    for canonical_smiles, id_smiles_list in canonical_smiles_dict.items():
        if len(id_smiles_list) > 1:
            for mol_id, original_smiles in id_smiles_list:
                duplicates.append({
                    'Canonical_SMILES': canonical_smiles,
                    'Original_SMILES': original_smiles,
                    'ID': mol_id,
                    'Duplicate_Count': len(id_smiles_list)
                })
    
    # Create a DataFrame from the list of duplicates
    duplicates_df = pd.DataFrame(duplicates)
    
    # Sort by Duplicate_Count (descending) and Canonical_SMILES
    duplicates_df = duplicates_df.sort_values(['Duplicate_Count', 'Canonical_SMILES'], ascending=[False, True])
    
    return duplicates_df

# Example usage:
df = pd.read_csv(os.path.abspath("./past_experiments/ChemXriv/7.0_Experiment/ACD_1H_with_SN_filtered_v3_regio_aug.csv"))
duplicate_smiles = find_duplicate_smiles(df, 'SMILES', 'sample-id')
print(duplicate_smiles)
print(f"Total number of duplicate entries: {len(duplicate_smiles)}")
print(f"Number of unique molecules with duplicates: {duplicate_smiles['Canonical_SMILES'].nunique()}")

#### Tanimoto Similarity from Smiles

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from rdkit import Chem
from rdkit.Chem import DataStructs, rdMolDescriptors
from rdkit import DataStructs
from rdkit.Chem.Draw import IPythonConsole
from rdkit.Chem import Draw
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

# Set up plotting parameters
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

def canonicalize_smiles(smiles):
    """Canonicalize SMILES string using RDKit"""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is not None:
            return Chem.MolToSmiles(mol, canonical=True)
        else:
            return None
    except:
        return None

def calculate_tanimoto_similarity(smiles1, smiles2):
    """Calculate Tanimoto similarity between two SMILES strings"""
    try:
        mol1 = Chem.MolFromSmiles(smiles1)
        mol2 = Chem.MolFromSmiles(smiles2)
        
        if mol1 is None or mol2 is None:
            return None
            
        # Generate Morgan fingerprints
        fp1 = rdMolDescriptors.GetMorganFingerprintAsBitVect(mol1, 2, nBits=2048)
        fp2 = rdMolDescriptors.GetMorganFingerprintAsBitVect(mol2, 2, nBits=2048)
        
        # Calculate Tanimoto similarity
        similarity = DataStructs.TanimotoSimilarity(fp1, fp2)
        return similarity
    except:
        return None

# File paths
augmented_file = '/projects/cc/se_users/knlr326/1_NMR_project/2_Notebooks/MultiModalSpectralTransformer_cleaned/past_experiments/ChemXriv/7.0_Experiment/ACD_1H_with_SN_filtered_v3_regio_aug.csv'
ground_truth_file = '/projects/cc/se_users/knlr326/1_NMR_project/2_Notebooks/MultiModalSpectralTransformer_cleaned/past_experiments/ChemXriv/8.0_Sim_Data/data_1H_924509.csv'

# Read the CSV files
print("Reading CSV files...")
df_aug = pd.read_csv(augmented_file)
df_truth = pd.read_csv(ground_truth_file)

print(f"Augmented data shape: {df_aug.shape}")
print(f"Ground truth data shape: {df_truth.shape}")
print("\nAugmented data columns:", df_aug.columns.tolist())
print("Ground truth data columns:", df_truth.columns.tolist())

# Display first few rows to understand structure
print("\nFirst 3 rows of augmented data:")
display(df_aug.head(3))
print("\nFirst 3 rows of ground truth data:")
display(df_truth.head(3))

# Identify the correct column names based on the data we just examined
# From the output above, we can see the column names are different
sample_id_col_aug = 'sample-id'  # In augmented file
sample_id_col_truth = 'sample-id'  # In ground truth file
smiles_col = 'SMILES'  # Same in both files
smiles_regio_col = 'SMILES_regio_isomers'  # The regio-isomer column in augmented file

# Canonicalize SMILES in both dataframes
print("\nCanonicalizing SMILES...")
df_aug['canonical_smiles'] = df_aug[smiles_col].apply(canonicalize_smiles)
df_aug['canonical_smiles_regio'] = df_aug[smiles_regio_col].apply(canonicalize_smiles)
df_truth['canonical_smiles'] = df_truth[smiles_col].apply(canonicalize_smiles)

# Remove rows with invalid SMILES
df_aug = df_aug.dropna(subset=['canonical_smiles', 'canonical_smiles_regio'])
df_truth = df_truth.dropna(subset=['canonical_smiles'])

print(f"After canonicalization - Augmented: {len(df_aug)}, Ground truth: {len(df_truth)}")

# Merge dataframes on sample_id
print("Merging dataframes on sample ID...")
merged_df = pd.merge(df_aug, df_truth, left_on=sample_id_col_aug, right_on=sample_id_col_truth, suffixes=('_aug', '_truth'))
print(f"Merged dataframe shape: {merged_df.shape}")

if len(merged_df) == 0:
    print("ERROR: No matching sample IDs found between the two files!")
    print("Sample IDs in augmented file (first 10):", df_aug[sample_id_col_aug].head(10).tolist())
    print("Sample IDs in ground truth file (first 10):", df_truth[sample_id_col_truth].head(10).tolist())
else:
    # Calculate Tanimoto similarities between regio-isomers and ground truth
    print("Calculating Tanimoto similarities...")
    similarities = []
    for idx, row in merged_df.iterrows():
        # Compare regio-isomer (augmented) with ground truth
        sim = calculate_tanimoto_similarity(row['canonical_smiles_regio'], row['canonical_smiles_truth'])
        similarities.append(sim)
    
    merged_df['tanimoto_similarity'] = similarities
    
    # Remove rows where similarity calculation failed
    merged_df = merged_df.dropna(subset=['tanimoto_similarity'])
    print(f"Final dataframe with similarities: {len(merged_df)} pairs")
    
    # Display statistics
    print(f"\nTanimoto Similarity Statistics:")
    print(f"Mean: {merged_df['tanimoto_similarity'].mean():.3f}")
    print(f"Median: {merged_df['tanimoto_similarity'].median():.3f}")
    print(f"Min: {merged_df['tanimoto_similarity'].min():.3f}")
    print(f"Max: {merged_df['tanimoto_similarity'].max():.3f}")
    print(f"Std: {merged_df['tanimoto_similarity'].std():.3f}")
    
    # Plot distribution of Tanimoto similarities
    plt.figure(figsize=(10, 6))
    plt.hist(merged_df['tanimoto_similarity'], bins=30, alpha=0.7, edgecolor='black')
    plt.xlabel('Tanimoto Similarity')
    plt.ylabel('Frequency')
    plt.title('Distribution of Tanimoto Similarities\n(Augmented vs Ground Truth SMILES)')
    plt.axvline(merged_df['tanimoto_similarity'].mean(), color='red', linestyle='--', 
                label=f'Mean: {merged_df["tanimoto_similarity"].mean():.3f}')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Create side-by-side molecular structure visualization
    def create_molecule_comparison_grid(df, num_examples=12, figsize=(20, 16)):
        """Create a grid showing side-by-side molecular comparisons"""
        
        # Select examples across different similarity ranges
        df_sorted = df.sort_values('tanimoto_similarity')
        n_total = len(df_sorted)
        
        # Select examples from different similarity ranges
        indices = []
        for i in range(num_examples):
            idx = int(i * n_total / num_examples)
            indices.append(df_sorted.iloc[idx].name)
        
        selected_df = df.loc[indices]
        
        # Calculate grid dimensions
        n_rows = int(np.ceil(num_examples / 2))
        n_cols = 4  # 2 pairs of molecules per row
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize)
        if n_rows == 1:
            axes = axes.reshape(1, -1)
        
        for i, (idx, row) in enumerate(selected_df.iterrows()):
            row_idx = i // 2
            col_start = (i % 2) * 2
            
            # Get molecules - regio-isomer vs ground truth
            mol_regio = Chem.MolFromSmiles(row['canonical_smiles_regio'])
            mol_truth = Chem.MolFromSmiles(row['canonical_smiles_truth'])
            
            # Draw regio-isomer molecule
            if mol_regio is not None:
                img_regio = Draw.MolToImage(mol_regio, size=(300, 300))
                axes[row_idx, col_start].imshow(img_regio)
                axes[row_idx, col_start].set_title(f'Regio-isomer\nID: {row[sample_id_col_aug]}', fontsize=10)
                axes[row_idx, col_start].axis('off')
            
            # Draw ground truth molecule
            if mol_truth is not None:
                img_truth = Draw.MolToImage(mol_truth, size=(300, 300))
                axes[row_idx, col_start + 1].imshow(img_truth)
                axes[row_idx, col_start + 1].set_title(f'Ground Truth\nTanimoto: {row["tanimoto_similarity"]:.3f}', fontsize=10)
                axes[row_idx, col_start + 1].axis('off')
        
        # Hide unused subplots
        for i in range(num_examples, n_rows * 2):
            row_idx = i // 2
            col_start = (i % 2) * 2
            axes[row_idx, col_start].axis('off')
            axes[row_idx, col_start + 1].axis('off')
        
        plt.tight_layout()
        plt.suptitle('Side-by-Side Comparison: Regio-isomers vs Ground Truth Molecules', 
                     fontsize=16, y=0.98)
        plt.show()
        
        return selected_df
    
    # Create the molecule comparison grid
    print("\nCreating side-by-side molecular structure visualization...")
    comparison_df = create_molecule_comparison_grid(merged_df, num_examples=12)
    
    # Create a detailed table for the examples shown
    print("\nDetailed comparison table for visualized examples:")
    display_df = comparison_df[[sample_id_col_aug, 'canonical_smiles_regio', 'canonical_smiles_truth', 'tanimoto_similarity']].copy()
    display_df.columns = ['Sample ID', 'Regio-isomer SMILES', 'Ground Truth SMILES', 'Tanimoto Similarity']
    display_df['Tanimoto Similarity'] = display_df['Tanimoto Similarity'].round(3)
    display(display_df)
    
    # Additional analysis: similarity ranges
    print("\nSimilarity Range Analysis:")
    bins = [0, 0.5, 0.7, 0.8, 0.9, 0.95, 1.0]
    labels = ['0-0.5', '0.5-0.7', '0.7-0.8', '0.8-0.9', '0.9-0.95', '0.95-1.0']
    merged_df['similarity_range'] = pd.cut(merged_df['tanimoto_similarity'], bins=bins, labels=labels, include_lowest=True)
    
    range_counts = merged_df['similarity_range'].value_counts().sort_index()
    print(range_counts)
    
    # Plot similarity ranges
    plt.figure(figsize=(10, 6))
    range_counts.plot(kind='bar', color='skyblue', alpha=0.7, edgecolor='black')
    plt.xlabel('Tanimoto Similarity Range')
    plt.ylabel('Number of Molecule Pairs')
    plt.title('Distribution of Molecule Pairs by Similarity Range')
    plt.xticks(rotation=45)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Save results to CSV for further analysis
    output_file = 'tanimoto_similarity_analysis.csv'
    merged_df.to_csv(output_file, index=False)
    print(f"\nResults saved to: {output_file}")
    
    # Summary for the paper
    print("\n" + "="*60)
    print("SUMMARY FOR PAPER:")
    print("="*60)
    print(f"Total molecule pairs analyzed: {len(merged_df)}")
    print(f"Tanimoto similarity range: {merged_df['tanimoto_similarity'].min():.3f} - {merged_df['tanimoto_similarity'].max():.3f}")
    print(f"Mean similarity: {merged_df['tanimoto_similarity'].mean():.3f} ± {merged_df['tanimoto_similarity'].std():.3f}")
    print(f"Median similarity: {merged_df['tanimoto_similarity'].median():.3f}")
    
    high_sim = (merged_df['tanimoto_similarity'] >= 0.7).sum()
    very_high_sim = (merged_df['tanimoto_similarity'] >= 0.8).sum()
    print(f"Pairs with similarity ≥ 0.7: {high_sim} ({high_sim/len(merged_df)*100:.1f}%)")
    print(f"Pairs with similarity ≥ 0.8: {very_high_sim} ({very_high_sim/len(merged_df)*100:.1f}%)")

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
range_counts = merged_df['similarity_range'].value_counts().sort_index()
print(range_counts)

# Plot similarity ranges
plt.figure(figsize=(10, 6))
range_counts.plot(kind='bar', color='skyblue', alpha=0.7, edgecolor='black')
plt.xlabel('Tanimoto Similarity Range')
plt.ylabel('Number of Molecule Pairs')
plt.title('Distribution of Molecule Pairs by Similarity Range')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import Draw
import matplotlib.gridspec as gridspec
import os
import warnings
warnings.filterwarnings('ignore')

# Set up the output directory for figures
base_dir = '/projects/cc/se_users/knlr326/1_NMR_project/2_Notebooks/MultiModalSpectralTransformer_cleaned'
figures_dir = os.path.join(base_dir, '_FIGURES')

# Ensure the directory exists
if not os.path.exists(figures_dir):
    os.makedirs(figures_dir)
    print(f"Created directory: {figures_dir}")
else:
    print(f"Using existing directory: {figures_dir}")

# Load the CSV file with similarity data
file_path = '/projects/cc/se_users/knlr326/1_NMR_project/2_Notebooks/MultiModalSpectralTransformer_cleaned/past_experiments/ChemXriv/9.0_Tanimoto_Similarity_Experiment/tanimoto_similarity_analysis.csv'
df = pd.read_csv(file_path)

# Display basic information
print(f"Loaded {len(df)} molecule pairs")
print(f"Columns in dataset: {df.columns.tolist()}")

# Check for missing values in critical columns
critical_cols = ['canonical_smiles_truth', 'canonical_smiles_regio', 'tanimoto_similarity']
missing_data = df[critical_cols].isnull().sum()
print(f"\nMissing values in critical columns:\n{missing_data}")

# Drop rows with missing values
df_clean = df.dropna(subset=critical_cols)
print(f"Molecules after removing missing data: {len(df_clean)}")

# Calculate statistics on Tanimoto similarities
sim_values = df_clean['tanimoto_similarity'].values
avg_similarity = np.mean(sim_values)
median_similarity = np.median(sim_values)
min_similarity = np.min(sim_values)
max_similarity = np.max(sim_values)
std_similarity = np.std(sim_values)

print(f"\nTanimoto Similarity Statistics:")
print(f"Average: {avg_similarity:.3f} ± {std_similarity:.3f}")
print(f"Median: {median_similarity:.3f}")
print(f"Range: {min_similarity:.3f} - {max_similarity:.3f}")

# Count molecules in different similarity ranges
similarity_ranges = [
    (0.0, 0.5, '0.0-0.5'),
    (0.5, 0.7, '0.5-0.7'),
    (0.7, 0.8, '0.7-0.8'),
    (0.8, 0.9, '0.8-0.9'),
    (0.9, 1.0, '0.9-1.0')
]

range_counts = []
for lower, upper, label in similarity_ranges:
    count = ((df_clean['tanimoto_similarity'] >= lower) & 
             (df_clean['tanimoto_similarity'] < upper)).sum()
    if upper == 1.0:  # Include exact 1.0 in the last bin
        count += (df_clean['tanimoto_similarity'] == 1.0).sum()
    range_counts.append((label, count, count/len(df_clean)*100))

print("\nSimilarity Range Distribution:")
for label, count, percentage in range_counts:
    print(f"{label}: {count} molecules ({percentage:.1f}%)")

# Function to create a grid of molecule pairs with similarities
def visualize_molecule_pairs(df, n_rows=5, n_cols=2, figsize=(12, 20), n_examples=10):
    # Sort by similarity to show a range of examples
    df_sorted = df.sort_values('tanimoto_similarity')
    
    # Select examples across the similarity spectrum
    indices = []
    if len(df_sorted) <= n_examples:
        indices = list(range(len(df_sorted)))
    else:
        # Take evenly spaced examples across the similarity range
        step = len(df_sorted) / n_examples
        for i in range(n_examples):
            idx = int(i * step)
            indices.append(idx)
    
    df_selected = df_sorted.iloc[indices].reset_index(drop=True)
    
    # Calculate number of pages needed
    examples_per_page = n_rows * n_cols
    n_pages = int(np.ceil(len(df_selected) / examples_per_page))
    
    # Process each page
    for page in range(n_pages):
        start_idx = page * examples_per_page
        end_idx = min(start_idx + examples_per_page, len(df_selected))
        
        # Create a figure with a grid layout
        fig = plt.figure(figsize=figsize)
        gs = gridspec.GridSpec(n_rows, n_cols * 2)
        
        # Add a title to the figure
        plt.suptitle(f"Target Molecules vs. Regio-isomers (Page {page+1}/{n_pages})", 
                     fontsize=16, y=0.98)
        
        # Process each pair for this page
        for i, idx in enumerate(range(start_idx, end_idx)):
            row_idx = i // n_cols
            col_idx = i % n_cols
            
            molecule_data = df_selected.iloc[idx]
            
            # Get molecule SMILES
            target_smiles = molecule_data['canonical_smiles_truth']
            regio_smiles = molecule_data['canonical_smiles_regio']
            sample_id = molecule_data['sample-id']
            similarity = molecule_data['tanimoto_similarity']
            
            # Create RDKit molecule objects
            target_mol = Chem.MolFromSmiles(target_smiles)
            regio_mol = Chem.MolFromSmiles(regio_smiles)
            
            # Create sub-plots for this pair
            ax1 = plt.subplot(gs[row_idx, col_idx*2])
            ax2 = plt.subplot(gs[row_idx, col_idx*2+1])
            
            # Draw molecules
            if target_mol and regio_mol:
                target_img = Draw.MolToImage(target_mol, size=(300, 300))
                regio_img = Draw.MolToImage(regio_mol, size=(300, 300))
                
                ax1.imshow(target_img)
                ax1.set_title(f"Original: {sample_id}", fontsize=10)
                ax1.axis('off')
                
                ax2.imshow(regio_img)
                ax2.set_title(f"Regio-isomer (Sim: {similarity:.3f})", fontsize=10)
                ax2.axis('off')
        
        plt.tight_layout(rect=[0, 0, 1, 0.96])  # Adjust for the suptitle
        save_path = os.path.join(figures_dir, f"molecule_pairs_page_{page+1}.png")
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Saved page {page+1} to {save_path}")
        plt.close()
    
    # Create histogram of similarities
    plt.figure(figsize=(10, 6))
    plt.hist(sim_values, bins=20, edgecolor='black', alpha=0.7, color="#8CB0FE")
    plt.axvline(avg_similarity, color='red', linestyle='--', 
                label=f'Mean: {avg_similarity:.3f}')
    plt.axvline(median_similarity, color='orange', linestyle='--', 
                label=f'Median: {median_similarity:.3f}')
    plt.xlabel('Tanimoto Similarity')
    plt.ylabel('Frequency')
    plt.title('Distribution of Tanimoto Similarities Between Target Molecules and Regio-isomers')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    hist_path = os.path.join(figures_dir, "tanimoto_similarity_distribution.png")
    plt.savefig(hist_path, dpi=300)
    print(f"Saved histogram to {hist_path}")
    plt.close()
    
    # Create bar chart of similarity ranges
    plt.figure(figsize=(10, 6))
    labels = [label for label, _, _ in range_counts]
    counts = [count for _, count, _ in range_counts]
    plt.bar(labels, counts, color="#8CB0FE", edgecolor='black', alpha=0.7)
    plt.xlabel('Tanimoto Similarity Range')
    plt.ylabel('Number of Molecule Pairs')
    plt.title('Distribution of Molecule Pairs by Similarity Range')
    plt.xticks(rotation=0)
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    bar_path = os.path.join(figures_dir, "similarity_ranges_barchart.png")
    plt.savefig(bar_path, dpi=300)
    print(f"Saved bar chart to {bar_path}")
    plt.close()
    
    return df_selected

# Create visualizations
print("\nGenerating visualizations...")
selected_molecules = visualize_molecule_pairs(df_clean, n_examples=20)

# Save the selected molecules with their similarities to a CSV
output_path = os.path.join(figures_dir, "selected_molecule_pairs.csv")
selected_molecules.to_csv(output_path, index=False)
print(f"Saved selected molecule pairs to {output_path}")


### 9.0 Improvement Cycle on Similar Molecules

In [ ]:

def split_dataset(config, chunk_size: int) -> List[pd.DataFrame]:
    df = pd.read_csv(config.SGNN_csv_gen_smi)
    return [df[i:i+chunk_size] for i in range(0, len(df), chunk_size)]


def create_chunk_folder(config, idx: int) -> str:
    base_dir = config.model_save_dir
    current_datetime = datetime.now().strftime("%Y%m%d_%H%M%S")
    chunk_folder_name = f"chunk_{idx:03d}_{current_datetime}"
    chunk_folder_path = os.path.join(base_dir, chunk_folder_name)
    
    os.makedirs(chunk_folder_path, exist_ok=True)
    print(f"Created folder for chunk {idx}: {chunk_folder_path}")
    
    return chunk_folder_path

def test_pretrained_model_on_sim_data_before(config, IR_config, stoi, itos, stoi_MF, itos_MF, chunk, idx):
    MW_filter, greedy_full = True, False
    
    print("prepare_data")
    config = prepare_data(config, chunk)
    print("generate_simulated_data")
    config = generate_simulated_data(config, IR_config)

    print("load_model_and_data")
    model_MMT, val_dataloader, val_dataloader_multi = load_model_and_data(config, stoi, stoi_MF)

    print("run_model_analysis")
    prob_dict_results_1c_, results_dict_1c_ = mrtf.run_model_analysis(config, model_MMT, val_dataloader_multi, stoi, itos)

    results = test_model_performance(config, model_MMT, val_dataloader, val_dataloader_multi, stoi, itos, stoi_MF, itos_MF)

    save_results_before(results, config, idx)

    return config

def prepare_data(config: Any, chunk: pd.DataFrame) -> Any:
    chunk_csv_path = os.path.join(config.pkl_save_folder, "SGNN_csv_gen_smi.csv")
    chunk.to_csv(chunk_csv_path)
    config.SGNN_csv_gen_smi = chunk_csv_path 
    config.data_size = len(chunk)
    return config

def generate_simulated_data(config: Any, IR_config: Any) -> Any:
    config.execution_type = "data_generation"
    if config.execution_type == "data_generation":
        print("\033[1m\033[31mThis is: data_generation\033[0m")
        #import IPython; IPython.embed();

        config = ex.gen_sim_aug_data(config, IR_config)
        backup_config_paths(config)
    return config

def backup_config_paths(config: Any) -> None:
    config.csv_1H_path_SGNN_backup = copy.deepcopy(config.csv_1H_path_SGNN)
    config.csv_13C_path_SGNN_backup = copy.deepcopy(config.csv_13C_path_SGNN)
    config.csv_HSQC_path_SGNN_backup = copy.deepcopy(config.csv_HSQC_path_SGNN)
    config.csv_COSY_path_SGNN_backup = copy.deepcopy(config.csv_COSY_path_SGNN)
    config.IR_data_folder_backup = copy.deepcopy(config.IR_data_folder)

def save_results_before(results: Dict[str, Any], config: Any, idx: int) -> None:
    variables_to_save = {
        'avg_tani_bl_ZINC': results['avg_tani_bl_ZINC_'],
        'results_dict_greedy_bl_ZINC': results.get('results_dict_greedy_bl_ZINC_'),
        'failed_bl_ZINC': results.get('failed_bl_ZINC_'),
        'avg_tani_greedy_bl_ZINC': results['avg_tani_greedy_bl_ZINC_'],
        'results_dict_ZINC_greedy_bl': results.get('results_dict_ZINC_greedy_bl_'),
        'total_results_bl_ZINC': results['total_results_bl_ZINC_'],
        'corr_sampleing_prob_bl_ZINC': results['corr_sampleing_prob_bl_ZINC_'],
        'results_dict_bl_ZINC': results['results_dict_bl_ZINC_'],
    }
    save_data_with_datetime_index(variables_to_save, config.pkl_save_folder, "before_sim_data", idx)

def create_run_folder(chunk_folder, idx):
    current_datetime = datetime.now().strftime("%Y%m%d_%H%M%S")
    run_folder_name = f"run_{idx}_{current_datetime}"
    run_folder_path = os.path.join(chunk_folder, run_folder_name)
    
    os.makedirs(run_folder_path, exist_ok=True)
    print(f"Created folder for run {idx}: {run_folder_path}")
    
    return run_folder_path

def fine_tune_model_aug_mol(config, stoi, stoi_MF, chunk, idx):
    #import IPython; IPython.embed();
    config, all_gen_smis, aug_mol_df = generate_augmented_molecules_from_aug_mol(config, chunk, idx)
    
    config.parent_model_save_dir = config.model_save_dir
    config.model_save_dir = config.current_run_folder 
    
    if config.execution_type == "transformer_improvement":
        print("\033[1m\033[31mThis is: transformer_improvement, sim_data_gen == TRUE\033[0m")
        config.training_setup = "pretraining"
        mtf.run_MMT(config, stoi, stoi_MF)
    
    config.model_save_dir = config.parent_model_save_dir
    #config = ex.update_model_path(config)

    return config, aug_mol_df, all_gen_smis


def generate_augmented_molecules_from_aug_mol(config, chunk, idx):

    ############# THis is just relevant for the augmented molecules #############
    chunk.rename(columns={'SMILES': 'SMILES_orig', 'SMILES_regio_isomers': 'SMILES'}, inplace=True)
    #############################################################################
    
    script_dir = os.getcwd()
    
    base_path = os.path.abspath(os.path.join(script_dir, 'deep-molecular-optimization'))

    csv_file_path = f'{base_path}/data/MMP/test_selection_2.csv'
    chunk.to_csv(csv_file_path, index=False)
    print(f"CSV file '{csv_file_path}' created successfully.")

    config.data_size = len(chunk)
    config.n_samples = config.data_size

    config, results_dict_MF = generate_smiles_mf(config)

    combined_list_MF = process_generated_smiles(results_dict_MF, config)

    all_gen_smis = filter_and_combine_smiles(combined_list_MF)

    aug_mol_df = create_augmented_dataframe(all_gen_smis)

    config, final_df = ex.blend_aug_with_train_data(config, aug_mol_df)

    config = ex.gen_sim_aug_data(config, IR_config)
    config.execution_type = "transformer_improvement"

    return config, all_gen_smis, aug_mol_df


def fine_tune_model(config, stoi, stoi_MF, chunk, idx):
    """
    Fine-tune the model on a chunk of data.
    """
    config, aug_mol_df, all_gen_smis = generate_augmented_molecules(config, chunk, idx)
    
    config.parent_model_save_dir = config.model_save_dir
    new_model_save_dir = create_model_save_dir(config.parent_model_save_dir, idx)
    config.model_save_dir = new_model_save_dir
    
    # Fine-tune the model
    if config.execution_type == "transformer_improvement":
        print("\033[1m\033[31mThis is: transformer_improvement, sim_data_gen == TRUE\033[0m")
        config.training_setup = "pretraining"
        mtf.run_MMT(config, stoi, stoi_MF)
        
    #config = ex.update_model_path(config)
    config.model_save_dir = config.parent_model_save_dir
    
    return config, aug_mol_df, all_gen_smis

def generate_augmented_molecules(config, chunk, idx):
    #import IPython; IPython.embed();
    script_dir = os.getcwd()
    
    base_path = os.path.abspath(os.path.join(script_dir, 'deep-molecular-optimization'))

    csv_file_path = f'{base_path}/data/MMP/test_selection_2.csv'
    chunk.to_csv(csv_file_path, index=False)
    print(f"CSV file '{csv_file_path}' created successfully.")

    config.data_size = len(chunk)
    config.n_samples = config.data_size

    config, results_dict_MF = generate_smiles_mf(config)

    combined_list_MF = process_generated_smiles(results_dict_MF, config)

    all_gen_smis = filter_and_combine_smiles(combined_list_MF)

    aug_mol_df = create_augmented_dataframe(all_gen_smis)

    config, final_df = ex.blend_aug_with_train_data(config, aug_mol_df)

    config = ex.gen_sim_aug_data(config, IR_config)
    config.execution_type = "transformer_improvement"

    return config, all_gen_smis, aug_mol_df


def generate_smiles_mf(config):
    print("\033[1m\033[31mThis is: SMI_generation_MF\033[0m")
    return ex.SMI_generation_MF(config, stoi, stoi_MF, itos, itos_MF)

def process_generated_smiles(results_dict_MF, config):
    results_dict_MF = {key: value for key, value in results_dict_MF.items() if not hf.contains_only_nan(value)}
    for key, value in results_dict_MF.items():
        results_dict_MF[key] = hf.remove_nan_from_list(value)

    combined_list_MF, _, _, _ = cv.plot_cluster_MF(results_dict_MF, config)
    return combined_list_MF

def filter_and_combine_smiles(combined_list_MF):
    print("\033[1m\033[31mThis is: combine_MMT_MF\033[0m")
    all_gen_smis = combined_list_MF
    all_gen_smis = [smiles for smiles in all_gen_smis if smiles != 'NAN']

    val_data = pd.read_csv(config.csv_path_val)
    all_gen_smis = mrtf.filter_smiles(val_data, all_gen_smis)
    return all_gen_smis

def create_augmented_dataframe(all_gen_smis):
    length_of_list = len(all_gen_smis)
    random_number_strings = [f"GT_{str(i).zfill(7)}" for i in range(1, length_of_list + 1)]
    return pd.DataFrame({'SMILES': all_gen_smis, 'sample-id': random_number_strings})

def setup_data_paths(config):
    
    base_path_exp = os.path.abspath("./past_experiments/ChemXriv/8.0_Experimenta_Data/")
    config.csv_1H_path_exp = f"{base_path_exp}real_1H_with_AZ_SMILES_v3.csv"
    config.csv_13C_path_exp = f"{base_path_exp}real_13C_with_AZ_SMILES_v3.csv"
    config.csv_HSQC_path_exp = f"{base_path_exp}real_HSQC_with_AZ_SMILES_v3.csv"
    config.csv_COSY_path_exp = f"{base_path_exp}real_COSY_with_AZ_SMILES_v3.csv"
    config.IR_data_folder_exp = f"{base_path_exp}IR_data"
    return config

def test_model_on_datasets(config, IR_config, stoi, itos, stoi_MF, itos_MF, chunk, composite_idx, aug_mol_df, all_gen_smis):
    checkpoint_path_backup = config.checkpoint_path    
    for data_type in ['exp', 'sim']:
        print(f"Testing on {data_type} data")
        config.pickle_file_path = ""
        config.training_mode = "1H_13C_HSQC_COSY_IR_MF_MW"
        config = test_on_data(config, IR_config, stoi, itos, stoi_MF, itos_MF, chunk, composite_idx, data_type, aug_mol_df, all_gen_smis)
    config.checkpoint_path = checkpoint_path_backup
    return config

def test_on_data(config, IR_config, stoi, itos, stoi_MF, itos_MF, chunk, composite_idx, data_type, aug_mol_df, all_gen_smis):
    if data_type == 'sim':
        restore_backup_configs(config)
    else:
        sample_ids = chunk['sample-id'].tolist()
        process_spectrum_data(config, sample_ids, data_type)
    #import IPython; IPython.embed();

    update_config_settings(config)
    last_checkpoint = get_last_checkpoint(config.current_run_folder)
    config.checkpoint_path = last_checkpoint
    
    model_MMT, val_dataloader, val_dataloader_multi = load_model_and_data(config, stoi, stoi_MF)
    
    prob_dict_results_1c_, results_dict_1c_ = mrtf.run_model_analysis(config, model_MMT, val_dataloader_multi, stoi, itos)

    results = test_model_performance(config, model_MMT, val_dataloader, val_dataloader_multi,
                                     stoi, itos, stoi_MF, itos_MF)
    
    if data_type == 'sim':
        results['aug_mol_df'] = aug_mol_df
        results['all_gen_smis'] = all_gen_smis
    
    save_results_acd_exp(results, config, data_type, composite_idx)
    return config

def restore_backup_configs(config):
    config.csv_1H_path_SGNN = config.csv_1H_path_SGNN_backup
    config.csv_13C_path_SGNN = config.csv_13C_path_SGNN_backup
    config.csv_HSQC_path_SGNN = config.csv_HSQC_path_SGNN_backup
    config.csv_COSY_path_SGNN = config.csv_COSY_path_SGNN_backup
    config.IR_data_folder = config.IR_data_folder_backup 
    config.csv_path_val = config.csv_1H_path_SGNN_backup
    config.pickle_file_path = ""

def process_spectrum_data(config: Any, sample_ids: List[str], data_type: str) -> None:
    spectrum_types = ['1H', '13C', 'HSQC', 'COSY']
    for spectrum in spectrum_types:
        csv_path = getattr(config, f'csv_{spectrum}_path_{data_type}')
        df_data = pd.read_csv(csv_path)
        df_data['sample-id'] = df_data['AZ_Number']
        data = select_relevant_samples(df_data, sample_ids)
        dummy_path, config = save_and_update_config(config, data_type, spectrum, data)
        print(f"Saved {spectrum} data to: {dummy_path}")
    if data_type == "ACD" or data_type == "sim":
        config.IR_data_folder = config.IR_data_folder_backup 
    elif  data_type == "exp":
        config.IR_data_folder = config.IR_data_folder_exp 

    
    
def select_relevant_samples(df: pd.DataFrame, sample_ids: List[str]) -> pd.DataFrame:
    return df[df['sample-id'].isin(sample_ids)]

def save_and_update_config(config, data_type: str, spectrum_type: str, data: pd.DataFrame) -> Tuple[str, Any]:
    temp_dir = tempfile.mkdtemp()
    dummy_path = os.path.join(temp_dir, f"{data_type}_{spectrum_type}_selected_samples.csv")
    
    data.to_csv(dummy_path, index=False)
    
    config_key = f'csv_{spectrum_type}_path_SGNN'
    setattr(config, config_key, dummy_path)
    
    return dummy_path, config

def update_config_settings(config: Any) -> None:
    config.csv_path_val = config.csv_1H_path_SGNN
    config.pickle_file_path = ""

def get_last_checkpoint(model_folder: str) -> str:
    checkpoints = [f for f in os.listdir(model_folder) if f.endswith('.ckpt')]
    if not checkpoints:
        raise ValueError(f"No checkpoints found in {model_folder}")
    
    last_checkpoint = max(checkpoints, key=lambda x: os.path.getmtime(os.path.join(model_folder, x)))
    return os.path.join(model_folder, last_checkpoint)

def load_model_and_data(config: Any, stoi: Dict, stoi_MF: Dict) -> Tuple[Any, Any, Any]:
    #import IPython; IPython.embed();

    val_dataloader = mrtf.load_data(config, stoi, stoi_MF, single=True, mode="val")
    val_dataloader_multi = mrtf.load_data(config, stoi, stoi_MF, single=False, mode="val")
    model_MMT = mrtf.load_MMT_model(config)
    return model_MMT, val_dataloader, val_dataloader_multi

def test_model_performance(config: Any, model_MMT: Any, val_dataloader: Any, val_dataloader_multi: Any, 
                           stoi: Dict, itos: Dict, stoi_MF: Dict, itos_MF: Dict) -> Dict[str, Any]:
    print("\033[1m\033[31mThis is: test_performance\033[0m")
    
    MW_filter = True
    greedy_full = False
    
    model_CLIP = mrtf.load_CLIP_model(config)
    
    results = {}
    
    results['results_dict_bl_ZINC_'] = mrtf.run_test_mns_performance_CLIP_3(
        config, model_MMT, model_CLIP, val_dataloader, stoi, itos, MW_filter)
    results['results_dict_bl_ZINC_'], counter = mrtf.filter_invalid_inputs(results['results_dict_bl_ZINC_'])

    results['avg_tani_bl_ZINC_'], html_plot = rbgvm.plot_hist_of_results(results['results_dict_bl_ZINC_'])

    if greedy_full:
        results['results_dict_greedy_bl_ZINC_'], results['failed_bl_ZINC_'] = mrtf.run_test_performance_CLIP_greedy_3(
            config, stoi, stoi_MF, itos, itos_MF)
        results['avg_tani_greedy_bl_ZINC_'], html_plot_greedy = rbgvm.plot_hist_of_results_greedy(
            results['results_dict_greedy_bl_ZINC_'])
    else:
        config, results['results_dict_ZINC_greedy_bl_'] = mrtf.run_greedy_sampling(
            config, model_MMT, val_dataloader_multi, itos, stoi)
        results['avg_tani_greedy_bl_ZINC_'] = results['results_dict_ZINC_greedy_bl_']["tanimoto_mean"]

    results['total_results_bl_ZINC_'] = mrtf.run_test_performance_CLIP_3(
        config, model_MMT, val_dataloader, stoi)
    results['corr_sampleing_prob_bl_ZINC_'] = results['total_results_bl_ZINC_']["statistics_multiplication_avg"][0]

    print("avg_tani, avg_tani_greedy, corr_sampleing_prob'")
    print(results['avg_tani_bl_ZINC_'], results['avg_tani_greedy_bl_ZINC_'], results['corr_sampleing_prob_bl_ZINC_'])
    print("Greedy tanimoto results")
    rbgvm.plot_hist_of_results_greedy_new(results['results_dict_ZINC_greedy_bl_'])

    return results

def save_results_acd_exp(results: Dict[str, Any], config: Any, data_type: str, composite_idx: str) -> None:
    variables_to_save = {
        'avg_tani_bl_ZINC': results['avg_tani_bl_ZINC_'],
        'results_dict_greedy_bl_ZINC': results.get('results_dict_greedy_bl_ZINC_'),
        'failed_bl_ZINC': results.get('failed_bl_ZINC_'),
        'avg_tani_greedy_bl_ZINC': results['avg_tani_greedy_bl_ZINC_'],
        'results_dict_ZINC_greedy_bl': results.get('results_dict_ZINC_greedy_bl_'),
        'total_results_bl_ZINC': results['total_results_bl_ZINC_'],
        'corr_sampleing_prob_bl_ZINC': results['corr_sampleing_prob_bl_ZINC_'],
        'results_dict_bl_ZINC': results['results_dict_bl_ZINC_'],
        'checkpoint_path': config.checkpoint_path,
    }
    
    if data_type == 'sim':
        variables_to_save['aug_mol_df'] = results.get('aug_mol_df')
        variables_to_save['all_gen_smis'] = results.get('all_gen_smis')
    
    save_data_with_datetime_index(
        variables_to_save, 
        config.pkl_save_folder, 
        f"{data_type}_sim_data", 
        composite_idx
    )

def save_data_with_datetime_index(data: Any, base_folder: str, name: str, idx: Union[int, str]) -> None:
    current_datetime = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"{current_datetime}_{name}_{idx}.pkl"
    os.makedirs(base_folder, exist_ok=True)
    file_path = os.path.join(base_folder, filename)

    
    with open(file_path, 'wb') as f:
        pickle.dump(data, f)
    
    print(f"Data saved to: {file_path}")

In [ ]:

def main_IC(chunk_size, config, IR_config, stoi, itos, stoi_MF, itos_MF, num_training_runs=3):
    chunks = split_dataset(config, chunk_size)
    config.model_save_dir = config.pkl_save_folder
    model_save_dir_backup = config.model_save_dir
    original_checkpoint_path = config.checkpoint_path  # Store the original checkpoint path

    for chunk_idx, chunk in enumerate(chunks):
        print(f"Processing chunk {chunk_idx+1} of {len(chunks)}")
        
        chunk_folder = create_chunk_folder(config, chunk_idx)
        config.current_chunk_folder = chunk_folder
            
        config.blank_percentage = 0
        config = test_pretrained_model_on_sim_data_before(config, IR_config, stoi, itos, stoi_MF, itos_MF, chunk, f"{chunk_idx}_{0}")
        print(config.csv_1H_path_SGNN)
        for run_idx in range(num_training_runs):
            print(f"Starting training run {run_idx+1} of {num_training_runs}")
            
            run_folder = create_run_folder(config.current_chunk_folder, f"{chunk_idx}_{run_idx}")
            config.current_run_folder = run_folder
            config.model_save_dir = run_folder

            config.blank_percentage = 50
            config, aug_mol_df, all_gen_smis = fine_tune_model_aug_mol(config, stoi, stoi_MF, chunk, f"{chunk_idx}_{run_idx}")
            #import IPython; IPython.embed();

            ### Retrun the labelling of the smiles to test it on the correct one
            #chunk.rename(columns={'SMILES': 'SMILES_regio_isomers', 'SMILES_orig': 'SMILES'}, inplace=True)

            config.blank_percentage = 0
            config = setup_data_paths(config)
            config = test_model_on_datasets(config, IR_config, stoi, itos, stoi_MF, itos_MF, chunk, f"{chunk_idx}_{run_idx}", aug_mol_df, all_gen_smis)

            config.checkpoint_path = original_checkpoint_path
        
        print(f"Chunk {chunk_idx+1} completed. All training runs finished.")
        config.model_save_dir = model_save_dir_backup


In [ ]:
### RUN a1
config.checkpoint_path = os.path.abspath("./models/mmst/base_models/1_0_V8i_MMTi_RAW_DROP_Loss_0.112.ckpt")
config.pkl_save_folder = os.path.abspath("./past_experiments/ChemXriv/7.0_Experiment/test_on_aug_data_34_v3_")
config.SGNN_csv_gen_smi = os.path.abspath("./past_experiments/ChemXriv/7.0_Experiment/ACD_1H_with_SN_filtered_v3_regio_aug.csv")

config.MF_generations = 30
config.MF_delta_weight = 100
config.max_scaffold_generations = 300
config.blank_percentage = 50
config.weight_MW = 100
config.lr_pretraining = 3e-4
config.tr_te_split = 0.9
config.batch_size = 64
config.num_epochs = 30
config.temperature = 1
config.multinom_runs = 20 
config.train_data_blend = 0

chunk_size = 1

main_IC(chunk_size, config, IR_config, stoi, itos, stoi_MF, itos_MF, 3)

In [ ]:
def main_aug():
    folder_path = os.path.abspath("./past_experiments/ChemXriv/7.0_Experiment/test_on_aug_data_34_v3/")
    model_type = "MMT"
    ranking_methods = ["HSQC"] #"COSY", "HSQC & COSY"
    file_types = ["sim_sim_data", "exp_sim_data"]

    data_type_map = {
        "sim_sim_data": "Our Simulated",
        "exp_sim_data": "Experimental"
    }
    filtered_out = []
    all_rankings_list = []
    for ranking_method in ranking_methods:
        for file_type in file_types:
            all_rankings = process_pkl_files_new(folder_path, file_type, ranking_method)
            #import IPython; IPython.embed();
            all_rankings, removed_smiles = exp_func.deduplicate_smiles_from_ranking(all_rankings)
            all_rankings, filtered_out_rankings = exp_func.filter_rankings_by_molecular_formula(all_rankings)
            filtered_out.append(filtered_out_rankings)
            all_rankings_list.append(all_rankings)
            
            accuracies = calculate_top_k_accuracy(all_rankings)
            sim_rank_one_count = exp_func.count_molecules_with_sim_rank_one(all_rankings)

            data_type = data_type_map[file_type]
            save_path = f"./_FIGURES/similar_starting_top_k_accuracy_MMT_{file_type}_{ranking_method}_FINAL_2.png"
            print(len(all_rankings.keys()))
            plot_top_k_accuracy(accuracies, data_type, model_type, ranking_method, sim_rank_one_count, save_path, total_samples=len(all_rankings.keys()))

            print(f"Completed plot for {ranking_method} - {data_type}")
    return all_rankings_list, filtered_out



In [ ]:
all_rankings_list, filtered_out = main_aug()

In [ ]:

# Helper function to calculate molecular weight
def calculate_weight(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return Chem.Descriptors.ExactMolWt(mol)

def plot_smiles_from_df(data_sorted, column_name):
    list_smiles = []
    sample_id_list = []
    for idx, smiles in enumerate(data_sorted[column_name]):
        weight = calculate_weight(smiles)
        string = f"{column_name}_{idx}_{weight:.2f}"
        sample_id_list.append(string)
        list_smiles.append(Chem.MolFromSmiles(smiles))
        if len(list_smiles) == 9 or idx == len(data_sorted) - 1:
            pic = Draw.MolsToGridImage(list_smiles, subImgSize=(250, 250), legends=sample_id_list)
            display(pic)
            list_smiles = []
            sample_id_list = []

# Read the CSV file
df = pd.read_csv(os.path.abspath("./past_experiments/ChemXriv/7.0_Experiment/ACD_1H_with_SN_filtered_v3_regio_aug.csv"))

# Plot SMILES from both columns
print("Original SMILES:")
plot_smiles_from_df(df, 'SMILES')

print("\nRegio Isomer SMILES:")
plot_smiles_from_df(df, 'SMILES_regio_isomers')

# Plotting both columns side by side
def plot_smiles_side_by_side(data_sorted, column1, column2):
    list_smiles1 = []
    list_smiles2 = []
    sample_id_list1 = []
    sample_id_list2 = []
    for idx, (smiles1, smiles2) in enumerate(zip(data_sorted[column1], data_sorted[column2])):
        weight1 = calculate_weight(smiles1)
        weight2 = calculate_weight(smiles2)
        string1 = f"{column1}_{idx}_{weight1:.2f}"
        string2 = f"{column2}_{idx}_{weight2:.2f}"
        sample_id_list1.append(string1)
        sample_id_list2.append(string2)
        list_smiles1.append(Chem.MolFromSmiles(smiles1))
        list_smiles2.append(Chem.MolFromSmiles(smiles2))
        if len(list_smiles1) == 4 or idx == len(data_sorted) - 1:
            combined_smiles = [mol for pair in zip(list_smiles1, list_smiles2) for mol in pair]
            combined_legends = [legend for pair in zip(sample_id_list1, sample_id_list2) for legend in pair]
            pic = Draw.MolsToGridImage(combined_smiles, molsPerRow=2, subImgSize=(250, 250), legends=combined_legends)
            display(pic)
            list_smiles1 = []
            list_smiles2 = []
            sample_id_list1 = []
            sample_id_list2 = []

print("\nSide-by-side comparison:")
plot_smiles_side_by_side(df, 'SMILES', 'SMILES_regio_isomers')

In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Draw
from IPython.display import display

def plot_smiles_grid(smiles_list, molsPerRow=5, subImgSize=(200, 200)):
    """
    Plot a list of SMILES as a grid with a specified number of molecules per row.
    
    :param smiles_list: List of SMILES strings to plot
    :param molsPerRow: Number of molecules per row in the grid (default: 5)
    :param subImgSize: Size of each molecule image (default: (200, 200))
    """
    mols = [Chem.MolFromSmiles(smiles) for smiles in smiles_list]
    
    # Generate labels (you can customize this if needed)
    legends = [f"Mol {i+1}" for i in range(len(mols))]
    
    # Create the grid image
    img = Draw.MolsToGridImage(mols, molsPerRow=molsPerRow, subImgSize=subImgSize, legends=legends)
    
    # Display the image
    display(img)

# Read the pickle file
file_path = os.path.abspath("./past_experiments/ChemXriv/7.0_Experiment/test_on_aug_data_34_v3/20240927_094958_exp_sim_data_0_0.pkl")
df = pd.read_pickle(file_path)

# Extract the SMILES list
results_dict_bl_ZINC = df['results_dict_bl_ZINC']
smiles_list = [value[0] for value in list(results_dict_bl_ZINC.values())[0][0]]

# Print the number of SMILES
print(f"Number of SMILES: {len(smiles_list)}")

# Print the first few SMILES for verification
print("\nFirst few SMILES:")
for smiles in smiles_list[:5]:
    print(smiles)

# Plot the SMILES grid
print("\nPlotting SMILES grid:")
plot_smiles_grid(smiles_list)

# Optionally, you can adjust the grid layout or image size like this:
# plot_smiles_grid(smiles_list, molsPerRow=4, subImgSize=(150, 150))

### Plot Guess Molecule in SVG

In [ ]:


# SMILES string of the molecule
smiles = "OC(CNC(C)Cc1ccc(OC)cc1)c1cc(ccc1O)NC=O"

# Create a RDKit molecule object from the SMILES string
mol = Chem.MolFromSmiles(smiles)

# Generate 2D coordinates for the molecule
AllChem.Compute2DCoords(mol)

# Create a drawer object
d = Draw.MolDraw2DSVG(300, 300)

# Draw the molecule
d.DrawMolecule(mol)

# End the drawing
d.FinishDrawing()

# Get the SVG as a string
svg = d.GetDrawingText()

# Specify the path where you want to save the SVG file
save_path = os.path.abspath("./_FIGURES/20240927_094958_exp_sim_data_0_0.svg")

# Ensure the directory exists
os.makedirs(os.path.dirname(save_path), exist_ok=True)

# Save the SVG to a file_k_accuracy_MMTi_baseline_sim_sim_data_COSY_FINAL_2_k_accuracy_MMTi_baseline_sim_sim_data_COSY_FINAL_2
with open(save_path, 'w') as f:
    f.write(svg)

print(f"SVG has been saved to: {save_path}")

## Extra Figures

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import Descriptors
import warnings
warnings.filterwarnings('ignore')

# Set up plotting parameters
plt.rcParams['figure.figsize'] = (15, 6)
plt.rcParams['font.size'] = 12

def calculate_heavy_atoms(smiles):
    """Calculate number of heavy atoms from SMILES"""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is not None:
            return mol.GetNumHeavyAtoms()
        else:
            return None
    except:
        return None

def calculate_molecular_weight(smiles):
    """Calculate molecular weight from SMILES"""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is not None:
            return Descriptors.MolWt(mol)
        else:
            return None
    except:
        return None

# Load the dataset
print("Loading ZINC dataset...")
file_path = '/projects/cc/se_users/knlr326/1_NMR_project/2_Notebooks/MultiModalSpectralTransformer_cleaned/data/ZINK_dataset/ML_NMR_5M_XL_1H_comb_train_V8.csv'

# Read a sample for analysis (adjust nrows based on your computational resources)
df = pd.read_csv(file_path, nrows=100000)
print(f"Dataset shape: {df.shape}")

# Calculate heavy atoms and molecular weights
print("Calculating heavy atoms and molecular weights...")
df['heavy_atoms'] = df['SMILES'].apply(calculate_heavy_atoms)
df['mol_weight'] = df['SMILES'].apply(calculate_molecular_weight)

# Remove invalid entries
df_clean = df.dropna(subset=['heavy_atoms', 'mol_weight'])
print(f"Clean dataset shape: {df_clean.shape}")

# Create side-by-side plots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# 1. Heavy atom distribution histogram
ax1.hist(df_clean['heavy_atoms'], bins=50, alpha=0.7, color='#8CB0FE', edgecolor='black')
ax1.axvline(df_clean['heavy_atoms'].mean(), color='red', linestyle='--', 
            label=f'Mean: {df_clean["heavy_atoms"].mean():.1f}')
ax1.axvline(df_clean['heavy_atoms'].median(), color='orange', linestyle='--', 
            label=f'Median: {df_clean["heavy_atoms"].median():.1f}')
ax1.set_xlabel('Number of Heavy Atoms')
ax1.set_ylabel('Frequency')
ax1.set_title('Distribution of Heavy Atoms in Training Dataset')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Molecular weight distribution
ax2.hist(df_clean['mol_weight'], bins=50, alpha=0.7, color='#8CB0FE', edgecolor='black')
ax2.axvline(df_clean['mol_weight'].mean(), color='red', linestyle='--', 
            label=f'Mean: {df_clean["mol_weight"].mean():.1f} Da')
ax2.axvline(df_clean['mol_weight'].median(), color='orange', linestyle='--', 
            label=f'Median: {df_clean["mol_weight"].median():.1f} Da')
ax2.set_xlabel('Molecular Weight (Da)')
ax2.set_ylabel('Frequency')
ax2.set_title('Distribution of Molecular Weights in Training Dataset')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()

# Save the figure
save_path = '/projects/cc/se_users/knlr326/1_NMR_project/2_Notebooks/MultiModalSpectralTransformer_cleaned/_FIGURES/heavy_atoms_molecular_weight_distribution.png'
plt.savefig(save_path, dpi=300, bbox_inches='tight')
print(f"Figure saved to: {save_path}")

plt.show()

# Print basic statistics
print("\n" + "="*60)
print("BASIC STATISTICS")
print("="*60)
print(f"Number of molecules analyzed: {len(df_clean):,}")
print(f"Heavy atoms - Min: {df_clean['heavy_atoms'].min()}, Max: {df_clean['heavy_atoms'].max()}")
print(f"Heavy atoms - Mean: {df_clean['heavy_atoms'].mean():.1f} ± {df_clean['heavy_atoms'].std():.1f}")
print(f"Molecular weight - Min: {df_clean['mol_weight'].min():.1f} Da, Max: {df_clean['mol_weight'].max():.1f} Da")
print(f"Molecular weight - Mean: {df_clean['mol_weight'].mean():.1f} ± {df_clean['mol_weight'].std():.1f} Da")